<a href="https://colab.research.google.com/github/edwardpriyatna/Auto-Complete-and-Car-Allocation-Problem/blob/main/BBRI_Investor_Classifier_v8_test_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BBRI Investor Classifier v9 - BenOwners grouping
Institutional grouping follows the uploaded BenOwners hierarchy. Private and individual investors retain the existing code rules.

In [1]:
# -*- coding: utf-8 -*-
"""BBRI Investor Classifier v8 test version.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1hdol56BEBSrMX-706BL4jcQ8wPkIEt7W
"""

# -*- coding: utf-8 -*-

"""BBRI Investor Classifier v8 real version.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1Kvj-OWbPdgAhkNf0GPpqHzzT-cxO4HoD

# BBRI Fund Manager Classifier v10

The target is the **fund manager controlling buy/sell decisions**, not the beneficial owner. Manager attribution runs before beneficiary and government-parent fallbacks.

## 1. Original reviewed registries and helper functions
"""

from dataclasses import dataclass
from typing import Callable, Optional
from IPython.display import display
import pandas as pd
import re
import unicodedata

from typing import Callable, Optional
from IPython.display import display
import pandas as pd
import re
import unicodedata

# Upload the workbook to Colab and change only this path if needed.
INPUT_XLSX = "/content/BBRI_combined.xlsx"
OUTPUT_XLSX = "/content/classified_test_output.xlsx"
WRONG_XLSX = "/content/wrong_outputs.xlsx"

UNRESOLVED = "ML_REQUIRED"

# Canonical field -> accepted workbook column names.
COLUMN_ALIASES = {
    "sid": ("NOMORSID", "SID"),
    "investor_name": ("NAMAINVESTOR", "Nama Investor"),
    "country": (
        "KEWARGANEGARAAN", "Negara", "NEGARA", "Country", "COUNTRY"
    ),
    "true_label": ("True Cient (Group)", "True Client (Group)"),
    "date": ("TGL",),
    "amount": ("AMT_BLNC_MEM",),
}

"""## 2. Normalization and custodian-prefix removal
Only text preparation and undisclosed-account primitives live here.
"""

def clean_name(name):
    """
    Normalize investor names and remove common custodian prefixes
    and trailing registry/account numbers.
    """
    if pd.isna(name):
        return ''
    name = str(name).upper().strip()

    patterns = [
        # =========================================================
        # JPMORGAN
        # Specific and multi-layer patterns must come first
        # =========================================================
        r'^JPMCB NA AIF CLT RE\s*[-:]?\s*',
        r'^JPMCB NA UCITS CLT RE\s*[-:]?\s*',
        r'^JPMCB NA RE\s*[-:]?\s*',
        r'^JPMCB NA RE:\s*',

        r'^JPMORGAN CHASE BANK,\s*N\.A\.\s*[-:]?\s*',

        r'^JPMSE LUX AIF CLT RE\s*[-:]?\s*',
        r'^JPMSE LUX RE\s*[-:]?\s*',

        r'^JPMSE STOCK RE UCITS CLT RE\s*[-:]?\s*',
        r'^JPMSE STOCK RE UCITS RE\s*[-:]?\s*',
        r'^JPMSE STOCK RE\s*[-:]?\s*',

        r'^JPMSE COPEN RE UCITS CLT RE\s*[-:]?\s*',
        r'^JPMSE COPEN RE\s*[-:]?\s*',

        r'^JPMSE OSLO RE AIF CLT RE\s*[-:]?\s*',
        r'^JPMSE OSLO RE\s*[-:]?\s*',

        r'^JPMSE DUB RE UCITS CLT RE\s*[-:]?\s*',
        r'^JPMSE DUB RE\s*[-:]?\s*',

        r'^JPMSE AMS RE\s*[-:]?\s*',

        r'^JPMBL SA AIF CLT RE\s*[-:]?\s*',
        r'^JPMBL SA RE\s*[-:]?\s*',

        r'^JPMS SA RE\s*[-:]?\s*',

        r'^JPMSE LUX RE UCITS CLT RE\s*[-:]?\s*',
        r'^UCITS CLT RE\s*[-:]?\s*',

        # =========================================================
        # BNY MELLON
        # Multi-layer BNYM prefixes
        # Optional apostrophe handles names beginning with 'BNYM
        # =========================================================
        r"^'?BNYM RE BNYMLB RE\s*",
        r"^'?BNYM RE LB RE\s*",
        r"^'?BNYM RE CIBC RE\s*",
        r"^'?BNYM RE BNYMCIBC RE\s*",
        r"^'?BNYM RE FOR\s+",
        r"^'?BNYM RE\s*[-:]?\s*",

        r'^BNYM S/A\s*',

        # Multi-layer BNYMSANV prefixes
        r'^BNYMSANV RE BNYMSANVLUX RE\s*',
        r'^BNYMSANV RE BNYMSANVAMS RE\s*',
        r'^BNYMSANV RE BNYMIL RE\s*',

        r'^BNYMSANV RE SANVAMS RE\s*',
        r'^BNYMSANV RE SANVDUB RE\s*',
        r'^BNYMSANV RE SANVLUX RE\s*',

        r'^BNYMSANV RE AMS RE\s*',
        r'^BNYMSANV RE DUB RE\s*',

        # General BNYMSANV patterns must remain after specific ones
        r'^BNYMSANV RE\s*',
        r'^BNYMSANVAMS RE\s*',
        r'^BNYMSANVLUX RE\s*',
        r'^BNYMCIBC RE\s*',

        # BNY Mellon remaining intermediate layers
        r'^BNYMSANV RE BNYMLB RE\s*',
        r'^BNYMLB RE\s*',
        r'^S/A\s+',

        # =========================================================
        # NORTHERN TRUST
        # =========================================================
        r'^NTGS SE LUX\s*[-:]?\s*',
        r'^NTC\s*[-:]?\s*',

        # =========================================================
        # STATE STREET BANK
        # Removes SSB plus a short account identifier
        # Examples: SSB KG33, SSB 2CW4, SSB J710
        # =========================================================
        r'^SSB\s+[A-Z0-9]{3,6}\s+',

        # =========================================================
        # CITIBANK
        # =========================================================
        r'^CITIBANK NEW YORK S/A\s+',
        r'^CITIBANK NY ADR S/A\s+',
        r'^CITIBANK LONDON S/A\s+',
        r'^CITIBANK HONG KONG S/A\s+',
        r'^CITIBANK SINGAPORE S/A\s+',
        r'^CITIBANK EUROPE PLC LUX BRANCH S/A\s+',
        r'^CITIBANK SINGAPORE A/C\s+CBSG-CTSL AC\s+',

        # =========================================================
        # HSBC
        # Specific account and service patterns first
        # =========================================================
        r'^HSBC CONTINENTAL EUROPE SA GERMANY S/A\s+',

        r'^HSBC-FUND SERVICES(?:,)?\s+A/C\s+',
        r'^HSBC-FUND SERVICES(?:,)?\s*-\s*',
        r'^HSBC-FUND SERVICES(?:,)?\s+',

        r'^HSBC-FUND SVS A/C\s+',
        r'^HSBC-FUND SVS AC\s+',

        r'^HSBC BANK PLC S/A\s+',
        r'^HSBC BANK PLC RE:\s*',
        r'^HSBC BANK PLC\s*/\s*',
        r'^HSBC BANK PLC\s*-\s*',

        r'^HSBC BK PLC S/A\s+',

        # =========================================================
        # BROWN BROTHERS HARRIMAN
        # =========================================================
        r'^BBH BOSTON S/A\s+',
        r'^BBH LUXEMBOURG S/A\s+',
        r'^BBH CO S/A\s+',
        r'^BBH CO SA\s+',

        # This is broader, so it must remain after the patterns above
        r'^BBH CO\s+',

        # =========================================================
        # RBC
        # =========================================================
        r'^RBC IST S/A\s+',

        # =========================================================
        # BNP PARIBAS SECURITIES SERVICES / BP2S
        # Specific location patterns first
        # =========================================================
        r'^BP2S LUXEMBOURG S/A\s+',
        r'^BP2S PARIS S/A\s+',
        r'^BP2S SYDNEY S/A\s+',

        r'^BP2S ZURICH\s*/\s*',
        r'^BP2S PARIS\s*/\s*',
        r'^BP2S FRANKFURT\s*/\s*',

        # Handles other BP2S locations followed by S/A
        r'^BP2S\s+[^/]+\s*/?\s*S/A\s+',

        # General slash-based BP2S fallback
        # Example: BP2S LOCATION/FUND NAME
        r'^BP2S\s+[^/]+\s*/\s*',

        # =========================================================
        # BNP PARIBAS /2S LOCATION FORMATS
        # =========================================================
        r'^BNPP SYD/2S/',
        r'^BNPP LUX/2S/',
        r'^BNPP FFT/2S/',
        r'^BNPP LDN/2S/',
        r'^BNPP SIN/2S/',

        # =========================================================
        # STANDARD CHARTERED
        # =========================================================
        r'^SCB SG S/A\s+',
        r'^SCB HONG KONG A/C\s+',

        # =========================================================
        # DBS
        # =========================================================
        r'^DBS BANK LTD S/A\s+',
        r'^DBS BANK LTD SA\s+',

        # =========================================================
        # SEB
        # =========================================================
        r'^SEB S/A\s*',

        # =========================================================
        # AMS
        # Keep this near the end because it is relatively broad
        # =========================================================
        r'^AMS RE\s*',

        # =========================================================
        # SUMITOMO MITSUI TRUST BANK
        # =========================================================
        r'^SMTBUSA S/A\s+',
        r'^SMTBUSA\s+',

        # =========================================================
        # CUSTODY BANK OF JAPAN
        # =========================================================
        r'^CUSTODY BANK OF JAPAN LTD RE:\s*',
        r'^CUSTODY BANK OF JAPAN,\s*LTD\.\s*RE:\s*',

        # =========================================================
        # PICTET
        # =========================================================
        r'^BANK PICTET AND CIE \(EUROPE\) AG\s*,?\s*'
        r'SUCCURSALE DE LUXEMBOURG\s+',

        r'^BANQUE PICTET AND CIE SA\s*/\s*',

        # =========================================================
        # INDIVIDUAL-ACCOUNT CUSTODY FORMATS
        # These require an explicit S/A indicator to avoid removing
        # the bank when it is actually the investor
        # =========================================================
        r'^UBS AG SINGAPORE S/A\s+',
        r'^LGT BANK AG S/A\s+',

        # =========================================================
        # OTHER CUSTODIANS
        # =========================================================
        r'^RBI S/A\s+',
    ]

    for pattern in patterns:
        name = re.sub(pattern, '', name)
    name = re.sub(r'\s*-\s*\d{6,}\s*$', '', name)
    name = re.sub(r'\s+', ' ', name)
    name = re.sub(r'(?:\s*-?\s*\d{7,})+\s*$','',name,).strip()
    return name.strip()

def keyword_in_name(keyword, name):
    """
    Match keyword as complete token or phrase.
    """
    if not keyword or not name:
        return False
    keyword = str(keyword).upper().strip()
    name = str(name).upper().strip()
    pattern = fr'(?<![A-Z0-9]){re.escape(keyword)}(?![A-Z0-9])'
    return re.search(pattern, name) is not None

def get_undisclosed_intermediary(name):
    markers = ('CLEAR ACCT', 'OMNIBUS CLIENT ACCOUNT', 'NON-TREATY OMNIBUS ACCOUNT', 'OMNIBUS ACCOUNT', 'PB CLIENTS', 'PB CAYMAN CLIENTS', 'CLIENT ACCOUNT', 'CLIENT AC', 'CLIENT A/C', 'PRIVATE BK-CLIENTS', 'PRIVATE BANKING DIVISION ACCOUNT CLIENTS', 'FOR CLIENTS A/C')
    if keyword_in_name('AIF CLIENTS', name):
        return 'UNDISCLOSED AIF CLIENT(S)'
    for marker in markers:
        if keyword_in_name(marker, name):
            return 'UNDISCLOSED CLIENT(S)'
    for pattern in UNDISCLOSED_REGEXES:
        if re.search(pattern, name):
            return 'UNDISCLOSED CLIENT(S)'
    for keyword in UNDISCLOSED_INTERMEDIARY_KEYWORDS:
        if keyword_in_name(keyword, name):
            return 'UNDISCLOSED CLIENT(S)'
    return None
UNDISCLOSED_REGEXES = (r'/\s*CLIENTS?\b', r'\bCLT\s+A/?C\b')
UNDISCLOSED_INTERMEDIARY_KEYWORDS = ('CACEIS BANK', 'CLEARSTREAM', 'EUROCLEAR',)

"""## 3. Core manager and SID mapping catalog
This is business data. Add reviewed aliases here, not inside classifier methods.
"""

MANAGER_KEYWORD_TO_PARENT = {
    # === A.J. Central Asia Raya ===
    'ASURANSI JIWA CENTRAL ASIA RAYA': 'A.J. Central Asia Raya',

    # === aberdeen plc ===
    'ABERDEEN': 'aberdeen plc',
    'ABRDN': 'aberdeen plc',

    # === ABN AMRO Investment ===
    'ABN AMRO': 'ABN AMRO Investment',

    # === AIA Group ===
    'AIA': 'AIA Group',

    # === Allianz SE ===
    'ALLIANZ GLOBAL INVESTORS': 'Allianz SE',
    'ALLIANZ': 'Allianz SE',
    'PIMCO': 'Allianz SE',

    # === Ameriprise Financial ===
    'COLUMBIA THREADNEEDLE': 'Ameriprise Financial',
    'PYRFORD INTERNATIONAL': 'Ameriprise Financial',

    # === AP-fonden ===
    'SEVENTH SWEDISH NATIONAL PENSION FUND': 'Sjunde AP-fonden',
    'SEVENTH SWEDISH NATIONAL PEN FD': 'Sjunde AP-fonden',
    'SJUNDE AP FONDEN': 'Sjunde AP-fonden',
    'SJUNDE AP-FONDEN': 'Sjunde AP-fonden',
    'ANDRA AP FONDEN': 'Andra AP-fonden',
    'ANDRA AP-FONDEN': 'Andra AP-fonden',

    # === AQR Capital Mgt ===
    'AQR': 'AQR Capital Mgt',

    # === ARGA Investment Mgt ===
    'ARGA': 'ARGA Investment Mgt',

    # === Arrowstreet Capital ===
    'ARROWSTREET': 'Arrowstreet Capital',

    # === ARTISAN PARTNERS ===
    'ARTISAN': 'ARTISAN PARTNERS',

    # === Ashmore Group ===
    'ASHMORE': 'Ashmore Group',

    # === AVANTIS INVESTORS ===
    'AVANTIS': 'AVANTIS INVESTORS',

    # === Avrist Asset Mgt ===
    'PT AVRIST ASSURANCE': 'Avrist Asset Mgt',
    'AVRIST LINK': 'Avrist Asset Mgt',
    'AVRIST': 'Avrist Asset Mgt',

    # === AXA Asset Mgt Indonesia ===
    'PT AXA FINANCIAL INDONESIA': 'AXA Asset Mgt Indonesia',
    'AXA FINANCIAL INDONESIA': 'AXA Asset Mgt Indonesia',
    'MAESTROLINK': 'AXA Asset Mgt Indonesia',

    # === AXA Group ===
    'AXA': 'AXA Group',

    # === Bahana TCW Investment Mgt ===
    'BAHANA': 'Bahana TCW Investment Mgt',

    # === Baillie Gifford & Co ===
    'BAILLIE GIFFORD': 'Baillie Gifford & Co',
    'BA GIFF': 'Baillie Gifford & Co',

    # === Bank Central Asia ===
    'PT BCA SEKURITAS': 'Bank Central Asia',

    # === Bank Indonesia ===
    'DP BANK INDONESIA': 'Dana Pensiun Bank Indonesia',
    'YKK BANK INDONESIA': 'Yayasan Kesejahteraan Karyawan Bank Indonesia',

    # === Bank Negara Indonesia ===
    'BNI ASSET': 'BNI Asset Management',
    'BNI LIFE': 'BNI Life',
    'BNI-AM': 'BNI Asset Management',

    # === Bank of Montreal ===
    'BMO': 'Bank of Montreal',

    # === Bank of New York Mellon ===
    'NEWTON INVESTMENT': 'Newton Invesment',
    'MELLON': 'Bank of New York Mellon',

    # === Bank Rakyat Indonesia ===
    'BANK RAKYAT INDONESIA': 'Bank Rakyat Indonesia',
    'DANA PENSIUN BRI': 'Dana Pensiun BRI',
    'DP PEGADAIAN': 'Dana Pensiun Pegadaiaian',
    'PEKERJA BRI': 'Serikat Pekerja BRI',
    'BRI LIFE':'BRI Life',

    # === Bank Tabungan Negara ===
    'DANA PENSIUN BTN': 'Dana Pensiun BTN',

    # === Banque Pictet & Cie ===
    'PICTET': 'Banque Pictet & Cie',

    # === Batavia Prosperindo Asset Mgt ===
    'BATAVIA': 'Batavia Prosperindo Asset Mgt',

    # === BlackRock Inc ===
    'APERIO GROUP': 'BlackRock Inc',
    'BLACKROCK': 'BlackRock Inc',
    'ISHARES': 'BlackRock Inc',

    # === BNP Paribas Group ===
    'AXA INVESTMENT MGR': 'AXA Investment Managers',
    'AXA IM SELECT': 'AXA Investment Managers',
    'BNP PARIBAS': 'BNP Paribas Asset Management',

    # === BPJS Kesehatan ===
    'BPJS KESEHATAN': 'BPJS Kesehatan',

    # === BPJS Ketenagakerjaan ===
    'BPJS KETENAGAKERJAAN': 'BPJS Ketenagakerjaan',
    'DJS KETENAGAKERJAAN': 'BPJS Ketenagakerjaan',

    # === Brandes Investment Partners ===
    'BRANDES': 'Brandes Investment Partners',

    # === Charles Schwab Investment Mgt ===
    'SCHWAB': 'Charles Schwab Investment Mgt',

    # === DAIWA ASSET MGT ===
    'DAIWA': 'DAIWA ASSET MGT',

    # === Dana Pensiun PLN ===
    'DAPEN PLN': 'Dana Pensiun PLN',

    # === DANAREKSA INVESTMENT MANAGEMENT ===
    'DANAREKSA': 'DANAREKSA INVESTMENT MANAGEMENT',

    # === Deutsche Bank ===
    'DWS INVESTMENTS': 'Deutsche Bank',
    'DBX ADVISORS': 'Deutsche Bank',

    # === Dimensional Fund Advisors ===
    'DIMENSIONAL': 'Dimensional Fund Advisors',
    'DFA': 'Dimensional Fund Advisors',

    # === EARNEST PARTNERS ===
    'EARNEST': 'EARNEST PARTNERS',

    # === Fidelity Mgt & Research ===
    'SPARTAN GROUP TRUST FOR EMPLOYEE BENEFIT PLANS': 'Fidelity Mgt & Research',
    'FIDELITY INVESTMENTS': 'Fidelity Mgt & Research',
    'FIDELITY': 'Fidelity Mgt & Research',

    # === Fidelity Worldwide Investment (FIL) ===
    'FIL INVESTMENT': 'Fidelity Worldwide Investment (FIL)',
    'FID FDS': 'Fidelity Worldwide Investment (FIL)',

    # === FWD Insurance ===
    'FWD USD BALANCED PLUS FUND': 'FWD Insurance',
    'FWD': 'FWD Insurance',

    # === Generali Indonesia ===
    'AJ GENERALI INDONESIA': 'Generali Indonesia',
    'GENERALI INDONESIA': 'Generali Indonesia',
    'GENERALI': 'Generali Indonesia',

    # === Government of Indonesia ===
    'BRI INVESTMENT': 'BRI Investment Management',
    'BRI MANAJEMEN': 'BRI Investment Management',
    'REKSA DANA INDEKS DANAREKSA MSCI INDONESIA ESG SCREENED': 'Danareksa',
    'BRI MAWAR': 'BRI Investment Management',

    # === GREAT EASTERN LIFE ===
    'GREAT EASTERN': 'Great Eastern Life Assurance',

    # === GTCR ===
    'ALLSPRING': 'Allspring Global Investments',

    # === GUIDESTONE FUNDS ===
    'GUIDESTONE': 'AQR Capital Mgt',

    # === Guinness Global Investors ===
    'GUINNESS': 'Guinness Atkinson Asset Mgt',

    # === Harding Loevner Mgt ===
    'HARDING LOEVNER': 'Harding Loevner Mgt',

    # === Henan Putihrai Asset Mgt ===
    'HENAN': 'Henan Putihrai Asset Mgt',
    'HPAM': 'Henan Putihrai Asset Mgt',

    # === HSBC Holdings Plc ===
    'HSBC INDEX TRACKER INVESTMENT FUNDS': 'HSBC Holdings Plc',
    'HSBC GLOBAL INVESTMENT FUNDS': 'HSBC Holdings Plc',
    'HSBC POOLED INVESTMENT FUND': 'HSBC Holdings Plc',
    'HSBC INDEX TRACKER': 'HSBC Holdings Plc',
    'HSBC MULTI FACTOR': 'HSBC Holdings Plc',
    'HSBC GLOBAL': 'HSBC Holdings Plc',
    'HSBC POOLED': 'HSBC Holdings Plc',
    'HSBC ETFS': 'HSBC Holdings Plc',
    "INKA MBH": "HSBC Holdings Plc",

    # === IFM Investors ===
    'HPL-HOSTPLUS POOLED SUPERANNUATION TRUST IFM EMERGING MARKET': 'IFM Investors',

    # === Indo Premier Investment Management ===
    'REKSA DANA INDEKS PREMIER ETF': 'Indo Premier Investment Management',
    'RD PREMIER ETF': 'Indo Premier Investment Management',

    # === Indonesia Financial Group (IFG) ===
    'PT ASURANSI JIWA IFG': 'Indonesia Financial Group (IFG)',
    'ASURANSI JIWA IFG': 'Indonesia Financial Group (IFG)',
    'IFG LINK': 'Indonesia Financial Group(IFG)',

    # === Intesa Sanpaolo Group ===
    'FIDEURAM': 'Fideuram - Intesa Sanpaolo',
    'EURIZON': 'Eurizon Capital',

    # === Invesco Ltd ===
    'INVESCO': 'Invesco Ltd',

    # === IRISH LIFE ASSURANCE ===
    'IRISH LIFE ASSURANCE': 'IRISH LIFE ASSURANCE',

    # === Janus Henderson Group plc ===
    'JANUS HENDERSON': 'Janus Henderson Group plc',

    # === JPMorgan Chase & Co ===
    'COMMINGLED PENSION TRUST FUND': 'JPMorgan Chase & Co',
    'JP MORGAN SECURITIES': 'JPMorgan Chase & Co',

    # === Julius Baer Group ===
    'JULIUS BAER': 'Julius Baer Group',

    # === KBC Asset Mgt ===
    'KBC': 'KBC Asset Mgt',

    # === KISI Asset Mgt ===
    'KISI': 'KISI Asset Mgt',

    # === KRESNA ASSET MANAGEMENT ===
    'KRESNA': 'KRESNA ASSET MANAGEMENT',

    # === Laerernes Pension ===
    'LAERERNES PENSION FORSIKRINGSAKTIESELSKAB': 'Laerernes Pension',

    # === Lazard ===
    'LAZARD': 'Lazard',

    # === Legal & General Investment Mgt ===
    'LEGAL AND GENERAL': 'Legal & General Investment Mgt',
    'LEGAL & GENERAL': 'Legal & General Investment Mgt',
    'L&G ASSURANCE': 'Legal & General Investment Mgt',

    # === Lombard Odier Darier Hentsch & Cie ===
    'LOMBARD ODIER': 'Lombard Odier Darier Hentsch & Cie',

    # === M&G INVESTMENTS ===
    'M AND G': 'M&G INVESTMENTS',
    'M&G': 'M&G INVESTMENTS',

    # === Macquarie Group ===
    'MACQUARIE': 'Macquarie Group',

    # === Mandiri Investasi ===
    'MANDIRI INVESTA': 'Mandiri Investasi',
    'MMI': 'Mandiri Investasi',

    # === Manulife Financial Corp ===
    'MANULIFE': 'Manulife Financial Corp',

    # === Marsh McLennan ===
    'MERCER UCITS COMMON CONTRACTUAL FUND': 'Marsh McLennan',
    'MERCER EMERGING MARKETS EQUITY FUND': 'Marsh McLennan',
    'MERCER EMERGING MARKETS SHARES FUND': 'Marsh McLennan',
    'MERCER EMERGING MARKETS': 'Marsh McLennan',
    'MERCER QIF FUND': 'Marsh McLennan',
    'MERCER': 'Marsh McLennan',

    # === Matthews International Capital Mgt ===
    'MATTHEWS': 'Matthews International Capital Mgt',

    # === Metlife ===
    'PINEBRIDGE': 'Metlife',
    'FWD ASSET': 'Metlife',

    # === Mirae Asset Financial Group ===
    'MIRAE ASSET': 'Mirae Asset Financial Group',
    'GLOBAL X': 'Mirae Asset Financial Group',

    # === Mitsubishi UFJ Financial Group ===
    'STEWART INVESTORS': 'Stewart Investors',
    'MITSUBISHI UFJ': 'Mitsubishi UFJ Financial Group',
    'FIRST SENTIER': 'First Sentier Investors',

    # === Mondrian Investment Partners ===
    'MONDRIAN': 'Mondrian Investment Partners',

    # === Nomura Holdings Inc. ===
    'NOMURA': 'Nomura Holdings Inc.',

    # === Northern Trust Corporation ===
    'NORTHERN TRUST GLOBAL INVESTMENTS': 'Northern Trust Corporation',
    'NORTHERN TRUST QUANTITATIVE': 'Northern Trust Corporation',
    'NORTHERN TRUST INVESTMENTS': 'Northern Trust Corporation',
    'NORTHERN EMERGING MARKETS': 'Northern Trust Corporation',
    'NORTHERN TRUST COMMON': 'Northern Trust Corporation',
    'NORTHERN TRUST UCITS': 'Northern Trust Corporation',
    'FLEXSHARES': 'Northern Trust Corporation',

    # === OCBC Group ===
    'BANK OF SINGAPORE': 'Bank of Singapore',

    # === ORIX Corporation ===
    'ROBECO': 'ORIX Corporation',

    # === Panin Asset Mgt ===
    'PANIN SEKURITAS': 'Panin Group',
    'PANIN': 'Panin Group',

    # === Pinnacle Investment ===
    'PINNACLE': 'Pinnacle Investment',

    # === Premier Miton Investors ===
    'PREMIER': 'Premier Miton Investors',

    # === Prudential Group ===
    'PRUDENTIAL HONG KONG': 'Prudential Group',
    'EASTSPRING': 'Eastspring Investments',
    'PRUDENTIAL': 'Prudential Group',
    "ESPRING": "Eastspring Investments",

    # === PT Astra International Tbk ===
    'DANA PENSIUN ASTRA': 'Dana Pensiun Astra',
    'AVA EQUITY DOLLAR NUSANTARA FUND': 'PT Asuransi Jiwa Astra',
    'AVA EQUITY DOLLAR': 'PT Asuransi Jiwa Astra',
    'AVA': 'PT Asuransi Jiwa Astra',

    # === PT Grow Investments Indonesia ===
    'GROW SRI-KEHATI': 'PT Grow Investments Indonesia',

    # === PT. MSIG LIFE INSURANCE INDONESIA TBK ===
    'PT. MSIG LIFE INSURANCE INDONESIA TBK': 'PT. MSIG LIFE INSURANCE INDONESIA TBK',
    'PT MSIG LIFE INSURANCE INDONESIA TBK': 'PT. MSIG LIFE INSURANCE INDONESIA TBK',
    'MSIG LIFE INSURANCE INDONESIA': 'PT. MSIG LIFE INSURANCE INDONESIA TBK',
    'MSIG': 'PT. MSIG LIFE INSURANCE INDONESIA TBK',

    # === Pzena Investment Management ===
    'RELIANCE TRUST INSTITUTIONAL RETIREMENT TRUST SERIES THIRTY SEVEN': 'Pzena Investment Management',

    # === Pzena Investment Mgt ===
    'PZENA': 'Pzena Investment Mgt',

    # === QIC Asset Mgt ===
    'QIC': 'QIC Asset Mgt',

    # === QUILTER INVESTORS ===
    'QUILTER': 'QUILTER INVESTORS',

    # === S-Bank Plc ===
    'S-BANK': 'S-Bank Plc',

    # === Samuel Asset Mgt ===
    'SAM DANA': 'Samuel Asset Mgt',

    # === SAS Rue la Boétie ===
    'KBI GLOBAL INVESTORS': 'SAS Rue la Boétie',
    'AMUNDI': 'SAS Rue la Boétie',

    # === Schroders ===
    'SCHRODERS': 'Schroders',
    'SCHRODER': 'Schroders',

    # === SEB Asset Mgt ===
    'SEB': 'SEB Asset Mgt',

    # === SEI Investments ===
    'SEI INSTITUTIONAL': 'SEI Investments',

    # === Sequis Asset Management ===
    'PT. AJ SEQUIS LIFE': 'Sequis Asset Management',
    'PT AJ SEQUIS LIFE': 'Sequis Asset Management',
    'KPD SQAM SQMF': 'Sequis Asset Management',
    'SQAM SQMF': 'Sequis Asset Management',

    # === Sinarmas Asset Mgt ===
    'SINAR MAS': 'Sinarmas Asset Mgt',
    'SIMAS': 'Sinarmas Asset Mgt',

    # === STAR Asset Management ===
    'STAR INFOBANK15': 'STAR Asset Management',

    # === State Street Corporation ===
    'STATE STREET': 'State Street Corporation',
    'SPDR': 'State Street Corporation',

    # === Storebrand Investments ===
    'STOREBRAND': 'Storebrand Investments',

    # === SucorInvest Asset Mgt ===
    'SUCORINVEST': 'SucorInvest Asset Mgt',

    # === Syailendra Capital ===
    'SYAILENDRA': 'Syailendra Capital',

    # === T Rowe Price Associates ===
    'T. ROWE PRICE': 'T Rowe Price Associates',

    # === T.ROWE PRICE ===
    'T.ROWE PRICE': 'T Rowe Price Associates',
    'TROWE PRICE': 'T Rowe Price Associates',

    # === TA Associates ===
    'RUSSELL INVESTMENTS': 'TA Associates',
    'RUSSELL': 'TA Associates',
    'RUSSEL': 'TA Associates',

    # === The Goldman Sachs Group, Inc ===
    'GOLDMAN SACHS': 'The Goldman Sachs Group, Inc',

    # === The Vanguard Group, Inc ===
    'VANGUARD': 'The Vanguard Group, Inc',

    # === TIAA-CREF ===
    'NUVEEN': 'TIAA-CREF',
    'TIAA': 'TIAA-CREF',

    # === Trimegah Asset Mgt ===
    'TRIMEGAH': 'Trimegah Asset Mgt',
    'TRIM': 'Trimegah Asset Mgt',

    # === UNDISCLOSED CLIENT(S) ===
    'DANA PENSIUN INDEXING': 'UNDISCLOSED CLIENT(S)',

    # === United Overseas Bank Limited ===
    'UOB ASSET': 'UOB Asset Management',
    'UOBAM': 'UOB Asset Management',

    # === VanEck Global ===
    'VANECK': 'VanEck Global',

    # === Victory Capital Holdings ===
    'VICTORY': 'Victory Capital Holdings',

    # === WISDOMTREE ===
    'WISDOMTREE': 'WISDOMTREE',

    # === ZURICH ===
    'ZURICH': 'Zurich Insurance',

    # === Colonial First State ===
    'COLONIAL FIRST STATE': 'Colonial First State',

    # === Handelsbanken Fonder ===
    'HANDELSBANKEN': 'Handelsbanken Fonder',
}

def match_parent(name, keywords):
    """
    Return the parent for the longest matching keyword.

    Raises a clear configuration error if a keyword tuple contains a keyword
    that is missing from MANAGER_KEYWORD_TO_PARENT.
    """
    for keyword in sorted(keywords, key=len, reverse=True):
        if not keyword_in_name(keyword, name):
            continue

        parent = MANAGER_KEYWORD_TO_PARENT.get(keyword)

        if parent is None:
            raise ValueError(
                f"Configuration error: {keyword!r} appears in a keyword "
                "tuple but is missing from MANAGER_KEYWORD_TO_PARENT."
            )

        return parent

    return None

PFD_MANAGER_KEYWORDS = ('BNP PARIBAS', 'SCHRODERS', 'SCHRODER', 'ASHMORE', 'ALLIANZ', 'HPAM', 'MMI', 'SUCORINVEST')
MFD_MANAGER_KEYWORDS = ('BNP PARIBAS', 'MANDIRI INVESTA', 'BRI MAWAR', 'SAM DANA', 'BNI-AM', 'MANULIFE', 'PANIN', 'SUCORINVEST', 'BATAVIA', 'TRIMEGAH', 'TRIM', 'BAHANA', 'SYAILENDRA', 'PREMIER', 'ASHMORE', 'DANAREKSA', 'EASTSPRING', 'UOBAM', 'KISI', 'KRESNA', 'HPAM', 'HENAN', 'SCHRODERS', 'SCHRODER', 'PINNACLE', 'SIMAS', 'REKSA DANA INDEKS PREMIER ETF', 'RD PREMIER ETF', 'REKSA DANA INDEKS DANAREKSA MSCI INDONESIA ESG SCREENED', 'GROW SRI-KEHATI', 'STAR INFOBANK15')
INSURANCE_KEYWORDS = ('PRUDENTIAL', 'AIA', 'ALLIANZ', 'MSIG', 'AXA', 'AVRIST', 'GENERALI', 'MANULIFE', 'ASHMORE', 'FWD', 'GREAT EASTERN', 'ZURICH', 'AVA', 'BNI LIFE', 'SINAR MAS', 'BLACKROCK', 'L&G ASSURANCE', 'IRISH LIFE ASSURANCE', 'PT. AJ SEQUIS LIFE', 'PT AJ SEQUIS LIFE', 'KPD SQAM SQMF', 'SQAM SQMF', 'PT. MSIG LIFE INSURANCE INDONESIA TBK', 'PT MSIG LIFE INSURANCE INDONESIA TBK', 'MSIG LIFE INSURANCE INDONESIA', 'PT AXA FINANCIAL INDONESIA', 'AXA FINANCIAL INDONESIA', 'MAESTROLINK', 'AJ GENERALI INDONESIA', 'GENERALI INDONESIA', 'FWD USD BALANCED PLUS FUND', 'PT ASURANSI JIWA IFG', 'ASURANSI JIWA IFG', 'IFG LINK', 'PT AVRIST ASSURANCE', 'AVRIST LINK', 'AVA EQUITY DOLLAR NUSANTARA FUND', 'AVA EQUITY DOLLAR', 'ASURANSI JIWA CENTRAL ASIA RAYA', 'BRI LIFE')
FUND_FAMILY_KEYWORDS = ('ISHARES', 'BLACKROCK', 'VANGUARD', 'FIDELITY', 'FID FDS', 'INVESCO', 'ROBECO', 'PIMCO', 'ASHMORE', 'DIMENSIONAL', 'DFA', 'T. ROWE PRICE', 'T.ROWE PRICE', 'TROWE PRICE', 'M&G', 'M AND G', 'SCHWAB', 'STATE STREET', 'SPDR', 'PZENA', 'BRANDES', 'ARGA', 'BAILLIE GIFFORD', 'BA GIFF', 'STOREBRAND', 'MERCER', 'BMO', 'WISDOMTREE', 'SEI INSTITUTIONAL', 'AQR', 'MONDRIAN', 'ARROWSTREET', 'ALLSPRING', 'GLOBAL X', 'GOLDMAN SACHS', 'NUVEEN', 'VANECK', 'SCHRODERS', 'SCHRODER', 'ABRDN', 'ABERDEEN', 'AMUNDI', 'RUSSELL', 'RUSSEL', 'ARTISAN', 'MATTHEWS', 'LAZARD', 'KBC', 'LEGAL AND GENERAL', 'LEGAL & GENERAL', 'FLEXSHARES', 'NORTHERN TRUST INVESTMENTS', 'NORTHERN TRUST UCITS', 'NORTHERN TRUST QUANTITATIVE', 'NORTHERN TRUST COMMON', 'NORTHERN EMERGING MARKETS', 'ABN AMRO', 'COLUMBIA THREADNEEDLE', 'S-BANK', 'HARDING LOEVNER', 'GUIDESTONE', 'QUILTER', 'JANUS HENDERSON', 'NOMURA', 'DAIWA', 'AVANTIS', 'VICTORY', 'GUINNESS', 'EARNEST', 'EURIZON', 'QIC', 'SEB', 'MERCER UCITS COMMON CONTRACTUAL FUND', 'MERCER EMERGING MARKETS SHARES FUND', 'MERCER EMERGING MARKETS EQUITY FUND', 'MERCER EMERGING MARKETS', 'MERCER QIF FUND')

INDO_AM_KEYWORDS = ('BNP PARIBAS', 'MANULIFE', 'PANIN', 'SUCORINVEST', 'BATAVIA', 'SYAILENDRA', 'BNI-AM')

HSBC_PRODUCT_TO_PARENT = {
    "HSBC GLOBAL INVESTMENT FUNDS": "HSBC Holdings Plc",
    "HSBC INDEX TRACKER INVESTMENT FUNDS": "HSBC Holdings Plc",
    "HSBC ETFS": "HSBC Holdings Plc",
    "HSBC POOLED INVESTMENT FUND": "HSBC Holdings Plc",
    "HSBC MULTI FACTOR": "HSBC Holdings Plc",
    "HANG SENG INSURANCE COMPANY": "HSBC Holdings Plc",
    "HSBC LIFE VALUE GROWTH FUND": "HSBC Holdings Plc",
    "INKA MBH": "HSBC Holdings Plc",
}

def match_mapping(name, keyword_to_parent):
    """
    Return the parent of the longest matching keyword in a mapping.
    """
    for keyword in sorted(keyword_to_parent, key=len, reverse=True):
        if keyword_in_name(keyword, name):
            return keyword_to_parent[keyword]

    return None

ALIAS_FALLBACK_KEYWORDS = (
    'RELIANCE TRUST INSTITUTIONAL RETIREMENT TRUST SERIES THIRTY SEVEN',
    'HPL-HOSTPLUS POOLED SUPERANNUATION TRUST IFM EMERGING MARKET',
    'SPARTAN GROUP TRUST FOR EMPLOYEE BENEFIT PLANS',
    'LAERERNES PENSION FORSIKRINGSAKTIESELSKAB',
    'SEVENTH SWEDISH NATIONAL PENSION FUND',
    'NORTHERN TRUST GLOBAL INVESTMENTS',
    'SEVENTH SWEDISH NATIONAL PEN FD',
    'COMMINGLED PENSION TRUST FUND',
    'ALLIANZ GLOBAL INVESTORS',
    'PYRFORD INTERNATIONAL',
    'COLUMBIA THREADNEEDLE',
    'PRUDENTIAL HONG KONG',
    'KBI GLOBAL INVESTORS',
    'FIDELITY INVESTMENTS',
    'RUSSELL INVESTMENTS',
    'AXA INVESTMENT MGR',
    'HSBC INDEX TRACKER',
    'LEGAL AND GENERAL',
    'NEWTON INVESTMENT',
    'SEI INSTITUTIONAL',
    'STEWART INVESTORS',
    'SJUNDE AP-FONDEN',
    'SJUNDE AP FONDEN',
    'PT BCA SEKURITAS',
    'ANDRA AP-FONDEN',
    'JANUS HENDERSON',
    'DWS INVESTMENTS',
    'BAILLIE GIFFORD',
    'ANDRA AP FONDEN',
    'LEGAL & GENERAL',
    'FIL INVESTMENT',
    'BRI INVESTMENT',
    'MITSUBISHI UFJ',
    'AXA IM SELECT',
    'T. ROWE PRICE',
    'BRI MANAJEMEN',
    'GOLDMAN SACHS',
    'L&G ASSURANCE',
    'FIRST SENTIER',
    'APERIO GROUP',
    'DBX ADVISORS',
    'T.ROWE PRICE',
    'STATE STREET',
    'TROWE PRICE',
    'HSBC POOLED',
    'HSBC GLOBAL',
    'BNP PARIBAS',
    'MIRAE ASSET',
    'ARROWSTREET',
    'DIMENSIONAL',
    'STOREBRAND',
    'FLEXSHARES',
    'PINEBRIDGE',
    'EASTSPRING',
    'HSBC ETFS',
    'UOB ASSET',
    'ALLSPRING',
    'BNI ASSET',
    'FWD ASSET',
    'BLACKROCK',
    'FIDEURAM',
    'MATTHEWS',
    'GLOBAL X',
    'ABERDEEN',
    'BNI LIFE',
    'MONDRIAN',
    'VANGUARD',
    'INVESCO',
    'ISHARES',
    'BA GIFF',
    'EURIZON',
    'VICTORY',
    'FID FDS',
    'BRANDES',
    'LAZARD',
    'NOMURA',
    'SCHWAB',
    'VANECK',
    'ROBECO',
    'MELLON',
    'AMUNDI',
    'NUVEEN',
    'BNI-AM',
    'PIMCO',
    'PZENA',
    'ABRDN',
    'SPDR',
    'ARGA',
    'TIAA',
    'SEB',
    'AQR',
    'AIA',
    'KBC',
    'DFA',
    'BMO',
    'QIC',
)
BANK_GROUP_KEYWORDS = ('JULIUS BAER', 'LOMBARD ODIER', 'PICTET', 'BANK OF SINGAPORE', 'MACQUARIE', 'JP MORGAN SECURITIES')
DOMESTIC_GROUP_KEYWORDS = ('BANK RAKYAT INDONESIA', 'PEKERJA BRI', 'DANA PENSIUN BRI', 'DP PEGADAIAN', 'BRI LIFE', 'DANA PENSIUN BTN', 'YKK BANK INDONESIA', 'DP BANK INDONESIA', 'DAPEN PLN', 'DANA PENSIUN ASTRA', 'DANA PENSIUN INDEXING', 'PANIN SEKURITAS', 'PT BCA SEKURITAS')
US_STATES = tuple(sorted(('ALABAMA', 'ALASKA', 'ARIZONA', 'ARKANSAS', 'CALIFORNIA', 'COLORADO', 'CONNECTICUT', 'DELAWARE', 'FLORIDA', 'GEORGIA', 'HAWAII', 'IDAHO', 'ILLINOIS', 'INDIANA', 'IOWA', 'KANSAS', 'KENTUCKY', 'LOUISIANA', 'MAINE', 'MARYLAND', 'MASSACHUSETTS', 'MICHIGAN', 'MINNESOTA', 'MISSISSIPPI', 'MISSOURI', 'MONTANA', 'NEBRASKA', 'NEVADA', 'NEW HAMPSHIRE', 'NEW JERSEY', 'NEW MEXICO', 'NEW YORK', 'NORTH CAROLINA', 'NORTH DAKOTA', 'OHIO', 'OKLAHOMA', 'OREGON', 'PENNSYLVANIA', 'RHODE ISLAND', 'SOUTH CAROLINA', 'SOUTH DAKOTA', 'TENNESSEE', 'TEXAS', 'UTAH', 'VERMONT', 'VIRGINIA', 'WASHINGTON', 'WEST VIRGINIA', 'WISCONSIN', 'WYOMING'), key=len, reverse=True))
PUBLIC_PENSION_WORDS = ('RETIREMENT', 'PENSION', 'TEACHERS', 'EMPLOYEES', 'INVESTMENT BOARD', 'INVESTMENT COUNCIL', 'TREASURER', 'STATE OF', 'STATE INVESTMENT')
PRIVATE_ENTITY_WORDS = ('INSURANCE', 'ASSURANCE', 'LIFE', 'CORPORATION', 'CORP', 'COMPANY', 'INC', 'LLC', 'LTD', 'LIMITED', 'BANK')
import unicodedata

"""## 4. Exact institution registry and public-owner helpers"""

def normalize_institution_name(value):
    """
    Normalize punctuation and case without deleting meaningful
    legal or institutional words.
    """
    if value is None:
        return ''
    value = unicodedata.normalize('NFKD', str(value))
    value = value.encode('ascii', 'ignore').decode('ascii')
    value = value.upper().strip().replace('&', ' AND ')
    value = re.sub('[^A-Z0-9]+', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

_REFERENCE_INSTITUTION_PARENT_SOURCE = {
    # === AP-fonden ===
    'Andra AP Fonden': 'AP-fonden',
    'AP-fonden': 'AP-fonden',
    'Sjunde AP-fonden': 'AP-fonden',

    # === Boy Scouts of America ===
    'BOY SCOUTS OF AMERICA MASTER PENSION TRUST': 'Boy Scouts of America',

    # === BPJS Kesehatan ===
    'BPJS KESEHATAN': 'BPJS Kesehatan',

    # === BPJS Ketenagakerjaan ===
    'BPJS KETENAGAKERJAAN': 'BPJS Ketenagakerjaan',
    'DJS KETENAGAKERJAAN': 'BPJS Ketenagakerjaan',

    # === Government of Abu Dhabi ===
    'Abu Dhabi Investment Authority': 'Government of Abu Dhabi',
    'Abu Dhabi Investment Authority (ADIA)': 'Government of Abu Dhabi',
    'Abu Dhabi Pension Fund': 'Government of Abu Dhabi',
    'Abu Dhabi Retirement Pensions and Benefits Fund': 'Government of Abu Dhabi',
    'Government of Abu Dhabi': 'Government of Abu Dhabi',

    # === Government of Australia ===
    'COMMONWEALTH SUPERANNUATION CORPORATION AS TRUSTEE FOR ARIA INVESTMENTS TRUST': 'Government of Australia',

    # === Government of Canada ===
    'Government of Canada': 'Government of Canada',
    'MUNICIPAL EMPLOYEES PENSION PLAN': 'Government of Canada',
    'OPSEU Pension Trust': 'Government of Canada',
    'PSP Investments': 'Government of Canada',
    'PUBLIC EMPLOYEES PENSION PLAN': 'Government of Canada',
    'Public Sector Pension Investment Board': 'Government of Canada',
    "THE WINNIPEG CIVIC EMPLOYEES' PENSION PLAN": 'Government of Canada',

    # === Government of China ===
    'Best Investment Corporation': 'Government of China',
    'China Investment Corporation': 'Government of China',
    'Flourish Investment Corporation': 'Government of China',
    'Government of China': 'Government of China',
    'National Council For Social Security Fund': 'Government of China',
    'NATIONAL COUNCIL FOR SOCIAL SECURITY FUND P.R.C.': 'Government of China',
    "People's Bank of China": 'Government of China',

    # === Government of Indonesia ===
    'BRI Investment Mgt': 'Government of Indonesia',
    'Government of Indonesia': 'Government of Indonesia',
    'Indonesia Investment Authority': 'Government of Indonesia',

    # === Government of Japan ===
    'FEDERATION OF NATIONAL PUBLIC SERVICE PERSONNEL MUTUAL AID ASSOCIATIONS': 'Government of Japan',
    'PENSION FUND ASSOCIATION FOR LOCAL GV OF': 'Government of Japan',
    'THE MASTER TRUST BANK OF JAPAN, LTD/FEDERATION OF NATIONAL PUBLIC SERVICE PERSONNEL MUTUAL AID ASSOCIATIONS 400080339': 'Government of Japan',

    # === Government of Kuwait ===
    'Government of Kuwait': 'Government of Kuwait',
    'Kuwait Investment Authority': 'Government of Kuwait',
    'Kuwait Investment Office': 'Government of Kuwait',
    'The Public Institution for Social Security': 'Government of Kuwait',

    # === Government of New Zealand ===
    'Government of New Zealand': 'Government of New Zealand',
    'Government Superannuation Fund': 'Government of New Zealand',

    # === Government of Saudi Arabia ===
    'General Organization for Social Insurance': 'Government of Saudi Arabia',
    'Government of Saudi Arabia': 'Government of Saudi Arabia',
    'Saudi Central Bank': 'Government of Saudi Arabia',

    # === Government of Singapore ===
    'Fullerton Fund Mgt': 'Government of Singapore',
    'GIC': 'Government of Singapore',
    'Government of Singapore': 'Government of Singapore',
    'Monetary Authority of Singapore': 'Government of Singapore',

    # === Government of South Korea ===
    'Government of South Korea': 'Government of South Korea',
    'National Pension Service': 'Government of South Korea',

    # === Government of Taiwan ===
    'Bureau of Labor Funds- Labor Pension Fund': 'Government of Taiwan',
    'Bureau of Labor Funds- Labor Retirement Fund': 'Government of Taiwan',
    'Bureau of Public Service Pension Fund': 'Government of Taiwan',
    'Government of Taiwan': 'Government of Taiwan',

    # === Government of United Arab Emirates ===
    'General Pension and Social Security Authority': 'Government of United Arab Emirates',
    'Government of United Arab Emirates': 'Government of United Arab Emirates',

    # === Government of United States ===
    'THRIFT SAVINGS PLAN': 'Government of United States',

    # === Pensioenfonds Zorg en Welzijn (PFZW) ===
    'Stichting Pensioenfonds Zorg en Welzijn': 'Pensioenfonds Zorg en Welzijn (PFZW)',

    # === PME Pensioenfonds ===
    'Stichting PME Pensioenfonds': 'PME Pensioenfonds',

    # === State of California ===
    'CalPERS': 'State of California',
    'LOS ANGELES COUNTY EMPL RETIREMENT ASSC': 'State of California',
    'Los Angeles County Empl Retirement Assc': 'State of California',
    'Los Angeles County Employees Retirement Association (LACERA)': 'State of California',
    'State of California': 'State of California',

    # === State of New Jersey ===
    'POLICE AND FIREMENS RETIREMENT SYS OF NJ': 'State of New Jersey',

    # === State of New Jersey - Division of Investment ===
    'State of New Jersey - Division of Investment': 'State of New Jersey - Division of Investment',

    # === State of New York ===
    'New York State Common Retirement Fund': 'State of New York',
    'State of New York': 'State of New York',
    'Teachers Retirement System of the City of New York': 'State of New York',

    # === State of Ohio ===
    'Ohio Public Employees Retirement System (OPERS)': 'State of Ohio',
    'Ohio State Teachers Retirement System': 'State of Ohio',
    'State of Ohio': 'State of Ohio',

    # === Stichting Pensioenfonds ABP ===
    'PENSIOENFONDS ABP': 'Stichting Pensioenfonds ABP',
    'STICHTING PENSIOENFONDS ABP': 'Stichting Pensioenfonds ABP',

    # === Stichting Pensioenfonds Hoogovens ===
    'Stichting Pensioenfonds Hoog': 'Stichting Pensioenfonds Hoogovens',

    # === BlackRock Inc ===
    '1895 FONDS FGR': 'BlackRock Inc',

    # === Corebridge Financial ===
    'SUNAMERICA SERIES TRUST': 'Sun Life Financial',
    'SUNAMERICA SR TST': 'Sun Life Financial',

     # Governments and public institutions
    "British Columbia Investment Management Corporation":
        "Government of British Columbia",
    "Bank of Botswana":
        "Government of Botswana",
    "Swiss National Bank":
        "Government of Switzerland",
    "Alaska Permanent Fund Corporation":
        "State of Alaska",
    "Caisse de depot et placement du Quebec":
        "Government of Quebec",
    "Regents of the University of California":
        "State of California",
    "Investment Management Corporation of Ontario":
        "Investment Management Corporation of Ontario",

    # Named pension and retirement institutions
    "Policemen's Annuity And Benefit Fund of Chicago":
        "Policemen's Annuity and Benefit Fund of Chicago",
    "National Railroad Retirement Investment Trust":
        "National Railroad Retirement Investment Trust",
    "Employees Provident Fund Board":
        "Employees Provident Fund Board",
    "Government Employees Pension Fund":
        "Government Employees Pension Fund, South Africa",
    "Hydro One Pension Plan":
        "Hydro One Pension Plan",
    "Australian Super":
        "AustralianSuper",
    "National Employment Savings Trust":
        "NATIONAL EMPLOYMENT SAVINGS TRUST",
    "PensionDanmark Pensionsforsikringsaktieselskab":
        "PensionDanmark",
    "Pensionskasse des Bundes Publica":
        "Pensionskasse des Bundes PUBLICA",

    # Named legal institutions
    "The Hong Kong Polytechnic University":
        "The Hong Kong Polytechnic University",
    "The Princess Margaret Cancer Foundation":
        "The Princess Margaret Cancer Foundation",
    "SIX SIS LTD":
        "SIX SIS LTD",
    "Trinity Health Corporation":
        "Trinity Health Corporation",
    "Warman Investments Pty. Limited":
        "WARMAN INVESTMENTS PTY. LIMITED",
    "Franciscan Alliance, Inc.":
        "FRANCISCAN ALLIANCE, INC.",
    "Wellmark, Inc.":
        "WELLMARK, INC.",

    "BANK LOMBARD ODIER AND CO LTD":
        "Lombard Odier Darier Hentsch & Cie",
    "BANK OF SINGAPORE LIMITED":
        "Bank of Singapore",
    "BANK JULIUS BAER AND CO LTD":
        "Julius Baer Group",
    "PHILLIP SECURITIES PTE LTD":
        "Phillip Capital",
    "SBI SECURITIES CO., LTD":
        "Strategic Business Innovator Group",
    "CGS INTERNATIONAL SECURITIES SINGAPORE PTE LTD":
        "China Galaxy Securities",
}


def _build_reference_map(source):
    result = {}
    for institution, parent in source.items():
        key = normalize_institution_name(institution)
        existing = result.get(key)
        if existing is not None and existing != parent:
            raise ValueError(f'Conflicting parent mapping for {institution!r}: {existing!r} versus {parent!r}')
        result[key] = parent
    return result

REFERENCE_INSTITUTION_PARENT = _build_reference_map(_REFERENCE_INSTITUTION_PARENT_SOURCE)

REFERENCE_INSTITUTION_PHRASES = tuple(
    sorted(
        _REFERENCE_INSTITUTION_PARENT_SOURCE.items(),
        key=lambda item: len(item[0]),
        reverse=True,
    )
)

def get_reference_institution_parent(cleaned_name):
    key = normalize_institution_name(cleaned_name)
    return REFERENCE_INSTITUTION_PARENT.get(key)

REFERENCE_INSTITUTION_PHRASES = tuple(
    sorted(
        _REFERENCE_INSTITUTION_PARENT_SOURCE.items(),
        key=lambda item: len(item[0]),
        reverse=True,
    )
)

def _apply_universal_overrides(result, cleaned_name):
    """
    Apply strong institution and alias mappings after SID-gated rules.
    """

    institution_parent = get_reference_institution_parent(cleaned_name)
    if institution_parent:
        return institution_parent

    for institution, parent in REFERENCE_INSTITUTION_PHRASES:
        if keyword_in_name(institution, cleaned_name):
            return parent

    for keyword in ALIAS_FALLBACK_KEYWORDS:
        if keyword_in_name(keyword, cleaned_name):
            return MANAGER_KEYWORD_TO_PARENT[keyword]

    return result

def strip_product_suffix(name):
    """'PT INDRA INVESTAMA - SUCOR ASSET' -> 'PT INDRA INVESTAMA'
    (the part after ' - ' is the fund manager / product, not the owner)."""
    head, sep, _ = name.partition(' - ')
    return head.strip() if sep and len(head.split()) >= 2 else name

def guess_public_fund_owner(name):
    """
    Public pension / treasury bodies -> their government or state.
      'OKLAHOMA PUBLIC EMPLOYEES RETIREMENT SYSTEM' -> 'State of Oklahoma'
      'GOVERNMENT OF NORWAY - 23'                   -> 'Government of Norway'
      'STICHTING PENSIOENFONDS VERVOER'             -> 'PENSIOENFONDS VERVOER'
    Returns None when no rule applies.
    """
    m = re.match(r'^(?:THE )?GOVERNMENT OF ([A-Z ]+?)\s*(?:-\s*\d+)?$', name)
    if m:
        return 'Government of ' + m.group(1).title()
    if name.startswith('STICHTING PENSIOENFONDS '):
        return name[len('STICHTING '):]
    if any((keyword_in_name(w, name) for w in PRIVATE_ENTITY_WORDS)):
        return None
    if any((w in name for w in PUBLIC_PENSION_WORDS)):
        for state in US_STATES:
            if keyword_in_name(state, name):
                return 'State of ' + state.title()
    return None


# ============================================================
# CONSERVATIVE MATCHER EXTENSIONS
# ============================================================

"""## 5. Product, contextual, and conservative matching registries"""

EXACT_PRODUCT_TO_PARENT = {
    'EMERGING MARKETS INDEX NON-LENDABLE FUND B': 'BlackRock Inc',
    'MSCI EQUITY INDEX FUND B - INDONESIA': 'BlackRock Inc',
    "PACIFIC HORIZON INVESTMENT TRUST PLC":"Baillie Gifford & Co",
    "F AND C INVESTMENT TRUST PLC":"Columbia Threadneedle Investments",
    "THE SEI EMERGING MARKETS EQUITY FUND SIMC":"SEI Investments",
    "MUAM EMERGING EQUITY INDEX MOTHER FUND":"Mitsubishi UFJ Financial Group",
    "BRIDGE BUILDER INTERNATIONAL EQUITY FUND":"Edward Jones Investments",
    "LUMIERE ASIA VALUE FUND":"Lumiere Capital",
    "AGGREGATE VALUE FUND VCC":"Aggregate Asset Mgt",
    "WESTPAC WHOLESALE INTERNATIONAL SHARE NO 2 TRUST":"Westpac Banking Corporation",
    "PUTM ACS SUSTAINABLE INDEX EMERGING MARKET EQUITY FUND HBEU AS AGENT":"Standard Life",
    "MSCI ACWI EX-U.S. IMI INDEX FUND B2":"BlackRock Inc",
    "GLOBAL ALPHA TILTS ESG NON-LENDABLE FUND B":"BlackRock Inc",
}

STRONG_MANAGER_ALIASES = {
    'BORDER TO COAST': 'BORDER TO COAST PENSIONS PARTNERSHIP',
    'GENERATION IM': 'Generation Investment Mgt',
    'EDGBASTON': 'Edgbaston Investment Partners',
    'ENSIGN PEAK': 'Ensign Peak Advisors',
    'DODGE AND COX': 'Dodge & Cox',
    'SWEDBANK ROBUR': 'Swedbank Robur',
    'AVIVA INVESTORS': 'Aviva Investors',
    'BAYERNINVEST': 'BayernInvest',
    'LANSFORSAKRINGAR': 'Lansforsakringar Fondforvaltning',
    'HANDELSBANKEN': 'Handelsbanken Fonder',
    'RAIFFEISEN': 'Raiffeisen Capital Management',
    'MACKENZIE': 'Mackenzie Investments',
    'WESPATH': 'Wespath Investment Mgt',
    'NISSAY': 'Nissay Asset Mgt',
    'SCOTTISH WIDOWS': 'Lloyds Banking Group',
    'COLONIAL FIRST STATE': 'Colonial First State',
    'UNIVERSAL-INVESTMENT': 'Universal Investment',
    'UNIVERSAL-INV': 'Universal Investment',
    'PRUSIK': 'Prusik Investment Mgt',
    'XTRACKERS': 'Deutsche Bank',
    'FONDITALIA': 'Fideuram Asset Management',
    'PREMIA': 'Premia Partners',
    'POLARIS': 'Polaris Capital Mgt',
    'BROWN ADVISORY': 'BROWN ADVISORY',

     # Asset managers and financial groups
    "PACIFIC HORIZON": "Baillie Gifford & Co",
    "F AND C INVESTMENT TRUST": "Columbia Threadneedle Investments",
    "FINREON": "Finreon",
    "TOKIO MARINE": "Tokio Marine Life Insurance Indonesia",
    "PERTA LIFE": "PertaLife Insurance",
    "TUGU PRATAMA": "Tugu Pratama Indonesia",
    "SUNLINK": "Sun Life Financial",
    "ASABRI": "PT.ASABRI ( Persero )",
    "DELA NATURA": "Cooperatie DELA",
    "COLLEGE RETIREMENT EQUITIES FUND": "TIAA-CREF",

    # Fund and manager names
    "SEI GLOBAL MASTER FUND": "SEI Investments",
    "NAM INVESTMENT FUND": "Nissay Asset Mgt",
    "MUAM": "Mitsubishi UFJ Financial Group",
    "JNL MULTI-MANAGER": "Jackson National Asset Management",
    "DIVERSIFIED GLOBAL SHARE TRUST": "MLC Investments",
    "ROLINCO": "Robeco",
    "LGPS CENTRAL": "LGPS Central",
    "VALIC": "Ameriprise Financial",
    "BRIDGE BUILDER": "Edward Jones Investments",
    "UBAM": "Union Bancaire Privee",
    "UBS (LUX) FUND SOLUTIONS": "UBS Group AG",
    "FAM SERIES UCITS": "Fineco Asset Management",
    "ACTIAM": "Cardano",
    "NORTHERN LIGHTS FUND TRUST IV": "Inspire Investing",
    "LUMIERE ASIA VALUE": "Lumiere Capital",
    "NBI DIVERSIFIED": "National Bank Investments",
    "SUMISHIN": "Sumitomo Mitsui Trust Asset Management",
    "DNB GLOBAL EMERGING MARKETS": "DNB Asset Management",
    "AGGREGATE VALUE FUND": "Aggregate Asset Mgt",
    "SMARTETFS": "Guinness Atkinson Asset Mgt",
    "UOBAM PING AN": "UOB Asset Management",
    "DANSKE INVEST": "Danske Bank Group",
    "MDPIM": "MD Financial Management",
    "VELLIV": "Velliv",
    "RUTH ASSET MANAGEMENT": "Ruth Asset Management",
    "MI SELECT MANAGERS": "Waystone Mgt",
    "UNITED INDONESIA EQUITY FUND": "United Overseas Bank Limited",
    "WESTPAC WHOLESALE": "Westpac Banking Corporation",
    "GOLDM SA ETF": "The Goldman Sachs Group, Inc",
    "BOC-PRUDENTIAL": "Prudential Group",
    "DIAM VIP": "ASSET MANAGEMENT ONE",
    "DEKA RETURN": "Deka Investment",
    "MAJ INVEST": "Maj Invest",
    "AEGON": "AEGON Asset Mgt",

    # Pension and sovereign institutions
    "BRITISH COLUMBIA INVESTMENT MANAGEMENT": "Government of British Columbia",
    "ALASKA PERMANENT": "State of Alaska",
    "CAISSE DE DEPOT ET PLACEMENT DU QUEBEC": "Government of Quebec",
    "SWISS NATIONAL BANK": "Government of Switzerland",
    "BANK OF BOTSWANA": "Government of Botswana",
    "QSMA2": "Qatar Investment Authority",
    "IMCO": "Investment Management Corporation of Ontario",
    "PENSDANMARK": "PensionDanmark",
    "AUSTRALIAN SUPER": "AustralianSuper",
    "NATIONAL EMPLOYMENT SAVINGS TRUST": "NATIONAL EMPLOYMENT SAVINGS TRUST",
    "KLP AKSJE": "KLP Kapitalforvaltning",
    "RABOBANK PEN": "Rabobank Pensioenfonds",
    "PENSIONSKASSE DES BUNDES PUBLICA": "Pensionskasse des Bundes PUBLICA",

    "BANK LOMBARD ODIER AND CO":"Lombard Odier Darier Hentsch & Cie",
    "BANK OF SINGAPORE":"Bank of Singapore",
    "BANK JULIUS BAER AND CO":"Julius Baer Group",
    "PHILLIP SECURITIES":"Phillip Capital",
    "KAPITALFORENINGEN DANSKE INVEST INSTITUTIONAL AFDELING DANICA PENSION":"Danske Bank Group",
    "PUTM ACS SUSTAINABLE INDEX EMERGING MARKET EQUITY FUND":"Standard Life",
    "CGS INTERNATIONAL SECURITIES":"China Galaxy Securities",
}

STRONG_MANAGER_KEYWORDS = tuple(
    sorted(
        STRONG_MANAGER_ALIASES,
        key=len,
        reverse=True,
    )
)

CONTEXTUAL_MANAGER_RULES = (
    # IG Wealth Management
    {
        'parent': 'Power Corporation of Canada',
        'sid_prefixes': ('MFF',),
        'all': ('IG',),
        'any': (
            'JPMORGAN IG EMERGING MKTS',
            'JPMORGAN IG EMERGING MARKETS',
            'IG EMERGING MKTS P II',
            'IG EMERGING MARKETS POOL II',
        ),
    },
    {
        'parent': 'JPMorgan Chase & Co',
        'sid_prefixes': ('MFF', 'OTF'),
        'all': ('JPMORGAN',),
    },
    {
        'parent': "ST. JAMES'S PLACE",
        'sid_prefixes': ('MFF',),
        'any': ("ST. JAMES'S", "ST.JAMES'S", 'ST. JAMES', 'ST.JAMES'),
    },
    {
        'parent': 'Marsh McLennan',
        'sid_prefixes': ('MFF',),
        'any': ('MARSH GLOBAL', 'MARSH EMERGING MARKETS', 'MGI FUNDS'),
    },
    {
        "parent": "CIBC Asset Mgt",
        "sid_prefixes": ("MFF",),
        "all": ("CIBC",),
        "any": (
            "EM MRKTS",
            "EMERGING MARKETS",
            "C EM MRKTS",
            "INT EQTY FD",
            "IMP EM ECOS",
        ),
    },
    {
        'parent': 'BlackRock Inc',
        'sid_prefixes': ('MFF', 'OTF'),
        'all': ('ISH', 'EMG', 'MK', 'EQ'),
    },
    {
        'parent': 'Public Mutual',
        'sid_prefixes': ('MFF',),
        'any': ('PUBLIC INDONESIA', 'PB INDONESIA'),
    },
    {
        'parent': 'Connor, Clark & Lunn',
        'sid_prefixes': ('MFF',),
        'any': ('CC&L', 'CONNOR CLARK', 'CONNOR, CLARK'),
    },
    {
        'parent': 'Jackson National Asset Management',
        'sid_prefixes': ('MFF',),
        'all': ('JNL',),
        'any': ('FUND', 'PORTFOLIO', 'SERIES'),
    },
    {
        'parent': 'Bank of New York Mellon',
        'sid_prefixes': ('MFF', 'OTF'),
        'any': ('BNYM GLB FDS', 'BNYM GLOBAL FUNDS'),
    },


    # Government Pension Investment Fund -> ultimate government parent
    {
        'parent': 'Government of Japan',
        'sid_prefixes': ('PFF',),
        'all': (
            'MST TST BOJ',
            'GOV PENS IF',
        ),
    },

    {
        "parent": "State of Illinois",
        "sid_prefixes": ("PFF",),
        "all": ("TEACHERS", "ILLNS"),
    },
    {
        "parent": "State of North Carolina",
        "sid_prefixes": ("PFF", "OTF"),
        "any": (
            "NORTH CAR SUP RET",
            "NORTH CRLN E INV",
        ),
    },
    {
        "parent": "Pensioenfonds Horeca & Catering",
        "sid_prefixes": ("PFF",),
        "all": ("PENS", "HORECA"),
    },
    {
        "parent": "Pensioenfonds Rail & Openbaar Vervoer",
        "sid_prefixes": ("PFF",),
        "all": ("PF", "RAIL", "OPENB"),
    },
    {
        "parent": "bpfBOUW",
        "sid_prefixes": ("PFF",),
        "all": (
            "STICHTING",
            "BEDRIJFSTAKPENSIOENFONDS",
            "BOUWNIJVERHEID",
        ),
    },
    {
        "parent": "PNO Media",
        "sid_prefixes": ("PFF",),
        "all": (
            "BEDRIJFSTAKPENSIOENFONDS",
            "MEDIA",
            "PNO",
        ),
    },
    {
        "parent": "SBZ Pensioen",
        "sid_prefixes": ("PFF",),
        "all": (
            "BEDRIJFSTAKPENSIOENFONDS",
            "ZORGVERZEKERAARS",
        ),
    },
    {
        "parent": "CAAT Pension Plan",
        "sid_prefixes": ("PFF",),
        "all": ("COLLEGES", "APPLD", "ARTS", "TCH", "PEN"),
    },
    {
        "parent": "Bayerische Versorgungskammer",
        "sid_prefixes": ("PFF",),
        "any": ("BAYVK", "BVK KAPITALVERWALTUNGSGESELLSCHAFT"),
    },
    {
        "parent": "Danske Bank Group",
        "sid_prefixes": ("MFF",),
        "all": (
            "DANSKE INVEST",
            "DANICA PENSION",
        ),
    },
    {
        "parent": "Employees Provident Fund Board Malaysia",
        "countries": ("MALAYSIA",),
        "sid_prefixes": ("PFF",),
        "any": (
            "EMPLOYEES PROVIDENT FUND BOARD",
            "EMPLOYEES PROVIDENTFD BOARD",
            "EMPLOYEES PROVIDENT FD BOARD",
            "EMPLOYEES PROVIDENT FUND BRD",
        ),
    },
)

UNDISCLOSED_EXACT_PHRASES = {
    'CGMI PB CUSTOMER ACCOUNT': 'UNDISCLOSED CITI PRIVATE BANK CLIENT(S)',
    'FBO ADR HOLDERS': 'ADR HOLDERS - UNDISCLOSED',
    "CITY OF NEW YORK GROUP TRUST":"UNDISCLOSED CLIENT(S)",
}

UNDISCLOSED_ACCOUNT_MARKERS = (
    'DAC ACCOUNT',
    'A/C CLIENT',
    'CLIENT ACCOUNT',
    'CUSTOMER ACCOUNT',
    'MAIN CUSTODY ACCOUNT',
    'A/C EMPLOYEES',
    'STAFF AND REL CO AC',
    'QUALIFIED INSTITUTIONAL INVESTORS ONLY',
    'A/C CLIENT',
)

DIRECT_ENTITY_SUFFIXES = (
    " CORPORATION",
    " CORPORATION.",
    " INC",
    " INC.",
    " LTD",
    " LTD.",
    " LIMITED",
    " PTY LIMITED",
    " PTY. LIMITED",
    " UNIVERSITY",
    " SECURITIES",
    " SECURITIES CO LTD",
    " SECURITIES PTE LTD",
)
DIRECT_ENTITY_SID_PREFIXES = ("CPF","FDF","IBF","SCF",)
PRODUCT_MARKERS = ('FUND', 'ETF', 'PORTFOLIO', 'INDEX', 'UCITS', 'ICAV','SICAV', 'MUTUAL FUND',)



def normalize_match_text(value):
    """Normalize punctuation, case, and whitespace for matching."""
    if pd.isna(value):
        return ''
    value = str(value).upper().strip().replace('&', ' AND ')
    value = re.sub(r'[^A-Z0-9]+', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()


def contains_phrase(name, phrase):
    """Match a normalized phrase as complete tokens."""
    normalized_name = normalize_match_text(name)
    normalized_phrase = normalize_match_text(phrase)
    if not normalized_name or not normalized_phrase:
        return False
    pattern = r'(?<![A-Z0-9])' + re.escape(normalized_phrase) + r'(?![A-Z0-9])'
    return bool(re.search(pattern, normalized_name))


def match_exact_product(name):
    """Match only a known full product name."""
    normalized_name = normalize_match_text(name)
    for product_name, parent in EXACT_PRODUCT_TO_PARENT.items():
        if normalized_name == normalize_match_text(product_name):
            return parent
    return None


def match_strong_manager(name):
    """Match distinctive manager aliases, longest phrase first."""
    for keyword in STRONG_MANAGER_KEYWORDS:
        if contains_phrase(name, keyword):
            return STRONG_MANAGER_ALIASES[keyword]
    return None


def match_contextual_manager(sid, name, country=None):
    """Match aliases only when their SID, token, and country context agrees."""
    sid = '' if pd.isna(sid) else str(sid).upper().strip()
    country = normalize_match_text(country)
    for rule in CONTEXTUAL_MANAGER_RULES:
        sid_prefixes = rule.get('sid_prefixes', ())
        if sid_prefixes and not sid.startswith(sid_prefixes):
            continue
        countries = {normalize_match_text(x) for x in rule.get('countries', ())}
        if countries and country not in countries:
            continue
        required = rule.get('all', ())
        if required and not all(contains_phrase(name, x) for x in required):
            continue
        alternatives = rule.get('any', ())
        if alternatives and not any(contains_phrase(name, x) for x in alternatives):
            continue
        return rule['parent']
    return None


def match_undisclosed_account(name):
    """Identify omnibus or custody structures with no safe beneficial owner."""
    for phrase, result in UNDISCLOSED_EXACT_PHRASES.items():
        if contains_phrase(name, phrase):
            return result
    for marker in UNDISCLOSED_ACCOUNT_MARKERS:
        if contains_phrase(name, marker):
            return 'UNDISCLOSED CLIENT(S)'
    return None


def keep_direct_entity(name):
    """Keep a named legal entity only when it does not look like a fund product."""
    normalized_name = normalize_match_text(name)
    has_suffix = any(
        normalized_name.endswith(normalize_match_text(suffix))
        for suffix in DIRECT_ENTITY_SUFFIXES
    )
    if not has_suffix:
        return None
    if any(contains_phrase(normalized_name, marker) for marker in PRODUCT_MARKERS):
        return None
    return name.strip()

"""## Cell 3: Ordered classifier
The sequence of rules is business logic. Specific rules run before broad SID fallbacks.
"""

# ============================================================
# COUNTRY-AWARE RULES
# ============================================================
# Country is a guardrail, not a prediction by itself. Every rule must also
# require distinctive name evidence. Add new country-aware patterns here.
COUNTRY_AWARE_RULES = (
    {
        "name": "cgs_singapore_parent",
        "parent": "China Galaxy Securities",
        "countries": ("SINGAPORE",),
        "sid_prefixes": ("IBF", "SCF"),
        "all": ("CGS INTERNATIONAL SECURITIES",),
    },
    {
        "name": "danica_danske_group",
        "parent": "Danske Bank Group",
        "countries": ("DENMARK",),
        "sid_prefixes": ("MFF",),
        "all": ("DANSKE INVEST", "DANICA PENSION"),
    },
    {
        "name": "gpif_japan",
        "parent": "Government of Japan",
        "countries": ("JAPAN",),
        "sid_prefixes": ("PFF",),
        "all": ("MST TST BOJ", "GOV PENS IF"),
    },
)


def normalize_country(value):
    """Return a stable country string for matching."""
    return normalize_match_text(value)


def match_country_aware_rule(sid, name, country):
    """Match a distinctive name pattern only in its valid country/SID scope."""
    sid = "" if pd.isna(sid) else str(sid).upper().strip()
    country = normalize_country(country)
    for rule in COUNTRY_AWARE_RULES:
        valid_countries = {normalize_country(x) for x in rule.get("countries", ())}
        if valid_countries and country not in valid_countries:
            continue
        prefixes = rule.get("sid_prefixes", ())
        if prefixes and not sid.startswith(prefixes):
            continue
        if not all(contains_phrase(name, phrase) for phrase in rule.get("all", ())):
            continue
        alternatives = rule.get("any", ())
        if alternatives and not any(contains_phrase(name, phrase) for phrase in alternatives):
            continue
        return rule
    return None

"""## 6. Configuration validation
Run this after editing mappings to catch missing parents and malformed entries.
"""

KEYWORD_GROUPS_USING_MANAGER_MAP = {
    "PFD_MANAGER_KEYWORDS":
        PFD_MANAGER_KEYWORDS,

    "MFD_MANAGER_KEYWORDS":
        MFD_MANAGER_KEYWORDS,

    "INSURANCE_KEYWORDS":
        INSURANCE_KEYWORDS,

    "FUND_FAMILY_KEYWORDS":
        FUND_FAMILY_KEYWORDS,

    "INDO_AM_KEYWORDS":
        INDO_AM_KEYWORDS,

    "ALIAS_FALLBACK_KEYWORDS":
        ALIAS_FALLBACK_KEYWORDS,

    "BANK_GROUP_KEYWORDS":
        BANK_GROUP_KEYWORDS,

    "DOMESTIC_GROUP_KEYWORDS":
        DOMESTIC_GROUP_KEYWORDS,
}


def validate_keyword_configuration():
    """
    Validate tuples that are resolved through
    MANAGER_KEYWORD_TO_PARENT.
    """
    missing_mappings = []

    for group_name, keywords in (
        KEYWORD_GROUPS_USING_MANAGER_MAP.items()
    ):
        for keyword in keywords:
            if keyword not in MANAGER_KEYWORD_TO_PARENT:
                missing_mappings.append(
                    (group_name, keyword)
                )

    if missing_mappings:
        details = "\n".join(
            f"- {group_name}: {keyword!r}"
            for group_name, keyword in missing_mappings
        )

        raise ValueError(
            "The following keywords are missing from "
            "MANAGER_KEYWORD_TO_PARENT:\n"
            f"{details}"
        )

    print(
        "Keyword configuration validation passed."
    )


def validate_direct_mappings():
    """
    Validate dedicated keyword-to-parent dictionaries.
    """
    mapping_groups = {
        "HSBC_PRODUCT_TO_PARENT":
            HSBC_PRODUCT_TO_PARENT,

        "STRONG_MANAGER_ALIASES":
            STRONG_MANAGER_ALIASES,

        "EXACT_PRODUCT_TO_PARENT":
            EXACT_PRODUCT_TO_PARENT,
    }

    problems = []

    for group_name, mapping in mapping_groups.items():
        if not isinstance(mapping, dict):
            problems.append(
                f"{group_name} must be a dictionary."
            )
            continue

        for keyword, parent in mapping.items():
            if not str(keyword).strip():
                problems.append(
                    f"{group_name} contains an empty keyword."
                )

            if not str(parent).strip():
                problems.append(
                    f"{group_name}[{keyword!r}] "
                    "has an empty parent."
                )

    if problems:
        raise ValueError(
            "Direct mapping configuration errors:\n"
            + "\n".join(
                f"- {problem}"
                for problem in problems
            )
        )

    print(
        "Direct mapping validation passed."
    )


validate_keyword_configuration()
validate_direct_mappings()

"""Deterministic investor-owner classifier."""

def validate_country_rules():
    seen = set()
    problems = []
    for rule in COUNTRY_AWARE_RULES:
        name = rule.get("name", "")
        if not name or name in seen:
            problems.append(f"Invalid or duplicate country-rule name: {name!r}")
        seen.add(name)
        if not rule.get("parent"):
            problems.append(f"{name!r} has no parent")
        if not rule.get("countries"):
            problems.append(f"{name!r} has no country guard")
        if not rule.get("all") and not rule.get("any"):
            problems.append(f"{name!r} has no name evidence")
    if problems:
        raise ValueError("Country-aware rule errors:\n- " + "\n- ".join(problems))
    print("Country-aware rule validation passed.")

"""## 7. Ordered rule engine
Rule precedence is declared once in `RULE_METHODS`. Each method is independently testable.
"""

@dataclass(frozen=True)
class InvestorContext:
    """Normalized input shared by every rule."""
    sid: str
    raw_name: str
    name: str
    country: str = ""


@dataclass(frozen=True)
class ClassificationDecision:
    """A classification plus an auditable explanation."""
    owner: str
    rule: str
    evidence: str = ""


class InvestorClassifier:
    """Ordered first-match rule engine.

    Each private method implements one concern. Rule precedence is declared
    once in RULE_METHODS rather than hidden inside one long function.
    """

    RULE_METHODS = (
        "_rule_undisclosed_account",
        "_rule_undisclosed_intermediary",
        "_rule_exact_institution",
        "_rule_exact_product",
        "_rule_country_aware",
        "_rule_contextual_manager",
        "_rule_strong_manager",
        "_rule_domestic_individual",
        "_rule_domestic_entity",
        "_rule_domestic_pension",
        "_rule_domestic_mutual_fund",
        "_rule_public_owner",
        "_rule_public_institution",
        "_rule_insurance",
        "_rule_hsbc_product",
        "_rule_fund_family",
        "_rule_indonesian_asset_manager",
        "_rule_foreign_pension",
        "_rule_foreign_bank",
        "_rule_direct_entity_fallback",
    )

    def make_context(self, sid, investor_name, country=None):
        raw = "" if pd.isna(investor_name) else str(investor_name)
        return InvestorContext(
            sid="" if pd.isna(sid) else str(sid).upper().strip(),
            raw_name=raw,
            name=clean_name(raw),
            country="" if pd.isna(country) else str(country),
        )

    def decide(self, sid, investor_name, country=None):
        ctx = self.make_context(sid, investor_name, country)
        if not ctx.sid or ctx.sid == "NAN":
            return ClassificationDecision(UNRESOLVED, "invalid_input", "missing SID")
        if not ctx.name:
            return ClassificationDecision(UNRESOLVED, "invalid_input", "missing investor name")

        for method_name in self.RULE_METHODS:
            decision = getattr(self, method_name)(ctx)
            if decision is not None:
                return decision
        return ClassificationDecision(UNRESOLVED, "no_rule_matched")

    def classify(self, sid, investor_name, country=None):
        """Backward-compatible API returning only the owner label."""
        return self.decide(sid, investor_name, country).owner

    @staticmethod
    def _decision(owner, rule, evidence=""):
        return ClassificationDecision(owner, rule, evidence)

    def _rule_undisclosed_account(self, c):
        owner = match_undisclosed_account(c.raw_name)
        return self._decision(owner, "undisclosed_account") if owner else None

    def _rule_undisclosed_intermediary(self, c):
        owner = get_undisclosed_intermediary(c.name)
        return self._decision(owner, "undisclosed_intermediary") if owner else None

    def _rule_exact_institution(self, c):
        owner = get_reference_institution_parent(c.name)
        return self._decision(owner, "exact_institution", c.name) if owner else None

    def _rule_exact_product(self, c):
        owner = match_exact_product(c.name)
        return self._decision(owner, "exact_product", c.name) if owner else None

    def _rule_country_aware(self, c):
        rule = match_country_aware_rule(c.sid, c.name, c.country)
        if not rule:
            return None
        evidence = f"{normalize_country(c.country)} | " + " + ".join(rule.get("all", ()))
        return self._decision(rule["parent"], f"country:{rule['name']}", evidence)

    def _rule_contextual_manager(self, c):
        owner = match_contextual_manager(c.sid, c.name, c.country)
        if not owner:
            return None
        owner = _apply_universal_overrides(owner, c.name)
        return self._decision(owner, "contextual_manager")

    def _rule_strong_manager(self, c):
        owner = match_strong_manager(c.name)
        if not owner:
            return None
        owner = _apply_universal_overrides(owner, c.name)
        return self._decision(owner, "strong_manager_alias")

    def _rule_domestic_individual(self, c):
        if not c.sid.startswith(("IDD", "IDF")):
            return None
        name = c.name
        account = re.search(r"\bS/A\s+(.+)$", name)
        if account:
            name = account.group(1).strip()
        return self._decision(name, "domestic_individual")

    def _rule_domestic_entity(self, c):
        if not c.sid.startswith(("CPD", "IBD", "SCD", "FDD", "OTD")):
            return None
        owner = match_parent(c.name, DOMESTIC_GROUP_KEYWORDS)
        owner = owner or strip_product_suffix(c.name)
        return self._decision(_apply_universal_overrides(owner, c.name), "domestic_entity")

    def _rule_domestic_pension(self, c):
        if not c.sid.startswith("PFD"):
            return None
        owner = match_parent(c.name, PFD_MANAGER_KEYWORDS)
        owner = owner or match_parent(c.name, DOMESTIC_GROUP_KEYWORDS)
        owner = owner or strip_product_suffix(c.name)
        return self._decision(_apply_universal_overrides(owner, c.name), "domestic_pension")

    def _rule_domestic_mutual_fund(self, c):
        if not c.sid.startswith("MFD"):
            return None
        owner = match_parent(c.name, MFD_MANAGER_KEYWORDS)
        if owner:
            return self._decision(_apply_universal_overrides(owner, c.name), "domestic_mutual_fund")
        retained = {
            "STAR INFOBANK15": "REKSA DANA INDEKS STAR INFOBANK15",
            "GROW SRI-KEHATI": "REKSA DANA INDEKS GROW SRI-KEHATI",
        }
        for keyword, owner in retained.items():
            if keyword_in_name(keyword, c.name):
                return self._decision(_apply_universal_overrides(owner, c.name), "retained_domestic_fund", keyword)
        return self._decision(_apply_universal_overrides(UNRESOLVED, c.name), "domestic_mutual_fund_unresolved")

    def _rule_public_owner(self, c):
        owner = guess_public_fund_owner(c.name)
        if not owner:
            return None
        return self._decision(_apply_universal_overrides(owner, c.name), "public_owner_heuristic")

    def _rule_public_institution(self, c):
        keywords = (
            "KETENAGAKERJAAN", "BPJS", "INVESTMENT AUTHORITY",
            "INVESTMENT BOARD", "INVESTMENT COUNCIL", "CENTRAL BANK",
            "MONETARY AUTHORITY", "GOVERNMENT OF", "RETIREMENT SYSTEM",
            "SOCIAL SECURITY",
        )
        match = next((k for k in keywords if keyword_in_name(k, c.name)), None)
        if not match:
            return None
        return self._decision(_apply_universal_overrides(c.name, c.name), "public_institution", match)

    def _rule_insurance(self, c):
        if not c.sid.startswith(("ISD", "ISF")):
            return None
        owner = match_parent(c.name, INSURANCE_KEYWORDS)
        if owner:
            return self._decision(_apply_universal_overrides(owner, c.name), "insurance_alias")
        if c.sid.startswith("ISD"):
            if keyword_in_name("TASPEN", c.name):
                return self._decision(_apply_universal_overrides("Taspen", c.name), "taspen")
            account = re.search(r"\bS/A\s+(.+)$", c.name)
            if account and account.group(1).strip():
                return self._decision(_apply_universal_overrides(account.group(1).strip(), c.name), "insurance_separated_account")
            if keyword_in_name("ASURANSI", c.name) or keyword_in_name("REASURANSI", c.name):
                return self._decision(_apply_universal_overrides(c.name, c.name), "named_domestic_insurer")
        return self._decision(_apply_universal_overrides(UNRESOLVED, c.name), "insurance_unresolved")

    def _rule_hsbc_product(self, c):
        owner = match_mapping(c.name, HSBC_PRODUCT_TO_PARENT)
        if not owner:
            return None
        return self._decision(_apply_universal_overrides(owner, c.name), "hsbc_product")

    def _rule_fund_family(self, c):
        owner = match_parent(c.name, FUND_FAMILY_KEYWORDS)
        if not owner:
            return None
        return self._decision(_apply_universal_overrides(owner, c.name), "fund_family")

    def _rule_indonesian_asset_manager(self, c):
        owner = match_parent(c.name, INDO_AM_KEYWORDS)
        if not owner:
            return None
        return self._decision(_apply_universal_overrides(owner, c.name), "indonesian_asset_manager")

    def _rule_foreign_pension(self, c):
        if not c.sid.startswith("PFF"):
            return None
        pension_words = (
            "PENSION", "PENSIOENFONDS", "RETIREMENT", "PROVIDENT",
            "SUPERANNUATION", "SAVINGS PLAN", "401(K)", "401 (K)",
            "EMPLOYEE BENEFIT", "ANNUITY", "AP-FONDEN", "PENSIONFORSIKRING",
            "PERSONALVORSORGE", "RETIREMENT SYSTEM", "RETIREMENT FUND",
            "PENSION PLAN", "MASTER TRUST",
        )
        match = next((k for k in pension_words if keyword_in_name(k, c.name)), None)
        if not match:
            return None
        return self._decision(_apply_universal_overrides(c.name, c.name), "foreign_pension_name", match)

    def _rule_foreign_bank(self, c):
        if not c.sid.startswith("IBF"):
            return None
        owner = match_parent(c.name, BANK_GROUP_KEYWORDS)
        if owner:
            return self._decision(_apply_universal_overrides(owner, c.name), "foreign_bank_group")
        keep_words = ("BANK", "BANQUE", "SECURITIES", "BANKING", "CLEARSTREAM", "HEALTH")
        match = next((k for k in keep_words if keyword_in_name(k, c.name)), None)
        if not match:
            return None
        return self._decision(_apply_universal_overrides(c.name, c.name), "named_foreign_bank", match)

    def _rule_direct_entity_fallback(self, c):
        if not c.sid.startswith(DIRECT_ENTITY_SID_PREFIXES):
            return None
        owner = keep_direct_entity(c.name)
        if not owner:
            return None
        return self._decision(_apply_universal_overrides(owner, c.name), "direct_entity_fallback")


DEFAULT_CLASSIFIER = InvestorClassifier()


def classify_owner(sid, investor_name, country=None):
    return DEFAULT_CLASSIFIER.classify(sid, investor_name, country)

"""## 2. Colab paths"""

# Colab file paths
INPUT_XLSX = "/content/BBRI_combined.xlsx"
OUTPUT_XLSX = "/content/classified_test_output_manager_first.xlsx"

"""## 3. Manager-first attribution layer"""

# ============================================================
# FUND-MANAGER ATTRIBUTION LAYER
# ============================================================
# Objective: identify the entity controlling the investment decision.
# These rules run BEFORE beneficiary-owner and government-parent fallbacks.

FUND_MANAGER_RULES = (
    # BlackRock and related brands/products
    {"name":"blackrock_brand", "manager":"BlackRock Inc", "any":(
        "BLACKROCK", "ISHARES", "APERIO GROUP", "BLACKROCK INVESTMENT MGT",
        "BLACKROCK INVESTMENT MANAGEMENT", "BLACKROCK JAPAN",
    )},
    {"name":"blackrock_collective_products", "manager":"BlackRock Inc", "countries":("UNITED STATES",), "any":(
        "MSCI ACWI EX-U.S. IMI INDEX FUND B2",
        "GLOBAL ALPHA TILTS ESG NON-LENDABLE FUND B",
        "EMERGING MARKETS INDEX NON-LENDABLE FUND B",
        "MSCI EQUITY INDEX FUND B - INDONESIA",
    )},

    # Vanguard
    {"name":"vanguard", "manager":"The Vanguard Group, Inc", "all":("VANGUARD",)},

    # Invesco. Manager evidence beats underlying beneficial-owner text.
    {"name":"invesco", "manager":"Invesco Ltd", "all":("INVESCO",)},
    {"name":"invesco_developing_markets", "manager":"Invesco Ltd", "countries":("CANADA",), "all":("INV DEVELOPING MARKETS CLASS",)},

    # Pzena, including delegated sleeves and Reliance trust mandate.
    {"name":"pzena", "manager":"Pzena Investment Mgt", "all":("PZENA",)},
    {"name":"pzena_reliance_trust", "manager":"Pzena Investment Mgt", "all":(
        "RELIANCE TRUST INSTITUTIONAL RETIREMENT TRUST SERIES THIRTY SEVEN",
    )},

    # Prudential / Eastspring
    {"name":"prudential", "manager":"Prudential Group", "any":(
        "PRUDENTIAL", "EASTSPRING", "BOC-PRUDENTIAL",
    )},

    # Other major explicit managers
    {"name":"bpjs_ketenagakerjaan", "manager":"BPJS Ketenagakerjaan", "any":(
        "BPJS KETENAGAKERJAAN", "DJS KETENAGAKERJAAN",
    )},
    {"name":"malaysia_epf", "manager":"Employees Provident Fund Board", "countries":("MALAYSIA",), "sid_prefixes":("PFF",), "any":(
        "EMPLOYEES PROVIDENT FUND BOARD", "EMPLOYEES PROVIDENTFD BOARD",
        "EMPLOYEES PROVIDENT FD BOARD", "EMPLOYEES PROVIDENT FUND BRD",
    )},
)


def match_fund_manager_rule(sid, cleaned_name, country):
    sid = "" if pd.isna(sid) else str(sid).upper().strip()
    country = normalize_match_text(country)
    for rule in FUND_MANAGER_RULES:
        countries = {normalize_match_text(x) for x in rule.get("countries", ())}
        if countries and country not in countries:
            continue
        prefixes = rule.get("sid_prefixes", ())
        if prefixes and not sid.startswith(prefixes):
            continue
        if not all(contains_phrase(cleaned_name, x) for x in rule.get("all", ())):
            continue
        alternatives = rule.get("any", ())
        if alternatives and not any(contains_phrase(cleaned_name, x) for x in alternatives):
            continue
        return rule
    return None


class FundManagerClassifier(InvestorClassifier):
    """Manager-first classifier.

    Manager attribution has precedence over beneficial owner, government parent,
    product sponsor, and legal-account fallbacks. The legacy classifier is used
    only when no reviewed manager rule matches.
    """
    def decide(self, sid, investor_name, country=None):
        context = self.make_context(sid, investor_name, country)
        if not context.sid or context.sid == "NAN":
            return ClassificationDecision(UNRESOLVED, "invalid_input", "missing SID")
        if not context.name:
            return ClassificationDecision(UNRESOLVED, "invalid_input", "missing investor name")

        manager_rule = match_fund_manager_rule(context.sid, context.name, context.country)
        if manager_rule:
            evidence = " + ".join(manager_rule.get("all") or manager_rule.get("any") or ())
            return ClassificationDecision(
                manager_rule["manager"],
                f"fund_manager:{manager_rule['name']}",
                evidence,
            )

        # Fallback to the reviewed legacy hierarchy for names where it already
        # identifies a manager or where no manager evidence is available.
        return super().decide(sid, investor_name, country)


DEFAULT_CLASSIFIER = FundManagerClassifier()


def classify_fund_manager(sid, investor_name, country=None):
    return DEFAULT_CLASSIFIER.classify(sid, investor_name, country)

"""## 4. Test-schema adapter and pipeline"""

@dataclass(frozen=True)
class ResolvedSchema:
    sid: str
    investor_name: str
    country: Optional[str]
    true_label: Optional[str]
    date: Optional[str]
    amount: Optional[str]
    @property
    def is_labeled(self): return self.true_label is not None


@dataclass(frozen=True)
class PipelineMetrics:
    rows: int
    resolved_rows: int
    coverage_pct: float
    accuracy_pct: Optional[float]
    wrong_predictions: Optional[int]


def first_existing_column(df, candidates):
    lookup={str(c).strip():c for c in df.columns}
    return next((lookup[x] for x in candidates if x in lookup),None)


def resolve_schema(df):
    values={k:first_existing_column(df,v) for k,v in COLUMN_ALIASES.items()}
    if not values["sid"] or not values["investor_name"]:
        raise ValueError(f"Missing SID or investor-name column. Actual: {list(df.columns)}")
    return ResolvedSchema(**values)


def classify_dataframe(df, classifier=None, include_trace=True):
    classifier = classifier or DEFAULT_CLASSIFIER
    schema=resolve_schema(df); out=df.copy()
    countries=out[schema.country] if schema.country else pd.Series([None]*len(out),index=out.index)
    decisions=[classifier.decide(s,n,c) for s,n,c in zip(out[schema.sid],out[schema.investor_name],countries)]
    out["Fund_Manager_Guess"]=[d.owner for d in decisions]
    out["Regex_Guess"]=out["Fund_Manager_Guess"]  # backward compatibility
    out["Needs_ML"]=out["Fund_Manager_Guess"].eq(UNRESOLVED)
    if include_trace:
        out["Matched_Rule"]=[d.rule for d in decisions]
        out["Match_Evidence"]=[d.evidence for d in decisions]
    return out,schema


def run_pipeline(input_excel=INPUT_XLSX,output_xlsx=OUTPUT_XLSX,include_trace=True):
    df=pd.read_excel(input_excel,engine="openpyxl")
    out,schema=classify_dataframe(df,include_trace=include_trace)
    out.to_excel(output_xlsx,index=False,engine="openpyxl")
    return out,schema


def manager_total(df, manager, amount_column="AMT_BLNC_MEM"):
    mask=df["Fund_Manager_Guess"].astype(str).str.strip().str.upper().eq(manager.upper())
    return pd.to_numeric(df.loc[mask,amount_column],errors="coerce").fillna(0).sum()

"""## 5. Regression tests"""

validate_keyword_configuration()
validate_direct_mappings()

# Manager precedence: Invesco wins even when the underlying beneficiary is BEST.
assert classify_fund_manager(
    "OTF0911L5040012",
    "HSBC-FUND SERVICES A/C 086 HBAP - INVESCO LTD A/C BEST INVESTMENT CORPORATION",
    "CHINA",
) == "Invesco Ltd"

# Mandatory EPF Malaysia compressed spelling.
assert classify_fund_manager(
    "PFF0110S9514926",
    "BNYM RE BNYMLB RE EMPLOYEES PROVIDENTFD BOARD-2039927326",
    "MALAYSIA",
) == "Employees Provident Fund Board"

# Exact BlackRock hidden products.
assert classify_fund_manager("OTF0901L1283259","JPMCB NA RE - MSCI ACWI EX-U.S. IMI INDEX FUND B2","UNITED STATES") == "BlackRock Inc"
assert classify_fund_manager("OTF0505K5766449","JPMCB NA RE-GLOBAL ALPHA TILTS ESG NON-LENDABLE FUND B","UNITED STATES") == "BlackRock Inc"

# Generic index text alone must not imply BlackRock.
assert classify_fund_manager("MFF001","GENERIC MSCI EMERGING MARKETS INDEX FUND","UNITED STATES") != "BlackRock Inc"
print("Manager-first regression tests passed.")



# ============================================================
# BENOWNERS REPORT GROUPING OVERRIDE
# Generated from BenOwners by shareholder.xlsx.
# Private-investor umbrella groups are deliberately excluded.
# Individual SIDs and undisclosed/private accounts retain code-rule priority.
# ============================================================
BENOWNER_UNIQUE_GROUP_MAP = {'GOVERNMENT OF INDONESIA': 'Government of Indonesia', 'INDONESIA INVESTMENT AUTHORITY': 'Indonesia Investment Authority', 'VANGUARD EMERGING MARKETS STOCK INDEX FUND': 'Vanguard Group', 'VANGUARD TOTAL INTERNATIONAL STOCK INDEX FUND': 'Vanguard Group', 'VANGUARD FIDUCIARY TRUST COMPANY INSTITUTIONAL TOTAL INTERNATIONAL STOCK MARKET IT II': 'Vanguard Group', 'VANGUARD INTERNATIONAL VALUE FUND': 'Vanguard Group', 'VANGUARD FTSE ALL WORLD EX US INDEX FUND': 'Vanguard Group', 'VANGUARD INSTITUTIONAL TOTAL INTERNATIONAL STOCK MARKET INDEX TRUST': 'Vanguard Group', 'VANGUARD TOTAL WORLD STOCK INDEX FUND': 'Vanguard Group', 'VANGUARD INTERNATIONAL HIGH DIVIDEND YIELD INDEX FUND': 'Vanguard Group', 'VANGUARD FTSE ALL WORLD UCITS ETF': 'Vanguard Group', 'VANGUARD FTSE EMERGING MARKETS UCITS ETF': 'Vanguard Group', 'VANGUARD FUNDS PLC VANGUARD FTSE ALL WORLD ETF': 'Vanguard Group', 'VANGUARD ESG INTERNATIONAL STOCK ETF': 'Vanguard Group', 'VANGUARD EMERGING MARKETS SHARE INDEX FUND': 'Vanguard Group', 'VANGUARD FTSE GLOBAL ALL CAP INDEX FUND': 'Vanguard Group', 'VANGUARD INVESTMENTS FUNDS ICVC VANGUARD ESG DEVELOPED WORLD ALL CAP EQUITY INDEX FUND UK': 'Vanguard Group', 'VANGUARD ASIA EX JAPAN SHARES INDEX FUND': 'Vanguard Group', 'VANGUARD EMERGING MARKETS EX CHINA ETF': 'Vanguard Group', 'VANGUARD ESG GLOBAL ALL CAP UCITS ETF': 'Vanguard Group', 'VANGUARD ESG EMERGING MARKETS ALL CAP EQUITY INDEX FUND': 'Vanguard Group', 'VANGUARD FTSE GLOBAL SMALL CAP UCITS ETF': 'Vanguard Group', 'VANGUARD FUNDS PLC VANGUARD FTSE ALL WORLD EX US ETF': 'Vanguard Group', 'ISHARES CORE MSCI EMERGING MARKETS ETF': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI EMERGING MARKETS INDEX FUND': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI INDONESIA INVESTABLE MARKET INDEX FUND': 'BlackRock Investment Mgt - Index', 'MSCI EQUITY INDEX FUND B INDONESIA': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI EMERGING MARKETS EX CHINA ETF': 'BlackRock Investment Mgt - Index', 'ISHARES EMERGING MARKETS DIVIDEND ETF': 'BlackRock Investment Mgt - Index', 'ISHARES CORE MSCI TOTAL INTERNATIONAL STOCK ETF': 'BlackRock Investment Mgt - Index', 'ISHARES CORE MSCI AC ASIA EX JAPAN INDEX ETF': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI ACWI ETF': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI ALL COUNTRY ASIA EX JAPAN ETF': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI ACWI EX US ETF': 'BlackRock Investment Mgt - Index', 'MERCER EMERGING MARKETS SHARES FUND': 'BlackRock Investment Mgt - Index', 'ISHARES CORE MSCI EMERGING MARKETS IMI INDEX ETF': 'BlackRock Investment Mgt - Index', 'BLACKROCK CDN MSCI EMERGING MARKETS INDEX FUND': 'BlackRock Investment Mgt - Index', 'EMERGING MARKETS INDEX NON LENDABLE FUND B': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI EMERGING MARKETS ASIA ETF': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI ASIA EX JAPAN CLIMATE ACTION ETF': 'BlackRock Investment Mgt - Index', 'BLACKROCK TOTAL INTERNATIONAL EX US INDEX MASTER PORTFOLIO': 'BlackRock Investment Mgt - Index', 'GLOBAL EX US ALPHA TILTS FUND B': 'BlackRock Investment Mgt - Index', 'ACTIVE STOCK MASTER PORTFOLIO OF MASTER INVESTMENT PORTFOLIO': 'BlackRock Investment Mgt - Index', 'SIX CIRCLES MANAGED EQUITY PORTFOLIO INTERNATIONAL UNCONSTRAINED FUND': 'BlackRock Investment Mgt - Index', 'MSCI ACWI EX U S IMI INDEX FUND B2': 'BlackRock Investment Mgt - Index', 'PUBLIC EMPLOYEES LONG TERM CARE FUND': 'BlackRock Investment Mgt - Index', 'ISHARES GLOBAL MONTHLY DIVIDEND INDEX FUND': 'BlackRock Investment Mgt - Index', 'BLACKROCK CDN ACWI ALPHA TILTS FUND': 'BlackRock Investment Mgt - Index', 'ISHARES BROAD EMERGING MARKETS FUND': 'BlackRock Investment Mgt - Index', 'ISHARES EMERGING MARKET IMI EQUITY INDEX FUND': 'BlackRock Investment Mgt - Index', 'BIFS EMERGING MARKETS EQUITY INDEX FUND': 'BlackRock Investment Mgt - Index', 'BLACKROCK EMERGING MARKETS ALPHA TILTS FUND B': 'BlackRock Investment Mgt - Index', 'BLACKROCK PREMIER FUNDS SYSTEMATIC GLOBAL EQUITY HIGH INCOME FUND': 'BlackRock Investment Mgt - Index', 'CENTRAL PROVIDENT FUND BOARD CPF': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI EMERGING MARKETS QUALITY FACTOR ETF': 'BlackRock Investment Mgt - Index', 'ISHARES LOW CARBON OPTIMIZED MSCI ACWI ETF': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI TOTAL INTERNATIONAL INDEX FUND': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI EMERGING MARKETS ETF': 'BlackRock Investment Mgt - Index', 'ISHARES ESG AWARE MSCI EM ETF': 'BlackRock Investment Mgt - Index', 'ISHARES ESG ADVANCED MSCI EAFE INDEX ETF': 'BlackRock Investment Mgt - Index', 'ISHARES ESG AWARE MSCI EMERGING MARKETS INDEX ETF': 'BlackRock Investment Mgt - Index', 'BLACKROCK EMERGING MARKETS LONG SHORT EQUITY FUND': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI EMERGING MARKETS ASIA INDEX FUND': 'BlackRock Investment Mgt - Index', 'ISHARES EMERGING MARKETS DIVIDEND INDEX FUND': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI ACWI LOW CARBON TARGET ETF': 'BlackRock Investment Mgt - Index', 'ISHARES MSCI ALL COUNTRY ASIA EX JAPAN INDEX FUND': 'BlackRock Investment Mgt - Index', 'CONSOLIDATED EDISON RETIREMENT PLAN': 'BlackRock Investment Mgt - Index', 'BPJS KETENAGAKERJAAN': 'BPJS Ketenagakerjaan', 'DJS KETENAGAKERJAAN PROGRAM JAMINAN KEMATIAN': 'BPJS Ketenagakerjaan', 'INVESCO DEVELOPING MARKETS FUND': 'Invesco', 'INVESCO FUNDS': 'Invesco', 'INVESCO ASIAN EQUITY INCOME FUND UK': 'Invesco', 'INVESCO GLOBAL EMERGING MARKETS FUND': 'Invesco', 'INVESCO ASIA TRUST PLC': 'Invesco', 'INVESCO EMERGING MARKETS EX CHINA FUND UK': 'Invesco', 'QUILTER INVESTORS ASIA PACIFIC EX JAPAN LARGE CAP EQUITY FUND': 'Invesco', 'PUTM ACS SUSTAINABLE INDEX EMERGING MARKET EQUITY FUND': 'Invesco', 'INVESCO PACIFIC FUND': 'Invesco', 'INVESCO GLOBAL INCOME FUND': 'Invesco', 'TA ASIA HIGH CONVICTION MANDATE': 'Invesco', 'EMPLOYEES PROVIDENT FUND EPF': 'Employees Provident Fund Board', 'ST JAMES S PLACE GLOBAL VALUE UNIT TRUST': 'Pzena Investment Mgt', 'PZENA EMERGING MARKETS VALUE FUND': 'Pzena Investment Mgt', 'BRIDGE BUILDER INTERNATIONAL EQUITY FUND': 'Pzena Investment Mgt', 'STATE OF NEW JERSEY COMMON PENSION FUND D': 'Pzena Investment Mgt', '1832 AM EMERGING MARKETS EQUITY POOL': 'Pzena Investment Mgt', 'VANGUARD GLOBAL EQUITY FUND': 'Pzena Investment Mgt', 'PZENA EMERGING MARKETS FOCUSED VALUE FUND': 'Pzena Investment Mgt', 'MDPIM INTERNATIONAL EQUITY POOL': 'Pzena Investment Mgt', 'PZENA GLOBAL FOCUSED VALUE FUND': 'Pzena Investment Mgt', 'BROWN ADVISORY EMERGING MARKETS SELECT FUND': 'Pzena Investment Mgt', 'AMERICAN AIRLINES INCORPORATED MASTER FIXED BENEFIT PENSION TRUST': 'Pzena Investment Mgt', 'PZENA VALUE FUNDS PLC PZENA EMERGING MARKETS SELECT VALUE FUND': 'Pzena Investment Mgt', 'PZENA INTERNATIONAL VALUE ALL COUNTRY EX U S FUND': 'Pzena Investment Mgt', 'PZENA GLOBAL VALUE FUND': 'Pzena Investment Mgt', 'MICROSOFT CORPORATION SAVINGS PLUS 401K PLAN': 'Pzena Investment Mgt', 'BARCLAYS MULTI MANAGER FUNDS GLOBALACCESS EMERGING MARKET EQUITY FUND': 'Pzena Investment Mgt', 'PZENA GLOBAL VALUE FUND 2 DST': 'Pzena Investment Mgt', 'PZENA INTERNATIONAL EX US FOCUSED VALUE FUND': 'Pzena Investment Mgt', 'BURROUGHS WELLCOME FUND': 'Pzena Investment Mgt', 'PZENA INVESTMENT MANAGEMENT LLC': 'Pzena Investment Mgt', 'PZENA GLOBAL VALUE FUND DST': 'Pzena Investment Mgt', 'PZENA INTERNATIONAL VALUE FUND': 'Pzena Investment Mgt', 'BANK RAKYAT INDONESIA': 'Bank Rakyat Indonesia', 'ST JAMES S PLACE EMERGING MARKETS EQUITY UNIT TRUST': 'ARGA Investment Mgt', 'ARGA EMERGING MARKETS VALUE FUND': 'ARGA Investment Mgt', 'ARGA FUNDS TRUST ARGA EMERGING MARKETS EX CHINA FUND': 'ARGA Investment Mgt', 'ARGA EMERGING MARKETS EQUITY FUND': 'ARGA Investment Mgt', 'STATE OF CONNECTICUT RETIREMENT PLANS AND TRUST FUNDS': 'ARGA Investment Mgt', 'REGIME DE RETRAITE DE L UNIVERSITE DE MONTREAL': 'ARGA Investment Mgt', 'ARGA INTERNATIONAL VALUE FUND': 'ARGA Investment Mgt', 'ARGA INTERNATIONAL FUND': 'ARGA Investment Mgt', 'ARGA EMERGING MARKETS SELECT FUND LP': 'ARGA Investment Mgt', 'ARGA ASIA EX JAPAN FUND': 'ARGA Investment Mgt', 'ARGA GLOBAL SECTOR NEUTRAL VALUE EQUITY FUND': 'ARGA Investment Mgt', 'ARGA INTERNATIONAL SECTOR NEUTRAL VALUE EQUITY FUND': 'ARGA Investment Mgt', 'PACE INTERNATIONAL EMERGING MARKETS EQUITY': 'ARGA Investment Mgt', 'PRUDENTIAL ASSURANCE COMPANY': 'Eastspring Investments', 'ESPRING INV ASIAN EQUITY FUND': 'Eastspring Investments', 'EASTSPRING INVESTMENTS SINGAPORE ASEAN EQUITY FUND': 'Eastspring Investments', 'PRULINK SINGAPORE ASIAN MANAGED FUND': 'Eastspring Investments', 'CBSG GEM BETA SMART BETA EQ PFASGBSB': 'Eastspring Investments', 'PACS GEM SMART BETA EQUITY PORTFOLIO ASGSBF': 'Eastspring Investments', 'EASTSPRING INVESTMENTS ASIA PACIFIC EQUITY MY FUND': 'Eastspring Investments', 'EASTSPRING INVESTMENTS ASIA PACIFIC EQUITY FUND': 'Eastspring Investments', 'BRANDES EMERGING MARKETS VALUE FUND': 'Brandes Investment Partners', 'BRANDES INSTITUTIONAL EQUITY TRUST': 'Brandes Investment Partners', 'BRANDES GLOBAL EQUITY FUND': 'Brandes Investment Partners', 'BRANDES EMERGING MARKETS EQUITY FUND': 'Brandes Investment Partners', 'BRANDES INVESTMENT FUNDS': 'Brandes Investment Partners', 'BRANDES INSTITUTIONAL INVESTMENT EMERGING MARKETS FUND': 'Brandes Investment Partners', 'BRANDES GLOBAL OPPORTUNITIES FUND': 'Brandes Investment Partners', 'MERCER GLOBAL EQUITY ALL COUNTRY WORLD INDEX FUND': 'Brandes Investment Partners', 'MERCER GLOBAL EQUITY FUND': 'Brandes Investment Partners', 'BRANDES INVESTMENT PARTNERS': 'Brandes Investment Partners', 'ALASKA ELECTRICAL PENSION FUND': 'Brandes Investment Partners', 'BRANDES GLOBAL VALUE FUND': 'Brandes Investment Partners', 'ISHARES PLC': 'BlackRock Investment Mgt - Index', 'ISHARES II PUBLIC LIMITED COMPANY': 'BlackRock Investment Mgt - Index', 'BLACKROCK GLOBAL FUNDS': 'BlackRock Investment Mgt - Index', 'BLACKROCK INDEX SELECTION FUND ISHARES EUROPE INDEX FUND IE': 'BlackRock Investment Mgt - Index', 'ISHARES III PLC MSCI EM UCITS ETF': 'BlackRock Investment Mgt - Index', 'ISHARES VII PLC': 'BlackRock Investment Mgt - Index', 'ISHARES IV PLC MSCI EM ESG ENHANCED UCITS ETF': 'BlackRock Investment Mgt - Index', 'ISHARES IV PLC MSCI EMERGING EX CHINA UCITS ETF': 'BlackRock Investment Mgt - Index', 'ISHARES IV PLC MSCI EM IMI ESG SCREENED UCITS ETF': 'BlackRock Investment Mgt - Index', 'BLACKROCK EMERGING MARKETS ESG INSIGHTS EQUITY FUND': 'BlackRock Investment Mgt - Index', 'PUBLIC INVESTMENT CORPORATION GOVERNMENT EMPLOYEES PENSION FUND': 'BlackRock Investment Mgt - Index', 'ISHARES V PLC MSCI ACWI UCITS ETF': 'BlackRock Investment Mgt - Index', 'BLACKROCK FUNDS I ICAV': 'BlackRock Investment Mgt - Index', 'BLACKROCK LIFE': 'BlackRock Investment Mgt - Index', 'BLACKROCK UCITS FUNDS': 'BlackRock Investment Mgt - Index', 'BLACKROCK BALANCED FUND': 'BlackRock Investment Mgt - Index', 'PENSIONSKASSE DES BUNDES PUBLICA': 'BlackRock Investment Mgt - Index', 'ISHARES IV PUBLIC LIMITED COMPANY': 'BlackRock Investment Mgt - Index', 'BLACKROCK GLOBAL INDEX FUNDS': 'BlackRock Investment Mgt - Index', 'ISHARES ESG ADVANCED MSCI EM ETF': 'BlackRock Investment Mgt - Index', 'GLOBAL ATLANTIC BLACKROCK DISCIPLINED INTERNATIONAL CORE PORTFOLIO': 'BlackRock Investment Mgt - Index', 'BLACKROCK GLOBAL INDEX FUNDS ISHARES WORLD EQUITY INDEX FUND LU': 'BlackRock Investment Mgt - Index', 'AEGON WORLD EQUITY BETA POOL P': 'BlackRock Investment Mgt - Index', 'ISHARES III PUBLIC LIMITED COMPANY': 'BlackRock Investment Mgt - Index', 'BRITISH AIRWAYS PENSIONS TRUSTEES': 'BlackRock Investment Mgt - Index', 'STATE STREET BANK AND TRUST CO INVESTMENT FUNDS FOR TAX EXEMPT RETIREMENT PLANS': 'State Street Investment Mgt', 'SPDR S AND P EMERGING MARKETS ETF': 'State Street Investment Mgt', 'STATE STREET AUT EMERGING MARKET TILTED INDEX EQUITY FUND': 'State Street Investment Mgt', 'SSGA MSCI INDONESIA INDEX NON LENDING COMMON TRUST FUND': 'State Street Investment Mgt', 'REGENTS OF THE UNIVERSITY OF CALIFORNIA': 'State Street Investment Mgt', 'STICHTING PENSIOENFONDS RAIL AND OPENBAAR VERVOER': 'State Street Investment Mgt', 'STATE STREET GLOBAL ALL CAP EQUITY EX U S INDEX PORTFOLIO': 'State Street Investment Mgt', 'IBM SAVINGS PLAN': 'State Street Investment Mgt', 'DELA NATURA EN LEVENSVERZEKERINGEN NV': 'State Street Investment Mgt', 'SSGA LUX SICAV GLOBAL EM ESG SCREENED INDEX EQUITY FUND': 'State Street Investment Mgt', 'SPDR MSCI ACWI EX US ETF': 'State Street Investment Mgt', 'STATE STREET GLOBAL ADVISORS LUXEMBOURG SICAV': 'State Street Investment Mgt', 'BRIGHTHOUSE FUNDS TRUST I SSGA EMERGING MARKETS ENHANCED INDEX PORTFOLIO': 'State Street Investment Mgt', 'SPDR S AND P EMERGING ASIA PACIFIC': 'State Street Investment Mgt', 'EMERGING MARKETS INDEX PLUS COMMON TRUST FUND': 'State Street Investment Mgt', 'STATE STREET AUT EMERGING MARKET INDEX EQUITY FUND': 'State Street Investment Mgt', 'STATE STREET EMERGING MARKETS EQUITY INDEX FUND': 'State Street Investment Mgt', 'MBB PUBLIC MARKETS I LLC': 'State Street Investment Mgt', 'JOHN HANCOCK VARIABLE INSURANCE TRUST INTERNATIONAL EQUITY INDEX TRUST B': 'State Street Investment Mgt', 'SPDR PORTFOLIO MSCI GLOBAL STOCK MARKET ETF': 'State Street Investment Mgt', 'LVIP SSGA EMERGING MARKET EQUITY INDEX FUND': 'State Street Investment Mgt', 'INTERNATIONAL MONETARY FUND': 'State Street Investment Mgt', 'SPDR MSCI EMERGING MARKETS FOSSIL FUEL RESERVES FREE ETF': 'State Street Investment Mgt', 'MSCI EAFE PROVISIONAL SCREENED INDEX NON LENDING COMMON TST FD': 'State Street Investment Mgt', 'RBC IST SUN LIFE STATE STREET EMERGING MARKETS EQUITY INDEX FUND': 'State Street Investment Mgt', 'STATE STREET TOTAL RETURN V I S FUND': 'State Street Investment Mgt', 'SSGA MSCI ACWI EX USA INDEX NON LENDING DAILY TRUST': 'State Street Investment Mgt', 'INTERNATIONAL EXPATRIATE BENEFIT MASTER TRUST': 'State Street Investment Mgt', 'SPDR MSCI EMERGING MARKETS QUALITY MIX ETF': 'State Street Investment Mgt', 'SPDR S AND P EMERGING MARKETS FUND': 'State Street Investment Mgt', 'SPDR S AND P EMERGING MARKETS EX CHINA ETF': 'State Street Investment Mgt', 'WELLS FARGO MANAGED EMERGING MARKETS EQUITY FUND': 'State Street Investment Mgt', 'CAISSE DE RETRAITE DES EMPLOYES D HYDRO QUEBEC': 'State Street Investment Mgt', 'UN RELIEF AND WORKS AGENCY FOR PALESTINE': 'State Street Investment Mgt', 'SPDR MSCI ACWI IMI ETF': 'State Street Investment Mgt', 'KNIGHTS OF COLUMBUS MASTER TRUST': 'State Street Investment Mgt', 'AMERICAN HEART ASSOCIATION AHAI': 'State Street Investment Mgt', 'BAILLIE GIFFORD EMERGING MARKETS FUND': 'Baillie Gifford & Co', 'BAILLIE GIFFORD OVERSEAS GROWTH FUND PACIFIC FUND': 'Baillie Gifford & Co', 'BAILLIE GIFFORD EMERGING MARKETS LARGE CAP': 'Baillie Gifford & Co', 'PACIFIC HORIZON INVESTMENT TRUST PLC': 'Baillie Gifford & Co', 'BAILLIE GIFFORD EMERGING MARKETS GROWTH FUND': 'Baillie Gifford & Co', 'BAILLIE GIFFORD WORLDWIDE EMERGING MARKETS LEADING CO FUND': 'Baillie Gifford & Co', 'BAILLIE GIFFORD GROUP TRUST': 'Baillie Gifford & Co', 'BAILLIE GIFFORD WORLDWIDE FUNDS PLC': 'Baillie Gifford & Co', 'BAILLIE GIFFORD COLLECTIVE INVESTMENT TRUST': 'Baillie Gifford & Co', 'BAILLIE GIFFORD EMERGING MARKETS ETF': 'Baillie Gifford & Co', 'BAILLIE GIFFORD EMERGING MARKETS EX CHINA FUND': 'Baillie Gifford & Co', 'BAILLIE GIFFORD AND CO': 'Baillie Gifford & Co', 'LF ACCESS GLOBAL EQUITY CORE FUND': 'Baillie Gifford & Co', 'BAILLIE GIFFORD POSITIVE CHANGE EQUITIES FUND': 'Baillie Gifford & Co', 'BAILLIE GIFFORD WORLDWIDE POSITIVE CHANGE FUND': 'Baillie Gifford & Co', 'DESJARDINS SOCIETERRA POSITIVE CHANGE FUND': 'Baillie Gifford & Co', 'BAILLIE GIFFORD IMPACT INVESTMENT MOTHER FUND': 'Baillie Gifford & Co', 'VANGUARD ACTIVE POSITIVE IMPACT FUND': 'Baillie Gifford & Co', 'BAILLIE GIFFORD CAUTIOUS MANAGED FUND': 'Baillie Gifford & Co', 'BAILLIE GIFFORD MANAGED FUND OEIC': 'Baillie Gifford & Co', 'ROBECO CAPITAL GROWTH FUNDS ASIA PACIFIC EQUITIES': 'Robeco Asset Mgt', 'MERCER EMERGING MARKETS FUND': 'Robeco Asset Mgt', 'STICHTING PENSIOENFONDS ZORG EN WELZIJN PFZW': 'PGGM Investments', 'M AND G LUX INVESTMENT FUNDS 1': 'M&G Investments', 'M AND G FUNDS 1 M AND G ASIAN FUND': 'M&G Investments', 'M AND G FUNDS 1 ASIA PACIFIC EX JAPAN EQUITY': 'M&G Investments', 'M AND G GLOBAL EMERGING MARKETS FUND': 'M&G Investments', 'M AND G GLOBAL POSITIVE IMPACT EQUITY': 'M&G Investments', 'PRUFUND CAUTIOUS FUND': 'M&G Investments', 'M AND G INVESTMENT FUNDS 10 M AND G POSITIVE IMPACT FUND': 'M&G Investments', 'M AND G LUX INVESTMENT FUNDS 1 INCOME ALLOCATION FUND': 'M&G Investments', 'M AND G INVESTMENT FUNDS 11 EPISODE INCOME FUND': 'M&G Investments', 'FAM SERIES UCITS ICAV DIVERSITY AND ICLUSION FAM FUND': 'M&G Investments', 'M AND G CLIMATE AWARE MULTI ASSET FUND': 'M&G Investments', 'AILIS': 'M&G Investments', 'PRUDENTIAL LIFE ASSURANCE': 'Eastspring Investments', 'PT PRUDENTIAL LIFE ASSURANCE': 'Eastspring Investments', 'PT PRUDENTIAL LIFE ASSURANCE REP III': 'Eastspring Investments', 'REKSA DANA SAHAM EASTSPRING INVESTMENTS ALPHA NAVIGATOR': 'Eastspring Investments', 'REKSA DANA EASTSPRING INVESTMENTS VALUE DISCOVERY': 'Eastspring Investments', 'EASTSPRING IDX ESG LEADERS PLUS': 'Eastspring Investments', 'REKSA DANA INDEKS EASTSPRING ESGQ45 IDXKEHATI': 'Eastspring Investments', 'L AND G ASSURANCE PENSION MANAGEMENT': 'Legal & General Investment Mgt', 'LEGAL AND GENERAL GLOBAL EMERGING MARKETS INDEX FUND': 'Legal & General Investment Mgt', 'LEGAL AND GENERAL ICAV': 'Legal & General Investment Mgt', 'L AND G COLLECTIVE INVESTMENT TRUST': 'Legal & General Investment Mgt', 'LEGAL AND GENERAL FUTURE WORLD ESG DEVELOPED INDEX FUND': 'Legal & General Investment Mgt', 'LEGAL AND GENERAL': 'Legal & General Investment Mgt', 'LEGAL AND GENERAL UCITS ETF': 'Legal & General Investment Mgt', 'POOL REINSURANCE COMPANY LIMITED': 'Legal & General Investment Mgt', 'QUEEN S UNIVERSITY OF BELFAST RETIREMENT PLAN': 'Legal & General Investment Mgt', 'LEGAL AND GENERAL FUTURE WORLD CLIMATE CHANGE EQUITY FACTORS INDEX FUND': 'Legal & General Investment Mgt', 'LEGAL AND GENERAL ASIA PACIFIC EQUITY INCOME FUND': 'Legal & General Investment Mgt', 'LEGAL AND GENERAL ASSURANCE SOCIETY': 'Legal & General Investment Mgt', 'PT TASPEN': 'Taspen', 'PT TASPEN ASURANSI': 'Taspen', 'DANA PENSIUN KARYAWAN TASPEN': 'Taspen', 'PT ASURANSI JIWA TASPEN': 'Taspen', 'OKLAHOMA PUBLIC EMPLOYEES RETIREMENT SYSTEM': 'Mondrian Investment Partners', 'MONDRIAN EMERGING MARKETS EQUITY FUND': 'Mondrian Investment Partners', 'MONDRIAN ALL COUNTRIES WORLD EX US EQUITY FUND LP': 'Mondrian Investment Partners', 'MONDRIAN GLOBAL ALL COUNTRIES WORLD EQUITY FUND': 'Mondrian Investment Partners', 'MONDRIAN GLOBAL EQUITY VALUE FUND': 'Mondrian Investment Partners', 'MONDRIAN FOCUSED ALL COUNTRIES WORLD EX US EQUITY CIT': 'Mondrian Investment Partners', 'MONDRIAN EMERGING MARKETS VALUE EQUITY FUND': 'Mondrian Investment Partners', 'MONDRIAN EMERGING MARKETS EQUITY ESG FUND': 'Mondrian Investment Partners', 'MONDRIAN EMERGING MARKETS EX CHINA EQUITY FUND LP': 'Mondrian Investment Partners', 'WEST MIDLANDS METROPOLITAN AUTHORITIES PENSION FUND': 'Mondrian Investment Partners', 'HSBC MSCI INDONESIA UCITS ETF': 'HSBC Global Asset Mgt', 'HSBC GIF GLOBAL EMERGING MARKETS EQUITY': 'HSBC Global Asset Mgt', 'HSBC LIFE INTERNATIONAL WEALTH GOAL INSURANCE PLAN': 'HSBC Global Asset Mgt', 'HSBC MULTI FACTOR WORLDWIDE EQUITY UCITS ETF': 'HSBC Global Asset Mgt', 'HSBC ETFS PLC ASIA PACIFIC EX JAPAN SUSTAINABLE EQUITY UCITS ETF': 'HSBC Global Asset Mgt', 'HSBC POOLED VC ASIA PACIFIC EX JAPAN EQUITY FUND': 'HSBC Global Asset Mgt', 'HSBC INDEX TRACKER INVESTMENT FUND FTSE ALL WORLD INDEX FUND': 'HSBC Global Asset Mgt', 'HSBC INDEX TRACKER INVESTMENT FUNDS MSCI EMERGING MARKETS EQUITY INDEX FUND': 'HSBC Global Asset Mgt', 'HSBC ETFS PLC EMERGING MARKET SCREENED EQUITY UCITS ETF': 'HSBC Global Asset Mgt', 'PLUS EMERGING MARKETS EQUITY INCOME QUANT ACTIVE UCITS ETF': 'HSBC Global Asset Mgt', 'HSBC ETFS PLC': 'HSBC Global Asset Mgt', 'HSBC INDEX TRACKER INVESTMENT FUNDS MSCI EMERGING MARKETS EX CHINA EQUITY INDEX FUND': 'HSBC Global Asset Mgt', 'HSBC GLOBAL EMERGING MARKETS EQUITY FUND': 'HSBC Global Asset Mgt', 'HSBC GLOBAL INVESTMENT FUNDS GLOBAL EMERGING MARKETS MULTI ASSET INCOME': 'HSBC Global Asset Mgt', 'T ROWE PRICE EMERGING MARKETS DISCOVERY STOCK TRUST': 'T Rowe Price International', 'T ROWE PRICE INTERNATIONAL FUNDS NEW ASIA FUND': 'T Rowe Price International', 'T ROWE PRICE ASIAN EX JAPAN EQUITY': 'T Rowe Price International', 'FIDELITY SALEM STREET TRUST SPARTAN GLOBAL EX US INDEX FUND': 'Geode Capital Mgt', 'SPARTAN EMERGING MARKETS INDEX POOL': 'Geode Capital Mgt', 'FIDELITY EMERGING MARKETS INDEX FUND': 'Geode Capital Mgt', 'FIDELITY TOTAL INTERNATIONAL INDEX FUND': 'Geode Capital Mgt', 'FIDELITY SALEM STREET TRUST FIDELITY SAI EMERGING MARKETS INDEX FUND': 'Geode Capital Mgt', 'FIDELITY ZERO INTERNATIONAL INDEX FUND': 'Geode Capital Mgt', 'FIDELITY INVESTMENT FUNDS FIDELITY INDEX EMERGING MARKETS FUND': 'Geode Capital Mgt', 'FIDELITY FLEX INTERNATIONAL FOCUSED INDEX FUND': 'Geode Capital Mgt', 'PACIFIC SELECT FUND PD EMERGING MARKETS PORTFOLIO': 'Geode Capital Mgt', 'FIDELITY UCITS II ICAV MSCI EMERGING MARKETS INDEX FUND': 'Geode Capital Mgt', 'FIDELITY GLOBAL EX U S EQUITY INDEX INSTITUTIONAL TRUST': 'Geode Capital Mgt', 'BMO PRIVATE INTERNATIONAL EQUITY PORTFOLIO': 'Pyrford International', 'HYDRO ONE PENSION PLAN': 'Pyrford International', 'INTERNATIONAL STOCK FUND': 'Pyrford International', 'PYRFORD INTERNATIONAL TRUST': 'Pyrford International', 'COLUMBIA THREADNEEDLE IRELAND III PLC PYRFORD GLOBAL TOTAL RETURN GBP FUND': 'Pyrford International', 'BMO INTERNATIONAL VALUE FUND': 'Pyrford International', 'BMO AM GLOBAL EQUITY FUND': 'Pyrford International', 'BMO PYRFORD INTERNATIONAL STOCK FUND': 'Pyrford International', 'NS PENSION PUBLIC EQUITY FUND': 'Pyrford International', 'COLUMBIA THREADNEEDLE IRELAND PLC PYRFORD INTERNATIONAL EAFE EQUITY FUND': 'Pyrford International', 'CT PYRFORD GLOBAL ABSOLUTE RETURN FUND': 'Pyrford International', 'COLUMBIA THREADNEEDLE IRELAND PLC': 'Pyrford International', 'WOODBRIDGE INVESTMENTS CORPORATION': 'Pyrford International', 'STAGECOACH GROUP PENSION SCHEME': 'Pyrford International', 'NOVA SCOTIA TEACHERS PENSION FUND': 'Pyrford International', 'AQR UCITS FUNDS EMERGING EQUITIES': 'AQR Capital Mgt', 'AQR EMERGING EQUITIES FUND': 'AQR Capital Mgt', 'AQR LEELANAU FUND LP': 'AQR Capital Mgt', 'GUIDESTONE EMERGING MARKETS EQUITY FUND': 'AQR Capital Mgt', 'AQR COLLECTIVE INVESTMENT TRUST': 'AQR Capital Mgt', 'AQR EMERGING EQUITIES FUND II': 'AQR Capital Mgt', 'AQR GLOBAL RISK PREMIUM MASTER ACCOUNT II': 'AQR Capital Mgt', 'AQR GLOBAL RISK PREMIUM MASTER ACCOUNT': 'AQR Capital Mgt', 'AQR GRP EL MASTER ACCOUNT': 'AQR Capital Mgt', 'AQR GRP EL 20 MASTER ACCOUNT': 'AQR Capital Mgt', 'VERDIPAPIRFONDET STOREBRAND GLOBAL SOLUTIONS': 'Storebrand Investments', 'SPP GLOBAL SOLUTIONS': 'Storebrand Investments', 'SPP EMERGING MARKETS PLUS': 'Storebrand Investments', 'SPP EMERGING MARKETS SRI': 'Storebrand Investments', 'STOREBRAND SICAV': 'Storebrand Investments', 'VERDIPAPIRFONDET STOREBRAND INDEKS NYE MARKEDER': 'Storebrand Investments', 'SPP AKTIEFOND GLOBAL': 'Storebrand Investments', 'VERDIPAPIRFONDET STOREBRAND EQUAL OPPORTUNITIES': 'Storebrand Investments', 'VERDIPAPIRFONDET STOREBRAND INDEKS ALLE MARKEDER': 'Storebrand Investments', 'PENSION RESERVES INVESTMENT TRUST FUND': 'Acadian Asset Mgt', 'NORTH CAROLINA SUPPLEMENTAL RETIREMENT PLANS GROUP TRUST': 'Acadian Asset Mgt', 'COLLEGES OF APPLIED ARTS AND TECHNOLOGY PENSION PLAN': 'Acadian Asset Mgt', 'COMMONWEALTH GLOBAL SHARES FUND 30': 'Acadian Asset Mgt', '3M EMPLOYEE RETIREMENT INCOME PLAN ERIP': 'Acadian Asset Mgt', 'COLONIAL FIRST STATE INVESTMENT FUND 187': 'Acadian Asset Mgt', 'THE BOARD OF REGENTS OF THE UNIVERSITY OF TEXAS SYSTEM': 'Acadian Asset Mgt', 'NORTHERN TRUST GLOBAL INVESTMENTS COLLECTIVE FUNDS': 'Northern Trust Global Investments', 'FLEXSHARES INTERNATIONAL QUALITY DIVIDEND INDEX FUND': 'Northern Trust Global Investments', 'NORTHERN TRUST QUANTITATIVE FUND': 'Northern Trust Global Investments', 'IMCO EMERGING MARKETS PUBLIC EQUITY LP': 'Northern Trust Global Investments', 'NATIONAL EMPLOYMENT SAVINGS TRUST': 'Northern Trust Global Investments', 'NORTHERN TRUST EMERGING MARKETS EQUITY INDEX FUND': 'Northern Trust Global Investments', 'FIREFIGHTERS PENSION INVESTMENT FUND': 'Northern Trust Global Investments', 'NTCC COLLECTIVE FUNDS FOR EMPLOYEE BENEFIT TRUSTS': 'Northern Trust Global Investments', 'NORTHERN TRUST COMMON EMERGING MARKETS INDEX FUND NON LENDING': 'Northern Trust Global Investments', 'TEXAS MUNICIPAL RETIREMENT SYSTEM': 'Northern Trust Global Investments', 'FLEXSHARES INTERNATIONAL QUALITY DIVIDEND DYNAMIC INDEX FUND': 'Northern Trust Global Investments', 'NORTHERN TRUST COMMON ALL COUNTRY WORLD INDEX ACWI EX US FUND LENDING': 'Northern Trust Global Investments', 'NORTHERN TRUST COMMON ALL COUNTRY WORLD EX US INVESTABLE MARKET INDEX FUND NON LENDING': 'Northern Trust Global Investments', 'FLEXSHARES MORNINGSTAR EMERGING MARKETS FACTOR TILT INDEX FUND': 'Northern Trust Global Investments', 'INTERNATIONAL BANK FOR RECONSTRUCTION AND DEVELOPMENT': 'Northern Trust Global Investments', 'CLINTON NUCLEAR POWER PLANT QUALIFIED FUND': 'Northern Trust Global Investments', 'STICHTING PENSIOENFONDS ABP': 'APG Asset Mgt Asia', 'STICHTING DEPOSITARY APG EMERGING MULTI CLIENT POOL': 'APG Asset Mgt Asia', 'STICHTING DEPOSITARY APG EMERGING MARKETS EQUITY POOL': 'APG Asset Mgt Asia', 'MANULIFE DANA EKUITAS USD': 'Manulife Investment Mgt Indonesia', 'REKSA DANA MANULIFE DANA EKUITAS UTAMA': 'Manulife Investment Mgt Indonesia', 'REKSA DANA MANULIFE DANA SAHAM': 'Manulife Investment Mgt Indonesia', 'REKSA DANA MANULIFE SAHAM UTAMA': 'Manulife Investment Mgt Indonesia', 'REKSADANA MANULIFE SAHAM ANDALAN': 'Manulife Investment Mgt Indonesia', 'ASURANSI JIWA MANULIFE INDONESIA IDR PAR EQUITY': 'Manulife Investment Mgt Indonesia', 'ASURANSI JIWA MANULIFE INDONESIA IDR NON PAR EQUITY': 'Manulife Investment Mgt Indonesia', 'REKSA DANA MANULIFE GREATER INDONESIA FUND': 'Manulife Investment Mgt Indonesia', 'MANULIFE GLOBAL FUND': 'Manulife Investment Mgt Indonesia', 'MANULIFE SCHRODER DANA EKUITAS PREMIER FUND': 'Manulife Investment Mgt Indonesia', 'GSP DANA SAHAM BERIMBANG': 'Manulife Investment Mgt Indonesia', 'PT ASURANSI JIWA MANULIFE INDONESIA': 'Manulife Investment Mgt Indonesia', 'REKSA DANA MANULIFE DANA CAMPURAN UTAMA': 'Manulife Investment Mgt Indonesia', 'MANULIFE DANA PRIMA DINAMIS': 'Manulife Investment Mgt Indonesia', 'TUGU REASURANSI INDONESIA': 'Manulife Investment Mgt Indonesia', 'DANA PENSIUN LEMBAGA KEUANGAN MANULIFE INDONESIA': 'Manulife Investment Mgt Indonesia', 'DAPEN PLN MAMI 2': 'Manulife Investment Mgt Indonesia', 'YAKES TELKOM MAMI': 'Manulife Investment Mgt Indonesia', 'MANULIFE DANA EKUITAS INDONESIA CHINA': 'Manulife Investment Mgt Indonesia', 'MANULIFE DANA EKUITAS UNGGULAN': 'Manulife Investment Mgt Indonesia', 'REKSA DANA MANULIFE INSTITUTIONAL EQUITY FUND': 'Manulife Investment Mgt Indonesia', 'MANULIFE DANA BERIMBANG SYARIAH': 'Manulife Investment Mgt Indonesia', 'DPLK MANULIFE INDONESIA': 'Manulife Investment Mgt Indonesia', 'MANULIFE ASIA BEST SELECT EQUITY FUND': 'Manulife Investment Mgt Indonesia', 'ASURANSI JIWA MANULIFE INDONESIA IDR SURPLUS EQUITY': 'Manulife Investment Mgt Indonesia', 'REKSA DANA MANULIFE DANA CAMPURAN II': 'Manulife Investment Mgt Indonesia', 'MANULIFE DANA EKUITAS INDO DEVELOPED MARKET': 'Manulife Investment Mgt Indonesia', 'MANULIFE DANA PRIMA DINAMIS MODERAT': 'Manulife Investment Mgt Indonesia', 'REKSA DANA MANULIFE DANA TUMBUH BERIMBANG': 'Manulife Investment Mgt Indonesia', 'DPLK MANULIFE CITIBANK BALANCE DKPK': 'Manulife Investment Mgt Indonesia', 'MANULIFE DANA SMALL MID CAPITAL USD': 'Manulife Investment Mgt Indonesia', 'REKSA DANA MANULIFE SAHAM SMC PLUS': 'Manulife Investment Mgt Indonesia', 'INDORAMA EQUITY FUND': 'Manulife Investment Mgt Indonesia', 'MANULIFE DANA EKUITAS SMALL MID CAPITAL': 'Manulife Investment Mgt Indonesia', 'JPMORGAN EMERGING MARKETS INVESTMENT TRUST PLC': 'JPMorgan Asset Mgt', 'JPMORGAN EMERGING MARKETS INCOME FUND': 'JPMorgan Asset Mgt', 'INDUSTRIENS PENSIONFORSIKRING': 'JPMorgan Asset Mgt', 'JPMORGAN INVESTMENT FUNDS GLOBAL BALANCED FUND': 'JPMorgan Asset Mgt', 'JPMORGAN GLOBAL EMERGING MARKETS INCOME TRUST PLC': 'JPMorgan Asset Mgt', 'INVESTORS GROUP IPROFILE EMERGING MARKETS EQUITY POOL': 'JPMorgan Asset Mgt', 'JPMORGAN MULTI ASSET INCOME FUND': 'JPMorgan Asset Mgt', 'IG JPMORGAN EMERGING MARKETS EQUITY FUND GROWTH': 'JPMorgan Asset Mgt', 'SUNAMERICA SERIES TRUST EMERGING MARKETS PORTFOLIO': 'JPMorgan Asset Mgt', 'COMMINGLED PENSION TRUST FUND EMERGING MARKETS EQUITY FOCUSED': 'JPMorgan Asset Mgt', 'IG JPMORGAN EMERGING MARKETS FUND': 'JPMorgan Asset Mgt', 'JPMORGAN ETFS IRELAND ICAV': 'JPMorgan Asset Mgt', 'MET INVESTORS SERIES TRUST JPMORGAN GLOBAL ACTIVE ALLOCATION PORTFOLIO': 'JPMorgan Asset Mgt', 'EMERGING MARKETS EQUITY CANADA FUND': 'JPMorgan Asset Mgt', 'JPMORGAN EMERGING MARKETS EQUITY FUND': 'JPMorgan Asset Mgt', 'MTBJ RE JPM EEFMF': 'JPMorgan Asset Mgt', 'NORGES BANK INVESTMENT MANAGEMENT NBIM': 'Norges Bank Investment Mgt', 'REKSA DANA INDEKS BNP PARIBAS SRI KEHATI': 'BNP Paribas Asset Mgt', 'REKSA DANA BNP PARIBAS INDONESIA ESG EQUITY': 'BNP Paribas Asset Mgt', 'RD BNP PARIBAS MAXI SAHAM': 'BNP Paribas Asset Mgt', 'REKSA DANA INDEKS IDX30': 'BNP Paribas Asset Mgt', 'REKSA DANA BNP PARIBAS EKUITAS': 'BNP Paribas Asset Mgt', 'REKSA DANA INDEKS BNP PARIBAS IDX GROWTH30': 'BNP Paribas Asset Mgt', 'DP BNI': 'BNP Paribas Asset Mgt', 'DP FREEPORT INDONESIA': 'BNP Paribas Asset Mgt', 'REKSA DANA INDEKS BNP PARIBAS IDX30 FILANTROPI': 'BNP Paribas Asset Mgt', 'REKSA DANA FORTIS PESONA': 'BNP Paribas Asset Mgt', 'REKSA DANA FORTIS SOLARIS': 'BNP Paribas Asset Mgt', 'REKSA DANA BNP PARIBAS KENCANA BALANCED': 'BNP Paribas Asset Mgt', 'REKSA DANA BNP PARIBAS INFRASTRUKTUR PLUS': 'BNP Paribas Asset Mgt', 'MANDIRI DINAMIC MONEY RP': 'Mandiri Investasi', 'MANDIRI INVESTA AKTIF': 'Mandiri Investasi', 'MANDIRI ATTRACTIVE EQUITY MONEY RUPIAH': 'Mandiri Investasi', 'REKSA DANA MANDIRI INVESTA ATRAKTIF': 'Mandiri Investasi', 'REKSA DANA MANDIRI INVESTA CERDAS BANGSA': 'Mandiri Investasi', 'MANDIRI EQUITY MONEY RUPIAH': 'Mandiri Investasi', 'DANA PENSIUN BANK MANDIRI': 'Mandiri Investasi', 'DANA PENSIUN PERKEBUNAN': 'Mandiri Investasi', 'REKSA DANA INDEKS MANDIRI INDEKS FTSE INDONESIA ESG': 'Mandiri Investasi', 'MANDIRI PROGRESSIVE BALANCED MONEY RUPIAH': 'Mandiri Investasi', 'YAKES TELEKOM MMI': 'Mandiri Investasi', 'REKSA DANA INDEKS MANDIRI ETF SRI KEHATI': 'Mandiri Investasi', 'KOP KESEHATAN PEG AND PENSIUNAN BANK MANDIRI': 'Mandiri Investasi', 'REKSA DANA MANDIRI INVESTA EQUITY ASEAN 5 PLUS': 'Mandiri Investasi', 'MANDIRI ACTIVE BALANCED MONEY SYARIAH RUPIAH': 'Mandiri Investasi', 'REKSA DANA INDEKS MANDIRI INDEKS LQ45': 'Mandiri Investasi', 'PT HEKSA SOLUTION INSURANCE MMI': 'Mandiri Investasi', 'RD MANDIRI INVESTA EQUITY MOVEMENT': 'Mandiri Investasi', 'REKSA DANA MANDIRI INVESTA DYNAMO FACTOR': 'Mandiri Investasi', 'MANDIRI EXCELLENT EQUITY RUPIAH': 'Mandiri Investasi', 'REKSA DANA BATAVIA CAMPURAN NUSANTARA': 'Batavia Prosperindo Asset Mgt', 'REKSA DANA INDEKS BATAVIA INDEX PEFINDO PRIME BANK': 'Batavia Prosperindo Asset Mgt', 'RDI BATAVIA SRI KEHATI ETF': 'Batavia Prosperindo Asset Mgt', 'BATAVIA SAHAM CEMERLANG': 'Batavia Prosperindo Asset Mgt', 'RD BATAVIA DANA SAHAM': 'Batavia Prosperindo Asset Mgt', 'BATAVIA SAHAM SEJAHTERA': 'Batavia Prosperindo Asset Mgt', 'KPD BPAM RUPIAH EQUITY FUND': 'Batavia Prosperindo Asset Mgt', 'REKSA DANA INDEKS BATAVIA INDEX PEFINDO I GRADE': 'Batavia Prosperindo Asset Mgt', 'PT ASURANSI BRI LIFE DARLINK AGRESIF 1': 'Batavia Prosperindo Asset Mgt', 'PT ASURANSI BRI LIFE DARLINK AGRESIF 2': 'Batavia Prosperindo Asset Mgt', 'DAPEN PLN': 'Batavia Prosperindo Asset Mgt', 'REKSA DANA BATAVIA CAMPURAN CEMERLANG': 'Batavia Prosperindo Asset Mgt', 'YAKES TELKOM BPAM': 'Batavia Prosperindo Asset Mgt', 'RD BATAVIA DANA SAHAM OPTIMAL': 'Batavia Prosperindo Asset Mgt', 'KPD SAHAM DANA MAKSIMA BATAVIA': 'Batavia Prosperindo Asset Mgt', 'KPD BPAM RUPIAH MANAGED FUND': 'Batavia Prosperindo Asset Mgt', 'REKSA DANA BATAVIA USA BALANCED ASIA': 'Batavia Prosperindo Asset Mgt', 'REKSA DANA INDEKS BATAVIA IDX30 ETF': 'Batavia Prosperindo Asset Mgt', 'REKSA DANA BATAVIA LQ45 PLUS': 'Batavia Prosperindo Asset Mgt', 'BATAVIA SMART LIQUID ETF': 'Batavia Prosperindo Asset Mgt', 'RD BATAVIA DANA DINAMIS': 'Batavia Prosperindo Asset Mgt', 'REKSA DANA BATAVIA PROVIDENTIA BALANCED FUND': 'Batavia Prosperindo Asset Mgt', 'REKSA DANA BATAVIA SAHAM ESG IMPACT': 'Batavia Prosperindo Asset Mgt', 'REKSA DANA BATAVIA DISRUPTIVE EQUITY': 'Batavia Prosperindo Asset Mgt', 'FIDELITY EMERGING ASIA FUND': 'Fidelity Investments', 'FIDELITY PACIFIC BASIN FUND': 'Fidelity Investments', 'FIDELITY ADVISOR EMERGING ASIA FUND': 'Fidelity Investments', 'FIDELITY FAR EAST FUND': 'Fidelity Investments', 'T ROWE PRICE EMERGING MARKETS DISCOVERY STOCK FUND': 'T Rowe Price', 'T ROWE PRICE FUNDS SICAV': 'T Rowe Price', 'FRANK RUSSELL TRUST COMPANY COMMINGLED EMPLOYEE BENEFIT FUNDS TRUST': 'T Rowe Price', 'ALASKA COMMON TRUST FUND': 'T Rowe Price', 'T ROWE PRICE INTERNATIONAL VALUE EQUITY TRUST': 'T Rowe Price', 'T ROWE PRICE FUNDS': 'T Rowe Price', 'T ROWE PRICE GLOBAL ALLOCATION FUND': 'T Rowe Price', 'STATE OF CONNECTICUT TREASURER': 'T Rowe Price', 'T ROWE PRICE ASIA OPPORTUNITIES FUND': 'T Rowe Price', 'T ROWE PRICE RETIREMENT HYBRID TRUST': 'T Rowe Price', 'T ROWE PRICE INTERNATIONAL VALUE EQUITY FUND': 'T Rowe Price', 'MASSMUTUAL SELECT T ROWE PRICE INTERNATIONAL EQUITY FUND': 'T Rowe Price', 'T ROWE PRICE GLOBAL IMPACT EQUITY FUND': 'T Rowe Price', 'TRP INTERNATIONAL STRUCTURED RESEARCH EQUITY': 'T Rowe Price', 'T ROWE PRICE IG GLOBAL IMPACT POOL': 'T Rowe Price', 'T ROWE PRICE INTEGRATED GLOBAL EQUITY FUND': 'T Rowe Price', 'T ROWE PRICE GLOBAL EX U S EQUITY MACRO POOL': 'T Rowe Price', 'T ROWE PRICE FUNDS OEIC GLOBAL IMPACT EQUITY FUND': 'T Rowe Price', 'T ROWE PRICE INTERNATIONAL EQUITY RESEARCH ETF': 'T Rowe Price', 'SCHWAB FUNDAMENTAL EMERGING MARKETS EQUITY ETF': 'Charles Schwab Investment Mgt', 'SCHWAB EMERGING MARKETS EQUITY ETF': 'Charles Schwab Investment Mgt', 'SCHWAB FUNDAMENTAL EMERGING MARKETS EQUITY INDEX FUND': 'Charles Schwab Investment Mgt', 'SCHWAB INTERNATIONAL DIVIDEND EQUITY ETF': 'Charles Schwab Investment Mgt', 'ROBECO GLOBAL EMERGING MARKETS EQUITY FUND II': 'Robeco Asset Mgt', 'ESKOM PENSION AND PROVIDENT FUND': 'Robeco Asset Mgt', 'ROLINCO NV': 'Robeco Asset Mgt', 'ROBECO INSTITUTIONAL EMERGING MARKETS FUND': 'Robeco Asset Mgt', 'ST JAMES S PLACE GLOBAL EMERGING MARKETS UNIT TRUST': 'Robeco Asset Mgt', 'ROBECO QI INSTITUTIONAL EMERGING MARKETS ENHANCED INDEX EQUITIES FUND': 'Robeco Asset Mgt', 'ROBECO CAPITAL GROWTH FUNDS EMERGING MARKETS EQUITIES': 'Robeco Asset Mgt', 'ROBECO QI CUSTOMIZED SUSTAINABLE EMERGING MARKETS ENHANCED INDEX EQUITIES FUND': 'Robeco Asset Mgt', 'BRUNEL PENSION PARTNERSHIP ACS TM BRUNEL EMERGING MARKETS EQUITY FUND': 'Robeco Asset Mgt', 'DFA EMERGING MARKETS CORE EQUITY 2 PORTFOLIO': 'Dimensional Fund Advisors', 'DIMENSIONAL ETF TRUST': 'Dimensional Fund Advisors', 'THE EMERGING MARKETS SERIES OF THE DFA INVESTMENT TRUST CO': 'Dimensional Fund Advisors', 'DIMENSIONAL EMERGING MARKETS CORE EQUITY 2 ETF': 'Dimensional Fund Advisors', 'DIMENSIONAL FUND ADVISORS WORLD EX US CORE EQUITY PORTFOLIO': 'Dimensional Fund Advisors', 'DIMENSIONAL FUNDS ICVC EMERGING MARKETS CORE EQUITY FUND': 'Dimensional Fund Advisors', 'ALL COUNTRY EX US EQUITY MARKET SUBTRUST': 'Dimensional Fund Advisors', 'JOHN HANCOCK MULTIFACTOR EMERGING MARKETS ETF': 'Dimensional Fund Advisors', 'DFA EMERGING MARKETS SOCIAL CORE PORTFOLIO': 'Dimensional Fund Advisors', 'DIMENSIONAL FUNDS PLC': 'Dimensional Fund Advisors', 'AZL DFA INTERNATIONAL CORE EQUITY FUND': 'Dimensional Fund Advisors', 'MANULIFE MULTIFACTOR EMERGING MARKETS INDEX ETF': 'Dimensional Fund Advisors', 'DIMENSIONAL FUNDS PLC WORLD EQUITY FUND': 'Dimensional Fund Advisors', 'DIMENSIONAL FUNDS PLC WORLD ALLOCATION 60 40 FUND': 'Dimensional Fund Advisors', 'DIMENSIONAL EMERGING MARKETS EX CHINA CORE EQUITY ETF': 'Dimensional Fund Advisors', 'DFA INTERNATIONAL VECTOR EQUITY PORTFOLIO': 'Dimensional Fund Advisors', 'DIMENSIONAL FUNDS PLC GLOBAL SUSTAINABILITY CORE EQUITY FUND': 'Dimensional Fund Advisors', 'WHOLESALE INTERNATIONAL CORE EQUITIES': 'Dimensional Fund Advisors', 'EMERGING MARKETS EX CHINA CORE EQUITY PORTFOLIO': 'Dimensional Fund Advisors', 'REKSA DANA PREMIER ETF IDX30': 'Indo Premier Investment Mgt', 'REKSA DANA INDEKS PREMIER ETF PEFINDO I GRADE': 'Indo Premier Investment Mgt', 'RD PREMIER ETF SRI KEHATI': 'Indo Premier Investment Mgt', 'RD PREMIER ETF INDONESIA STATE OWNED COMPANIES': 'Indo Premier Investment Mgt', 'REKSA DANA INDEKS PREMIER ETF IDX': 'Indo Premier Investment Mgt', 'REKSA DANA PREMIER ETF LQ 45': 'Indo Premier Investment Mgt', 'REKSA DANA INDEKS PREMIER ETF MSCI INDONESIA LARGE CAP': 'Indo Premier Investment Mgt', 'RD PREMIER ETF SMINFRA18': 'Indo Premier Investment Mgt', 'REKSA DANA INDEKS PREMIER ETF FTSE INDONESIA ESG': 'Indo Premier Investment Mgt', 'REKSA DANA INDEKS PREMIER ETF IDX HIGH DIVIDEND 20': 'Indo Premier Investment Mgt', 'REKSA DANA PREMIER CAMPURAN DINAMIS': 'Indo Premier Investment Mgt', 'RD PREMIER ETF INDONESIA FINANCIAL': 'Indo Premier Investment Mgt', 'REKSA DANA INDEKS PREMIER ETF INDEX IDX30': 'Indo Premier Investment Mgt', 'RD PREMIER EKUITAS MAKRO PLUS': 'Indo Premier Investment Mgt', 'RD PREMIER ETF INDONESIA CONSUMER': 'Indo Premier Investment Mgt', 'YAYASAN KESEJAHTERAAN PEKERJA BRI': 'BRI Investment Mgt', 'REKSA DANA DANAREKSA MAWAR EKUITAS UTAMA': 'BRI Investment Mgt', 'YKP BRI THT 1': 'BRI Investment Mgt', 'REKSA DANA INDEKS DANAREKSA MSCI INDONESIA ESG SCREENED': 'BRI Investment Mgt', 'YKP BRI QQ BANTUAN LAINNYA 1': 'BRI Investment Mgt', 'REKSA DANA DANAREKSA MAWAR KONSUMER 10': 'BRI Investment Mgt', 'REKSA DANA DANAREKSA BALANCED REGULAR INCOME FUND': 'BRI Investment Mgt', 'YKP BRI THT TRADING': 'BRI Investment Mgt', 'YKP BRI P4 TRADING': 'BRI Investment Mgt', 'PT ASURANSI BRI LIFE KPD BRIMI': 'BRI Investment Mgt', 'REKSA DANA BRI ETF INDONESIA TOP 40': 'BRI Investment Mgt', 'REKSA DANA DANAREKSA KOMODITAS 10': 'BRI Investment Mgt', 'DANAREKSA MAWAR': 'BRI Investment Mgt', 'REKSA DANA DANAREKSA ANGGREK FLEKSIBEL': 'BRI Investment Mgt', 'YKP BRI P3 TRADING': 'BRI Investment Mgt', 'REKSA DANA DANAREKSA ETF INDONESIA TOP 40': 'BRI Investment Mgt', 'PT AIA FINANCIAL UL EQUITY PROGRAMME': 'AIA Indonesia', 'PT AIA FINANCIAL': 'AIA Indonesia', 'AIA HY PAR IDR EQUITY': 'AIA Indonesia', 'PT AIA FINANCIAL HY HY NON PAR IDR EQUITY': 'AIA Indonesia', 'PT AIA FINL BLFR': 'AIA Indonesia', 'AIA FINANCIAL IDR ULTIMATE EQUITY FUND': 'AIA Indonesia', 'DPLK AIA DANAKARYA SEHATSEJAHTERA': 'AIA Indonesia', 'DPLK AIA EQUITY FUND ER13': 'AIA Indonesia', 'PT AIA FINANCIAL USD ONSHORE EQUITY FUND': 'AIA Indonesia', 'DPLK AIA FINANCIAL DEXA': 'AIA Indonesia', 'DPLK AIA FINANCIAL EQUITY PPUK': 'AIA Indonesia', 'DPLK AIA FINANCIAL': 'AIA Indonesia', 'PT AIA FINANCIAL UL EQUITY SOLID': 'AIA Indonesia', '1895 FONDS FGR WERELD AANDELEN ENHANCED INDEX FONDS': 'BlackRock Investment Mgt', 'BLACKROCK EMERGING MARKETS TRACKER FUND': 'BlackRock Investment Mgt', 'BLACKROCK SOLUTIONS FUNDS ICAV': 'BlackRock Investment Mgt', 'STICHTING BEDRIJFSTAKPENSIOENFONDS VOOR DE DETAILHANDEL': 'BlackRock Investment Mgt', 'STICHTING PHILIPS PENSIOENFONDS': 'BlackRock Investment Mgt', 'QSMA1 LLC': 'BlackRock Investment Mgt', 'MOOREA FUND MULTI MANAGER EMERGING MARKETS EQUITY': 'BlackRock Investment Mgt', 'SPW MANAGED ICVC SPW GLOBAL EX UK EQUITY FUND': 'BlackRock Investment Mgt', 'BLACKROCK GLOBAL FUNDS EMERGING MARKETS EQUITY INCOME FUND': 'BlackRock Investment Mgt', 'AXA CH STRATEGY FUND GLOBAL EQUITY CHF': 'BlackRock Investment Mgt', 'BLACKROCK GLOBAL FUNDS EMERGING MARKETS SUSTAINABLE EQUITY FUND': 'BlackRock Investment Mgt', 'REKSA DANA INDEKS BAHANA ETF BISNIS 27': 'Bahana TCW Investment Mgt', 'REKSA DANA BAHANA PRIMAVERA FUND': 'Bahana TCW Investment Mgt', 'REKSA DANA BAHANA STELLAR EQUITY FUND': 'Bahana TCW Investment Mgt', 'REKSA DANA BAHANA EXPLORER EQUITY FUND': 'Bahana TCW Investment Mgt', 'DANA PENSIUN BRI BAHANA 2': 'Bahana TCW Investment Mgt', 'REKSA DANA DANA EKUITAS PRIMA': 'Bahana TCW Investment Mgt', 'PT ASURANSI JIWA IFG IFG LINK EKUITAS': 'Bahana TCW Investment Mgt', 'ASURANSI BRI LIFE KPD BTIM EQUITY': 'Bahana TCW Investment Mgt', 'REKSA DANA INDEKS BAHANA INDEKS IDX30': 'Bahana TCW Investment Mgt', 'YAYASAN KESEHATAN PENSIUNAN ANEKA TAMBANG': 'Bahana TCW Investment Mgt', 'REKSA DANA BAHANA DANA EKUITAS ANDALAN': 'Bahana TCW Investment Mgt', 'ASURANSI BRI LIFE KPD BTIM EQUITY IBS': 'Bahana TCW Investment Mgt', 'PT ASURANSI JIWA IFG IFG LINK BERIMBANG': 'Bahana TCW Investment Mgt', 'PRIME EQUITY RUPIAH BAHANA': 'Bahana TCW Investment Mgt', 'REKSA DANA INDEKS BAHANA ETF PEFINDO I GRADE': 'Bahana TCW Investment Mgt', 'REKSA DANA BAHANA PRIMAVERA PLUS': 'Bahana TCW Investment Mgt', 'REKSADANA BAHANA TRAILBLAZER FUND': 'Bahana TCW Investment Mgt', 'DP BANK MANDIRI': 'Bahana TCW Investment Mgt', 'KPD DAPEN WIKA PPMP S A': 'Bahana TCW Investment Mgt', 'BUREAU OF LABOR FUNDS LABOR PENSION FUND': 'Bureau of Labor Funds Taiwan', 'BUREAU OF LABOR FUNDS LABOR RETIREMENT FUND': 'Bureau of Labor Funds Taiwan', 'DEUTSCHE BANK TRUST COMPANY AMERICAS': 'American Depositary Receipts', 'ARROWSTREET EMERGING MARKET TRUST FUND': 'Arrowstreet Capital', 'ARROWSTREET CAPITAL GLOBAL ALL COUNTRY ALPHA EXTENSION FUND': 'Arrowstreet Capital', 'DIVERSIFIED GLOBAL SHARE TRUST': 'Arrowstreet Capital', 'ARROWSTREET ACWI ALPHA EXTENSION FUND III': 'Arrowstreet Capital', 'HPE COMMON CONTRACTUAL FUND THE HPE GLOBAL EQUITY SUB FUND': 'Arrowstreet Capital', 'ARROWSTREET US GROUP TRUST': 'Arrowstreet Capital', 'ARROWSTREET COLLECTIVE INVESTMENT TRUST': 'Arrowstreet Capital', 'ARROWSTREET INTERNATIONAL EQUITY ACWI EX US TRUST FUND': 'Arrowstreet Capital', 'ARROWSTREET ACWI EX US ALPHA EXTENSION TRUST FUND': 'Arrowstreet Capital', 'ARROWSTREET CAPITAL NEWBURY FUND LIMITED': 'Arrowstreet Capital', 'ARROWSTREET GLOBAL EQUITY ACWI TRUST FUND': 'Arrowstreet Capital', 'ARROWSTREET GLOBAL EQUITY FUND': 'Arrowstreet Capital', 'ARROWSTREET DELAWARE ALPHA EXTENSION FUND': 'Arrowstreet Capital', 'LOCKHEED MARTIN CORPORATION DEFINED CONTRIBUTION PLANS MASTER TRUST': 'Arrowstreet Capital', 'ARROWSTREET CANADA GLOBAL ALL COUNTRY FUND I': 'Arrowstreet Capital', 'ARROWSTREET DELAWARE CUSTOM ALPHA EXTENSION FUND Q LP': 'Arrowstreet Capital', 'MLC LIMITED': 'Arrowstreet Capital', 'ROCHE US RETIREMENT PLANS MASTER TRUST': 'Arrowstreet Capital', 'ARROWSTREET ACWI ALPHA EXTENSION FUND V CAYMAN': 'Arrowstreet Capital', 'ARROWSTREET CANADA GLOBAL ALL COUNTRY FUND II': 'Arrowstreet Capital', 'DUNHAM INTERNATIONAL STOCK FUND': 'Arrowstreet Capital', 'BNP PARIBAS FINANCIAL MARKETS': 'BNP Paribas Financial Markets', 'JPMORGAN SECURITIES': 'JPMorgan Securities', 'REKSA DANA INDEKS PANIN IDX 30': 'Panin Asset Mgt', 'REKSA DANA PANIN DANA TELADAN': 'Panin Asset Mgt', 'REKSA DANA INDEKS PANIN SRI KEHATI': 'Panin Asset Mgt', 'REKSA DANA SAHAM PANIN DANA BERKEMBANG': 'Panin Asset Mgt', 'REKSA DANA SAHAM PANIN DANA BERDEDIKASI': 'Panin Asset Mgt', 'REKSA DANA CAMPURAN PANIN DANA BERIMBANG': 'Panin Asset Mgt', 'REKSA DANA PANIN DANA PRIMA': 'Panin Asset Mgt', 'REKSA DANA CAMPURAN PANIN DANA BERIMBANG DUA': 'Panin Asset Mgt', 'REKSA DANA PANIN DANA ULTIMA': 'Panin Asset Mgt', 'REKSA DANA PANIN DANA BERSAMA': 'Panin Asset Mgt', 'REKSA DANA PANIN DANA INFRASTRUKTUR BERTUMBUH': 'Panin Asset Mgt', 'REKSA DANA INDEKS PANIN ETF IDX30 DINAMIS': 'Panin Asset Mgt', 'PANIN NEO EQUITY FUND': 'Panin Asset Mgt', 'PANIN DAI ICHI LIFE PANIN SPECIAL BALANCED FUND': 'Panin Asset Mgt', 'REKSA DANA SAHAM PANIN BETA ONE': 'Panin Asset Mgt', 'REKSA DANA PANIN DANA PRIORITAS': 'Panin Asset Mgt', 'REKSA DANA CAMPURAN PANIN PRIORITAS SEHAT': 'Panin Asset Mgt', 'REKSA DANA PANIN DANA UNGGULAN': 'Panin Asset Mgt', 'REKSA DANA INDEKS PANIN IDX30': 'Panin Asset Mgt', 'REKSA DANA PANIN DANA MAKSIMA': 'Panin Asset Mgt', 'ABU DHABI INVESTMENT AUTHORITY': 'Abu Dhabi Investment Authority (ADIA)', 'BCIMC BRITISH COLUMBIA INVESTMENT MANAGEMENT CORPORATION': 'British Columbia Investment Mgt Corporation', 'BNY MELLON GLOBAL EQUITY INCOME FUND': 'Newton Investment Mgt', 'SUSTAINABLE GLOBAL EQUITY INCOME FUND': 'Newton Investment Mgt', 'BNY MELLON GLOBAL EMERGING MARKETS FUND': 'Newton Investment Mgt', 'FUTURELEGACY 6 FUND': 'Newton Investment Mgt', 'FUTURELEGACY 7 FUND': 'Newton Investment Mgt', 'FUTURELEGACY 5 FUND': 'Newton Investment Mgt', 'FUTURELEGACY 4 FUND': 'Newton Investment Mgt', 'FUTURELEGACY 3 FUND': 'Newton Investment Mgt', 'BNY MELLON GLOBAL EMERGING MARKETS OPPORTUNITIES FUND': 'Newton Investment Mgt', 'BNYM MELLON ASIAN INCOME FUND': 'Newton Investment Mgt', 'BNY MELLON ASIAN OPPORTUNITIES FUND': 'Newton Investment Mgt', 'BNY MELLON FUNDS TRUST BNY MELLON EMERGING MARKETS FUND': 'Newton Investment Mgt', 'BNY MELLON INVESTMENT FUNDS NEWTON EMERGING INCOME FUND': 'Newton Investment Mgt', 'MANULIFE MPF PACIFIC ASIA EQUITY FUND': 'Manulife Investment Mgt', 'UNIVERSAL S AKTIE NFONDS I': 'Manulife Investment Mgt', 'MANULIFE GLOBAL FUND ASEAN EQUITY FUND': 'Manulife Investment Mgt', 'MANULIFE INTERNATIONAL LIMITED': 'Manulife Investment Mgt', 'REKSA DANA SUCORINVEST EQUITY FUND': 'SucorInvest Asset Mgt', 'REKSA DANA SUCORINVEST MAXI FUND': 'SucorInvest Asset Mgt', 'REKSA DANA BALANCED FUND SUCORINVEST ANAK PINTAR': 'SucorInvest Asset Mgt', 'PT INDRA INVESTAMA': 'SucorInvest Asset Mgt', 'REKSA DANA SUCORINVEST DANA SEHAT BERIMBANG': 'SucorInvest Asset Mgt', 'DAPEN BANK JABAR': 'SucorInvest Asset Mgt', 'ASURANSI CIPUTRA INDONESIA SUCOR': 'SucorInvest Asset Mgt', 'REKSA DANA SUCORINVEST PREMIUM SOVEREIGN BALANCED FUND': 'SucorInvest Asset Mgt', 'REKSA DANA INDEKS SUCORINVEST IDX 30': 'SucorInvest Asset Mgt', 'SUCORINVEST SUSTAINABILITY EQUITY FUND': 'SucorInvest Asset Mgt', 'REKSA DANA SUCORINVEST USD BALANCED FUND': 'SucorInvest Asset Mgt', 'REKSA DANA SUCORINVEST AGGRESSIVE BALANCED AI FUND': 'SucorInvest Asset Mgt', 'REKSA DANA CAMPURAN SUCORINVEST DANA PRIORITAS SEHAT': 'SucorInvest Asset Mgt', 'SMARTLINK RUPIAH EQUITY FUND': 'Allianz Life Insurance', 'SMARTLINK RUPIAH BALANCED FUND': 'Allianz Life Insurance', 'SMARTWEALTH RUPIAH EQUITY ROTATION FUND': 'Allianz Life Insurance', 'DPLK ALLIANZ IND HM SAMPOERNA EQUITY FUND': 'Allianz Life Insurance', 'ALLIANZ SMARTLINK RUPIAH BALANCED PLUS FUND': 'Allianz Life Insurance', 'SMARTWEALTH RUPIAH EQUITY ROTATION CLASS B FUND': 'Allianz Life Insurance', 'SMARTWEALTH EQUITY INFRASTRUCTURE FUND': 'Allianz Life Insurance', 'DPLK ALLIANZ IND EQUITY FUND': 'Allianz Life Insurance', 'SMARTWEALTH EQUITY INDOASIA FUND': 'Allianz Life Insurance', 'DPLK ALLIANZ IND PPUKP EQUITY FUND': 'Allianz Life Insurance', 'ALLIANZ LIFE IND SAVINGS PLAN EQUITY FUND': 'Allianz Life Insurance', 'ALLIANZ LIFE IND SMARTWEALTH EQUITY SMALL MEDIUM CAPITAL FUND': 'Allianz Life Insurance', 'SMARTWEALTH RUPIAH EQUITY SUSTAINABLE INDONESIA FUND': 'Allianz Life Insurance', 'SMARTWEALTH LIQUIFLEX LQ45': 'Allianz Life Insurance', 'DPLK ALLIANZ IND HERO FUND': 'Allianz Life Insurance', 'ALLIANZ LIFE SMARTWEALTH EQUITY INDOGLOBAL FUND': 'Allianz Life Insurance', 'DPLK ALLIANZ IND PPUKP NESTLE FUND': 'Allianz Life Insurance', 'PUBLIC EMPLOYEES RETIREMENT ASSOCIATION OF COLORADO': 'Colorado PERA', 'WISDOMTREE EMERGING MARKETS HIGH DIVIDEND FUND': 'Mellon', 'JNL MELLON EMERGING MARKETS INDEX FUND': 'Mellon', 'POLICE AND FIREMEN S RETIREMENT SYSTEM OF NEW JERSEY': 'Mellon', 'CF DV EMERGING MARKETS STOCK INDEX FUND': 'Mellon', 'HC CAPITAL TRUST THE EMERGING MARKETS PORTFOLIO': 'Mellon', 'WISDOMTREE EMERGING MARKETS MULTIFACTOR FUND': 'Mellon', 'LATTICE EMERGING MARKETS STRATEGY ETF': 'Mellon', 'SNOWBALL CAPITAL': 'Mellon', 'WISDOMTREE EMERGING MARKETS EFFICIENT CORE FUND': 'Mellon', 'WISDOMTREE GLOBAL HIGH DIVIDEND FUND': 'Mellon', 'BANK OF NEW YORK MELLON EMPLOYEE BENEFIT COLLECTIVE INVESTMENT FUND PLAN': 'Mellon', 'WISDOMTREE EMERGING MARKETS QUALITY DIVIDEND GROWTH FUND': 'Mellon', 'GREENFIELDS CAPITAL': 'Mellon', 'RAINDROPS CAPITAL': 'Mellon', 'GLOBAL X SUPERDIVIDEND ETF': 'Global X Mgt', 'GLOBAL X ETFS ICAV SUPERDIVIDEND UCITS ETF': 'Global X Mgt', 'GLOBAL X FTSE SOUTHEAST ASIA ETF': 'Global X Mgt', 'GLOBAL X MSCI SUPERDIVIDEND EMERGING MARKETS ETF': 'Global X Mgt', 'SEVENTH SWEDISH NATIONAL PENSION FUND AP7 EQUITY FUND': 'Sjunde AP-fonden', 'LF ACCESS EMERGING MARKETS EQUITY FUND ROBECO': 'Robeco', 'REKSA DANA SCHRODER DANA PRESTASI PLUS': 'Schroder Investment Mgt', 'DANA PENSIUN BNI': 'Schroder Investment Mgt', 'AVA GROWTH PLUS FUND': 'Schroder Investment Mgt', 'PT ZURICH TOPAS LIFE ZURICHLINK RUPIAH EQUITY INVEST FUND': 'Schroder Investment Mgt', 'DANA PENSIUN CENTRAL BANK ASIA': 'Schroder Investment Mgt', 'REKSA DANA SCHRODER DANA PRESTASI': 'Schroder Investment Mgt', 'REKSA DANA SCHRODER 90 PLUS EQUITY FUND': 'Schroder Investment Mgt', 'PFI MEGA LIFE INSURANCE PFI MEGA LIFE EQUITY FUND': 'Schroder Investment Mgt', 'SCHRODER DANA ISTIMEWA': 'Schroder Investment Mgt', 'REKSA DANA SCHRODER DANA TERPADU II': 'Schroder Investment Mgt', 'PT ASURANSI BRI LIFE DARLINK DINAMIS': 'Schroder Investment Mgt', 'REKSA DANA SCHRODER DYNAMIC BALANCED FUND': 'Schroder Investment Mgt', 'AVA SMALL CAP EQUITY FUND': 'Schroder Investment Mgt', 'PRIME EQUITY RUPIAH SCHRODERS': 'Schroder Investment Mgt', 'PFI MEGA LIFE BALANCED FUND': 'Schroder Investment Mgt', 'DENNY DARMAWAN': 'Denny Darmawan', 'JPMORGAN ASEAN FUND A C 604124': 'JPMorgan Asset Mgt', 'JPMORGAN ACTIVE BUILDERS EMERGING MARKETS EQUITY ETF': 'JPMorgan Asset Mgt', 'JPMORGAN FUNDS CHINA A SHARE OPPORTUNITIES FUND': 'JPMorgan Asset Mgt (JF Asset Mgt)', 'JPM GEM DIVERSIFIED MOTHER FUND': 'JPMorgan Asset Mgt (JF Asset Mgt)', 'MACKENZIE GQE EMERGING MARKETS FUND': 'Mackenzie Investments', 'CANADA LIFE EMERGING MARKETS LARGE CAP EQUITY FUND': 'Mackenzie Investments', 'MACKENZIE EMERGING MARKETS LARGE CAPITAL FUND': 'Mackenzie Investments', 'MACKENZIE EMERGING MARKETS EX CHINA EQUITY FUND': 'Mackenzie Investments', 'MACKENZIE EMERGING MARKETS EQUITY INDEX ETF': 'Mackenzie Investments', 'IG MACKENZIE LOW VOLATILITY EMERGING MARKETS EQUITY POOL': 'Mackenzie Investments', 'MACKENZIE FUTUREPATH GLOBAL VALUE FUND': 'Mackenzie Investments', 'FIDELITY FUND INDONESIA FUND': 'FIL Investment Mgt', 'FIDELITY FUNDS ASEAN FUND': 'FIL Investment Mgt', 'FIDELITY FUNDS SICAV ASIAN SPECIAL SITUATIONS': 'FIL Investment Mgt', 'TREASURER OF THE STATE OF NORTH CAROLINA EQUITY INVESTMENT FUND POOLED TRUST': 'North Carolina Department of State Treasurer', 'AMERICAN CENTURY ETF TRUST AVANTIS EMERGING MARKETS EQUITY ETF': 'American Century Investment Mgt', 'AMERICAN CENTURY ETF TRUST AVANTIS EMERGING MARKETS VALUE ETF': 'American Century Investment Mgt', 'AMERICAN CENTURY ETF TRUST AVANTIS EMERGING MARKETS EQUITY FUND': 'American Century Investment Mgt', 'AMERICAN CENTURY ETF TRUST AVANTIS EMERGING MARKETS EX CHINA EQUITY ETF': 'American Century Investment Mgt', 'AMERICAN CENTURY ETF TRUST AVANTIS RESPONSIBLE EMERGING MARKETS EQUITY ETF': 'American Century Investment Mgt', 'AMERICAN CENTURY ICAV': 'American Century Investment Mgt', 'UBS LUX FUND SOLUTIONS MSCI EMERGING MARKETS UCITS ETF': 'UBS Global Asset Mgt', 'PENSIOENFONDS METAAL EN TECHNIEK PFMT': 'UBS Global Asset Mgt', 'RUSSELL INVESTMENT COMPANY EMERGING MARKETS FUND': 'UBS Global Asset Mgt', 'INTERNATIONALE KAPITALANLAGEGELLSCHAFT MBH FOR DC EM EQUITIES': 'UBS Global Asset Mgt', 'UBS GLOBAL EMERGING MARKETS EQUITY CLIMATE TRANSITION FUND': 'UBS Global Asset Mgt', 'UBS LIFE ALL WORLD EQUITY FACTOR MIX CARBON AWARE FUND': 'UBS Global Asset Mgt', 'UBS IRL ETF PLC MSCI ACWI UNIVERSAL UCITS ETF': 'UBS Global Asset Mgt', 'UBS LUX FUND SOLUTIONS': 'UBS Global Asset Mgt', 'WELLS FARGO ADVANTAGE EMERGING MARKETS EQUITY FUND': 'Allspring Global Investments', 'ALLSPRING DECLARATION OF TRUST ESTABLISHING INVESTMENT FUNDS FOR EMPLOYEE BENEFIT TRUSTS': 'Allspring Global Investments', 'EMERGING MARKETS EQUITY FUND': 'Allspring Global Investments', 'WELLS FARGO EMERGING MARKETS LARGE MID CAP FUND': 'Allspring Global Investments', 'GENUS EMERGING MARKET EQUITY FUND': 'Allspring Global Investments', 'PIMCO RAE EMERGING MARKETS FUND': 'Research Affiliates', 'DANA PENSIUN BRI SWAKELOLA': 'PT Bank Rakyat Indonesia (Persero) Tbk.', 'DANA PENSIUN BRI SWAKELOLA 3': 'PT Bank Rakyat Indonesia (Persero) Tbk.', 'DANA PENSIUN BRI SWAKELOLA 2': 'PT Bank Rakyat Indonesia (Persero) Tbk.', 'LO KHENG HONG': 'Lo Kheng Hong', 'NBI DIVERSIFIED EMERGING MARKETS EQUITY FUND': 'Goldman Sachs Asset Mgt', 'GS FUNDS GROWTH AND EMERGING MARKETS CORE EQUITY PORTFOLIO': 'Goldman Sachs Asset Mgt', 'GOLDMAN SACHS EMERGING MARKETS EQUITY INSIGHTS FUND': 'Goldman Sachs Asset Mgt', 'GOLDMAN SACHS ACTIVE BETA INTERNATIONAL EQUITY ETF': 'Goldman Sachs Asset Mgt', 'GOLDMAN SACHS ACTIVEBETA EMERGING MARKETS EQUITY ETF': 'Goldman Sachs Asset Mgt', 'PACIFIC SELECT FUND EMERGING MARKETS PORTFOLIO': 'Goldman Sachs Asset Mgt', 'GOLDMAN SACHS ETF TRUST II': 'Goldman Sachs Asset Mgt', 'GOLDMAN SACHS MARKETBETA EMERGING MARKETS EQUITY ETF': 'Goldman Sachs Asset Mgt', 'US BANK GUIDEMARK EMERGING MARKET FUND': 'Goldman Sachs Asset Mgt', 'UBS GLOBAL ASSET MANAGEMENT': 'UBS Global Asset Mgt', 'UBS CH INDEX FUND EQUITIES EMERGING MARKETS NSL': 'UBS Global Asset Mgt', 'BVK PERSONALVORSORGE DES KANTONS ZURICH': 'UBS Global Asset Mgt', 'KOLUMBAN BALANCED': 'UBS Global Asset Mgt', 'MIGROS AST FONDS II AKTIEN AUSLAND': 'UBS Global Asset Mgt', 'UBS LUX INST SICAV EMERGING MARKETS EQUITY PASSIVE': 'UBS Global Asset Mgt', 'AEIS INSTITUTIONAL FUND AKTIEN GLOBAL': 'UBS Global Asset Mgt', 'GALLUS AKTIEN EMERGING MARKETS AKTIV': 'UBS Global Asset Mgt', 'DEKA INVESTMENT GMBH RE BF MASTERFONDS': 'UBS Global Asset Mgt', 'GOTTHARD INSTITUTIONAL FUND GLOBAL EQUITIES': 'UBS Global Asset Mgt', 'UBS LUX FUND SOLUTIONS II': 'UBS Global Asset Mgt', 'BPER INTERNATIONAL SICAV GLOBAL BALANCED RISK CONTROL': 'UBS Global Asset Mgt', 'REKSA DANA ASHMORE SAHAM SEJAHTERA NUSATARA': 'Ashmore Asset Mgt', 'REKSA DANA ASHMORE DANA EKUITAS NUSANTARA': 'Ashmore Asset Mgt', 'REKSA DANA ASHMORE DANA PROGRESIF NUSANTARA': 'Ashmore Asset Mgt', 'DANA PENSIUN FREEPORT INDONESIA': 'Ashmore Asset Mgt', 'REKSA DANA ASHMORE PROVIDENTIA BALANCED NUSANTARA': 'Ashmore Asset Mgt', 'ASHMORE DIGITAL EQUITY SUSTAINABLE FUND': 'Ashmore Asset Mgt', 'MAESTROLINK EQUITY PLUS ASHMORE': 'Ashmore Asset Mgt', 'REKSA DANA ASHMORE SAHAM UNGGULAN NUSATARA': 'Ashmore Asset Mgt', 'REKSA DANA ASHMORE DANA OBLIGASI PROVIDENTIA NUSANTARA': 'Ashmore Asset Mgt', 'AVA EQUITY DOLLAR NUSANTARA FUND': 'Ashmore Asset Mgt', 'PRIME EQUITY RUPIAH ASHMORE': 'Ashmore Asset Mgt', 'REKSA DANA ASHMORE DANA USD EQUITY NUSANTARA': 'Ashmore Asset Mgt', 'REKSA DANA INDEKS BNI AM INDEKS IDX30': 'BNI Asset Mgt', 'REKSA DANA INDEKS BNI AM IDX PEFINDO PRIME BANK': 'BNI Asset Mgt', 'BNI LIFE INSURANCE SHF': 'BNI Asset Mgt', 'REKSA DANA INDEKS BNI ASSET MANAGEMENT IDX HIGH DIVIDEND 20': 'BNI Asset Mgt', 'REKSA DANA INDEKS BNI AM SRI KEHATI': 'BNI Asset Mgt', 'REKSA DANA INDEKS BNI AM INDEKS IDX GROWTH 30': 'BNI Asset Mgt', 'REKSA DANA INDEKS BNI AM PEFINDO I GRADE': 'BNI Asset Mgt', 'DANA PENSIUN KRAKATAU STEEL': 'BNI Asset Mgt', 'REKSA DANA PENDAPATAN TETAP BNI AM': 'BNI Asset Mgt', 'KPD SAHAM DANA MAKSIMA BNIAM': 'BNI Asset Mgt', 'SWISS NATIONAL BANK': 'Swiss National Bank', 'REKSA DANA SYAILENDRA DANA EKUITAS SEJAHTERA': 'Syailendra Capital', 'REKSA DANA INDEKS SYAILENDRA MSCI INDONESIA VALUE INDEX FUND': 'Syailendra Capital', 'REKSA DANA INDEKS SYAILENDRA SRI KEHATI': 'Syailendra Capital', 'REKSA DANA SYAILENDRA EQUITY GARUDA FUND': 'Syailendra Capital', 'REKSA DANA SYAILENDRA DANA INVESTASI DINAMIS': 'Syailendra Capital', 'REKSA DANA SYAILENDRA ETF MSCI INDONESIA ESG UNIVERSAL INDEX': 'Syailendra Capital', 'REKSA DANA SYAILENDRA EQUITY MOMENTUM FUND': 'Syailendra Capital', 'SYAILENDRA EQUITY OPPORTUNITY FUND': 'Syailendra Capital', 'SYAILENDRA BALANCED OPPORTUNITY FUND': 'Syailendra Capital', 'REKSA DANA SYAILENDRA PENDAPATAN TETAP PREMIUM': 'Syailendra Capital', 'REKSA DANA SYAILENDRA ALPHA FOCUS EQUITY FUND': 'Syailendra Capital', 'DANA PENSIUN KONFERENSI WALIGEREJA INDONESIA': 'Syailendra Capital', 'ALASKA PERMANENT FUND CORPORATION': 'Alaska Permanent Fund Corp.', 'MUTB400021536': 'State Street Investment Mgt', 'CMA STATE STREET EMERGING EQUITY INDEX MOTHER FUND': 'State Street Investment Mgt', 'MUTB400021492': 'State Street Investment Mgt', 'BBH MTBJ RE MTBJ400025521': 'State Street Investment Mgt', 'XTRACKERS IE PLC XTRACKERS MSCI EMERGING MARKETS UCITS ETF': 'DWS Investments', 'STICHTING BEDRIJFSTAKPENSIOENFONDS VOOR DE BOUWNIJVERHEID': 'DWS Investments', 'XTRACKERS IE PLC MSCI AC WORLD ESG SCREENED UCITS ETF': 'DWS Investments', 'ARERO DER WELTFONDS': 'DWS Investments', 'XTRACKERS IE PLC': 'DWS Investments', 'DB ADVISORS EMERGING MARKETS EQUITIES PASSIVE': 'DWS Investments', 'ARERO DER WELTFONDS ESG': 'DWS Investments', 'XTRACKERS UCITS COMMON CONTRACTUAL FUND': 'DWS Investments', 'XTRACKERS MSCI GLOBAL SDGS SOCIAL FAIRNESS CONTRIBUTORS UCITS ETF': 'DWS Investments', 'DWS INVEST EUROPEAN EQUITIES': 'DWS Investments', 'DEAM FONDS KG PENSIONEN': 'DWS Investments', 'SCOTTISH WIDOWS TRACKER AND SPECIALIST INVESTMENT FUNDS ICVC EMERGING MARKETS FUND': 'aberdeen (SWIP)', 'SCOTTISH WIDOWS INVESTMENT SOLUTIONS FUNDS ICVC FUNDAMENTAL INDEX EMERGING MARKETS EQUITY FUND': 'aberdeen (SWIP)', 'SCOTTISH WIDOWS INVESTMENT SOLUTIONS FUNDS ICVC EMERGING MARKETS TILTED EQUITY TRACKER FUND': 'aberdeen (SWIP)', 'SCOTTISH WIDOWS MANAGED INVESTMENT FUNDS INTERNATIONAL EQUITY TRACKER FUND': 'aberdeen (SWIP)', 'ABERDEEN ASIA PACIFIC EQUITY ENHANCED INDEX FUND': 'aberdeen (SWIP)', 'MERRILL LYNCH INTERNATIONAL LONDON': 'BofA Securities', 'BOFA SECURITIES': 'BofA Securities', 'COLLEGE RETIREMENT EQUITIES FUND CREF': 'TIAA Investment Mgt', 'NUVEEN EMERGING MARKETS EQUITY INDEX FUND': 'TIAA Investment Mgt', 'NUVEEN INTERNATIONAL OPPORTUNITIES FUND': 'TIAA Investment Mgt', 'SEI TRUST FBO NUVEEN TIAA INTERNATIONAL EQUITY INDEX FUND': 'TIAA Investment Mgt', 'NUVEEN LIFE INTERNATIONAL EQUITY FUND': 'TIAA Investment Mgt', 'TIAA CREF FUNDS INTERNATIONAL EQUITY FUND': 'TIAA Investment Mgt', 'TIAA CREF FUNDS TIAA CREF EMERGING MARKETS EQUITY FUND': 'TIAA Investment Mgt', 'GIC PRIVATE LTD': 'GIC', 'GOVERNMENT OF SINGAPORE': 'GIC', 'NEW YORK STATE COMMON RETIREMENT FUND': 'New York State Common Retirement Fund', 'HGIF ASIA PACIFIC EX JAPAN EQUITY HIGH DIVIDEND': 'HSBC Global Asset Mgt', 'HSBC GIF ASIAN FOCUSED INCOME': 'HSBC Global Asset Mgt', 'HSBC POOLED INVESTMENT FUND HSBC POOLED GLOBAL EQUITY FUND': 'HSBC Global Asset Mgt', 'HSBC LIFE VALUE GROWTH FUND': 'HSBC Global Asset Mgt', 'HSBC GIF ASEAN EQUITY': 'HSBC Global Asset Mgt', 'HSBC LIFE AND ANNUITY FUND': 'HSBC Global Asset Mgt', 'HSBC LIFE PAR WL FUND': 'HSBC Global Asset Mgt', 'HANDELSBANKEN TILLV XTMARKNADSFOND': 'Handelsbanken Fonder - Index', 'HANDELSBANKEN EMERGING MARKETS INDEX': 'Handelsbanken Fonder - Index', 'HANDELSBANKEN ASIENFOND': 'Handelsbanken Fonder - Index', 'HANDELSBANKEN GLOBAL INDEX CORE': 'Handelsbanken Fonder - Index', 'ILLINOIS MUNICIPAL RETIREMENT FUND': 'Illinois Municipal Retirement Fund (IMRF)', 'MUAM EMERGING EQUITY INDEX MOTHER FUND MTBJ101111080': 'Mitsubishi UFJ Asset Mgt', 'NIKKEI ASIA 300 INVESTABLE MOTHER FUND': 'Mitsubishi UFJ Asset Mgt', 'SBI EMERGING COUNTRIES HIGH DIVIDEND EQUITY MOTHER FUND': 'Mitsubishi UFJ Asset Mgt', 'UNIVERSAL INVESTMENT BAYVK A2 FONDS': 'Universal Investment', 'UNIVERSAL FONDS UNIVERSAL INVESTMENT GESELLSCHAFT': 'Universal Investment', 'BIL UNIVERSAL FONDS': 'Universal Investment', 'UNIVERSAL INVESTMENT GMBH FOR APT UNIVERSAL FUND': 'Universal Investment', 'DT EQUITY': 'Universal Investment', 'BROCKEN UI FONDS': 'Universal Investment', 'JPMSE LUX RE JPMSE RE UI S': 'Universal Investment', 'FOVERUKA PENSION UNIVERSAL': 'Universal Investment', 'BIS UI FONDS': 'Universal Investment', 'UNIVERSAL INVESTMENT GESELLSCHAFT MBH FOR KFPT UNIVERSAL FONDS': 'Universal Investment', 'UNIVERSAL IIVESTMENT GESELLSCHAFT MBH ON BEHALF OF UI FONDS RBSG': 'Universal Investment', 'EBK AKTIEN UNIVERSAL FONDS': 'Universal Investment', 'UNIVERSAL INVESTMENT GESELLSCHAFT MBH FOR PRISMA ASIANAVIGATOR UI': 'Universal Investment', 'BAYVK A4 FONDS': 'Universal Investment', 'NORTHERN TRUST EUROPE EQUITY INDEX UCITS FGR FUND': 'Northern Trust Global Investments', 'STICHTING PENSIOENFONDS HORECA AND CATERING': 'Northern Trust Global Investments', 'STICHTING AHOLD DELHAIZE PENSIOEN': 'Northern Trust Global Investments', 'IAM NATIONAL PENSION FUND': 'Northern Trust Global Investments', 'MERSEYSIDE PENSION FUND': 'Northern Trust Global Investments', 'GENERATION IM ASIA FUND': 'Generation Investment Mgt', 'ASABRI PERSERO DAPEN TNI': 'Asabri', 'ASABRI PERSERO DAPEN POLRI': 'Asabri', 'ASABRI': 'Asabri', 'INVESCO EQV ASIA PACIFIC EQUITY FUND': 'Invesco', 'INSTITUTIONAL RETIREMENT TRUST': 'Invesco', 'INVESCO ASIA PACIFIC GROWTH FUND': 'Invesco', 'AWARE SUPER': 'State Street Investment Mgt', 'CFS WHOLESALE INDEXED GLOBAL SHARE FUND': 'State Street Investment Mgt', 'CFSIL COMMONWEALTH GLOBAL SHARE FUND 16': 'State Street Investment Mgt', 'COLONIAL FIRST STATE INVESTMENT FUND 50': 'State Street Investment Mgt', 'PRIME SUPER': 'State Street Investment Mgt', 'PRUSIK ASIAN EQUITY INCOME FUND': 'Prusik Investment Mgt', 'ANDRA AP FONDEN': 'Andra AP Fonden', 'REKSA DANA TRIMEGAH SAHAM NUSANTARA': 'Trimegah Asset Mgt', 'REKSA DANA TRIM KAPITAL PLUS': 'Trimegah Asset Mgt', 'TRAM CONSUMPTION PLUS': 'Trimegah Asset Mgt', 'REKSA DANA INDEKS TRIMEGAH FTSE INDONESIA LOW VOLATILITY FACTOR INDEX': 'Trimegah Asset Mgt', 'REKSA DANA TRIMEGAH EQUITY FOCUS 2': 'Trimegah Asset Mgt', 'TRIMEGAH SEKURITAS INDONESIA': 'Trimegah Asset Mgt', 'TRIMEGAH FTSE4 GOOD INDONESIA INDEX KELAS M': 'Trimegah Asset Mgt', 'REKSA DANA TRIM KOMBINASI 2': 'Trimegah Asset Mgt', 'REKSA DANA TRIMEGAH BALANCED ABSOLUTE STRATEGY': 'Trimegah Asset Mgt', 'PUBLIC INDONESIA SELECT FUND': 'Public Mutual', 'PUBLIC INDONESIA BALANCE FUND': 'Public Mutual', 'PUBLIC MUTUAL ASEAN DIVIDEND SEQUEL FUND': 'Public Mutual', 'PUBLIC ASEAN GROWTH FUND': 'Public Mutual', 'PB ASEAN DIVIDEND FUND': 'Public Mutual', 'PB CHINA ASEAN EQUITY FUND': 'Public Mutual', 'PUBLIC SOUTH EAST ASIA SELECT FUND': 'Public Mutual', 'STATE OF NEW MEXICO INVESTMENT COUNCIL': 'New Mexico State Investment Council', 'FOREIGN AND COLONIAL INVESTMENT TRUST PLC': 'Columbia Threadneedle Investments', 'COLUMBIA THREADNEEDLE UK ICVC III CT UNIVERSAL MAP INCOME FUND': 'Columbia Threadneedle Investments', 'COLUMBIA THREADNEEDLE IRELAND III PLC': 'Columbia Threadneedle Investments', 'COLUMBIA THREADNEEDLE LUX III': 'Columbia Threadneedle Investments', 'COLUMBIA THREADNEEDLE UK ICVC III CT SUSTAINABLE UNIVERSAL MAP DEFENSIVE FUND': 'Columbia Threadneedle Investments', 'COLUMBIA THREADNEEDLE UK ICVC III CT SUSTAINABLE UNIVERSAL MAP ADVENTUROUS FUND': 'Columbia Threadneedle Investments', 'COLUMBIA THREADNEEDLE UK ICVC III CT SUSTAINABLE UNIVERSAL MAP BALANCED FUND': 'Columbia Threadneedle Investments', 'COLUMBIA THREADNEEDLE UK ICVC III CT SUSTAINABLE UNIVERSAL MAP CAUTIOUS FUND': 'Columbia Threadneedle Investments', 'COLUMBIA THREADNEEDLE UK ICVC III CT UNIVERSAL MAP GROWTH FUND': 'Columbia Threadneedle Investments', 'CT LUX ASIA EQUITIES FUND': 'Columbia Threadneedle Investments', 'BMO SUSTAINABLE OPPORTUNITIES GLOBAL EQUITY FUND': 'Columbia Threadneedle Investments', 'BMO GLOBAL BALANCED FUND': 'Columbia Threadneedle Investments', 'THREADNEEDLE PENSIONS LTD': 'Columbia Threadneedle Investments', 'ZURICH ASSURANCE': 'Columbia Threadneedle Investments', 'THREADNEEDLE SPECIALIST INVESTMENT FUNDS ICVC GLOBAL EQUITY INCOME FUND': 'Columbia Threadneedle Investments', 'THREADNEEDLE ASIA PACIFIC FUND': 'Columbia Threadneedle Investments', 'CT LUX ASIAN EQUITY INCOME FUND': 'Columbia Threadneedle Investments', 'THREADNEEDLE LUX GLOBAL MULTI ASSET INCOME': 'Columbia Threadneedle Investments', 'MERCER QIF FUND': 'Stewart Investors', 'UBS AG LONDON BRANCH': 'UBS Securities', 'ASEAN MOTHER FUND': 'Sumitomo Mitsui DS Asset Mgt', 'EDGBASTON ASIAN EQUITY JERSEY TRUST': 'Edgbaston Investment Partners', 'ENSIGN PEAK ADVISORS INC': 'Ensign Peak Advisors', 'PUBLIC EMPLOYEES RETIREMENT SYSTEM OF IDAHO': 'Public Employee Retirement System of Idaho', 'THE PUBLIC INSTITUTE FOR SOCIAL SECURITY': 'The Public Institution for Social Security', 'KBC EQUITY FUND': 'KBC Asset Mgt', 'KBC ECO FUND': 'KBC Asset Mgt', 'PLATO INSTITUTIONAL INDEX FUND': 'KBC Asset Mgt', 'CVBA MRBB': 'KBC Asset Mgt', 'UBAM POSITIVE IMPACT EMERGING EQUITY': 'UBP Asset Mgt', 'IMPERIAL EMERGING ECONOMIES POOL': 'CIBC Asset Mgt', 'CIBC EMERGING MARKETS FUND': 'CIBC Asset Mgt', 'CIBC EMERGING MARKETS EQUITY IETF': 'CIBC Asset Mgt', 'CIBC INTERNATIONAL EQUITY FUND': 'CIBC Asset Mgt', 'CIBC EMERGING MARKETS EQUITY PRIVATE POOL': 'CIBC Asset Mgt', 'CIBC EMERGING MARKETS INDEX FUND': 'CIBC Asset Mgt', 'CIBC EMERGING MARKETS EQUITY INDEX ETF': 'CIBC Asset Mgt', 'AVANTIS CIBC EMERGING MARKETS EQUITY ETF': 'CIBC Asset Mgt', 'CIBC ASIA PACIFIC INDEX FUND': 'CIBC Asset Mgt', 'RENAISSANCE EMERGING MARKETS FUND': 'CIBC Asset Mgt', 'MORGAN STANLEY AND CO INC': 'Morgan Stanley Asia', 'MS AND CO LLC': 'Morgan Stanley Asia', 'WESPATH FUNDS TRUST': 'Wespath Investment Mgt', 'COMMINGLED PENSION TRUST FUND EMERGING MARKETS RESEARCH ENHANCED EQUITY': 'JPMorgan Asset Mgt', 'JPMORGAN INCOME BUILDER FUND': 'JPMorgan Asset Mgt', 'COMMINGLED PENSION TRUST FUND EMERGING MARKETS EQUITY INDEX': 'JPMorgan Asset Mgt', 'JPMORGAN BETABUILDERS EMERGING MARKETS EQUITY ETF': 'JPMorgan Asset Mgt', 'COMMINGLED PENSION TRUST FUND ACTIVEBUILDERS EMERGING MARKETS EQUITY': 'JPMorgan Asset Mgt', 'JPMORGAN GLOBAL EMERGING MARKETS HYBRID FUND QDII': 'JPMorgan Asset Mgt', 'JPMORGAN ACTIVE DEVELOPING MARKETS EQUITY ETF': 'JPMorgan Asset Mgt', 'JPMORGAN EMERGING MARKETS RESEARCH ENHANCED EQUITY FUND': 'JPMorgan Asset Mgt', 'COMMINGLED PENSION TRUST FUND GLOBAL ALL COUNTRY RESEARCH ENHANCED EQUITY': 'JPMorgan Asset Mgt', 'COMMINGLED PENSION TRUST FUND ALL COUNTRY INTERNATIONAL EQUITY': 'JPMorgan Asset Mgt', 'HERMANTO TANOKO': 'Hermanto Tanoko', 'PUBLIC SECTOR PENSION INVESTMENT BOARD': 'PSP Investments', 'PUBLIC EMPLOYEES RETIREMENT SYSTEM OF OHIO OPERS': 'Ohio Public Employees Retirement System (OPERS)', 'DIMAS WIBOWO': 'Dimas Wibowo', 'ASIAN EQUITIES DERIVATIVES': 'HSBC Securities', 'FONDITALIA': 'Fideuram Asset Mgt', 'INTERFUND SICAV': 'Fideuram Asset Mgt', 'FONDITALIA EQUITY PACIFIC EX JAPAN FUND': 'Fideuram Asset Mgt', 'FIDEURAM FUND': 'Fideuram Asset Mgt', 'MACQUARIE BANK LIMITED DBU': 'Macquarie Bank', 'PIMCO BERMUDA DYNAMIC MULTI ASSET STRATEGY FUND': 'PIMCO', 'MATTHEWS ASIA DIVIDEND FUND': 'Matthews International Capital Mgt', 'MATTHEWS EMERGING MARKETS SUSTAINABLE FUTURE FUND': 'Matthews International Capital Mgt', 'BMO GUARDIAN ASIAN GROWTH AND INCOME FUND': 'Matthews International Capital Mgt', 'MATTHEWS EMERGING MARKETS SUSTAINABLE FUTURE ACTIVE ETF': 'Matthews International Capital Mgt', 'MATTHEWS ASIA FUNDS': 'Matthews International Capital Mgt', 'MATTHEWS ASIA EX JAPAN DIVIDEND FUND': 'Matthews International Capital Mgt', 'MATTHEWS EMERGING MARKETS EQUITY FUND': 'Matthews International Capital Mgt', 'REKSA DANA SAHAM HPAM SMART BETA EKUITAS': 'Henan Putihrai Asset Mgt', 'REKSA DANA HENAN ULTIMA EKUITAS': 'Henan Putihrai Asset Mgt', 'REKSA DANA HPAM TACTICAL EQUITY': 'Henan Putihrai Asset Mgt', 'IRISH LIFE ASSURANCE': 'Keyridge Asset Mgt', 'WISDOMTREE ISSUER ICAV': 'Keyridge Asset Mgt', 'GOLDMAN SACHS EMERGING MARKETS ENHANCED INDEX SUSTAINABLE EQUITY': 'Keyridge Asset Mgt', 'PUBLIC EMPLOYEES PENSION PLAN': 'Public Employees Pension Plan (PEPP)', 'LGPS CENTRAL EMERGING MARKETS EQUITY ACTIVE MULTI MANAGER FUND': 'LGPS Central', 'LGPS CENTRAL GLOBAL SUSTAINABLE EQUITY ACTIVE TARGETED FUND': 'LGPS Central', 'SSGA SPDR ETFS EUROPE II PUBLIC LIMITED COMPANY': 'State Street Investment Mgt', 'SSGA SPDR ETFS EUROPE I PLC MSCI ALL COUNTRY WORLD UCITS ETF': 'State Street Investment Mgt', 'L NSF RS KRINGAR TILLV XTMARKNAD INDEX': 'State Street Investment Mgt', 'MANAGED PENSION FUNDS LIMITED EMERGING MARKETS ESG SCREENED INDEX EQUITY SUB FUND': 'State Street Investment Mgt', 'MANAGED PENSION FUNDS LIMITED FUNDAMENTAL INDEX GLOBAL EQUITY SUB FUND': 'State Street Investment Mgt', 'UI FONDS BAV RBI AKTIEN': 'State Street Investment Mgt', 'STATE STREET GLOBAL ADVISORS EXEMPT UNIT TRUST': 'State Street Investment Mgt', 'SSGA SPDR ETFS EUROPE I PUBLIC LIMITED COMPANY MSCI ALL COUNTRY WORLD INVESTABLE MARKET UCITS ETF': 'State Street Investment Mgt', 'STATE STREET AUT GLOBAL INDEX EQUITY FUND': 'State Street Investment Mgt', 'RUTH CORE EMERGING MARKETS': 'Ruth Asset Mgt', 'RUTH ASSET MANAGEMENT SICAV EMERGING MARKET EQUITY': 'Ruth Asset Mgt', 'BALANCE CRAFTED BY RUTH': 'Ruth Asset Mgt', 'EXPANSION CRAFTED BY RUTH': 'Ruth Asset Mgt', 'BARCLAYS CAPITAL SECURITIES LTD': 'Barclays', 'BCSL CLIENT PB CAYMAN CLIENTS': 'Barclays', 'JASA RAHARJA PT': 'Jasa Raharja (Persero)', 'JASA RAHARJA PERSERO': 'Jasa Raharja (Persero)', 'GUINNESS ASSET MANAGEMENT FUNDS PLC': 'Guinness Asset Mgt', 'VICTORY MARKET NEUTRAL INCOME FUND': 'Victory Capital Mgt', 'USAA GLOBAL MANAGED VOLATILITY FUND': 'Victory Capital Mgt', 'AGGRESSIVE EKUITAS': 'Sun Life Financial', 'LINK BERIMBANG': 'Sun Life Financial', 'SUNLIFE FINANCIAL INDONESIA GLOBAL EMERGING MARKET EQUITY FUND': 'Sun Life Financial', 'SUNLINK IDR FIXED INCOME PLUS': 'Sun Life Financial', 'HARTFORD SCHRODERS INTERNATIONAL MULTI CAP VALUE FUND': 'Schroder Investment Mgt', 'SCHRODER INTERNATIONAL SELECTION FUND DYNAMIC OPPORTUNITIES': 'Schroder Investment Mgt', 'SCHRODER ASIAN INCOME FUND': 'Schroder Investment Mgt', 'SCHRODER GLOBAL VALUE FUND': 'Schroder Investment Mgt', 'BCT PIF SMART ASIAN EQUITY FUND': 'Schroder Investment Mgt', 'SCHRODER QEP GLOBAL ACTIVE VALUE FUND': 'Schroder Investment Mgt', 'AI BALANCED PENSION FUND': 'Schroder Investment Mgt', 'SCHRODER DIVERSIFIED GROWTH PORFOLIO': 'Schroder Investment Mgt', 'SCHRODER DIVERSIFIED GROWTH FUND': 'Schroder Investment Mgt', 'HARTFORD SCHRODERS DIVERSIFIED OPPRTUNITIES FUND': 'Schroder Investment Mgt', 'SCHRODER INSTITUTIONAL DEVELOPING MARKETS FUND': 'Schroder Investment Mgt', 'SCHRODER EMERGING MARKETS EQUITY FUND': 'Schroder Investment Mgt', 'SCHRODER GLOBAL EMERGING MARKETS FUND': 'Schroder Investment Mgt', 'AZL SCHRODER EMERGING MARKETS EQUITY FUND': 'Schroder Investment Mgt', 'SCHRODER EMERGING MARKETS FUND CANADA': 'Schroder Investment Mgt', 'SCHRODER CAPITAL MANAGEMENT COLLECTIVE TRUST': 'Schroder Investment Mgt', 'SCHRODER SUSTAINABLE MULTI FACTOR EQUITY FUND': 'Schroder Investment Mgt', 'SCHRODER GAIA HELIX FUND': 'Schroder Investment Mgt', 'SCHRODER GLOBAL EQUITY COMPONENT FUND': 'Schroder Investment Mgt', 'SCHRODER EMERGING MARKETS EQUITY ALPHA FUND': 'Schroder Investment Mgt', 'SPW TACTICAL FUND 1': 'Schroder Investment Mgt', 'SPW TACTICAL FUND 2': 'Schroder Investment Mgt', 'STATE OF CONNECTICUT': 'Schroder Investment Mgt', 'MTBJ RE SCHRODERS GLOBAL EMERGING MARKETS EQUITY MOTHER FUND': 'Schroder Investment Mgt', 'MUTB400037408': 'Schroder Investment Mgt', 'ABERDEEN ASIAN INCOME FUND': 'aberdeen', 'ABERDEEN INDONESIA EQUITY FUND': 'aberdeen', 'ASI ASIA PACIFIC EX JAPAN EQUITY TRACKER FUND': 'aberdeen', 'ABERDEEN SELECT PORTFOLIO ABERDEEN INCOME PLUS FUND': 'aberdeen', 'ABRDN EMERGING MARKETS EQUITY TRACKER FUND': 'aberdeen', 'REASSURE LTD': 'aberdeen', 'ABRDN EVOLVE ASIA PACIFIC EX JAPAN EQUITY INDEX FUND': 'aberdeen', 'ABRDN EMERGING MARKETS EQUITY ENHANCED INDEX FUND': 'aberdeen', 'ABERDEEN EM SMA COMPLETION FUND': 'aberdeen', 'ABERDEEN STANDARD SICAV I EMERGING MARKETS EQUITY INCOME FUND': 'aberdeen', 'ABRDN INSTITUTIONAL COMMINGLED FUNDS ABRDN EAFE ONLY FUND': 'aberdeen', 'ABRDN EMERGING MARKETS EQUITY INCOME FUND': 'aberdeen', 'BAYERNINVEST FONDS AKTIEN ASIEN': 'FIL Investment Mgt', 'FIDELITY INVESTMENT FUNDS FIDELITY ASIA FUND': 'FIL Investment Mgt', 'AIR PRODUCTS AND CHEMICALS INC MASTER TRUST': 'FIL Investment Mgt', 'PT TANCORP GLOBAL SENTOSA': 'PT Tancorp Global Sentosa', 'TUTY CHRISTIANA': 'Tuty Christiana', 'AVIVA INVESTORS GLOBAL EMERGING MARKETS INDEX FUND': 'Aviva Investors', 'AVIVA INVESTORS GLOBAL EMERGING MARKETS CORE FUND': 'Aviva Investors', 'AQUILA LIFE UK EQUITY INDEX FUND': 'Aviva Investors', 'AI STRATEGIC GLOBAL EQUITY FUND': 'Aviva Investors', 'CITIGROUP GLOBAL MARKETS': 'Citigroup Global Markets', 'NOMURA TB ASIA ATTRACTIVE DIVIDEND STOCK FUND MOTHER FUND': 'Nomura Asset Mgt', 'NOMURA TB NAMCO GLOBAL HIGH PROFIT MOTHER': 'Nomura Asset Mgt', 'NOMURA TRUST AND BANKING CO EMERGING EQUITY MOTHER FUND': 'Nomura Asset Mgt', 'NOMURA ASEAN GROWTH EQUITY MOTHER FUND': 'Nomura Asset Mgt', 'NOMURA MANAGED MASTER TRUST GLOBAL EMERGING MARKET HIGH DIVIDENG STOCK PREMIUM': 'Nomura Asset Mgt', 'NOMURA TB NAMCO GLOBAL EQUITY MOTHER': 'Nomura Asset Mgt', 'INVESCO RAFI EMERGING MARKETS ETF': 'Invesco Capital Mgt', 'INVESCO MARKETS III PLC FTSE RAFI EUROPE MID SM': 'Invesco Capital Mgt', 'INVESCO MARKETS II PLC': 'Invesco Capital Mgt', 'INVESCO BCPL BADGER FUND': 'Invesco Capital Mgt', 'MDPIM EMERGING MARKET EQUITY POOL': 'GMO', 'GMO EMERGING MARKETS EX CHINA FUND': 'GMO', 'IR GAYATRI RAWIT': 'Ir Gayatri Rawit', 'ROBERT CHRISTIAN TANOKO': 'Robert Christian Tanoko', 'LAZARD DEVELOPING MARKETS EQUITY PORTFOLIO': 'Lazard Asset Mgt', 'LAZARD ASSET MANAGEMENT': 'Lazard Asset Mgt', 'LAZARD INVESTMENT FUNDS LAZARD DEVELOPING MARKETS FUND': 'Lazard Asset Mgt', 'NYKREDIT INVEST ENGROS VAEKSTLANDE': 'Lazard Asset Mgt', 'LAZARD ASSET MANAGEMENT LLC': 'Lazard Asset Mgt', 'LAZARD WILMINGTON INTERNATIONAL EQUITY COLLECTIVE TRUST': 'Lazard Asset Mgt', 'MAYO CLINIC FOUNDATION': 'Lazard Asset Mgt', 'GREAT WEST EMERGING MARKETS EQUITY FUND': 'Lazard Asset Mgt', 'LAZARD GLOBAL ACTIVE FUNDS PLC': 'Lazard Asset Mgt', 'LAZARD EMERGING MARKETS EQUITY ADVANTAGE FUND': 'Lazard Asset Mgt', 'SEI GMF EMERGING MARKETS EQUITY FUND': 'Lazard Asset Mgt', 'ABU DHABI RETIREMENT PENSION AND BENEFITS FUND': 'Abu Dhabi Retirement Pensions and Benefits Fund', 'EQ EMERGING DIVIDEND': 'eQ Asset Mgt', 'GE GREATLINK ASEAN GROWTH FUND': 'Lion Global Investors', 'LION OCBC SECURITIES APAC FINANCIALS DIVIDEND PLUS ETF': 'Lion Global Investors', 'THE GREAT EASTERN LIFE SLIF FUND PAR': 'Lion Global Investors', 'LION CHINA MERCHANTS EMERGING ASIA SELECT INDEX ETF': 'Lion Global Investors', 'GREAT EASTERN TRUST PRIVATE LIMITED': 'Lion Global Investors', 'LION GLOBAL SOUTH EAST ASIA FUND': 'Lion Global Investors', 'GE LIFE SLIF NON PAR': 'Lion Global Investors', 'TA SOUTH EAST ASIA EQUITY FUND': 'Lion Global Investors', 'LIONGLOBAL ASIA HIGH DIVIDEND EQUITY FUND': 'Lion Global Investors', 'LION ASEAN FUND': 'Lion Global Investors', 'BANK OF KOREA BOK': 'Korea Investment Corporation', 'PICTET CH INSTITUTIONAL FUNDS EMERGING MARKETS TRACKER FUND': 'Pictet Asset Mgt', 'PICTET EMERGING MARKETS INDEX': 'Pictet Asset Mgt', 'PICTET REGENERATION': 'Pictet Asset Mgt', 'PKG FONDS AKTIEN GLOBAL': 'Pictet Asset Mgt', 'CAPACITY FONDS INSTITUTIONNEL': 'Pictet Asset Mgt', 'SBI PICTET ASIA HI TECH VENTURE FUND': 'Pictet Asset Mgt', 'SMTB PICTET GLOBAL THEMATIC OPPORTUNITIES MOTHER FUND': 'Pictet Asset Mgt', 'DB ADVISORY MULTIBRANDS': 'Pictet Asset Mgt', 'PICTET GLOBAL THEMATIC OPPORTUNITIES MASTER FUND': 'Pictet Asset Mgt', 'PICTET EMERGING MARKETS MULTI ASSET': 'Pictet Asset Mgt', 'PICTET POSITIVE CHANGE': 'Pictet Asset Mgt', 'SMTB PICTET POSITIVE CHANGE STRATEGY MOTHER FUND': 'Pictet Asset Mgt', 'PT AXA MANDIRI FINANCIAL SERVICES': 'AXA Asset Mgt Indonesia', 'AXA MAESTROLINK DYNAMIC': 'AXA Asset Mgt Indonesia', 'MAESTROLINK EQUITY PLUS BAHANA': 'AXA Asset Mgt Indonesia', 'AXA FINANCIAL INDONESIA': 'AXA Asset Mgt Indonesia', 'DPLK AXA MANDIRI EQUITY FUND': 'AXA Asset Mgt Indonesia', 'MAESTROLINK BALANCED IDR': 'AXA Asset Mgt Indonesia', 'MAESTROLINK EQUITY PLUS SCHRODERS': 'AXA Asset Mgt Indonesia', 'MAESTROLINK MAXIADVANTAGE RP': 'AXA Asset Mgt Indonesia', 'AXA BUMN TITANS EQUITY': 'AXA Asset Mgt Indonesia', 'GUNAWAN TJOKROHADI': 'Gunawan Tjokrohadi', 'ARTISAN SUSTAINABLE EMERGING MARKETS FUND': 'Artisan Partners', 'NEW SHIPS INC PENSION MASTER TRUST': 'Artisan Partners', 'RED LIFE REINSURANCE SERVICES LIMITED': 'Artisan Partners', 'OBLATE INTERNATIONAL PASTORAL INVESTMENT TRUST': 'Artisan Partners', 'ARTISAN PARTNERS GLOBAL FUNDS': 'Artisan Partners', 'GMS DISCOVERY FUND': 'Artisan Partners', 'BORDER TO COAST EMERGING MARKETS EQUITY ALPHA FUND': 'Goldman Sachs Asset Mgt', 'GOLDMAN SACHS ENHANCED INDEX SUSTAINABLE EMERGING MARKETS EQUITY FUND': 'Goldman Sachs Asset Mgt', 'GOLDMAN SACHS FUNDS': 'Goldman Sachs Asset Mgt', 'S BANK EMERGING MARKETS ESG EQUITY FUND': 'FIM Asset Mgt', 'S BANK GROWING ECONOMIES EQUITY FUND': 'FIM Asset Mgt', 'GOLDMAN SACHS EXECUTION AND CLEARING LP': 'Goldman Sachs', 'LOMBARD ODIER DARIER HENTSCH AND CIE': 'Lombard Odier Darier Hentsch & Cie', 'LOMBARD ODIER FUNDS': 'Lombard Odier Darier Hentsch & Cie', 'CC AND L QUANTITATIVE EMERGING MARKETS EQUITY FUND': 'Connor Clark & Lunn Investment Mgt', 'CONNOR CLARK AND LUNN INVESTMENT TRUST': 'Connor Clark & Lunn Investment Mgt', 'ENTSORGUNGSFONDS': 'Connor Clark & Lunn Investment Mgt', 'CC AND L QUANTITATIVE GLOBAL EQUITY UCITS FUND': 'Connor Clark & Lunn Investment Mgt', 'CC AND L QUANTITATIVE EMERGING MARKETS EQUITY EXTENSION MASTER FUND': 'Connor Clark & Lunn Investment Mgt', 'SCOTIA DIVERSIFIED BALANCED FUND': 'Connor Clark & Lunn Investment Mgt', 'CC AND L Q GLOBAL EQUITY FUND LP': 'Connor Clark & Lunn Investment Mgt', 'CC AND L Q GLOBAL EQUITY FUND': 'Connor Clark & Lunn Investment Mgt', 'CC AND L Q GROUP GLOBAL EQUITY FUND': 'Connor Clark & Lunn Investment Mgt', 'AMIN': 'Amin', 'HENDRA': 'Hendra', 'ALLIANZ EEE FONDS': 'Allianz Global Investors', 'ALLIANZGI FONDS APNIESA': 'Allianz Global Investors', 'ALLIANZ CGI FONDS 1': 'Allianz Global Investors', 'ALLIANZGI FONDS PTV2': 'Allianz Global Investors', 'ALLIANZ PV WS FONDS': 'Allianz Global Investors', 'ALLIANZ GLOBAL INVEST EUROPE GMBH': 'Allianz Global Investors', 'ALLIANZ VGL FONDS': 'Allianz Global Investors', 'ALLIANZ GLOBAL INVESTORS GMBH VKA FONDS': 'Allianz Global Investors', 'ALLIANZ GLOBAL INVESTORS FUND': 'Allianz Global Investors', 'ALLIANZ VGI 1 FONDS': 'Allianz Global Investors', 'SUMISHIN SHINKOUKOKUKABUSHI INDEX MOTHER FUND': 'Sumitomo Mitsui Trust Asset Mgt', 'STB EMERGING MARKETS EQUITY INDEX MOTHER FUND': 'Sumitomo Mitsui Trust Asset Mgt', 'CGMI PRIME FINANCE CLEARANCE ACC': 'Citigroup Global Markets', 'WS WALES PP GLOBAL GROWTH FUND': 'Waystone Mgt', 'WS ACCESS GLOBAL ACTIVE VALUE FUND': 'Waystone Mgt', 'WS WALES PP EMERGING MARKETS EQUITY FUND': 'Waystone Mgt', 'WAYSTONE UCITS PLATFORM LUX SICAV': 'Waystone Mgt', 'INKAMBH FONDS STU': 'HSBC INKA', 'INTERNATIONALE KAPITALANLAGEGESELLSCHAFT MBH': 'HSBC INKA', 'METZLER INVESTMENT GMBH FOR MI FONDS F29': 'HSBC INKA', 'INTERNATIONALE KAPITALANLAGEGESELLSCHAFT MBH EMA': 'HSBC INKA', 'INTERNATIONALE KAPITALANLAGEGESELLSCHAFT': 'HSBC INKA', 'INTERNATIONALE KAPITALANLAGEGESELLSCHAFT MBH A ASIEN LA I': 'HSBC INKA', 'INKA MBH AVAW': 'HSBC INKA', 'HSBC MST ASIEN': 'HSBC INKA', 'AVRIST LINK EQUITY IDR FUND': 'Avrist Asset Mgt', 'REKSA DANA INDEKS AVRIST IDX30': 'Avrist Asset Mgt', 'AVRIST ADA SAHAM BLUE SAFIR FUND': 'Avrist Asset Mgt', 'PT AVRIST ASSET MANAGEMENT': 'Avrist Asset Mgt', 'REKSA DANA AVRIST EQUITY': 'Avrist Asset Mgt', 'AVRIST LINK EQUITY PREMIER 2 IDR FUND': 'Avrist Asset Mgt', 'AVRIST LINK EQUITY GROWTH IDR FUND': 'Avrist Asset Mgt', 'AVRIST LINK EQUITY PREMIER 3 IDR FUND': 'Avrist Asset Mgt', 'PT AVRIST ASSURANCE': 'Avrist Asset Mgt', 'REKSA DANA INDEKS AVRIST LQ45': 'Avrist Asset Mgt', 'REKSA DANA AVRIST EQUITY GROWTH FUND': 'Avrist Asset Mgt', 'JAMINAN': 'Jaminan', 'RICO RIZAL BUDIDARMO': 'Rico Rizal Budidarmo', 'FIDELITY SUSTAINABLE RESEARCH ENHANCED EMERGING MARKETS EQUITY UCITS ETF': 'FIL Investment International', 'FIDELITY FUNDS SUSTAINABLE RESEARCH EMERGING EQUITY INCOME POOL': 'FIL Investment International', 'FIDELITY COMMON CONTRACTUAL FUND II FIDELITY GLOBAL EMRGING MARKET EQUITY FUND RE GSAM': 'FIL Investment International', 'FIDELITY COMMON CONTRACTUAL FUND II': 'FIL Investment International', 'RUDY JULIANTO': 'Rudy Julianto', 'AMUNDI AUSTRIA GMBH': 'Amundi Asset Mgt', 'KARYA DUMOGI RAHAYU': 'Karya Dumogi Rahayu', 'JNL MULTI MANAGER EMERGING MARKETS EQUITY FUND': 'T Rowe Price International', 'T ROWE PRICE FUNDS SICAV EMERGING MARKETS DISCOVERY EQUITY FUND': 'T Rowe Price International', 'T ROWE PRICE EMERGING MARKETS STOCK FUND': 'T Rowe Price International', 'RUSSELL INVESTMENT COMPANY INTERNATIONAL DEVELOPED MARKETS FUND': 'Russell Investments', 'RUSSELL INSTITUTIONAL FUNDS LLC EMERGING MARKETS EQUITY PLUS FUND': 'Russell Investments', 'RUSSELL INVESTMENTS TRUST COMPANY COMMINGLED EMPLOYEE BENEFIT FUNDS TRUST': 'Russell Investments', 'RUSSELL INSTITUTIONAL FUNDS LLC RUSSELL INTERNATIONAL EQUITY FUND': 'Russell Investments', 'VENERABLE EMERGING MARKETS EQUITY FUND': 'Russell Investments', 'HALLMARK CARDS INCORPORATED MASTER TRUST': 'Russell Investments', 'RUSSELL INVESTMENT FUNDS INTERNATIONAL DEVELOPED MARKETS FUND': 'Russell Investments', 'MULTI STYLE MULTI MANAGER FUNDS PLC': 'Russell Investments', 'RUSSELL INVESTMENT COMPANY SUSTAINABLE AWARE EQUITY FUND': 'Russell Investments', 'RUSSELL INVESTMENTS EMERGING MARKETS EQUITY POOL': 'Russell Investments', 'RUSSELL INVESTMENTS GLOBAL SHARES INDEX FUND': 'Russell Investments', 'RUSSELL INVESTMENTS OVERSEAS EQUITY POOL': 'Russell Investments', 'RUSSELL INVESTMENT COMPANY V PLC': 'Russell Investments', 'RUSSELL INVESTMENTS MENSURAM BONAM GLOBAL SHARES FUND': 'Russell Investments', 'RUSSELL INVESTMENTS FOCUSED GLOBAL EQUITY POOL': 'Russell Investments', 'YAYASAN KESEHATAN PEGAWAI TELKOM 3': 'Yayasan Kesehatan Pegawai Telkom', 'CATERPILLAR INVESTMENT TRUST': 'Caterpillar Master Retirement Trust', 'CATERPILLAR INC MASTER RETIREMENT TRUST': 'Caterpillar Master Retirement Trust', 'SUTTER HEALTH MASTER RETIREMENT TRUST': 'Sutter Health Master Retirement Trust', 'MI SELECT MANAGERS GLOBAL AND EMERGING OPPORTUNITIES FUND': 'RBC Global Asset Mgt', 'RBC EMERGING MARKETS EX CHINA EQUITY FUND': 'RBC Global Asset Mgt', 'RBC VFS INTERNATIONAL EQUITY FUND INCUBATOR': 'RBC Global Asset Mgt', 'MANULIFE PAR 3 EQUITY': 'Manulife Investment Mgt', 'MANULIFE PAR 2 EQUITY': 'Manulife Investment Mgt', 'MANULIFE PAR EQUITY': 'Manulife Investment Mgt', 'KENTUCKY RETIREMENT SYSTEM': "Kentucky Teachers' Retirement System", 'STICHTING PENSIONFONDS HOOGOVENS': 'Stichting Pensioenfonds Hoogovens', 'BPJS KESEHATAN': 'BPJS Kesehatan', 'PT INVESTASI KARYA UTAMA': 'PT Investasi Karya Utama', 'INDRAJONO SANKAWANG': 'Indrajono Sankawang', 'PING AN FTSE ASEAN DIVIDEND INDEX': 'UOB Asset Mgt', 'UNITED INDONESIA EQUITY FUND': 'UOB Asset Mgt', 'UNITED ASEAN FUND': 'UOB Asset Mgt', 'CBUS CONSTRUCTION AND BUILDING UNIONS SUPERANNUATION': 'Construction and Building Industry Super - Cbus', 'PT SURYAPARAMITRA ABADI': 'PT Suryaparamitra Abadi', 'STICHTING PENSIOEN': 'MN Services', 'VIRGINIA RETIREMENT SYSTEM': 'Virginia Retirement System', 'GENERAL ORGANISATION FOR SOCIAL INSURANCE': 'General Organization for Social Insurance', 'PT WARUNA NUSASENTANA': 'Waruna Nusasentana', 'PT TANRISE PROPERTY INDONESIA': 'Pt Tanrise Property Indonesia', 'SEB FUND 3 GLOBAL EXPOSURE FUND': 'SEB Asset Mgt', 'SEB EMERGING MARKETSFONDS': 'SEB Asset Mgt', 'SEB EMERGING MARKETS EXPOSURE FUND': 'SEB Asset Mgt', 'SEB SICAV 1 EMERGING MARKETS FUND': 'SEB Asset Mgt', 'FLORIDA RETIREMENT SYSTEM': 'Florida Retirement System', 'SINARIO THAMRIN': 'Sinario Thamrin', 'HENDRY CHRISTIAN W': 'Hendry Christian ,W', 'YAYASAN KESEJAHTERAAN KARYAWAN BANK INDONESIA': 'Yayasan Kesejahteraan Karyawan Bank Indonesia', 'COLONIAL FIRST STATE INVESTMENT FUND 110': 'JO Hambro Capital Mgt', 'JO HAMBRO CAPITAL MANAGEMENT GLOBAL EMERGING MARKET OPPORTUNITIES FUND': 'JO Hambro Capital Mgt', 'RYDER COURT GLOBAL EMERGING MARKETS OPPORTUNITIES FUND DST': 'JO Hambro Capital Mgt', 'PENDAL GLOBAL EMERGING MARKETS OPPORTUNITIES FUND': 'JO Hambro Capital Mgt', 'JOHCM EMERGING MARKETS OPPORTUNITIES FUND': 'JO Hambro Capital Mgt', 'COMMON FUND STRATEGIC DIRECT SERIES LLC CF TT INTERNATIONAL': 'JO Hambro Capital Mgt', 'COMMONWEALTH EMERGING MARKETS FUND 9': 'JO Hambro Capital Mgt', 'GOLDMAN SACHS FUNDS II': 'JO Hambro Capital Mgt', 'REGNAN GLOBAL EQUITY IMPACT SOLUTIONS FUND': 'JO Hambro Capital Mgt', 'HENDRY TANASAL': 'Hendry Tanasal', 'FAM SERIES UCITS ICAV EMERGING MARKETS EQUITY FAM FUND': 'Fineco Asset Mgt', 'FLEXIBLE INCOME FAM FUND': 'Fineco Asset Mgt', 'FAM SERIES UCITS ICAV FINECO AM GLOBAL EQUITY FUND': 'Fineco Asset Mgt', 'PT TAMAN DAYU': 'Pt Taman Dayu', 'SWEDBANK ROBUR ACCESS ASIEN': 'Swedbank Robur', 'SWEDBANK ROBUR ACCESS EDGE EMERGING MARKETS': 'Swedbank Robur', 'PT TANRISE MAKMUR SENTOSA': 'Pt Tanrise Makmur Sentosa', 'NOMURA GLOBAL HIGH INCOME STOCK FUND': 'Nomura Asset Mgt', 'NOMURA FUNDS IRELAND GLOBAL SUSTAINABLE EQUITY FUND': 'Nomura Asset Mgt', 'STICHTING JURIDISCH EIGENAAR ACTIAM': 'Cardano', 'NOW PENSIONS TRUST': 'Cardano', 'CARDANO GLOBAL SUSTAINABLE EQUITY FUND': 'Cardano', 'SOUTHERN COMPANY SYSTEM MASTER RETIREMENT TRUST': 'Southern Company System Master Retirement Trust', 'NATIONAL RAILROAD RETIREMENT INVESTMENT TRUST': 'National Railroad Retirement Investment Trust', 'VARIABLE PORTFOLIO PARTNERS INTERNATIONAL VALUE FUND': 'Thompson, Siegel & Walmsley', 'ASURANSI JIWA SEQUIS LIFE': 'Sequislife', 'PT AJ SEQUIS AAMI SQEF': 'Sequislife', 'PT AJ SEQUIS LIFE S A KPD BPAM SQEF': 'Sequislife', 'PT AJ SEQUIS LIFEE AAMI SQMF': 'Sequislife', 'PT AJ SEQUIS LIFE S A KPD SQAM SQEF': 'Sequislife', 'PT AJ SEQUIS LIFE S A KPD BPAM SQMF': 'Sequislife', 'PT AJ SEQUIS LIFE S A KPD SQAM SQMF': 'Sequislife', 'DANA PENSUIN BANK INDONESIA': 'Dapenbi', 'COUNTY EMPLOYEES ANNUITY AND BENEFIT FUND OF COOK COUNTY': 'Cook County Pension Fund', 'WINNIPEG CIVIC EMPLOYEES PENSION PLAN': 'Winnipeg Civic Employees Pension Plan', 'BNP PARIBAS EASY MSCI EMERGING ESG FILTERED MIN TE': 'BNP Paribas Asset Mgt', 'BNP PARIBAS FUNDS ASIA': 'BNP Paribas Asset Mgt', 'FWD SPRINT EQUITY FUND': 'FWD Asset Mgt', 'FWD EXCELLENT EQUITY FUND': 'FWD Asset Mgt', 'FWD EXCELLENT BALANCED FUND': 'FWD Asset Mgt', 'REKSA DANA INDEKS FWD ASSET IDX30 INDEX EQUITY FUND': 'FWD Asset Mgt', 'FWD USD BALANCED PLUS FUND': 'FWD Asset Mgt', 'CHANDRA': 'Chandra', 'KAPITALFORENINGEN INVESTIN PRO VELLIV EMERGING MARKETS EQUITY INDEX': 'Nykredit Asset Mgt', 'KAPITALFORENINGEN INVESTIN PRO GLOBAL EQUITIES I': 'Nykredit Asset Mgt', 'INVESTERINGSFORENINGEN NYKREDIT INVEST ENGROS GLOBAL EQUITIES I': 'Nykredit Asset Mgt', 'INVESTERINGSFORENINGEN NYKREDIT INVEST TAKTISK AKTIEALLOKERING ALLOK': 'Nykredit Asset Mgt', 'KAPITALFORENINGEN INVESTIN PRO GLOBALE AKTIER INDEX': 'Nykredit Asset Mgt', 'INVESTERINGSFORENINGEN NYKREDIT INVEST TAKTISK AKTIEALLOKERING ALLOK AKK': 'Nykredit Asset Mgt', 'KAPITALFORENINGEN EMD INVEST GLOBALE AKTIER III': 'Nykredit Asset Mgt', 'NYKREDIT INVEST TAKTISK ALLOKERING': 'Nykredit Asset Mgt', 'EMERGING MARKETS COMPLETION FUND LP': 'Parametric Portfolio Associates', 'EATON VANCE TAX MANAGED EMERGING MARKETS FUND': 'Parametric Portfolio Associates', 'PIMCO DIVIDEND AND INCOME FUND': 'Parametric Portfolio Associates', 'CATHOLIC VALUES EQUITY FUND': 'Parametric Portfolio Associates', 'PARAMETRIC SEM CORE EQUITY FUND': 'Parametric Portfolio Associates', 'EATON VANCE PARAMETRIC EMERGING MARKETS FUND': 'Parametric Portfolio Associates', 'EATON VANCE PARAMETRIC STRUCTURED EMERGING MARKETS FUND': 'Parametric Portfolio Associates', 'EATON VANCE COLLECTIVE INVESTMENT TRUST FOR EMPLOYEE BENEFIT PLANS EMERGING MARKETS EQUITY FUND': 'Parametric Portfolio Associates', 'THE BUNTING FAMILY VI SOCIALLY RESPONSIBLE LLC': 'Parametric Portfolio Associates', 'HUDSON EAST RIVER SYSTEMS LLC': 'Parametric Portfolio Associates', 'PARAMETRIC TMEMC FUND LP': 'Parametric Portfolio Associates', 'BRIDGE BUILDER TAX MANAGED INTERNATIONAL EQUITY FUND': 'Parametric Portfolio Associates', 'RUTH HUTAMA MANUA': 'Ruth Hutama Manua', 'DWIDJAJA SETIABUDI': 'Dwidjaja Setiabudi', 'THE TRUSTEE OF GLOBAL SUSTAINABLE EQUITY MOTHER FUND': 'Nomura Asset Mgt', 'PUBLIC SCHOOL TEACHERS PENSION AND RETIREMENT FUND OF CHICAGO': 'Public School Teachers Retirement Fund of Chicago', 'SINGGIH MINTARNO': 'Singgih Mintarno', 'SUWIGNYO BUDIMAN IR': 'Suwignyo Budiman Ir', 'AB SICAV I EMERGING MARKETS MULTI ASSET PORTFOLIO': 'AllianceBernstein - Growth', 'AB SICAV I ALL MARKET INCOME PORTFOLIO': 'AllianceBernstein - Growth', 'PENSION DANMARK PERSFORSKRGSAKTSELSKAB': 'PensionDanmark', 'SOESANAH RAHARDJO': 'Soesanah Rahardjo', 'ANDREAS BOBBY CIPUTRA': 'Andreas Bobby Ciputra', 'HARDING LOEVNER FUND INSTITUTIONAL EMERGING MARKETS PORTFOLIO': 'Harding Loevner Mgt', 'HARDING LOEVNER COLLECTIVE INVESTMENT TRUST FOR RETIREMENT PLANS': 'Harding Loevner Mgt', 'HARDING LOEVNER EMERGING MARKETS EQUITY FUND': 'Harding Loevner Mgt', 'HARDING LOEVNER FUNDS EMERGING MARKETS EX CHINA PORTFOLIO': 'Harding Loevner Mgt', 'HARDING LOEVNER FUND EMERGING MARKETS PORTFOLIO': 'Harding Loevner Mgt', 'VANECK VECTORS INDONESIA INDEX ETF': 'VanEck Global', 'CFSIL COMMONWEALTH GLOBAL SHARES FUND 22': 'First Sentier Investors RQI', 'COLONIAL FIRST STATE EMERGING MARKETS FUND 6': 'First Sentier Investors RQI', 'RQI EMERGING MARKETS DIVERSIFIED ALPHA FUND': 'First Sentier Investors RQI', 'COLONIAL FIRST STATE INVESTMENT FUND 122': 'First Sentier Investors RQI', 'CFSIL COMMONWEALTH GLOBAL SHARES FUND 23': 'First Sentier Investors RQI', 'TEAM SUPER': 'First Sentier Investors RQI', 'VERDIPAPIRFONDET KLP AKSJE FREMVOKSENDE MARKEDER INDEX 1': 'KLP Kapitalforvaltning', 'SOCIETE GENERALE PARIS STK BASKET TRADING': 'SG Securities', 'RB INDONESIAN EQUITY FUND': 'Resona Asset Mgt', 'JAPAN MUTUAL AID ASSOCIATION OF PUBLIC SCHOOL TEACHERS': 'Resona Asset Mgt', 'RB RAM INTERNATIONAL EMERGING COUNTRIES EQUITY MOTHER FUND': 'Resona Asset Mgt', 'ERNEST TEWIRA': 'Ernest Tewira', 'MR ARIES LIMAN': 'Mr Aries Liman', 'SWISSCANTO CH EQUITY FUND SUSTAINABLE EMERGING MARKETS': 'ZKB Asset Mgt', 'SWISSCANTO IE ESGEN SDG INDEX EQUITY WORLD UCITS ETF': 'ZKB Asset Mgt', 'SWISSCANTO CH INDEX FUND V EQUITY EMERGING MARKETS': 'ZKB Asset Mgt', 'SWISSCANTO CH INDEX FUND V EMERGING MARKETS RESPONSIBLE': 'ZKB Asset Mgt', 'ROY TEGUH': 'Roy Teguh', 'PETRUS KURNIAWAN SUSILO': 'Petrus Kurniawan Susilo', 'HERLIE': 'Herlie', 'JEREMY GOZAL': 'Jeremy Gozal', 'JOHN HANCOCK FUNDS II INTERNATIONAL STRATEGIC EQUITY ALLOCATION FUND': 'Manulife Investment Mgt', 'JOHN HANCOCK TRUST COMPANY COLLECTIVE INVESTMENT TRUST': 'Manulife Investment Mgt', 'BMO MSCI EMERGING MARKETS INDEX ETF': 'BMO Asset Mgt', 'TJEUW SUDYANTO': 'Tjeuw Sudyanto', 'AXA CHINA REGION INSURANCE COMPANY BERMUDA LIMITED': 'AXA Investment Mgrs Asia', 'DBI FONDS DSPT 8 ALLIANZ GLOBAL INVESTORS': 'AXA Investment Mgrs Asia', 'AXA WF FRAMLINGTON EMERGING MARKETS': 'AXA Investment Mgrs Asia', 'AXA AFFIN LIFE INSURANCE': 'AXA Investment Mgrs Asia', 'AXA CHINA REGION INSURANCE CO BERMUDA LTD HIGH YIELD DIVIDEND': 'AXA Investment Mgrs Asia', 'STATE TEACHERS RETIREMENT SYSTEM OF OHIO': 'Ohio State Teachers Retirement System', 'KPD GREATLINK SAHAM NUSANTARA': 'Great Eastern Life Assurance', 'GREATLINK SUSTAINABILITY EQUITY FUND': 'Great Eastern Life Assurance', 'GREATLINK INTELLIGENCE EQUITY FUND': 'Great Eastern Life Assurance', 'KPD GREATLINK BALANCE': 'Great Eastern Life Assurance', 'KPD GREATLINK DANA SAHAM': 'Great Eastern Life Assurance', 'GREATLINK SUPREME EQUITY FUND': 'Great Eastern Life Assurance', 'I GEDE SAGUS YUDISTIRA LAKSANA': 'I Gede Sagus Yudistira Laksana', 'TJOA SIU LAN': 'Tjoa Siu Lan', 'BMO PRIVATE EMERGING MARKETS EQUITY PORTFOLIO': 'Comgest', 'HARSHA EDWANA JOESOEF': 'Harsha Edwana Joesoef', 'TEACHERS RETIREMENT SYSTEM OF THE CITY OF NEW YORK': 'Teachers Retirement System of the City of New York', 'VICTORY EMERGING MARKETS FUND': 'Victory Capital Mgt', 'REKSA DANA INDEX PINNACLE FTSE INDONESIA ETF': 'Pinnacle Investment', 'REKSA DANA PINNACLE ENHANCED LIQUID ETF': 'Pinnacle Investment', 'REKSA DANA PINNACLE INDONESIA ESG ETF': 'Pinnacle Investment', 'REKSA DANA PINNACLE CORE HIGH DIVIDEND ETF': 'Pinnacle Investment', 'REKSA DANA PINNACLE STRATEGIC EQUITY FUND': 'Pinnacle Investment', 'HANG SENG INSURANCE COMPANY ASIA EQUITY PASSIVE': 'Hang Seng Investment Mgt', 'TRINITY HEALTH CORPORATION': 'Trinity Health Corporation', 'PT STUDILMU TEKNOLOGI GLOBAL': 'PT Studilmu Teknologi Global', 'DODGE AND COX EMERGING MARKETS STOCK FUND': 'Dodge & Cox', 'DODGE AND COX WORLDWIDE FUNDS PLC': 'Dodge & Cox', 'PICTET AND CIE UNDISCLOSED': 'Pictet & Cie', 'PICTET ET CIE GENEVA': 'Pictet & Cie', 'CC AND L DIVERSIFIED INCOME PORTFOLIO': 'Connor Clark & Lunn Investment Mgt', 'EARNEST PARTNERS EMERGING MARKET FUND': 'Earnest Partners', 'EMERGING MARKETS EQUITY PLUS PORTFOLIO': 'Earnest Partners', 'EARNEST EMERGING MARKETS INVESTMENT TRUST FUND': 'Earnest Partners', 'EARNEST INSTITUTIONAL LLC': 'Earnest Partners', 'HARBOR EMERGING MARKETS EQUITY FUND': 'Earnest Partners', 'JONATHAN HANDOKO PUTRA': 'Jonathan Handoko Putra', 'QIC LISTED EQUITIES FUND': 'QIC Asset Mgt', 'QIC INTERNATIONAL EQUITIES FUND': 'QIC Asset Mgt', 'GUNADI WANAMARTA': 'Gunadi Wanamarta', 'ANAWATI DARMAWAN': 'Anawati Darmawan', 'BUDI SETIAWAN': 'Budi Setiawan', 'KURNIA LUKMAN GOUTAMA': 'Kurnia Lukman Goutama', 'ANTON SUTOPO': 'Anton Sutopo', 'HAUW HOA ENG': 'Hauw Hoa Eng', 'SANDI SUWITO SUSILO': 'Sandi Suwito Susilo', 'DJONI': 'Djoni', 'REKSA DANA INDEKS UOBAM INDEKS BISNIS 27': 'UOB Asset Mgt', 'REKSA DANA SAHAM UOBAM SUSTAINABLE EQUITY INDONESIA': 'UOB Asset Mgt', 'PT TUGU PRATAMA INDONESIA': 'Tugu Pratama Indonesia', 'HUDIBJO': 'Hudibjo', 'HENDRAWAN TRANGGANA': 'Hendrawan Tranggana', 'PELAYARAN MULTI JAYA SAMUDERA': 'Pelayaran Multi Jaya Samudera', 'EURIZON EASYFUND': 'Eurizon Capital', 'EURIZON EMERGING LEADERS ESG 50 LUGLIO 2026': 'Eurizon Capital', 'EURIZON EMERGING LEADERS ESG 50 GENNAIO 2027': 'Eurizon Capital', 'JOHANES DANIEL': 'Johanes Daniel', 'SURYADI TJU': 'Suryadi Tju', 'INVESTERINGSFORENINGEN BANKINVEST': 'BI Asset Mgt', 'TSCB DIAM VIP FOCUS FUND': 'Asset Mgt One Singapore', 'AJ GENERALI INDONESIA PT GENERALY EQUITY ULTIMA': 'Generali Indonesia', 'GENERALI EQUITY ALPHA': 'Generali Indonesia', 'AJ GENERALI INDONESIA PT': 'Generali Indonesia', 'DPLK GENERALI EQUITY BETA PLUS': 'Generali Indonesia', 'AJ GENERALI INDONESIA PT GENERALI FORTUNA EQ FUND': 'Generali Indonesia', 'AJ GENERALI INDONESIA PT GENERALI EQUITY INDEX': 'Generali Indonesia', 'AJ GENERALI INDONESIA PT GENERALI EQUITY GROWTH': 'Generali Indonesia', 'AJ GENERALI INDONESIA PT GENERALY EQUITY BETA PLUS': 'Generali Indonesia', 'JONATHAN ANDERSEN P': 'Jonathan Andersen P', 'INSIGHT COMMUNITY DEVELOPMENT': 'Insight Investments Mgt', 'REKSA DANA INSIGHT BENEFIT BALANCED FUND': 'Insight Investments Mgt', 'REKSADANA GURU': 'Insight Investments Mgt', 'REKSA DANA INSIGHT BHINNEKA BALANCED FUND': 'Insight Investments Mgt', 'REKSA DANA INSIGHT WEALTH': 'Insight Investments Mgt', 'REKSA DANA INDEKS INSIGHT SRI KEHATI': 'Insight Investments Mgt', 'REKSA DANA INSIGHT COMMUNITY DEVELOPMENT': 'Insight Investments Mgt', 'REKSA DANA INDEKS INSIGHT ETF FTSE LOW VOLATILITY FACTOR INDEX': 'Insight Investments Mgt', 'REKSA DANA INSIGHT TUNAS BANGSA I NEXT G': 'Insight Investments Mgt', 'REKSA DANA INSIGHT TUNAS BANGSA BALANCED FUND 2': 'Insight Investments Mgt', 'REKSA DANA INSIGHT FELLOWSHIP': 'Insight Investments Mgt', 'REKSA DANA INSIGHT GREEN': 'Insight Investments Mgt', 'REKSA DANA INSIGHT GENERATE BALANCED FUND': 'Insight Investments Mgt', 'REKSA DANA INSIGHT GROWTH BALANCED FUND': 'Insight Investments Mgt', 'ALWIN': 'Alwin', 'PT ASURANSI JIWA SEQUIS LIFE EQUITY FUND': 'Sequis Asset Mgt', 'KPD AAMI RUPIAH EQUITY FUND': 'Sequis Asset Mgt', 'KPD SQAM RUPIAH EQUITY FUND': 'Sequis Asset Mgt', 'SEQUIS Q DYNAMIC FUND': 'Sequis Asset Mgt', 'REKSA DANA INDEKS SEQUIS EQUITY IDX30': 'Sequis Asset Mgt', 'REKSA DANA SEQUIS EQUITY INDONESIA': 'Sequis Asset Mgt', 'KPD AAMI RUPIAH MANAGED FUND': 'Sequis Asset Mgt', 'KPD SQAM RUPIAH MANAGED FUND': 'Sequis Asset Mgt', 'REKSA DANA SEQUIS BALANCE ULTIMA': 'Sequis Asset Mgt', 'REKSA DANA SEQUIS EQUITY MAXIMA': 'Sequis Asset Mgt', 'SELFIA SALIM': 'Selfia Salim', 'NOMURA SYSTEMATIC EMERGING MARKETS EQUITY FUND': 'Nomura Asset Mgt', 'REKSA DANA INDEKS RHB SRI KEHATI INDEX FUND': 'Allianz Global Investors Asset Mgt Indonesia', 'ALLIANZ ALPHA SECTOR ROTATION': 'Allianz Global Investors Asset Mgt Indonesia', 'DANA PENSIUN LEMBAGA KEUANGAN ALLIANZ INDONESIA': 'Allianz Global Investors Asset Mgt Indonesia', 'SMART WEALTH RUPIAH MULTI ASSET INCOME FUND': 'Allianz Global Investors Asset Mgt Indonesia', 'REKSA DANA RHB INDO ASIA EQUITY FUND': 'Allianz Global Investors Asset Mgt Indonesia', 'ARIA INVESTMENTS TRUST': 'Aria Investments', 'INVESTORS WHOLESALE EMERGING MARKETS EQUITIES TRUST': 'BlackRock Investment Mgt (Australia)', 'SIGNATURE SELECT APAC ALLOCATION PLUS FUND': 'BlackRock Investment Mgt (Australia)', 'AEGON WORLD EQUITY BEAT POOL C': 'AEGON Asset Mgt', 'AEGON WORLD EQUITY INDEX FUND II': 'AEGON Asset Mgt', 'AEGON EQUITY EMERGING MARKETS INDEX FUND': 'AEGON Asset Mgt', 'AEGON CUSTODY BV': 'AEGON Asset Mgt', 'ARTALYTA SURYANI': 'Artalyta Suryani', 'ALLSPRING LUX WORLDWIDE FUND': 'Allspring Global Investments', 'ALLSPRING EMERGING MARKETS EQUITY FUND': 'Allspring Global Investments', 'YOSEPH NEIL SANMIKHA': 'Yoseph Neil Sanmikha', 'HARTONO LEMBONO': 'Hartono Lembono', 'NUVEEN TIAA EMERGING MARKET INDEX FUND': 'Nuveen Asset Mgt', 'FIDELITY INVESTMENT TRUST FIDELITY FLEX INTERNATIONAL FUND': 'Fidelity Investments', 'VARIABLE INSURANCE PRODUCTS FUND II INTERNATIONAL CAPITAL APPRECIATION': 'Fidelity Investments', 'FIDELITY MANAGEMENT AND RESEARCH COMPANY LLC': 'Fidelity Investments', 'FIDELITY SELECT PORTFOLIOS SEMICONDUCTORS PORTFOLIO': 'Fidelity Investments', 'FMR CAPITAL INC': 'Fidelity Investments', 'FIDELITY SUSTAINABLE EMERGING MARKETS EQUITY FUND': 'Fidelity Investments', 'FIDELITY SELECT EMERGING MARKETS EQUITY FUND': 'Fidelity Investments', 'FIDELITY EMERGING MARKETS EQUITY CENTRAL FUND': 'Fidelity Investments', 'FIDELITY SERIES EMERGING MARKETS FUND': 'Fidelity Investments', 'FIDELITY EMERGING MARKETS EQUITY INVESTMENT TRUST': 'Fidelity Investments', 'JOHNWAY SUWARSONO': 'Johnway Suwarsono', 'PAUL SLAMET': 'Paul Slamet', 'ZAINI ROESLI': 'Zaini Roesli', 'FERDIAN TIMUR SATYAGRAHA': 'Ferdian Timur Satyagraha', 'AIA ASIA EX JAPAN EQUITY FUND': 'AIA', 'PRUDENTIAL UNIT TRUST FUND': 'BOCI Prudential Asset Mgt', 'BOCI PRUDENTIAL MSCI AC ASIA PACIFIC EX JAPAN INDEX': 'BOCI Prudential Asset Mgt', 'NOMURA INDONESIA STOCK MOTHER FUND': 'Nomura Asset Mgt', 'STEVANUS LENGKONG': 'Stevanus Lengkong', 'ASURANSI KERUGIAN JASA RAHARJA': 'Asuransi Kerugian Jasa Raharja', 'THAT FA': 'That Fa', 'RAHADI SANTOSO': 'Rahadi Santoso', 'TJIN LIONG': 'Tjin Liong', 'RAM TECHCHAND MANWANI': 'Ram Techchand Manwani', 'WEST YORKSHIRE PENSION FUND': 'West Yorkshire Pension Fund', 'WAYAN ALIT ANTARA': 'Wayan Alit Antara', 'DEKA RETURN LIGHT BETA': 'Deka Investment Mgt', 'DEKA INVESTMENT GMBH': 'Deka Investment Mgt', 'DEKA MSCI EMERGING MARKETS ESG FOCUS SELECT UCTS ETF': 'Deka Investment Mgt', 'DEKA NACHHALTIGKEIT IMPACT AKTIEN': 'Deka Investment Mgt', 'DEKA INVESTMENT GMBH RE LWH FONDS AKTIEN': 'Deka Investment Mgt', 'DEKA INVESTMENT GMBH RE ELB FUNDMASTER1': 'Deka Investment Mgt', 'WIN BAGHRAF BERSAUDARA PT': 'Win Baghraf Bersaudara Pt', 'HADI WIJONO': 'Hadi Wijono', 'JOAO ANGELO DE SOUSA MOTA': 'Joao Angelo De Sousa Mota', 'TJHIN SYLVESTER TH': 'Tjhin Sylvester Th', 'DNB GLOBAL EMERGING MARKET INDEX': 'DNB Asset Mgt', 'ARGA SISWANDHI': 'Arga Siswandhi', 'MAESTROLINK EQUITY PLUS PRINCIPAL LQ45': 'Principal Asset Mgt', 'MAESTORLINK EQUITY PLUS PRINCIPAL IDX30': 'Principal Asset Mgt', 'REKSA DANA INDEKS PRINCIPAL INDEX IDX 30': 'Principal Asset Mgt', 'PRIME EQUITY RUPIAH PRINCIPAL': 'Principal Asset Mgt', 'REKSA DANA PRINCIPAL INDO DOMESTIC EQUITY FUND': 'Principal Asset Mgt', 'REKSA DANA PRINCIPAL TOTAL RETURN EQUITY FUND': 'Principal Asset Mgt', 'REKSA DANA CIMB PRINCIPAL BALANCED FOCUS II': 'Principal Asset Mgt', 'AXA VORSORGE FONDS II EQUITIES GLOBAL MARKETS PASSIVE': 'AXA Investment Mgrs Schweiz', 'DONNY SUYANTO': 'Donny Suyanto', 'SIMON KOSHAN': 'Simon Koshan', 'OSCAR SITORUS': 'Oscar Sitorus', 'LIA TIRTASAPUTRA': 'Lia Tirtasaputra', 'KUSWANDI SUDARGA': 'Kuswandi Sudarga', 'SMARTETFS ASIA PACIFIC DIVIDEND BUILDER ETF': 'Guinness Atkinson Asset Mgt', 'SOEGIHARTO SOEGONDO': 'Soegiharto Soegondo', 'SANTOSA ARYA': 'Santosa Arya', 'ALFREDY SIMAMORA': 'Alfredy Simamora', 'RUSTIYAN OEN': 'Rustiyan Oen', 'PT EKADHARMA INTERNATIONAL TBK': 'Pt Ekadharma International', 'DP PEGADAIAN': 'Dana Pensiun Pegadaian', 'DANA PENSIUN PEGADAIAN': 'Dana Pensiun Pegadaian', 'DP PEGADAIAN BNIS': 'Dana Pensiun Pegadaian', 'DP PEGADAIAN CGS': 'Dana Pensiun Pegadaian', 'DP PEGADAIAN IPS': 'Dana Pensiun Pegadaian', 'INDONESIA MANDIRI NETWORKS': 'Indonesia Mandiri Networks', 'HUSIN SARIMAN': 'Husin Sariman', 'PUTM ACS EMERGING MARKETS FUND': 'aberdeen', 'ASI ASIA PACIFIC INCOME EQUITY FUND': 'aberdeen', 'ASI ASIA PACIFIC DYNAMIC DIVIDEND FUND': 'aberdeen', 'ABERDEEN INSTITUTIONAL COMMINGLED FUNDS ABERDEEN INTERNATIONAL SMALLER COMPANIES FUND': 'aberdeen', 'ABRDN EMERGING MARKETS DIVIDEND ACTIVE ETF': 'aberdeen', 'ABERDEEN STANDARD SICAV 1 DIVERSIFIED INCOME FUND': 'aberdeen', 'KWIYONO': 'Kwiyono', 'HOSKING GLOBAL FUND PLC': 'Hosking Partners', 'THE MASTER TRUST BANK OF JAPAN AS TRUSTEE FOR PENSION FUND': 'Hosking Partners', 'HOSKING PARTNERS EQUITY FUND LLC': 'Hosking Partners', 'HOSKING PARTNERS GLOBAL EQUITY TRUST': 'Hosking Partners', 'NAM WORLDWIDE INVESTMENT FUND': 'Nissay Asset Mgt', 'NISSAY EMERGING MARKETS EQUITY INDEX MOTHER FUND': 'Nissay Asset Mgt', 'NISSAY GLOBAL LEADERS EQUITY MOTHER FUND': 'Nissay Asset Mgt', 'PT BNI LIFE INSURANCE BLIFE GROUP LINK MAXI BALANCED FUND': 'BNI Life Insurance', 'USLI': 'Usli', 'MUHAMMAD IQBAL': 'Muhammad Iqbal', 'KI SUN SEO': 'Ki Sun Seo', 'LIDYAWATI': 'Lidyawati', 'ANDRE RIZKY RAMADHAN': 'Andre Rizky Ramadhan', 'ANDRI WIJONO SUTIONO': 'Andri Wijono Sutiono', 'RAIFFEISEN NACHHALTIG AKTIEN': 'Raiffeisen Capital Mgt', 'VALIDA AKTIEN EMERGING MARKETS 1': 'Raiffeisen Capital Mgt', 'RAIFFEISEN EURASIEN AKTIEN': 'Raiffeisen Capital Mgt', 'BENNY PONTIAN MUSLIM': 'Benny Pontian Muslim', 'CHANDRA ADISUSANTO': 'Chandra Adisusanto', 'BELINDA NATALIA': 'Belinda Natalia', 'SAMSUNG ASEAN SECURITIES MASTER INVESTMENT TRUST EQUITY': 'Samsung Investment Mgt', 'ASSET MANAGEMENT EXCHANGE CCF': 'Metropolis Capital', 'LIONG BUDI SUHARJA': 'Liong Budi Suharja', 'LENNY SUGIHAT': 'Lenny Sugihat', 'PT BERLIAN TAMBANG INDONESIA': 'PT Berlian Tambang Indonesia', 'UTAH RETIREMENT SYSTEMS': 'Utah Retirement Systems', 'KAPITALFORENINGEN DANSKE INVEST INSTITUTIONAL AFDELING DANICA PENSION AKTIER 10': 'Danske Capital Mgt', 'DANSKE INVEST INDEX GLOBAL AC RESTRICTED': 'Danske Capital Mgt', 'AFDELING DANICA PENSION AKTIER 2': 'Danske Capital Mgt', 'MULYADI': 'Mulyadi', 'EDDY SUDJONO': 'Eddy Sudjono', 'DARMAYANTI ANYA ADISUSANTO': 'Darmayanti Anya Adisusanto', 'TM EQUITY FUND': 'Tokio Marine Asset Mgt', 'TM EQUITY INDONESIAN FUND': 'Tokio Marine Asset Mgt', 'TM EQUITY AGGRESSIVE FUND': 'Tokio Marine Asset Mgt', 'BAMBANG SUTANTO': 'Bambang Sutanto', 'JOHAN SOEDIBYO': 'Johan Soedibyo', 'AUSTRALIANSUPER': 'AustralianSuper', 'ALEX LISAN': 'Alex Lisan', 'LUMIERE ASIA VALUE FUND': 'Lumiere Capital', 'YAKES TELKOM SAM': 'Samuel Asset Mgt', 'DANA PENSIUN SEMEN GRESIK KPAI SAM': 'Samuel Asset Mgt', 'REKSA DANA INDEKS SAM ETF SRI KEHATI': 'Samuel Asset Mgt', 'YKP3 JATI SEJAHTERA SAM': 'Samuel Asset Mgt', 'REKSA DANA SAM DANA SAHAM NUSANTARA': 'Samuel Asset Mgt', 'REKSA DANA SAM INDONESIAN EQUITY FUND': 'Samuel Asset Mgt', 'HARTONO': 'Hartono', 'SCHRODER INTERNATIONAL MULTI CAP EQUITY TRUST': 'Schroder Investment Mgt', 'SUN LIFE ASSURANCE COMPANY OF CANADA': 'Schroder Investment Mgt', 'DRG HANDAYANI': 'Drg Handayani', 'TCSB EMERGING EQUITY PASSIVE MOTHER FUND': 'Asset Mgt One', 'TRUST AND CUSTODY SERVICES BANK LTD AS TRUSTEE FOR GLOBAL HIGH QUALITY EQUITY MOTHER FUND': 'Asset Mgt One', 'KARDIMAN SASTRO': 'Kardiman Sastro', 'HAJI YUSUF NAFI': 'Haji Yusuf Nafi', 'BNPPF S FUND EQUITY EMERGING MARKETS': 'BNP Paribas Asset Mgt', 'MALTA PENSION INVESTMENTS': 'BNP Paribas Asset Mgt', 'PEAR TREE POLARIS INTERNATIONAL OPPORTUNITIES FUND': 'Polaris Capital Mgt', 'F IRENE FANCY': 'F Irene Fancy', 'KOESMAN HERMAWAN': 'Koesman Hermawan', 'STEVEN SETIAWAN SUTRISNA': 'Steven Setiawan Sutrisna', 'REKSA DANA INDEKS MAYBANK FINANCIAL INFOBANK15 INDEX FUND': 'Maybank Asset Mgt', 'REKSA DANA MAYBANK DANA EKUITAS': 'Maybank Asset Mgt', 'REKSA DANA MAM BALANCED FUND': 'Maybank Asset Mgt', 'PREMIA DOW JONES EMERGING ASEAN TITANS 100 ETF': 'Premia Partners', 'RUSSELL EMERGING DIVIDEND GROWTH MOTHER FUND': 'Russell Investments', 'FIRE AND POLICE PENSION ASSOCIATION OF COLORADO': 'Russell Investments', 'RUSSELL INVESTMENTS LOW CARBON AUSTRALIAN SHARE FUND': 'Russell Investments', 'RUSSELL INVESTMENTS UCITS ICAV': 'Russell Investments', 'RUSSELL TAX MANAGED INTERNATIONAL EQUITY FUND': 'Russell Investments', 'REKSA DANA INDEKS GROW SRI KEHATI': 'Grow Investments', 'REKSA DANA GROW SAHAM INDONESIA': 'Grow Investments', 'DANA PENSIUN LEMBAGA KEUANGAN SAHAM': 'Dana Pensiun BRI', 'EDHIE BASKORO YUDHOYONO': 'Edhie Baskoro Yudhoyono', 'ANDREW': 'Andrew', 'DEWATA MEKARYA AGENG PT': 'Dewata Mekarya Ageng PT', 'MARDIANA': 'Mardiana', 'FRANKY MONDONG': 'Franky Mondong', 'SUSANTO': 'Susanto', 'VALIC COMPANY I EMERGING ECONOMIES FUND': 'BlackRock Investment Mgt', 'BLACKROCK ADVANTAGE EMERGING MARKETS FUND': 'BlackRock Investment Mgt', 'TANTONO TJANDRA': 'Tantono Tjandra', 'REKSA DANA INDEKS STAR INFOBANK15': 'Star Asset Mgt', 'REKSA DANA STAR VALUE HIGH CONVICTION EQUITY FUND': 'Star Asset Mgt', 'LINDA': 'Linda', 'NTB IESI MSCI ENHMF 936561': 'Amova Asset Mgt', 'SUMARDI': 'Sumardi', 'JONATAN SETIAWAN': 'Jonatan Setiawan', 'SATRIA AMI PUTRA A': 'Satria Ami Putra A', 'IR SUNARSO MSI': 'Ir Sunarso Msi', 'BENNY MARDONO': 'Benny Mardono', 'RUSSELL INVESTMENTS CONTINENTAL EUROPEAN EQUITY FUND': 'UBS Global Asset Mgt', 'GUNAWAN': 'Gunawan', 'YUANTA P SHARES MSCI INDONESIA INDEX FUND': 'Yuanta Securities Investment Trust', 'YUANTA NEW ASEAN BALANCED FUND': 'Yuanta Securities Investment Trust', 'HENDRI': 'Hendri', 'JOHAN BUNTORO': 'Johan Buntoro', 'DESJARDINS EMERGING MARKETS EQUITY INDEX ETF': 'Desjardins Global Asset Mgt', 'DESJARDINS EMERGING MARKETS OPPORTUNITIES FUND': 'Desjardins Global Asset Mgt', 'DESJARDINS RI EMERGING MARKETS LOW CO2 INDEX ETF': 'Desjardins Global Asset Mgt', 'DESJARDINS SUSTAINABLE EMERGING MARKETS EQUITY FUND': 'Desjardins Global Asset Mgt', 'AMAL SUSILO': 'Amal Susilo', 'PT ASURANSI WAHANA TATA': 'Asuransi Wahana Tata', 'DP JASA MARGA': 'Jasa Marga', 'INVESTERINGSFORENINGEN SPARINVEST INDEX MSCI ACWI': 'Sparinvest', 'SPARINVEST INDEX GLOBALE AKTIER ETIK KL': 'Sparinvest', 'BCEE SPARINVEST': 'Sparinvest', 'SPARINVEST INDEX B REDYGTIGE GLOBAL KL': 'Sparinvest', 'DAIWA EMERGING EQUITY FUNDAMENTAL INDEX MOTHER FUND': 'Daiwa Asset Mgt', 'SUSENO ANGGORO': 'Suseno Anggoro', 'PT DELTA WIBAWA BERSAMA': 'Delta Wibawa Bersama', 'MOTHER CABRINI': 'RhumbLine Advisers', 'RHUMBLINE ALL COUNTRY INTERNATIONAL INDEX FUND LLC': 'RhumbLine Advisers', 'RHUMBLINE CV ALL GLOBAL EX US EQUITY INDEX FUND LLC': 'RhumbLine Advisers', 'FRANK RUSSELL INVESTMENT COMPANY II PLC': 'Old Mutual Investment Group', 'SUBARNI AG': 'Subarni AG', 'AMUNDI PRIME ALL COUNTRY WORLD UCITS ETF': 'Amundi Asset Mgt', 'CEMERLANG PRIMA INVESTAMA': 'Cemerlang Prima Investama', 'JEANNE NATALIE PUTRI': 'Jeanne Natalie Putri', 'ALI SEBASTIAN': 'Ali Sebastian', 'TEACHERS RETIREMENT SYSTEM OF OKLAHOMA': 'Teachers Retirement System Of Oklahoma', 'PT SINAR INTI KORPORA': 'PT Sinar Inti Korpora', 'REKSA DANA KRESNA INDEX 45': 'Kresna Securities', 'ALBERTUS ANDRI ANGGASAPUTRO': 'Albertus Andri Anggasaputro', 'WIDYANTI LISTYANI LIE': 'Widyanti Listyani Lie', 'YULIANTO SANTOSO': 'Yulianto Santoso', 'STANLEY HANDIONO ANGKASA': 'Stanley Handiono Angkasa', 'AGGREGATE VALUE FUND': 'Aggregate Asset Mgt', 'MINAWATI MUNANTO': 'Minawati Munanto', 'BILLY': 'Billy', 'SUGIANTO ADI SUWIRYO': 'Sugianto Adi Suwiryo', 'ANDY': 'Andy', 'AHMAD SOLICHIN LUTFIYANTO': 'Ahmad Solichin Lutfiyanto', 'ABDUL RACHMAN': 'Abdul Rachman', 'HARESH CHANDRA NANDWANI': 'Haresh Chandra Nandwani', 'INVESTERINGSFORENINGEN MAJ INVEST EMERGING MARKETS VALUE KL': 'Maj Invest', 'MAJ INVEST EMERGING MARKETS VALUE FUND': 'Maj Invest', 'KUSNO TRIANTIO SE': 'Kusno Triantio Se', 'MINEWORKERS PENSION SCHEME': 'Mineworkers Pension Scheme', 'TEKARDJO ANGKASA': 'Tekardjo Angkasa', 'SUGIANTO': 'Sugianto', 'ADRIAN PUTRA BRASALI': 'Adrian Putra Brasali', 'MUHAMMAD RUDY': 'Muhammad Rudy', 'FELICE LEROY': 'Felice Leroy', 'STEFANNY SUTEDY': 'Stefanny Sutedy', 'TANSIL WILIS': 'Tansil Wilis', 'LIMINARDI SOLIMIN': 'Liminardi Solimin', 'SUPARI': 'Supari', 'CHARLES': 'Charles', 'IR I PUTU ABDI ANOM': 'IR I Putu Abdi Anom', 'IVAN WIHARDJA': 'Ivan Wihardja', 'BINTARNA TARDY': 'Bintarna Tardy', 'STEVEN THAMRIN': 'Steven Thamrin', 'MICHAEL HENDRATA': 'Michael Hendrata', 'SUGIARTO': 'Sugiarto', 'HADI SUNYOTO': 'Hadi Sunyoto', 'SCHRODER ASIAN ASSET INCOME FUND': 'Schroder Investment Mgt', 'SCHRODER ASIA MORE PLUS': 'Schroder Investment Mgt', 'SAMUEL TIRTA AGUNG BUDIHARDJO H': 'Samuel Tirta Agung Budihardjo H', 'DICKSON THAMRIN': 'Dickson Thamrin', 'INDRA YUWANA': 'Indra Yuwana', 'HANDRY JIMMY': 'Handry Jimmy', 'AGUSTINUS SIOR FERDINANDUS': 'Agustinus Sior Ferdinandus', 'LAY RAMLI SUTEJO': 'Lay Ramli Sutejo', 'GO HADI SAPUTRO': 'Go Hadi Saputro', 'HEE RENALD MAMANGKEY': 'Hee Renald Mamangkey', 'DARWIN SHP MANURUNG': 'Darwin Shp Manurung', 'ONEPATH GLOBAL EMERGING MARKETS SHARES UNDEDGED INDEX POOL': 'UBS Global Asset Mgt', 'SEI SELECT EMERGING MARKETS EQUITY ETF': 'SEI Investments', 'PRUDENTIAL HONG KONG': 'Prudential Hong Kong', 'LANNEBO EMERGING MARKETS': 'Lannebo Asset Mgt', 'OHMAN ETISK EMERGING MARKET': 'Lannebo Asset Mgt', 'RICHARD TANOTO': 'Richard Tanoto', 'RADITYA DIKA ANGKASA PUTRA': 'Raditya Dika Angkasa Putra', 'LIDYA LAURENCE': 'Lidya Laurence', 'MARGARETHA PRATIWI S': 'Margaretha Pratiwi S', 'ERICK KRISTIANTO ADITYATAMA': 'Erick Kristianto Adityatama', 'JOHAN': 'Johan', 'NICHOLAS JOSHUA WEEKS': 'Nicholas Joshua Weeks', 'DR KARINA': 'Dr Karina', 'MARTINUS': 'Martinus', 'DANA PENSIUN MANFAAT PASTI UNILEVER INDONESIA': 'Dana Pensiun Manfaat Pasti Unilever Indonesia', 'GIDION HASAN': 'Gidion Hasan', 'JULIANTO WIDJAJA': 'Julianto Widjaja', 'ZIF AKTIEN EMERGING MARKETS PASSIV': 'DWS Investments', 'EDDY SUSANTO': 'Eddy Susanto', 'WIDAYANTI DARMAWAN': 'Widayanti Darmawan', 'BAYERNINVEST KAPITALVERWALTUNGSGESELLSCHAFT MBH': 'BayernInvest', 'MICHAEL PAUL': 'Michael Paul', 'FRANKLIN LIBERTYSHARES ICAV': 'Franklin Templeton Investments', 'FRANKLIN TEMPLETON ICAV': 'Franklin Templeton Investments', 'FRANKLIN CLEARBRIDGE EMERGING MARKETS FUND': 'Franklin Templeton Investments', 'NG LIAN HO': 'Ng Lian Ho', 'PT PELAYARAN MARINDO PACIFIC': 'PT Pelayaran Marindo Pacific', 'MACQUARIE TRUE INDEX EMERGING MARKETS FUND': 'Macquarie Asset Mgt', 'PRINCIPAL MPF ASIA PACIFIC EX JAPAN EQUITY FUND': 'Principal Asset Mgt', 'PRINCIPAL TRUST COMPANY ASIA LIMITED': 'Principal Asset Mgt', 'BOENNARIO NYOTOSETIADI': 'Boennario Nyotosetiadi', 'PT ERINDO MULIA PRATAMA': 'Pt Erindo Mulia Pratama', 'RUDY SETIA WIBISONO SH': 'Rudy Setia Wibisono Sh', 'NINA MERADONA': 'Nina Meradona', 'SALLY SINGGIH': 'Sally Singgih', 'NN PARAPLUFONDS 1 N V': 'Goldman Sachs Asset Mgt', 'SUTOMO': 'Sutomo', 'SOEWITO': 'Soewito', 'TATANG BUDIONO TJAHAJANTO': 'Tatang Budiono Tjahajanto', 'DANIEL ALEXANDER SANTOSA': 'Daniel Alexander Santosa', 'HENDRY AGUS TAN': 'Hendry Agus Tan', 'COLUMBIA RESEARCH ENHANCED EMERGING ECONOMIES ETF': 'Columbia Threadneedle Investments', 'COLUMBIA INTERNATIONAL DIVIDEND INCOME FUND': 'Columbia Threadneedle Investments', 'COLUMBIA EMERGING MARKETS CORE EX CHINA ETF': 'Columbia Threadneedle Investments', 'HOSTPLUS POOLED SUPERANNUATION TRUST': 'IFM Investors', 'HARTONO WIDJAJA': 'Hartono Widjaja', 'FERRY YOGIANTO': 'Ferry Yogianto', 'ILHAM LIE': 'Ilham Lie', 'KSD MIRAE ASSET SECURITIES': 'Mirae Asset Securities', 'HORACIUM': 'Horacium', 'CATUR BUDI HARTO': 'Catur Budi Harto', 'DINO LEO': 'Dino Leo', 'FUTURE FUND BOARD OF GUARDIANS': 'Future Fund', 'YULIANA TARIGAN': 'Yuliana Tarigan', 'MOLEONOTO': 'Moleonoto', 'ANDY WIJAYA CHANDRA': 'Andy Wijaya Chandra', 'WILIWATI': 'Wiliwati', 'BAMBANG SOEPENO': 'Bambang Soepeno', 'YAMIN WIDJAJA': 'Yamin Widjaja', 'PERSEK DAPENMA PAMSI': 'Persek Dapenma Pamsi', 'TJIANG LIKSON CHANDRA': 'Tjiang Likson Chandra', 'WILLY PURNAMA CHANDRA': 'Willy Purnama Chandra', 'HENDRIYANTO HALIM': 'Hendriyanto Halim', 'DONNY CAHYADI FOENG': 'Donny Cahyadi Foeng', 'SIMON MARK DELICATO': 'Simon Mark Delicato', 'FERDI PURNAMA': 'Ferdi Purnama', 'REKSA DANA PNM SAHAM UNGGULAN': 'PNM Investment Mgt', 'REKSA DANA PNM SAHAM AGRESIF': 'PNM Investment Mgt', 'KEVIN WUSMAN': 'Kevin Wusman', 'SUHANDI KOSASIH': 'Suhandi Kosasih', 'LOO PANG KEE': 'Loo Pang Kee', 'SUWANTO': 'Suwanto', 'RONNY KURNAWAN': 'Ronny Kurnawan', 'SLAMET SUBYAKTO': 'Slamet Subyakto', 'NORDEA 1 SICAV': 'Nordea Investment Mgt', 'MAKMUR ROESLI': 'Makmur Roesli', 'PT MITRA MAJU INVESTAMA': 'Pt Mitra Maju Investama', 'TJUK SUGIYONO': 'Tjuk Sugiyono', 'HARY DJAJA': 'Hary Djaja', 'VIVIANA DYAH AYU R K': 'Viviana Dyah Ayu R K', 'PINEBRIDGE WORLD EQUITY OPEN': 'PineBridge Investments', 'KARTONO LEMBONO': 'Kartono Lembono', 'SULAIMAN ARIF ARIANTO': 'Sulaiman Arif Arianto', 'JAHAYA': 'Jahaya', 'MARTIN LUTHER': 'Martin Luther', 'INDRA KUSUMA': 'Indra Kusuma', 'WILLY MINDJAJA': 'Willy Mindjaja', 'PREMIER MITON DIVERSIFIED INCOME FUND': 'Premier Miton Investors', 'EVA MARIANI': 'Eva Mariani', 'CAPITAL ASEAN FUND': 'Capital Investment Trust Corporation', 'ABDUL RAHMAN NASUTION': 'Abdul Rahman Nasution', 'TAN SIAUW DJIANG ALS ANDI TJIPTO': 'Tan Siauw Djiang Als Andi Tjipto', 'YULIANA TAN': 'Yuliana Tan', 'BAMBANG SOERJADI ALI': 'Bambang Soerjadi Ali', 'ALPS EMERGING SECTOR DIVIDEND DOGS ETF': 'SS&C ALPS Advisors', 'RAKUTEN EMERGING MARKETS EQUITY INDEX MOTHER FUND': 'Rakuten Investment Mgt', 'AGUS SUDIARTO': 'Agus Sudiarto', 'PT DIMA INVESTINDO': 'PT Dima Investindo', 'AMAM SUKRIYANTO': 'Amam Sukriyanto', 'DAVID ISKANDAR': 'David Iskandar', 'TRIAWAN MUNAF': 'Triawan Munaf', 'MONICA ARISTA': 'Monica Arista', 'FX BOEDI SOEBIANTO': 'Fx Boedi Soebianto', 'HENDRIK': 'Hendrik', 'RUDY': 'Rudy', 'ARIFIN': 'Arifin', 'NINIK MURNI AYU HARTANTO': 'Ninik Murni Ayu Hartanto', 'BERNY GOMULYA': 'Berny Gomulya', 'YOSSY ANDREW GIRSANG': 'Yossy Andrew Girsang', 'MARINUS COSTAN': 'Marinus Costan', 'RANDI ANTO': 'Randi Anto', 'WIDYA KASUMA LAWRENZI': 'Widya Kasuma Lawrenzi', 'ALFIAN SUHARGO': 'Alfian Suhargo', 'WILLY': 'Willy', 'RD MAJORIS USD BALANCE INDONESIA': 'Majoris Asset Mgt', 'MAJORIS SAHAM ALOKASI DINAMIK': 'Majoris Asset Mgt', 'REKSA DANA INDEKS MAJORIS PEFINDO I GRADE ETF INDONESIA': 'Majoris Asset Mgt', 'SYAFEI': 'Syafei', 'BAMBANG SUSILO HALIM': 'Bambang Susilo Halim', 'INDRA UTOYO MSC': 'Indra Utoyo Msc', 'YUTI WIHARDJA': 'Yuti Wihardja', 'TJONG SWIE KOEI': 'Tjong Swie Koei', 'ARGA MAHANANA NUGRAHA': 'Arga Mahanana Nugraha', 'OPSEU PENSION PLAN TRUST FUND': 'OPSEU Pension Trust', 'AGUS WINARDONO': 'Agus Winardono', 'EMY TRI HARYANI': 'Emy Tri Haryani', 'AMUNDI MODERATE GROWTH FUND': 'Amundi Hong Kong', 'LIAUW RHEITINIA': 'Liauw Rheitinia', 'LUSI INGAWATI HADITORO': 'Lusi Ingawati Haditoro', 'GINDRA TARDY': 'Gindra Tardy', 'PT DWIMURIA INVESTAMA ANDALAN 2': 'PT Dwimuria Investama Andalan 2', 'DEEPAK PRITAMDAS GURBANI': 'Deepak Pritamdas Gurbani', 'ENGELBERT WINOTO': 'Engelbert Winoto', 'ROSNA MARLYANI HARTOJO': 'Rosna Marlyani Hartojo', 'IR ANDREAS': 'Ir Andreas', 'GEMA MERDEKA GOEYARDI': 'Gema Merdeka Goeyardi', 'MARGARETHA PRATIWI SEKARWATI': 'Margaretha Pratiwi Sekarwati', 'MERLIN FIDELIS EMERGING MARKETS FUND': 'Merlin Fidelis Asset Mgt', 'SHERLYN TEWIRA': 'Sherlyn Tewira', 'YUSTINA': 'Yustina', 'SUMANTRI SLAMET': 'Sumantri Slamet', 'IRWAN RINALDI SE': 'Irwan Rinaldi Se', 'SUINDIYO': 'Suindiyo', 'RATHBONE SICAV GLOBAL EMERGING MARKETS EQUITY FUND': 'Rathbone Investment Mgt', 'ROBERT NAPITUPULU': 'Robert Napitupulu', 'SC EDISON NUCLEAR FACILITIES QUALIFIED CPUC DECOMISIONED MASTER TRUST': 'PanAgora Asset Mgt', 'PACIFIC GAS AND ELECTRIC NUCLEAR FACILITIES QUALIFIED DECOMMISSIONING TRUST': 'PanAgora Asset Mgt', 'MOHAMMAD SOFYAN': 'Mohammad Sofyan', 'KENNY JUMICO': 'Kenny Jumico', 'SANTIKA TANTRA SETIAWAN': 'Santika Tantra Setiawan', 'JIMMY WISAN': 'Jimmy Wisan', 'ENNI SURYANI': 'Enni Suryani', 'ROBERT WIDJAYA': 'Robert Widjaya', 'DENNY': 'Denny', 'ROBINSON SITORUS': 'Robinson Sitorus', 'JONI': 'Joni', 'ANTON': 'Anton', 'PELAYARAN NASIONAL SINAR PAGODA': 'Pelayaran Nasional Sinar Pagoda', 'PT OPTIK TUNGGAL SEMPURNA': 'Pt Optik Tunggal Sempurna', 'SUGIANTO TJHIN': 'Sugianto Tjhin', 'TEO USMAN EFENDI': 'Teo Usman Efendi', 'AHMAD ASKANDAR': 'Ahmad Askandar', 'HONG KIE HONG': 'Hong Kie Hong', 'DAVID': 'David', 'DAVID GUNAWAN': 'David Gunawan', 'YULIANA MONDONG': 'Yuliana Mondong', 'SENTOT PRASETYO': 'Sentot Prasetyo', 'RIO VALENTINO': 'Rio Valentino', 'MARSONO': 'Marsono', 'BENNI SUSANTO': 'Benni Susanto', 'FRANZ PURNOMO': 'Franz Purnomo', 'KOMANG ANI SUSANA': 'Komang Ani Susana', 'BAMBANG HERUAWAN H': 'Bambang Heruawan H', 'YOUNG THOMAS CHUN': 'Young Thomas Chun', 'EKA SETYA ADRIANTO SE': 'Eka Setya Adrianto Se', 'HENDRIK BIMA HERAWAN': 'Hendrik Bima Herawan', 'LIEN TJHAI HOAN': 'Lien Tjhai Hoan', 'ARIEL DARMAWAN': 'Ariel Darmawan', 'LEVINA LAWRENT KUSUMA': 'Levina Lawrent Kusuma', 'RUDY HADI SENTOSA': 'Rudy Hadi Sentosa', 'JUANO SIMKOPUTERA': 'Juano Simkoputera', 'ALEXANDER SENJAYA': 'Alexander Senjaya', 'WIEKE POEJIASTUTI': 'Wieke Poejiastuti', 'JAPAN POLICE PERSONNEL MUTUAL AID ASSOCIATION': 'Resona Bank', 'INDRAWATI': 'Indrawati', 'VOYA VACS INDEX SERIES I PORTFOLIO': 'Voya Investment Mgt', 'PT ASURANSI MITRA PELINDUNG MUSTIKA': 'PT Asuransi Mitra Pelindung Mustika', 'WIBISONO': 'Wibisono', 'MELANNY PRAMONO': 'Melanny Pramono', 'DINO': 'Dino', 'HENDRY': 'Hendry', 'DWIJANTO GONDOKUSUMO': 'Dwijanto Gondokusumo', 'I MADE SUTARNA': 'I Made Sutarna', 'SIS APIK WIJAYANTO': 'Sis Apik Wijayanto', 'HERIYANTO': 'Heriyanto', 'DARWO': 'Darwo', 'DJOKOSANTOSO MOELJONO': 'Djokosantoso Moeljono', 'WIJONO TANOKO': 'Wijono Tanoko', 'BUDI SULISTYO': 'Budi Sulistyo', 'AMF AKTIEFOND TILLV XTMARKNADER': 'AMF Pensions', 'BUDI PRIYONO': 'Budi Priyono', 'WIDYA': 'Widya', 'LUKMAN SE MM': 'Lukman Se Mm', 'SUSY LIESTIOWATY': 'Susy Liestiowaty', 'MD INTERNATIONAL VALUE FUND': 'Jensen Investment Mgt', 'DANIEL IKI': 'Daniel Iki', 'SYALOM': 'Syalom', 'SANTOSO': 'Santoso', 'GIOVANIS TIANIZAR': 'Giovanis Tianizar', 'HYACINTHA SUSANTI JAHJA': 'Hyacintha Susanti Jahja', 'HANIFAH': 'Hanifah', 'HERESMAN BONG': 'Heresman Bong', 'BENNY SOECIANTO': 'Benny Soecianto', 'ERIC SAPUTRO': 'Eric Saputro', 'ANDRIJANTO': 'Andrijanto', 'DARMAWAN SANTOSO': 'Darmawan Santoso', 'HERI SUPRIADI': 'Heri Supriadi', 'AMUNDI INDONESIA FUND': 'Amundi Singapore', 'YURICHO ALEXANDER YOGIANTO': 'Yuricho Alexander Yogianto', 'LAUDHI ADRIAN SUSILO': 'Laudhi Adrian Susilo', 'EDI SANTOSA': 'Edi Santosa', 'JOHANNES SETOJONO': 'Johannes Setojono', 'YUWONO SUMODIJOYO AND BENG YOE T': 'Yuwono Sumodijoyo & Beng Yoe T', 'IVAN EFFENDY T': 'Ivan Effendy T', 'ADI HALIEM': 'Adi Haliem', 'BILLY SABARTO': 'Billy Sabarto', 'KENNY ANTONIO RULLY S': 'Kenny Antonio Rully S', 'ARISTIANTO': 'Aristianto', 'ANTHONY WIJAYA': 'Anthony Wijaya', 'EDY GOENAWAN': 'Edy Goenawan', 'PT ARTHAWENASAKTI GEMILANG': 'Pt Arthawenasakti Gemilang', 'TAN GUNAWAN TANWIJAYA': 'Tan Gunawan Tanwijaya', 'ARCA FONDI SOCIETA DI GESTIONE DEL RISPARMIO SPA': 'Arca Fondi SGR', 'A M NOVA CHRISTIANA': 'A M Nova Christiana', 'KWA BENG BENG': 'Kwa Beng Beng', 'KENNY ROESLI': 'Kenny Roesli', 'PT INOCREA INDONESIA': 'PT Inocrea Indonesia', 'ROBET ROBINSON': 'Robet Robinson', 'KURNIA JAYA SUTANA': 'Kurnia Jaya Sutana', 'PT BERKAT MITRA PATRIA': 'PT Berkat Mitra Patria', 'WIDYAWATI': 'Widyawati', 'OKTARINI': 'Oktarini', 'MARGARETA': 'Margareta', 'HENGKY WIJAYA': 'Hengky Wijaya', 'ALIUYANTO': 'Aliuyanto', 'LOUIS THEOSABRATA': 'Louis Theosabrata', 'AGUS NOORSANTO': 'Agus Noorsanto', 'DRS JAHJA SANTOSO': 'Drs. Jahja Santoso', 'PT INDOTARA PERSADA': 'PT Indotara Persada', 'JOHNY USMAN': 'Johny Usman', 'SOEHARMIN ICHSAN': 'Soeharmin Ichsan', 'PT INDOVESTA UTAMA MANDIRI': 'PT Indovesta Utama Mandiri', 'JOHAN SUTONO': 'Johan Sutono', 'SUSANTI': 'Susanti', 'TEGUH BUDI PAMUNGKAS SE MM': 'Teguh Budi Pamungkas SE MM', 'PT VICTORINDO INVESTAMA SURYA ARTHA': 'Pt Victorindo Investama Surya Artha', 'TD EMERALD EMERGING MARKETS INDEX POOLED FUND TRUST': 'TD Asset Mgt', 'HALIM SUDARJANTO': 'Halim Sudarjanto', 'VERDIPAPIRFONDET HOLBERG VEKSTMARKEDER': 'Holberg Fondene', 'HENDY ROSWANDY SE': 'Hendy Roswandy Se', 'EDY SUSANTO': 'Edy Susanto', 'YANTO': 'Yanto', 'WIDJAJA KOSWARA': 'Widjaja Koswara', 'LIM GANI EFFENDY': 'Lim Gani Effendy', 'RITA LACHMAN BHARWANI': 'Rita Lachman Bharwani', 'HENGKY KARTAWINATA': 'Hengky Kartawinata', 'DANICA': 'Danica', 'SOEWARNO ALMASIH': 'Soewarno Almasih', 'HENDRA KO': 'Hendra Ko', 'MICHAEL WIJAYA': 'Michael Wijaya', 'WIRA LAJU REJEKI': 'Wira Laju Rejeki', 'JOHANNES SETIJONO': 'Johannes Setijono', 'MAGDALENA BONARIA': 'Magdalena Bonaria', 'SEPPALGA AHMAD': 'Seppalga Ahmad', 'LILYANA': 'Lilyana', 'TEDDY KURNIA TJANDRA': 'Teddy Kurnia Tjandra', 'INVESCO DEVELOPING MARKET CLASS': 'Invesco (OppenheimerFunds)', 'APRIANTO': 'Aprianto', 'ARVIN ADIPUTRA': 'Arvin Adiputra', 'JOVIN CHANDRA TAN': 'Jovin Chandra Tan', 'RUMPIN RUSLI': 'Rumpin Rusli', 'TONO': 'Tono', 'AMIR SUDJONO': 'Amir Sudjono', 'WILSON GUNAWAN': 'Wilson Gunawan', 'SANTOSA': 'Santosa', 'RICHARD NELSON MANURUNG IR': 'Richard Nelson Manurung Ir', 'ANTON HARTONO ANSARI': 'Anton Hartono Ansari', 'BUDIANTO': 'Budianto', 'WANDY HARTANTO': 'Wandy Hartanto', 'AMIR HAMZAH': 'Amir Hamzah', 'MELIA BUDIMAN DRA': 'Melia Budiman Dra', 'INTAN SARI': 'Intan Sari', 'SUGENG TJAHJONO': 'Sugeng Tjahjono', 'FRANK RUSSELL INVESTMENTS JAPAN INTERNATIONAL EQUITY FUND': 'Russell Investments', 'PRANOTO': 'Pranoto', 'PRASETYA SETIAWAN': 'Prasetya Setiawan', 'SWANDANA WITANANTA SEJAHTERA': 'Swandana Witananta Sejahtera', 'TAN BUDIONO TEJA': 'Tan Budiono Teja', 'DENNY HARYONO': 'Denny Haryono', 'IWAN SANTOSA KUSWANTO': 'Iwan Santosa Kuswanto', 'MOHAMAD SUBALI': 'Mohamad Subali', 'BUN KHIM BUN SE': 'Bun Khim Bun Se', 'SUWANDI SUHERMAN': 'Suwandi Suherman', 'ONG MADIAN': 'Ong Madian', 'ADHI SUCITA': 'Adhi Sucita', 'MASHADI': 'Mashadi', 'HENDY': 'Hendy', 'LEONARD': 'Leonard', 'IRA PUSPADEWI': 'Ira Puspadewi', 'WASITO PRAMONO': 'Wasito Pramono', 'INSPIRE GLOBAL HOPE ETF': 'Inspire Advisors', 'WELLIJ ICHWAN NOORJONO': 'Wellij Ichwan Noorjono', 'SIE HIOK': 'Sie Hiok', 'JUNIWATI': 'Juniwati', 'WIWIN SUMARTINI': 'Wiwin Sumartini', 'SELIGSON AND CO EMERGING MARKETS FUND': 'LocalTapiola Asset Mgt', 'LYDIA': 'Lydia', 'RISMA ERAWATI SINAGA': 'Risma Erawati Sinaga', 'POLYANSKI CAKRA KESUMA': 'Polyanski Cakra Kesuma', 'RATNAWATI TANIARDJA': 'Ratnawati Taniardja', 'MARYANTO': 'Maryanto', 'HANDAYANI TJANDRASA DR IR': 'Handayani Tjandrasa Dr Ir', 'SUKARDI': 'Sukardi', 'ANTHONIUS HEROE W SH': 'Anthonius Heroe W Sh', 'ELDA CHRISTIAN TIARAWATY C': 'Elda Christian Tiarawaty C', 'AGUS SALIM': 'Agus Salim', 'ALEXANDER FULBERTUS': 'Alexander Fulbertus', 'I GEDE ALIT ARDIMAYASA': 'I Gede Alit Ardimayasa', 'I GEDE SULIARTHA': 'I Gede Suliartha', 'HADDY TIMOR': 'Haddy Timor', 'RUDY BONGSO': 'Rudy Bongso', 'PRINCIPAL GLOBAL INVESTORS COLLECTIVE INVESTMENT TRUST': 'Principal Global Investors', 'PRINCIPAL FUNDS INC INTERNATIONAL EMERGING MARKETS FUND': 'Principal Global Investors', 'CATHOLIC RESPONSIBLE INVESTMENTS INTERNATIONAL EQUITY FUND': 'Principal Global Investors', 'PRINCIPAL VARIABLE CONTRACTS FUNDS INC DIVERSIFIED INTERNATIONAL ACCOUNT': 'Principal Global Investors', 'YOSEPH MARIO YAPPY': 'Yoseph Mario Yappy', 'PINEBRIDGE INTERNATIONAL FUNDS ACORNS OF ASIA BALANCED FUND': 'PineBridge Investments', 'PINEBRIDGE SOUTH EAST ASIA FUND': 'PineBridge Investments', 'PINEBRIDGE ASIAN FUND': 'PineBridge Investments', 'AIG ASIA BALANCED FUND': 'PineBridge Investments', 'ELFRIDA ELENY': 'Elfrida Eleny', 'DONI MS RAMADAN': 'Doni Ms Ramadan', 'SHENDY LUKITO H': 'Shendy Lukito H', 'OEI SETIAWAN WINARTO': 'Oei Setiawan Winarto', 'ELYNA THERASITA': 'Elyna Therasita', 'CSOP FTSE ASIA PACIFIC SELECT INDEX ETF': 'CSOP Asset Mgt', 'FENNI YANTY': 'Fenni Yanty', 'UNIVERSITAS GADJAH MADA': 'Universitas Gadjah Mada', 'FONNY': 'Fonny', 'ARIF BAHARUDIN': 'Arif Baharudin', 'PT SUARA VISUAL INDONESIA': 'PT Suara Visual Indonesia', 'EDWIN WIBOWO': 'Edwin Wibowo', 'AMINA': 'Amina', 'RICKY': 'Ricky', 'DJAUHAR MAULIDI': 'Djauhar Maulidi', 'JANNI': 'Janni', 'REKSA DANA AURORA SMC EQUITY': 'Aurora Asset Mgt', 'CHAN CHON CHI': 'Chan Chon Chi', 'JUSUF BARLI': 'Jusuf Barli', 'HENRIETTA K': 'Henrietta K', 'FRANKLIN TEMPLETON INVESTMENT FUND CLEARBRIDGE UK EQUITY INCOME FUND': 'Franklin Templeton Investments', 'AGUNG PRABOWO': 'Agung Prabowo', 'METZLER ASIAN BALANCED INVESTMENTS FUND': 'Metzler Asset Mgt', 'MI FONDS K18 KAE SPK': 'Metzler Asset Mgt', 'WM POOL EQUITIES TRUST NO 75': 'INTECH Investment Mgt', 'ERNA INDRIANY': 'Erna Indriany', 'MIGROS PENSIONSKASSE FONDS AKTIEN WELT': 'Migros Bank', 'YANTONI KERISNA': 'Yantoni Kerisna', 'TIRTO WIGUNO': 'Tirto Wiguno', 'WINFIELD': 'Winfield', 'NG JOHANES WIJAYA': 'Ng Johanes Wijaya', 'JIO ENG AN': 'Jio Eng An', 'RICO': 'Rico', 'FINREON WORLD EQUITY ECONOMIC EXPSR PENSION': 'Finreon', 'FINREON EMERGING MARKETS EQUITY ISOPRO': 'Finreon', 'YUDHI IRAWAN AMIN WIJAYA': 'Yudhi Irawan Amin Wijaya', 'HALIMAN SANTO': 'Haliman Santo', 'RONNY HERMAWAN ANDOYO': 'Ronny Hermawan Andoyo', 'PRINCIPAL CHINA INDIA INDONESIA OPPORTUNITIES FUND': 'Principal Asset Mgt', 'HARTA BAYUMAS': 'Principal Asset Mgt', 'CIMB PRINCIPAL ASSET MANAGEMENT BERHAD': 'Principal Asset Mgt', 'PRINCIPAL ASEAN DYNAMIC FUND': 'Principal Asset Mgt', 'SUCIPTO': 'Sucipto', 'MICHAEL STEVEN': 'Michael Steven', 'AUGUSTINA SIMANJUNTAK': 'Augustina Simanjuntak', 'YULIUS KURNIAWAN': 'Yulius Kurniawan', 'REGINA SARAH TAMBUNAN': 'Regina Sarah Tambunan', 'YONGKIE HANDAYA': 'Yongkie Handaya', 'JOHANSYAH': 'Johansyah', 'SAIFUL ANWAR': 'Saiful Anwar', 'TJIOE JOHNNY': 'Tjioe Johnny', 'PHILIP AGUSTINUS P TRISNADI': 'Philip Agustinus P Trisnadi', 'WURI RATNASARI': 'Wuri Ratnasari', 'FRANKLIN EMERGING MARKET CORE DIVIDEND TILT INDEX ETF': 'Franklin Templeton Investments', 'FRANKLIN FTSE ASIA EX JAPAN ETF': 'Franklin Templeton Investments', 'FRANKLIN EMERGING MARKETS EQUITY INDEX ETF': 'Franklin Templeton Investments', 'FRANKLIN LIBERTYQ EMERGING MARKETS ETF': 'Franklin Templeton Investments', 'JAUW DHANNY KURNIAWAN': 'Jauw Dhanny Kurniawan', 'INDUK KOPERASI KEPOLISIAN': 'Induk Koperasi Kepolisian', 'VINSEN ALFREDO ARYA': 'Vinsen Alfredo Arya', 'TJHAI LIUK NEN': 'Tjhai Liuk Nen', 'PT BINTANG DELAPAN HARMONI': 'PT Bintang Delapan Harmoni', 'GUNUNG PAHALA TUMPAL S IR': 'Gunung Pahala Tumpal S, Ir', 'THOMAS TJAHAJANTO': 'Thomas Tjahajanto', 'KARJANTO TJUGITO': 'Karjanto Tjugito', 'TAN SOEI LIE': 'Tan Soei Lie', 'HAZQIL ARAFI': 'Hazqil Arafi', 'HENDRA PANGESTU': 'Hendra Pangestu', 'CLIFF SINTITI TAN': 'Cliff Sintiti Tan', 'CHRISTIAN ANGKY ANG': 'Christian Angky Ang', 'WAHID BROTO KUNCORO': 'Wahid Broto Kuncoro', 'TOMY WIJAYA': 'Tomy Wijaya', 'REINALDO JIFFRI': 'Reinaldo Jiffri', 'MUHAMMAD CANDRA UTAMA': 'Muhammad Candra Utama', 'ONG SIEN HWA': 'Ong Sien Hwa', 'JACKSON': 'Jackson', 'HENRI WIJAYA': 'Henri Wijaya', 'YULIANA': 'Yuliana', 'DRA MARIA GORETTI TINAWATI': 'Dra Maria Goretti Tinawati', 'EDY HERLAMBANG DR': 'Edy Herlambang Dr', 'ROBI CAHYADI': 'Robi Cahyadi', 'HENRY SATEX': 'Henry Satex', 'CEPI JAMALUDIN MALIK': 'Cepi Jamaludin Malik', 'LEONARD AGUS SETIAWAN': 'Leonard Agus Setiawan', 'SUMIATI': 'Sumiati', 'JUNQI YI': 'Junqi Yi', 'TONNY TANUWIJAYA': 'Tonny Tanuwijaya', 'PT MINRIKO TOUVEL': 'Pt Minriko Touvel', 'ANTON SULISTYO': 'Anton Sulistyo', 'AGUNG DWI SANDI': 'Agung Dwi Sandi', 'DARIUS ISKANDAR': 'Darius Iskandar', 'AMP WHOLESALE EMERGING MARKETS FUND': 'Guardian Trust', 'TEDDY DJULIANTO': 'Teddy Djulianto', 'IVONNE OLIVIA INDRADIHARDJO': 'Ivonne Olivia Indradihardjo', 'DJOHAN TEGUH SUGIANTO': 'Djohan Teguh Sugianto', 'LIANG PHO LENG': 'Liang Pho Leng', 'DJOHAN RIMBA': 'Djohan Rimba', 'LISA WIDIASARI ATMADJA': 'Lisa Widiasari Atmadja', 'DEDY ZULKARNAEN': 'Dedy Zulkarnaen', 'SU FIE': 'Su Fie', 'HARI WAHYUDI': 'Hari Wahyudi', 'SUGIHARTO OETOMO': 'Sugiharto Oetomo', 'FERDI CANDRA': 'Ferdi Candra', 'DJOHAN EMIR SETIJOSO': 'Djohan Emir Setijoso', 'JUANDY ST': 'Juandy St', 'GAMSEN': 'Gamsen', 'SUYANTI': 'Suyanti', 'AGUS SUWANTO': 'Agus Suwanto', 'UMI HARYATI': 'Umi Haryati', 'LINDA SIDHARTA': 'Linda Sidharta', 'HENGKY GOENAWAN HENOEK S E': 'Hengky Goenawan Henoek S E', 'YONGKY MARGONO': 'Yongky Margono', 'LIM HARRY MURPHY': 'Lim Harry Murphy', 'I GEDE WIREYASA': 'I Gede Wireyasa', 'FILIPUS HUNADI SETIADI': 'Filipus Hunadi Setiadi', 'SARMAULI SITORUS': 'Sarmauli Sitorus', 'EDDY GUNAWAN': 'Eddy Gunawan', 'TJOA HAN TJIANG': 'Tjoa Han Tjiang', 'HENDRA JUTAWAN': 'Hendra Jutawan', 'LEGINO HALIM': 'Legino Halim', 'HANDY SUYONO': 'Handy Suyono', 'STEVEN': 'Steven', 'ALI WIYOGO': 'Ali Wiyogo', 'ARROW DOW JONES GLOBAL YIELD ETF': 'Arrow Investment Advisors', 'HARI SIAGA AMIJARSO': 'Hari Siaga Amijarso', 'M ICHSAN': 'M Ichsan', 'AGUS SETIAWAN': 'Agus Setiawan', 'HANDOJO TUMBELAKA': 'Handojo Tumbelaka', 'COOPER INVESTORS ASIAN EQUITY FUND': 'Cooper Investors', 'HENDRI EFFENDI': 'Hendri Effendi', 'EKO HANDONO TJAHJO': 'Eko Handono Tjahjo', 'IMRON': 'Imron', 'L SURYAWIDJAJA DJUHADI': 'L Suryawidjaja Djuhadi', 'ONG BU TJIONG': 'Ong Bu Tjiong', 'DIOCESE OF ARLINGTON': 'Federated Hermes', 'FEDERATED HERMES GLOBAL ALLOCATION FUND': 'Federated Hermes', 'RIYAN PERMADI': 'Riyan Permadi', 'MARIANI': 'Mariani', 'EDDY TIO': 'Eddy Tio', 'HIRONIMUS': 'Hironimus', 'AXEL ILHAM GUNARDI': 'Axel Ilham Gunardi', 'HENNY WIDYAWATI OEI': 'Henny Widyawati Oei', 'ASAN ROSMINI': 'Asan Rosmini', 'RAHMAT SUMENGKAR ANHAR': 'Rahmat Sumengkar Anhar', 'RONALD CAESAR RIO': 'Ronald Caesar Rio', 'JURIKA NATA': 'Jurika Nata', 'ROMMY HARTONO THEOS': 'Rommy Hartono Theos', 'PT METROCOM INDONESIA': 'PT Metrocom Indonesia', 'ANTON BUDIMAN': 'Anton Budiman', 'YOHAN SUGANDA': 'Yohan Suganda', 'FIRDAUS SALIM': 'Firdaus Salim', 'MOCH IRFAN': 'Moch Irfan', 'LIE JIMMY SUNARYO': 'Lie Jimmy Sunaryo', 'BENI WONGSO': 'Beni Wongso', 'DJOKO WINATA': 'Djoko Winata', 'FANDI HERMAWAN': 'Fandi Hermawan', 'TJOANDI BUNGSU SANTOSO': 'Tjoandi Bungsu Santoso', 'AINY': 'Ainy', 'LULUK INDAHSYAH': 'Luluk Indahsyah', 'JOKO WAHYONO': 'Joko Wahyono', 'MUZAKKIR ZAM ZAM': 'Muzakkir Zam Zam', 'FREDDI SUMADI HIU': 'Freddi Sumadi Hiu', 'DIFEN': 'Difen', 'KWA KIAN SAN': 'Kwa Kian San', 'ENDANG ARIFIN': 'Endang Arifin', 'ALI TARIGAN': 'Ali Tarigan', 'EDDY SUTOMO': 'Eddy Sutomo', 'RISWINANDI': 'Riswinandi', 'IE ANDRE IRAWAN': 'IE Andre Irawan', 'PAWAN NANDWANI': 'Pawan Nandwani', 'DAVID ETENG': 'David Eteng', 'HENNY SARI DEWI IRAWAN': 'Henny Sari Dewi Irawan', 'ABBAS': 'Abbas', 'RYAN ADIPUTRA': 'Ryan Adiputra', 'EDHI SOESANTO': 'Edhi Soesanto', 'AMIEN MAKRUF': 'Amien Makruf', 'FANDRAWATI WIDJOJO': 'Fandrawati Widjojo', 'YUWONO SUMODIJOYO': 'Yuwono Sumodijoyo', 'SWANDY HALIM MSC': 'Swandy Halim MSC', 'INDJUN': 'Indjun', 'DAVID MAK': 'David Mak', 'JOHN CORNELIUS YONATAN': 'John Cornelius Yonatan', 'YOYO': 'Yoyo', 'TJEN FOENG FA': 'Tjen Foeng Fa', 'KHOIRUL HADI': 'Khoirul Hadi', 'CHRIS CHANDRA': 'Chris Chandra', 'GODFREY LIUS': 'Godfrey Lius', 'TORAS S PANGGABEAN SH': 'Toras S Panggabean Sh', 'MARTIN TEH': 'Martin Teh', 'TEMY': 'Temy', 'LINA SARI': 'Lina Sari', 'HANSEL SOSRO SAPUTRO': 'Hansel Sosro Saputro', 'EDDY JUNAEDY': 'Eddy Junaedy', 'NINA': 'Nina', 'BAMBANG SETYAWAN DJOJO': 'Bambang Setyawan Djojo', 'FLOURISH INVESTMENT CORPORATION CIC SMART BETA': 'China Investment Corporation', 'KAMARI': 'Kamari', 'DAVIN ALDO ANDRIANTO': 'Davin Aldo Andrianto', 'FREDY TEJA NG': 'Fredy Teja Ng', 'KESEJAHTERAAN PEGAWAI PERUM PERHUTANI JATI SEJAHTERA': 'Yay Kesejahteraan Pegawai Perum Perhutani Jati Sejahtera', 'HADI WIBOWO': 'Hadi Wibowo', 'TJING HAI': 'Tjing Hai', 'REINHARD WINARTO': 'Reinhard Winarto', 'YOHANES FREDDY WIJAYA': 'Yohanes Freddy Wijaya', 'ROLAND SIAHAAN': 'Roland Siahaan', 'MICHAEL': 'Michael', 'JEFRY HARTANTO': 'Jefry Hartanto', 'BENNY': 'Benny', 'FERRY MARANGIN SIMATUPANG': 'Ferry Marangin Simatupang', 'ERWIN DJOHAN': 'Erwin Djohan', 'ANDI MELDA DARDIANSYAH SOUISSA': 'Andi Melda Dardiansyah Souissa', 'DJOKO PURNOMO': 'Djoko Purnomo', 'ADI GUNAWAN TANZIL': 'Adi Gunawan Tanzil', 'DAHLIA S BAMBANG': 'Dahlia S Bambang', 'BAMBANG EKO P': 'Bambang Eko P', 'ALEXANDER': 'Alexander', 'ARDIANSYAH': 'Ardiansyah', 'IWAN SUGIARTO WIJATMIKO': 'Iwan Sugiarto Wijatmiko', 'PT KURNIA LUWUK SEJATI': 'Pt Kurnia Luwuk Sejati', 'RIANTO RUSLIE': 'Rianto Ruslie', 'ANTO SUBIAKTO': 'Anto Subiakto', 'SUDIRMAN SIMANGUNSONG': 'Sudirman Simangunsong', 'ANDREE SEMIKAR WIJAYA': 'Andree Semikar Wijaya', 'ANDREAS MANIK': 'Andreas Manik', 'WENDIYO KHOMAN': 'Wendiyo Khoman', 'BENTEK HIJANI': 'Bentek Hijani', 'SUTIONO': 'Sutiono', 'AUGUSTINUS BEDJO TJOKRODIHARDJO': 'Augustinus Bedjo Tjokrodihardjo', 'MTBOJ PENSION FUND ASSOCIATION 400039078': 'Sumitomo Mitsui Trust Bank', 'LEO ZAHAR': 'Leo Zahar', 'ASEP SAEPUDIN': 'Asep Saepudin', 'KARTIKA': 'Kartika', 'DWI HENDRY NATHANIEL': 'Dwi Hendry Nathaniel', 'DEWINTA ANGELIA AND GAN': 'Dewinta Angelia & Gan', 'DENAT ARNANDO': 'Denat Arnando', 'LINNA BUDIAWAN': 'Linna Budiawan', 'ALFREDO SETIABUDI': 'Alfredo Setiabudi', 'INCOME INSURANCE LIMITED': 'Fullerton Fund Mgt', 'HASIM SUTIONO': 'Hasim Sutiono', 'BURHAN GUNAWAN KOSIDIN': 'Burhan Gunawan Kosidin', 'TAN HENDY TAIZAR': 'Tan Hendy Taizar', 'DFC EMERGING MARKETS EQUITY FUND': 'LMCG Investments', 'EBENESER GIRSANG': 'Ebeneser Girsang', 'YENNY MAK': 'Yenny Mak', 'ROBBY KURNIAWAN': 'Robby Kurniawan', 'DJONI TJIUNARDI': 'Djoni Tjiunardi', 'PT ASURANSI BRI LIFE': 'PT Asuransi BRI Life', 'MINIATY LAM': 'Miniaty Lam', 'WIBOWO SAPUTRO': 'Wibowo Saputro', 'DANNY EXWANTO': 'Danny Exwanto', 'IR NELSON IDRIS SILABAN': 'Ir Nelson Idris Silaban', 'IGN SUGIAN SINANTO': 'Ign Sugian Sinanto', 'PANGKAL MULTIKARYA': 'Pangkal Multikarya', 'ASSETPLUS SUPER ASIA RICH TOGETHER MASTER INVESTMENT TRUST': 'Assetplus Investment Mgt', 'ASSETPLUS SUPER ASIA PRIVATE EQUITY INVESTMENT TRUST': 'Assetplus Investment Mgt', 'DARMANTO': 'Darmanto', 'YAFINDO GLOBAL INVESTAMA': 'Yafindo Global Investama', 'MARINI RYANI IRAWAN': 'Marini Ryani Irawan', 'KT ISMAIL DJALIMAN': 'KT Ismail Djaliman', 'FATHA BUDI ANUGRAH': 'Fatha Budi Anugrah', 'PT HIT GLOBAL INDONESIA': 'PT Hit Global Indonesia', 'SRI MAWARTI': 'Sri Mawarti', 'CAROLINE KWOK': 'Caroline Kwok', 'SOPHIA CENDANA': 'Sophia Cendana', 'PRABOWO WICAKSONO YUWONO P': 'Prabowo Wicaksono Yuwono P', 'ANTONIUS JUSUF JOHARI': 'Antonius Jusuf Johari', 'IMAM HANAFI': 'Imam Hanafi', 'DENNY SUTOMO': 'Denny Sutomo', 'EMERGING MARKETS EQUITY ACTIVE ETF': 'Russell Investments Implementation Services', 'RUDOLPH J AND DAPHNE A MUNZER FOUNDATION': 'Russell Investments Implementation Services', 'DONGKWAN HAN': 'Dongkwan Han', 'ARISTEA SICAV NEW FRONTIERS EQUITY FUND': 'Banor Capital', 'RATNA SARI SUHARTONO': 'Ratna Sari Suhartono', 'TOMMY YAKIN': 'Tommy Yakin', 'VINI TABADIRI INVESTA PT': 'Vini Tabadiri Investa Pt', 'PT INDONESIA OCEAN TRUCK': 'PT Indonesia Ocean Truck', 'PT GISTEX CHEWON SYNTHETIC': 'Pt Gistex Chewon Synthetic', 'HERRY CHEN': 'Herry Chen', 'IVAN': 'Ivan', 'RATNA WATI': 'Ratna Wati', 'LALA DARMILA': 'Lala Darmila', 'HERMAWAN WIJAYA': 'Hermawan Wijaya', 'TRI CAHYA WISNHU GROHO': 'Tri Cahya Wisnhu Groho', 'ANTARES HANAEDI II': 'Antares Hanaedi II', 'PT PROSPECT MOTOR': 'PT Prospect Motor', 'EDDY SOERYANTO SOEGOTO': 'Eddy Soeryanto Soegoto', 'CELYN ANGELIA': 'Celyn Angelia', 'ALEXANDRY': 'Alexandry', 'BAMBANG PRANOTO': 'Bambang Pranoto', 'TAHIR SAFUDIN': 'Tahir Safudin', 'MUKIT HENDRAYATNO ST': 'Mukit Hendrayatno ST', 'DRS H ARMED WJAYA MH': 'Drs H Armed Wjaya Mh', 'EDISON NAPITUPULU': 'Edison Napitupulu', 'IMELDA FRANCISCA BUDIMAN': 'Prospera Asset Mgt', 'SALEH DJASIT': 'Saleh Djasit', 'YULIANUS': 'Yulianus', 'RAHMAWANI': 'Rahmawani', 'DIMAS WIBISONO': 'Dimas Wibisono', 'DARMAWAN HERLINA': 'Darmawan Herlina', 'FERRY SUSANTO': 'Ferry Susanto', 'NOOR ROCHMAH': 'Noor Rochmah', 'OTTY ANDESITA': 'Otty Andesita', 'PHILIP WIJAYA': 'Philip Wijaya', 'TONAM': 'Tonam', 'HENDRI GUNAWAN': 'Hendri Gunawan', 'SUGIHARTO HUSODO': 'Sugiharto Husodo', 'STEVEN WIJAYA GUNAWAN': 'Steven Wijaya Gunawan', 'HANDY FRANDINATA': 'Handy Frandinata', 'NIE NIE': 'Nie Nie', 'TEGUH PRAMONO': 'Teguh Pramono', 'ADE CHANDRA': 'Ade Chandra', 'ARIEF CHRISTOPHEL SARAGIH': 'Arief Christophel Saragih', 'RABIN INDRAJAD HATTARI': 'Rabin Indrajad Hattari', 'SUWALIM HADI PRATAMA': 'Suwalim Hadi Pratama', 'JUNAIDI': 'Junaidi', 'FIRST SENTIER INVESTORS GLOBAL UMBRELLA FUND PLC RQI GLOBAL VALUE FUND': 'First Sentier Investors', 'RQI GLOBAL FUND': 'First Sentier Investors', 'FIRST SENTIER INVESTORS GLOBAL UMBRELLA FUND PLC': 'First Sentier Investors', 'GUNAWAN SUSANTO': 'Gunawan Susanto', 'PERUM JAMKRINDO': 'Perum Jamkrindo', 'CHARLES LENDRA': 'Charles Lendra', 'MULYADI WIBOWO': 'Mulyadi Wibowo', 'EDWARD THEJASURYA LIM': 'Edward Thejasurya Lim', 'SOEGIANDI': 'Soegiandi', 'MR ANDRIANTO': 'Mr Andrianto', 'YERIKHO': 'Yerikho', 'TEMPA IMANDAFA': 'Tempa Imandafa', 'RATNA LISTIANI SUGITO DRG': 'Ratna Listiani Sugito Drg', 'SURYASIN': 'Suryasin', 'ANTHONIUS': 'Anthonius', 'PT AGUNG BERSATU JAYA': 'Pt Agung Bersatu Jaya', 'EDDY LOLO': 'Eddy Lolo', 'TINNO SUSILO ST': 'Tinno Susilo St', 'PETER KOSASI': 'Peter Kosasi', 'LIU EDI AMAS': 'Liu Edi Amas', 'SRI ELVILA SARI': 'Sri Elvila Sari', 'DARWIN': 'Darwin', 'AMOVA MSCI AC ASIA EX JAPAN EX CHINA INDEX ETF': 'Amova Asset Mgt', 'DEVINA KONATRA': 'Devina Konatra', 'RICKY LAURUS': 'Ricky Laurus', 'MERIANI': 'Meriani', 'ALLSOLUTIONS ETF ICAV UNIACTIVE Q EQUITIES EMERGING MARKETS ENHANCED UCITS ETF': 'Quoniam Asset Mgt', 'SENG BENNY SUSANTO': 'Seng Benny Susanto', 'LO SIE KIAT': 'Lo Sie Kiat', 'RONY RIANTO': 'Rony Rianto', 'IRAWATI GUNAWAN': 'Irawati Gunawan', 'LENA SETIAWATI': 'Lena Setiawati', 'HARIANTO TJAHAYA': 'Harianto Tjahaya', 'TJINDARSAH MINTARNO': 'Tjindarsah Mintarno', 'ARIF RACHMAT': 'Arif Rachmat', 'BUDI PRASETYO': 'Budi Prasetyo', 'KURNIAWAN HALIM': 'Kurniawan Halim', 'OSCAR HUTAGAOL': 'Oscar Hutagaol', 'DEDY HARYADI': 'Dedy Haryadi', 'FERDYNAN JULYADY SETIAWAN': 'Ferdynan Julyady Setiawan', 'PRIJONO SUGIARTO': 'Prijono Sugiarto', 'DHANI MUHAMMAD KURNIAWAN ST': 'Dhani Muhammad Kurniawan St', 'ANDREAS TIMOTI': 'Andreas Timoti', 'DEMAK SIMARMATA': 'Demak Simarmata', 'ANDRE YULIANTO': 'Andre Yulianto', 'RONALDO JEFFREY': 'Ronaldo Jeffrey', 'PERPETUAL SMART BETA EMERGING MARKET EQUITIES FUND': 'Perpetual Investments', 'FRENGKY KOSASI': 'Frengky Kosasi', 'RIVAN KURNIAWAN': 'Rivan Kurniawan', 'ELISA SANTOSO': 'Elisa Santoso', 'SUKENGSI HARI SLAMET': 'Sukengsi Hari Slamet', 'SANDY': 'Sandy', 'ERNI KURNIAWATI AZISAH': 'Erni Kurniawati Azisah', 'VONNY SUMENDAP': 'Vonny Sumendap', 'ROY SETIAWAN PRASETYA': 'Roy Setiawan Prasetya', 'BAMBANG SUTANTIO': 'Bambang Sutantio', 'BARNABAS SIMON KURNIAWAN': 'Barnabas Simon Kurniawan', 'DAI ICHI MUTUAL LIFE INSURANCE CO': 'Dai-Ichi Life Insurance Co.', 'DARMADI': 'Darmadi', 'I WAYAN SISWANTO': 'I Wayan Siswanto', 'YOHANES BINTORO BUDIRAHARJO': 'Yohanes Bintoro Budiraharjo', 'PT VERITAS INTI AMERTA': 'PT Veritas Inti Amerta', 'KEN KRISMADI': 'Ken Krismadi', 'LAI HIANG': 'Lai Hiang', 'SASTRO WIJOYO': 'Sastro Wijoyo', 'PETER TJUT': 'Peter Tjut', 'JULIANA': 'Juliana', 'ANDI CHANDRA': 'Andi Chandra', 'RUDI LING ROJI': 'Rudi Ling Roji', 'DYNAMIC MONEY IDR': 'Dynamic Money Idr', 'MARISA ANTA SARI AND S SOS': 'Marisa Anta Sari & S SOS', 'MUHAMMAD FAUZAN': 'Muhammad Fauzan', 'KRIS JACOBUS SILFANUS': 'Kris Jacobus Silfanus', 'AUDRY SUBAGIA': 'Audry Subagia', 'CI EMERGING MARKETS DIVIDEND INDEX ETF': 'CI Global Asset Mgt', 'PRAWIRAYUDHA HALIEM': 'Prawirayudha Haliem', 'TJUN FUNG AND IWAN PURNAWAN': 'Tjun Fung & Iwan Purnawan', 'ANAS WIDJAJA BARAK RIMBA': 'Anas Widjaja Barak Rimba', 'KO CHING': 'Ko Ching', 'STEFAN LIMANTARA': 'Stefan Limantara', 'SWANTO GO': 'Swanto Go', 'IDA AYU KETUT SURYA ADNYANI SE MAP': 'Ida Ayu Ketut Surya Adnyani Se Map', 'TAUFIK ANDRIAWAN': 'Taufik Andriawan', 'JOSEF SUGIARTO': 'Josef Sugiarto', 'HASAN HALIM': 'Hasan Halim', 'AWALUDDIN OCTO RACHNALIM': 'Awaluddin Octo Rachnalim', 'PT KEDAI MIMPI TERWUJUD': 'Pt Kedai Mimpi Terwujud', 'HADIYANTO SH': 'Hadiyanto Sh', 'HIMAWAN KEMBAREN': 'Himawan Kembaren', 'HADI SUKANTO': 'Hadi Sukanto', 'HATTA WENTIO BUDIANTO': 'Hatta Wentio Budianto', 'ARIEF DARU WIBAWANTO': 'Arief Daru Wibawanto', 'ALVIN MARZUKI': 'Alvin Marzuki', 'GOUW KWE SEN': 'Gouw Kwe Sen', 'PIETER PHILIP': 'Pieter Philip', 'ELKE SOLOMON': 'Elke Solomon', 'CHRISTINA TANUWIDJAJA': 'Christina Tanuwidjaja', 'GLOBAL ALPHA STRATEGY FUND GLOBAL EQUITY SATELLITE II': 'Fondaco Lux', 'GLOBAL ALPHA STRATEGY FUND GLOBAL EQUITY SATELLITE IV': 'Fondaco Lux', 'XTRACKERS MSCI ALL WORLD EX US HEDGED EQUITY ETF': 'DBX Advisors', 'XTRACKERS MSCI EMERGING MARKETS HEDGED EQUITY ETF': 'DBX Advisors', 'XTRACKERS EMERGING MARKETS CARBON REDUCTION AND CLIMATE IMPROVERS ETF': 'DBX Advisors', 'HELENA': 'Helena', 'PT BERKAT NIRWANA SENTOSA': 'Pt Berkat Nirwana Sentosa', 'HABI HAYASI': 'Habi Hayasi', 'AGUSTIANI': 'Agustiani', 'HENDRIK JUARSA': 'Hendrik Juarsa', 'EDY': 'Edy', 'FERRY WIDJAJA': 'Ferry Widjaja', 'SETIA BUDI SE': 'Setia Budi Se', 'BRYAN MILLER': 'Bryan Miller', 'YIU CHEN': 'Yiu Chen', 'WILLYANTO': 'Willyanto', 'JOE KARTAWIDJAJA': 'Joe Kartawidjaja', 'SHELIN': 'Shelin', 'SRI REDJEKI BUDIMARTONO': 'Sri Redjeki Budimartono', 'BUDI KURNIAWAN': 'Budi Kurniawan', 'FENDY HERYANTO': 'Fendy Heryanto', 'PT LUMBUNG INDUSTRI PERSADA': 'Lumbung Industri Persada', 'M JAROT EKO WINARNO': 'M Jarot Eko Winarno', 'MOH ADIB IR': 'Moh Adib Ir', 'NIERWAN JUDI': 'Nierwan Judi', 'SARANAKELOLA INVESTA': 'Saranakelola Investa', 'YOPPY ROSIMIN DRS': 'Yoppy Rosimin DRS', 'SAK TJHOENG': 'Sak Tjhoeng', 'JUNIANTO': 'Junianto', 'NURWATI B': 'Nurwati B', 'BIDIN YUSUF': 'Bidin Yusuf', 'RIUS VERNANDES': 'Rius Vernandes', 'ALAMSYAH': 'Alamsyah', 'YENNY HAMIDAH KOEAN': 'Yenny Hamidah Koean', 'IDA BAGUS PUTU ANOM REDHI': 'Ida Bagus Putu Anom Redhi', 'I KETUT WIDARTA': 'I Ketut Widarta', 'DIANA SUMAWI': 'Diana Sumawi', 'MANUEL MUSON': 'Manuel Muson', 'LIE DIANA LISSAN': 'Lie Diana Lissan', 'ALEX PURNADI CHANDRA': 'Alex Purnadi Chandra', 'PT CHANDRA ANUGERAH SENTOSA': 'Pt Chandra Anugerah Sentosa', 'JUNAIDY HALIM': 'Junaidy Halim', 'SHERLY': 'Sherly', 'ROBERT': 'Robert', 'CHUBB LIFE INSURANCE IND RUPIAH EQUITY GF': 'Chubb Life Insurance', 'CHUBB LIFE INSURANCE IND RUPIAH EQUITY': 'Chubb Life Insurance', 'CHUBB RUPIAH EQUITY FUND III': 'Chubb Life Insurance', 'CHUBB RUPIAH MANAGED FUND': 'Chubb Life Insurance', 'ALVIN BINGEI': 'Alvin Bingei', 'RANDY MULYADI JATIM': 'Randy Mulyadi Jatim', 'TAN KHANG SENG': 'Tan Khang Seng', 'DJUANDA WIDJAJA': 'Djuanda Widjaja', 'STEVEN ANTONIUS KURNIAWAN': 'Steven Antonius Kurniawan', 'RUBEN ARTIA LUMBAN TORUAN': 'Ruben Artia Lumban Toruan', 'TRI WAHYUNI': 'Tri Wahyuni', 'MA YAN': 'Ma Yan', 'DONNY HARYANTO': 'Donny Haryanto', 'ADRI ACHMAD DARADJATUN': 'Adri Achmad Daradjatun', 'ROBY CHANDRA': 'Roby Chandra', 'SUHAIWI': 'Suhaiwi', 'ISWANDI': 'Iswandi', 'RIEDNO GRAAL TALIAWO': 'Riedno Graal Taliawo', 'HERMANTO TANDINA': 'Hermanto Tandina', 'DWINATA': 'Dwinata', 'OEN MARIA CHRISTINA': 'Oen Maria Christina', 'REKSA DANA SETIABUDI DANA CAMPURAN': 'Setiabudi Investment Mgt', 'REKSA DANA SETIABUDI DANA OBLIGASI PLUS': 'Setiabudi Investment Mgt', 'REKSA DANA SETIABUDI DANA KOMBINASI DINAMIS': 'Setiabudi Investment Mgt', 'SAMIN SARIMAN': 'Samin Sariman', 'EDI SUGIARTO': 'Edi Sugiarto', 'HARIJANTO SOEGIO': 'Harijanto Soegio', 'ANDY HALIM': 'Andy Halim', 'EDDY KOKO YOENAEDY': 'Eddy Koko Yoenaedy', 'DANIEL SUNJAYA': 'Daniel Sunjaya', 'MOH ALI MURAD': 'Moh. Ali Murad', 'DANIEL MONIAGA FREDDY PUTRA': 'Daniel Moniaga Freddy Putra', 'ROBBY TAMAN': 'Robby Taman', 'TET FA DJAP': 'Tet Fa Djap', 'LANNY SANTOSO': 'Lanny Santoso', 'SONNY TERIMSIA': 'Sonny Terimsia', 'YANDRAMIN HALIM IR': 'Yandramin Halim IR', 'TJANDRA STRATEGIK INVESTAMA': 'Tjandra Strategik Investama', 'LAYONO': 'Layono', 'HERRY MULJA': 'Herry Mulja', 'DADANG YUWENDI': 'Dadang Yuwendi', 'HARRY SUPARTAN': 'Harry Supartan', 'HELEN JENIWATI': 'Helen Jeniwati', 'PT CHITATEX PENI': 'Pt Chitatex Peni', 'EDY GANI': 'Edy Gani', 'ARIEF TJATUR WIDODO': 'Arief Tjatur Widodo', 'SHANDY EFFO KALIM': 'Shandy Effo Kalim', 'ANGGA INDO SUCIPTO': 'Angga Indo Sucipto', 'LILI MULYADI SUTANTO': 'Lili Mulyadi Sutanto', 'TEGUH KURNIAWAN': 'Teguh Kurniawan', 'JIMMY': 'Jimmy', 'DODDY PRAYOGO': 'Doddy Prayogo', 'JOKO': 'Joko', 'BUDI HARIONO LIMAN LIM': 'Budi Hariono Liman Lim', 'STRIVE EMERGING MARKETS EX CHINA ETF': 'Strive Asset Mgt', 'PT ASURANSI JIWA': 'Reliance Investment Mgt', 'MUHAMMAD NADHIF TAQWA': 'Muhammad Nadhif Taqwa', 'KUAN CEN': 'Kuan Cen', 'HANDI': 'Handi', 'NASIR SE': 'Nasir Se', 'KRANESHARES MSCI EMERGING MARKETS EX CHINA INDEX ETF': 'Krane Funds Advisors', 'RUDI SUTANTO': 'Rudi Sutanto', 'ALEXANDER SOETOPO PUTRA P': 'Alexander Soetopo Putra P', 'ANIMA SGR S P A': 'Anima', 'ANIMA PRIMA FUNDS PL': 'Anima', 'LAY TJE THAT': 'Lay Tje That', 'MUTB300000069 STOCK': 'Mitsubishi UFJ Trust & Banking Corporation', 'MUTB300000120 STOCK': 'Mitsubishi UFJ Trust & Banking Corporation', 'EDDY SUGIANTO': 'Eddy Sugianto', 'MOCHAMAD PRIMANTO': 'Mochamad Primanto', 'BNZ WHOLESALE INTERNATIONAL EQUITIES INDEX FUND': 'Bank of New Zealand', 'RIDWAN GUNAWAN': 'Ridwan Gunawan', 'LIEM SALIM HADRIAN WIJAYA': 'Liem Salim Hadrian Wijaya', 'TJAN IND HONG': 'Tjan Ind Hong', 'DILIP LACHMANDAS DODANI': 'Dilip Lachmandas Dodani', 'SURYANTO': 'Suryanto', 'VONTOBEL FUND SIF': 'Vontobel Asset Mgt', 'A KHIONG': 'A Khiong', 'LUCAS ISKANDAR': 'Lucas Iskandar', 'DEDDY HARYADI E': 'Deddy Haryadi E', 'ANGKASA STRATEGIK INVESTINDO': 'Angkasa Strategik Investindo', 'TANUWI KUSUMA': 'Tanuwi Kusuma', 'IMELDA TJHIA': 'Imelda Tjhia', 'LIUW HARY': 'Liuw Hary', 'FIRST TRUST GLOBAL EQUITY INCOME UCITS ETF': 'First Trust Advisors', 'EMERGING MARKETS EQUITY SELECT ETF': 'First Trust Advisors', 'PT JAVA AGRI SUKSES MAKMUR': 'Pt Java Agri Sukses Makmur', 'HARI SOEDJANADI': 'Hari Soedjanadi', 'SURYO SUWIGNJO': 'Suryo Suwignjo', 'HANDI LAKSONO': 'Handi Laksono', 'EKO A I BASYUNI': 'Eko A I Basyuni', 'JONI INDRA SE AKT MM': 'Joni Indra Se Akt Mm', 'TONY': 'Tony', 'JAKA MULYANA': 'Jaka Mulyana', 'PENDAL SUSTAINABLE INTERNATIONAL SHARE FUND': 'Pendal Group', 'PT MERPATI MARINE SERVICE': 'Pt Merpati Marine Service', 'ELLY IVONNE SITOMPUL': 'Elly Ivonne Sitompul', 'SURJADI HALIM': 'Surjadi Halim', 'BINTORO': 'Bintoro', 'DODO ROHIMAT SH': 'Dodo Rohimat Sh', 'FONNY SE': 'Fonny se', 'PT ANDITYAWARMAN': 'PT Andityawarman', 'IVAN ALAMSYAH SIREGAR': 'Ivan Alamsyah Siregar', 'DIANA DEWI JUNUS': 'Diana Dewi Junus', 'EFFENDY TANAMAL': 'Effendy Tanamal', 'LIE SHE LIAM': 'Lie She Liam', 'MIKHAEL GALAXY': 'Mikhael Galaxy', 'IR AGUS BUDHIARTO': 'Ir Agus Budhiarto', 'SUANDI': 'Suandi', 'HARYONO UTOMO': 'Haryono Utomo', 'TIKA DWI KERISNA': 'Tika Dwi Kerisna', 'LINA': 'Lina', 'ENRICO FAJAR FERDIANSYAH': 'Enrico Fajar Ferdiansyah', 'TIRNADI KOSASIH': 'Tirnadi Kosasih', 'ANDHY KOESNANDAR': 'Andhy Koesnandar', 'SLAMET BUDIARTO': 'Slamet Budiarto', 'ANTONIUS AGUS SUSANTO': 'Antonius Agus Susanto', 'SALI': 'Sali', 'NICODEMUS KASAN KURNIAWAN': 'Nicodemus Kasan Kurniawan', 'CAHYANI ENDAHAYU': 'Cahyani Endahayu', 'RUSMIN': 'Rusmin', 'FRANS SUWARAJI': 'Frans Suwaraji', 'RAFAEL SUWARDY': 'Rafael Suwardy', 'CORNELLIUS HENRY KHO': 'Cornellius Henry Kho', 'JEVINA JESIKA TAN': 'Jevina Jesika Tan', 'ANTHONIUS S TJAHJA': 'Anthonius S Tjahja', 'DAVID FERNANDO AUDY': 'David Fernando Audy', 'SUTIADI WIDJAJA': 'Sutiadi Widjaja', 'FANUR RACHMAN': 'Fanur Rachman', 'SOEGIANTO': 'Soegianto', 'HIAN HU WI': 'Hian Hu Wi', 'ELLEN YOLANDA': 'Ellen Yolanda', 'FERRY': 'Ferry', 'SENDRA GUNAWAN': 'Sendra Gunawan', 'SANNY NANANG': 'Sanny Nanang', 'TAN HI SENG': 'Tan Hi Seng', 'TANTIO BAGUS CAHYONO': 'Tantio Bagus Cahyono', 'LAURENTIA WIJAYA': 'Laurentia Wijaya', 'FENLI': 'Fenli', 'RUDI HARTONO': 'Rudi Hartono', 'PAULUS RAGIL SAPUTRO': 'Paulus Ragil Saputro', 'WAHYUDI LUKMANTARA': 'Wahyudi Lukmantara', 'TJAHJANA KURNIAWAN TIRTA': 'Tjahjana Kurniawan Tirta', 'SIGIT PRASTOWO': 'Sigit Prastowo', 'MARJUKY': 'Marjuky', 'KEVIN VALL': 'Kevin Vall', 'MOUNT MURRAY INVESTMENT EMERGING MARKETS EQUITY FUND': 'Mount Murray Investment', 'VICTOR SANTOSO': 'Victor Santoso', 'KETUT SARJANA': 'Ketut Sarjana', 'RIAN ANDRIAN T': 'Rian Andrian T', 'WINO SUBALI': 'Wino Subali', 'HELEN FILIANI F': 'Helen Filiani F', 'GUNAWAN SETIONO IR': 'Gunawan Setiono Ir', 'MICHAEL CHRISTIAN SOEKAMTO': 'Michael Christian Soekamto', 'GIOVANNO OROH VICKY HERMAWAN': 'Giovanno Oroh Vicky Hermawan', 'EY SARWONO SOEDARTO': 'Ey Sarwono Soedarto', 'LIM ANDY WIJAYA': 'Lim Andy Wijaya', 'CANDRA WIJAYA': 'Candra Wijaya', 'IVAN FERNANDO A T': 'Ivan Fernando A T', 'FELIX DARMA': 'Felix Darma', 'SUPRIYANTO': 'Supriyanto', 'WENDY FEBRIYANTI LAOLI ST': 'Wendy Febriyanti Laoli ST', 'LIE PETER LEEANSYAH': 'Lie Peter Leeansyah', 'ANDRE TEGUH KOHAR': 'Andre Teguh Kohar', 'MUHAMAD ARIFUDIN': 'Muhamad Arifudin', 'FREDERICK MURTANU': 'Frederick Murtanu', 'MOH IRFAN SEPTIAWAN': 'Moh Irfan Septiawan', 'NINY': 'Niny', 'HANSEN RAMBITAN': 'Hansen Rambitan', 'HERNY ROSALINA': 'Herny Rosalina', 'EVODIUS PURWOKO': 'Evodius Purwoko', 'MASWAR PURNAMA': 'Maswar Purnama', 'BENNY ARIYANTO': 'Benny Ariyanto', 'WAHYU TRENGGONO': 'Wahyu Trenggono', 'AGUSTIADI WIJAYA': 'Agustiadi Wijaya', 'DPLK TUGU MANDIRI DANA SITAMPAN': 'Tugu Mandiri', 'SIANE YUANITA': 'Siane Yuanita', 'IMPRESITAMA VISI KARYA PT': 'Impresitama Visi Karya Pt', 'KUI HUI ALI': 'Kui Hui Ali', 'JOKO SUDARMO': 'Joko Sudarmo', 'TANATO S KOM': 'Tanato S Kom', 'MICHAEL TJUT': 'Michael Tjut', 'LELI SUBARNAS': 'Leli Subarnas', 'IRAWATI DARMAWAN BCHK': 'Irawati Darmawan Bchk', 'KUARDI CHANDRA': 'Kuardi Chandra', 'SUNARYO WONGSO': 'Sunaryo Wongso', 'YUDI PRIAMBODO P': 'Yudi Priambodo P', 'KUNCORO WIBOWO': 'Kuncoro Wibowo', 'NG HUCK YUN': 'Ng Huck Yun', 'RUSLI': 'Rusli', 'RIDWAN SOMADIBRATA': 'Ridwan Somadibrata', 'BHAGWAN RAMCHAND BHARWANI': 'Bhagwan Ramchand Bharwani', 'SUBUR DJINGGA': 'Subur Djingga', 'TOTO SUGIARTO': 'Toto Sugiarto', 'MELANIA AMANDA MANDAGIE': 'Melania Amanda Mandagie', 'RAYMOND PUTRA': 'Raymond Putra', 'WISKOK': 'Wiskok', 'CV ULTRA ENGINEERING': 'CV Ultra Engineering', 'BAMBANG BUDIONO': 'Bambang Budiono', 'YOSEPH KUSDIYANTO': 'Yoseph Kusdiyanto', 'RR ERNIATI': 'Rr Erniati', 'ISUMITRO': 'Isumitro', 'SILVANA SOEI': 'Silvana Soei', 'THOENG CONRAD ITUNG': 'Thoeng Conrad Itung', 'ROBERT PETER RAJIMIN': 'Robert Peter Rajimin', 'SUPRATMAN': 'Supratman', 'DHARMAYANTO TIRTAWISATA': 'Dharmayanto Tirtawisata', 'AGUS RIWAYANTO DIPUTRA': 'Agus Riwayanto Diputra', 'EDY NOP': 'Edy Nop', 'ANUGRAH PAKERTI': 'Anugrah Pakerti', 'LINDA SILING': 'Linda Siling', 'SREEJIT CHALAKKAL': 'Sreejit Chalakkal', 'PRIMARTONO GUNAWAN': 'Primartono Gunawan', 'IR YOSEPH': 'Ir Yoseph', 'HARRY WIJAYA': 'Harry Wijaya', 'EDDY CUACA': 'Eddy Cuaca', 'I NENGAH SUANDA': 'I Nengah Suanda', 'PT SUMBER PERINTIS RAGAMPANGAN': 'PT Sumber Perintis Ragampangan', 'JOKO SUTANTO': 'Joko Sutanto', 'HARDINO GO': 'Hardino Go', 'I KADEK SARENDRA JAYA': 'I Kadek Sarendra Jaya', 'SUWANDI': 'Suwandi', 'CHARNES SETIAWAN': 'Charnes Setiawan', 'JOYCE WONSONO': 'Joyce Wonsono', 'EDY SULAIMAN': 'Edy Sulaiman', 'INDINVEST STRATEGY CENTRE': 'Indinvest Strategy Centre', 'PAULA NANIWATI DASUKI': 'Paula Naniwati Dasuki', 'ILONA KUSWANTO': 'Ilona Kuswanto', 'MEGAWATI MULJADI': 'Megawati Muljadi', 'DEDDY SETIAWAN': 'Deddy Setiawan', 'FERRYANTO SALIM': 'Ferryanto Salim', 'GOLDIAN YENDITHYA': 'Goldian Yendithya', 'SINYO KALANGI': 'Sinyo Kalangi', 'OMAR HIDAJAT WANGSIT': 'Omar Hidajat Wangsit', 'HERY SUDJONO': 'Hery Sudjono', 'SUKIRMAN DJOHAN': 'Sukirman Djohan', 'PETER': 'Peter', 'DASARI': 'Dasari', 'IR HARJANTO TEGUH RAHAJU': 'Ir Harjanto Teguh Rahaju', 'LIE GIN HWI': 'Lie Gin Hwi', 'WILSON CAHYADI LIAUW': 'Wilson Cahyadi Liauw', 'HADI TANUMIHARJA': 'Hadi Tanumiharja', 'ELIA DJONLIE NAULIE': 'Elia Djonlie Naulie', 'TENDY': 'Tendy', 'BINTANG PRIMA PUTRA HUTASOIT': 'Bintang Prima Putra Hutasoit', 'YANSEN GUNAWAN': 'Yansen Gunawan', 'PT MULYA HUSADA JAYA': 'PT Mulya Husada Jaya', 'MARYANTA': 'Maryanta', 'KUNARDI SASTRAWIJAYA': 'Kunardi Sastrawijaya', 'FEDERAL HOLD INDONESIA': 'Federal Hold Indonesia', 'CYNTHIA ARIYANI': 'Cynthia Ariyani', 'BUDI HARTONO': 'Budi Hartono', 'ELIZABETH OSCAR': 'Elizabeth Oscar', 'AKHMAD FAZRI': 'Akhmad Fazri', 'REYN ALBERT BONARDO': 'Reyn Albert Bonardo', 'ACHMAD ROYADI': 'Achmad Royadi', 'REINO RAMAPUTRA BARACK': 'Reino Ramaputra Barack', 'JULIUS SANTOSA': 'Julius Santosa', 'KELVIN': 'Kelvin', 'ROLIN': 'Rolin', 'ISKANDAR CHANDRA': 'Iskandar Chandra', 'MEDANY HALIM': 'Medany Halim', 'ALBERT HARANDA LINCOLN': 'Albert Haranda Lincoln', 'J F HASUDUNGAN': 'J.F Hasudungan', 'MARDJONO TJAHJADI': 'Mardjono Tjahjadi', 'HERWANTO SUTANTO': 'Herwanto Sutanto', 'ERIC HARTANTO NYOTOSETIADI': 'Eric Hartanto Nyotosetiadi', 'GUINNESS ASIAN EQUITY INCOME FUND': 'Guinness Global Investors', 'YOSEFA CAHAYA': 'Yosefa Cahaya', 'DANA PENSIUN KARYAWAN STAF KEBON AGUNG': 'Dana Pensiun Karyawan Staf Kebon Agung', 'SUSANTO GONDO WIJAYA': 'Susanto Gondo Wijaya', 'AMP SUPER FUND': 'AMP Super Fund', 'BAMBANG TEDJO NARSOYO ST MT': 'Bambang Tedjo Narsoyo St Mt', 'I DEWA PUTU PRAYOGA': 'I Dewa Putu Prayoga', 'HERMAN WIBISONO': 'Herman Wibisono', 'MARZUKI EFREN LOU': 'Marzuki Efren Lou', 'ADITYA SUTANDHI': 'Aditya Sutandhi', 'WILLIAM SUSILO': 'William Susilo', 'IWAN SIOK': 'Iwan Siok', 'ILYAS ABDUR RAHMAN': 'Ilyas Abdur Rahman', 'INDRA WIBISANA SE': 'Indra Wibisana Se', 'RICZAL TJOWASI': 'Riczal Tjowasi', 'NICO GUNNAR HUTOMO': 'Nico Gunnar Hutomo', 'ERNAWATY TAMBA': 'Ernawaty Tamba', 'RACHIM SATRIA': 'Rachim Satria', 'ANNA MARIA TJIADARMA': 'Anna Maria Tjiadarma', 'LINDA RACHMAT': 'Linda Rachmat', 'STEFANUS EFFENDY': 'Stefanus Effendy', 'PT SYNERGY INTI PROPERTI': 'Pt Synergy Inti Properti', 'HARRIS KRISTANTO WIDJAJA': 'Harris Kristanto Widjaja', 'KAMARUDDIN': 'Kamaruddin', 'K T ISMAIL DJALIMAN': 'K T Ismail Djaliman', 'FATKUL MUBIN AND ST': 'Fatkul Mubin & ST', 'EDDY HANDALI': 'Eddy Handali', 'RENO': 'Reno', 'IR KOMARI SUBAKIR': 'Ir Komari Subakir', 'VICTOR ANDRIANTO WIBAWA': 'Victor Andrianto Wibawa', 'RONNY': 'Ronny', 'TAMINABUAN': 'Taminabuan', 'JOHN STEPHEN': 'John Stephen', 'SAIMAN SAMPIN': 'Saiman Sampin', 'LISA SELIYANTI MULYONO': 'Lisa Seliyanti Mulyono', 'OENTOENG WIDJAJA': 'Oentoeng Widjaja', 'HENDRA MUNANTO': 'Hendra Munanto', 'NONI TIO BIE KIAN': 'Noni Tio Bie Kian', 'KELVIN PHANGESTU': 'Kelvin Phangestu', 'BASYIRUDDIN': 'Basyiruddin', 'SINODE GEREJA JKI': 'Sinode Gereja Jki', 'JECKY EDISON LENENG': 'Jecky Edison Leneng', 'YULIATI': 'Yuliati', 'EDWARD BARCELLO SIMORANGKIR': 'Edward Barcello Simorangkir', 'ERMA': 'Erma', 'SUANTO': 'Suanto', 'I NYOMAN SUGIRI YASA': 'I Nyoman Sugiri Yasa', 'JILLY VANESSI LOPRANG': 'Jilly Vanessi Loprang', 'I WAYAN SUNARTA': 'I Wayan Sunarta', 'HENGKI': 'Hengki', 'HENDRO': 'Hendro', 'ERMIRA KURNIA SUSILO': 'Ermira Kurnia Susilo', 'PT RAJAWALI MUTIARA SEJAHTERA': 'Pt Rajawali Mutiara Sejahtera', 'TJIOK ARISTO SUNGKONO S': 'Tjiok Aristo Sungkono S', 'WILLIARTO KUOSASTRA': 'Williarto Kuosastra', 'IR MASPIYONO': 'Ir Maspiyono', 'HENGKY': 'Hengky', 'JENNY JOHANNES': 'Jenny Johannes', 'SAMUEL HARRY MULANA SH': 'Samuel Harry Mulana Sh', 'FARIDA LIM': 'Farida Lim', 'NELLIE': 'Nellie', 'SUPOMO KOK': 'Supomo Kok', 'DENNY PANGAT': 'Denny Pangat', 'SIAT ALAN DARMASAPUTRA': 'Siat Alan Darmasaputra', 'PHEIM ASEAN ALL CAP EQUITY FUND': 'Pheim Asset Mgt', 'PHEIM ASIA EX JAPAN FUND': 'Pheim Asset Mgt', 'PHEIM A C 260': 'Pheim Asset Mgt', 'VARIAN ADIPUTRA': 'Varian Adiputra', 'SU CHIN': 'Su Chin', 'REKSA DANA DANAPATHI BALANCE FUND': 'Danapathi Asset Management', 'REKSA DANA DANAPATHI EQUITY GROWTH': 'Danapathi Asset Management', 'ARIEF BUDIMAN': 'Arief Budiman', 'HARYANTO HADI': 'Haryanto Hadi', 'EDY SUYANTO': 'Edy Suyanto', 'PIERRE DERMAWAN': 'Pierre Dermawan', 'HENGKI SETIAWAN': 'Hengki Setiawan', 'JASON SOESANTO': 'Jason Soesanto', 'SUWOKO SINGOASTRO': 'Suwoko Singoastro', 'SURJADI LUKMAN': 'Surjadi Lukman', 'SUMIHAR MANULLANG': 'Sumihar Manullang', 'SUTARDJO': 'Sutardjo', 'CALVIN HARIANTO': 'Calvin Harianto', 'RUDY WIJAYA': 'Rudy Wijaya', 'TEGUH WAHJU NUGRAHA': 'Teguh Wahju Nugraha', 'KARTIKA GUNAWAN': 'Kartika Gunawan', 'KAMALUL HAKIM': 'Kamalul Hakim', 'LAURA SANTOSO': 'Laura Santoso', 'BRAM RAHARDIAN MAHENDRA': 'Bram Rahardian Mahendra', 'YUNITA': 'Yunita', 'HENDRI SALIM': 'Hendri Salim', 'IGNATIUS YANSEN NG': 'Ignatius Yansen Ng', 'DAVA KHARIS': 'Dava Kharis', 'KAFI SOEKAMA': 'Kafi Soekama', 'INDRA KRISMAYADI S IK': 'Indra Krismayadi S Ik', 'R HERDI SUHENDRI SAKTI': 'R Herdi Suhendri Sakti', 'RUDY HARTONO BUDIYANTO': 'Rudy Hartono Budiyanto', 'ERICHALIM': 'Erichalim', 'ANDRY TJAHJADI': 'Andry Tjahjadi', 'ALIE MAKMUR LAI': 'Alie Makmur Lai', 'GAM MULTISTOCK': 'GAM Investment Mgt', 'DAVID BUDIWANTO': 'David Budiwanto', 'RADJA ZAINULLAH': 'Radja Zainullah', 'FEBRI HENDARJAT': 'Febri Hendarjat', 'IWAN GOUTAMA GOUW': 'Iwan Goutama Gouw', 'EDO UTAMA': 'Edo Utama', 'AFRIYAN': 'Afriyan', 'HENNY SUYATI': 'Henny Suyati', 'WINSTON TAN': 'Winston Tan', 'PUTU DENIK SUASTIKA': 'Putu Denik Suastika', 'SUGIJANTO HARTOJO': 'Sugijanto Hartojo', 'SWANDI': 'Swandi', 'HARIJADI': 'Harijadi', 'JAHJA SETIAATMADJA': 'Jahja Setiaatmadja', 'YOUNG SONNY CHUN': 'Young Sonny Chun', 'ERWIN GANDI': 'Erwin Gandi', 'USMAN ANDY': 'Usman Andy', 'IRSAN TAN': 'Irsan Tan', 'WIRANTO NURHADI': 'Wiranto Nurhadi', 'AKRIM': 'Akrim', 'ICAPITAL ASIA PACIFIC EX JAPAN BTB FUND': 'Capital Dynamics', 'MUHAMMAD PURI ANDAMAS': 'Muhammad Puri Andamas', 'SIE SURYA DHARMA': 'Sie Surya Dharma', 'SIAT SUN FAT': 'Siat Sun Fat', 'BENNY HANJAYA MARDJUKI': 'Benny Hanjaya Mardjuki', 'EDDO CHANDRA': 'Eddo Chandra', 'GOENAWAN TANTONO': 'Goenawan Tantono', 'MARCELUS': 'Marcelus', 'ISKANDAR': 'Iskandar', 'DJAMIN ASNAWI': 'Djamin Asnawi', 'SISKA WIDYA SANTI': 'Siska Widya Santi', 'I MADE KOMPIANG GAUTAMA': 'I Made Kompiang Gautama', 'HANAFI LIBRARI PRATAMA': 'Hanafi Librari Pratama', 'CHANDRA RAMAZETA PAMUNGKAS': 'Chandra Ramazeta Pamungkas', 'MAX MEIDEN DASUKI': 'Max Meiden Dasuki', 'EDY SUWANTO': 'Edy Suwanto', 'PT BERJUANG SUKSES BERSAMA': 'Pt Berjuang Sukses Bersama', 'AZ FUND 1 AZ EQUITY ASEAN COUNTRIES': 'Azimut Investments', 'ONG SANG SANG BUDIMAN': 'Ong Sang Sang Budiman', 'FELICIA NATALI': 'Felicia Natali', 'LEO CHENDRA': 'Leo Chendra', 'ALBERT KRISTIANTO GANI': 'Albert Kristianto Gani', 'NJOMAN TIRTHA': 'Njoman Tirtha', 'LEONARDY PUTRA NEGARA SIREGAR': 'Leonardy Putra Negara Siregar', 'WINDY ARY WIJAYA': 'Windy Ary Wijaya', 'AGF SYSTEMATIC GLOBAL ESG FACTORS ETF': 'AGF International Advisors', 'HASIM': 'Hasim', 'JULIYANTI': 'Juliyanti', 'JONNY ITEH': 'Jonny Iteh', 'RUDY GUNAWAN WIDJAJA': 'Rudy Gunawan Widjaja', 'EFFENDY': 'Effendy', 'MARTIN SUGIANTO': 'Martin Sugianto', 'IVANDER THEODORA': 'Ivander Theodora', 'PITER GOENAWAN HENOEK': 'Piter Goenawan Henoek', 'HUSDI STEVEN TIORONTO': 'Husdi Steven Tioronto', 'ASHMORE SICAV': 'Ashmore Investment Mgt', 'SCOTIA EMERGING MARKETS EQUITY INDEX TRACKER ETF': 'State Street Investment Mgt', 'WISNU ISKANDAR': 'Wisnu Iskandar', 'AMI NOERDHINI MARLIM': 'Ami Noerdhini Marlim', 'JOE RICHARD JOHAN': 'Joe Richard Johan', 'SUKMAWATI ONG': 'Sukmawati Ong', 'LIANAWATI ALI': 'Lianawati Ali', 'JOANES JUSTIRA GUNAWAN': 'Joanes Justira Gunawan', 'CHUNGHWA POST': 'Blackrock Investment Mgt', 'JONI ALWADRIS': 'Joni Alwadris', 'YOHANES': 'Yohanes', 'ANTHONY KARTOLO': 'Anthony Kartolo', 'FJERNOSTEN INDEKS KL': 'Danske Capital Mgt', 'FIFI OKTAVIANDORA': 'Fifi Oktaviandora', 'REKSA DANA OSO FLORES EQUITY FUND': 'Oso Investment Mgt', 'SUKIMAN': 'Sukiman', 'EASTSPRING INVESTMENTS GLOBAL MULTI FACTOR EQUITY ENHANCED INCOME FUND': 'Eastspring Investments', 'CANDRA AGUSTIAN DATAM': 'Candra Agustian Datam', 'PERKUMPULAN DANA PENSIUN PROCTER AND GAMBLE HOME PRODUCT INDONESIA': 'Perkumpulan Dana Pensiun Procter & Gamble Home Product Indonesia', 'BONG NIE FUI': 'Bong Nie Fui', 'HEREFORD FUNDS VALVERDE ASEAN FUND': 'Valverde Investment Partners', 'RADIUS CHRISTANTO': 'Radius Christanto', 'MARGETTS AFH DA GLOBAL EMERGING MARKETS EQUITY FUND': 'Jupiter Asset Mgt', 'BUDI SUTRISNO RACHMAT K': 'Budi Sutrisno Rachmat K', 'WAN SEN RODJALI': 'Wan Sen Rodjali', 'DARWIN HARIANJA ST': 'Darwin Harianja St', 'CHRISTIAN HANDOKO': 'Christian Handoko', 'FSITC ASIAN EMERGING MARKET FUND': 'First Securities Investment Trust', 'JEFRI': 'Jefri', 'HUSIN': 'Husin', 'LISTIARINI DEWAJANTI': 'Listiarini Dewajanti', 'BADARUDDIN': 'Badaruddin', 'MARATHON UCITS FUNDS GLOBAL OPPORTUNITIES EQUITY FUND': 'Marathon Asset Mgt', 'THE MARATHON LONDON GLOBAL SELECT INVESTMENT TRUST': 'Marathon Asset Mgt', 'MU THI ABDILLAH': 'Mu Thi Abdillah', 'ANDREW PRASETIO SUWANDI PUGUH': 'Andrew Prasetio Suwandi Puguh', 'IR EDI WAHYU CAHYONO M P': 'Ir Edi Wahyu Cahyono M P', 'LUCKMAN CHEN TAN': 'Luckman Chen Tan', 'SAPTA AFRIANTA': 'Sapta Afrianta', 'AGCM FUND ASIA DIVIDEND FUND': 'Asia Growth Capital Mgt', 'YOGIANTO CHONDRO': 'Yogianto Chondro', 'PUSPAWATI': 'Puspawati', 'RICHARD CHRISTIAN': 'Richard Christian', 'FERINA TEJA WIJAYA': 'Ferina Teja Wijaya', 'DARMA GANI': 'Darma Gani', 'DANA PENSIUN LEMBAGA KEUNGAN': 'Pinebridge Investments', 'KURNIA HARYANTO': 'Kurnia Haryanto', 'SHELL OVERSEAS CONTRIBUTORY PENSION FUND': 'Shell Asset Mgt', 'SHELL FOUNDATION SHELL CENTRE': 'Shell Asset Mgt', 'WIDIA': 'Widia', 'CAROLUS ALDO WINDURA': 'Carolus Aldo Windura', 'NATALIA': 'Natalia', 'KARTIKA HADI': 'Kartika Hadi', 'JIP TOMMY SUTANTO': 'Jip Tommy Sutanto', 'PANDRI PRABONO MOELYO': 'Pandri Prabono Moelyo', 'LISTIJANTI HIDAYAT': 'Listijanti Hidayat', 'REKSA DANA I AM BUMN BALANCED PLUS FUND': 'Indoasia Asset Mgt', 'REKSA DANA I AM EQUITY FUND': 'Indoasia Asset Mgt', 'BOOSTER WHOLESALE SCHEME': 'Booster Investment Mgt', 'REKSA DANA INDEKS SIMAS IDX30': 'Sinarmas Asset Mgt', 'MORGAN STANLEY INVESTMENT FUNDS': 'Morgan Stanley Investment Mgt', 'MORGAN STANLEY PATHWAY FUNDS EMERGING MARKETS EQUITY FUND': 'Morgan Stanley Investment Mgt', 'AAI LIMITED': 'State Street Global Markets (Transition)', 'C WORLDWIDE': 'C WorldWide Asset Mgt', 'INVESTERINGSFORENINGEN C WORLDWIDE AFDELING STABILE AKTIER KL': 'C WorldWide Asset Mgt', 'C WORLDWIDE AFDELING EMERGING MARKETS KL': 'C WorldWide Asset Mgt', 'REKSA DANA INDEKS CIPTA ETF INDEX LQ45': 'Ciptadana Asset Mgt', 'REKSA DANA CIPTA DINAMIKA': 'Ciptadana Asset Mgt', 'VALBURY PRIME DYNAMIC EQUITY': 'Valbury Capital Mgt', 'REKSA DANA VALBURY INVESTASI BERIMBANG': 'Valbury Capital Mgt', 'ADLI AZHARI': 'Adli Azhari', 'RUSSELL INVESTMENTS SUSTAINABLE GLOBAL SHARES FUND': 'Russell Investments', 'RUSSELL INVESTMENTS SUSTAINABLE GLOBAL SHARES EX FOSSIL FUELS FUND': 'Russell Investments', 'RUSSELL MULTI ASSET FACTOR EXPOSURE FUND': 'Russell Investments', 'HENDRA WIJAYA': 'Hendra Wijaya', 'NELLY': 'Nelly', 'CV SHENGPO SUKSES BERSAMA': 'CV Shengpo Sukses Bersama', 'THE BING TIK AND BUDIMAN': 'The Bing Tik & Budiman', 'HERMAN SUTARJO': 'Herman Sutarjo', 'AMUNDI CDA HIGH INCOME FUND': 'Amundi Asset Mgt', 'HCEIF AMUNDI ETF ICAV AMUNDI FTSE ALL WORLD GDP WEIGHTED UCITS ETF': 'Amundi Asset Mgt', 'AMUNDI ESG GLOBAL LOW CARBON FUND': 'Amundi Asset Mgt', 'ONEMARKETS AMUNDI FLEXIBLE INCOME FUND': 'Amundi Asset Mgt', 'AMUNDI FUNDS': 'Amundi Asset Mgt', 'AMUNDI INDEX SOLUTIONS AMUNDI INDEX BARCLAYS GLOBAL AGG 500M': 'Amundi Asset Mgt', 'STANLEY HANDIONO': 'Stanley Handiono', 'BAMBANG HARYANTO': 'Bambang Haryanto', 'OSMOSIS ICAV': 'Osmosis Investment Mgt', 'NOMURA SECURITIES': 'Nomura Securities', 'OFI INVEST': 'OFI Invest Asset Mgt', 'LILY KRISHNAN': 'Lily Krishnan', 'BNY MELLON EMERGING MARKETS FUND': 'Mellon', 'SEI EMERGING MARKETS EQUITY FUND': 'Mellon', 'REKSA DANA KISI EQUITY FUND': 'KISI Asset Mgt', 'REKSA DANA KISI BALANCED FUND': 'KISI Asset Mgt', 'COMMERZBANK AG': 'Commerzbank', 'RISWAN DJAYASAPUTRA': 'Riswan Djayasaputra', 'RATNA NOVITA': 'Ratna Novita', 'REKSADANA MNC DANA EKUITAS': 'MNC Asset Mgt', 'REKSA DANA MNC SMART EQUITY FUND': 'MNC Asset Mgt', 'MNC SAHAM DANA AKTIF': 'MNC Asset Mgt', 'HENDRO CIUWINTO': 'Hendro Ciuwinto', 'SUROTO CHANDRA': 'Suroto Chandra', 'HENGKY TJITRA': 'Hengky Tjitra', 'SURYADI WIJAYA': 'Suryadi Wijaya', 'SUYANTO': 'Suyanto', 'TREASURE FUND SUPER MAXXI': 'Treasure Fund Investama', 'REKSA DANA NET DANA FLEXI': 'Net Assets Mgt', 'IWAN SETIAWAN': 'Iwan Setiawan', 'REKSA DANA ARCHITAS SAHAM DINAMIS': 'AXA IM Select', 'EDDY': 'Eddy', 'PUTUT TRIA PUTRA': 'Putut Tria Putra', 'RACHMI RACHMAN': 'Rachmi Rachman', 'LO MOI KIAN': 'Lo Moi Kian', 'IRMA ROCHYANI': 'Irma Rochyani', 'TITIEN AGUSTINA': 'Titien Agustina', 'REKSA DANA CAPITAL OPTIMAL EQUITY': 'Capital Asset Mgt', 'RD CAPITAL BALANCED GROWTH': 'Capital Asset Mgt', 'REKSA DANA DANAKITA INVESTASI FLEKSIBEL': 'Danakita Investama', 'REKSA DANA DANAKITA SAHAM PRIORITAS': 'Danakita Investama', 'GLOBAL X MSCI ASIA PACIFIC EX JAPAN ETF': 'Mirae Asset Global Investments', 'PT ADIKARYA SUKSES BERSAMA': 'PT Adikarya Sukses Bersama', 'GLOBAL X MSCI EMERGING MARKETS INDEX ETF': 'Global X', 'DANA PENSIUN KIMIA FARMA': 'Narada Asset Mgt', 'KARTINI': 'Kartini', 'LUSIA': 'Lusia', 'AVANZA BANK AB': 'Avanza Bank', 'AGUS IWAN SUTANTO KUSUMA': 'Agus Iwan Sutanto Kusuma', 'THE MANZANITA FOUNDATION': 'The Manzanita Foundation', 'WILSON': 'Wilson', 'ASURANSI CENTRAL ASIA': 'A.J. Central Asia Raya', 'KBI FUNDS ICAV': 'KBI Global Investors', 'NGS SUPER NON GOVERNMENT SCHOOLS SUPERANNUATION': 'ClariVest Asset Mgt', 'WILMINGTON TRUST COLLECTIVE INVESTMENT TRUST': 'ClariVest Asset Mgt', 'PT GOLDEN HEART': 'Pt Golden Heart', 'ROBIN': 'Robin', 'SUTANTO': 'Sutanto', 'NICOLAUS T B HARJANTO': 'Nicolaus T B Harjanto', 'TJIA PIN LIE': 'Tjia Pin Lie', 'NABILA ILMI HAKIMAH': 'Nabila Ilmi Hakimah', 'J G NARA NILANDAROE': 'J. G. Nara Nilandaroe', 'MASOU ASTA PACIFIC EQUITY FUND': 'Masou Investment Mgt', 'WIHARTO LIMANTARA': 'Wiharto Limantara', 'REKSA DANA NIKKO INDONESIA EQUITY FUND': 'Ina Sekuritas Indonesia', 'REKSA DANA NIKKO INDONESIA BALANCED FUND': 'Ina Sekuritas Indonesia', 'BUDY SUTIADY': 'Budy Sutiady', 'FREDY': 'Fredy', 'CV ARTHA FIBER INDONESIA': 'CV Artha Fiber Indonesia', 'FREDDY KURNIAWAN': 'Freddy Kurniawan', 'RONNY HADIANA': 'Ronny Hadiana', 'HENDRY KURNIAWAN': 'Hendry Kurniawan', 'HENRI': 'Henri', 'CAISHEN SPC LTCP FUND SP': 'Swiss Asia Capital', 'TATIK': 'Tatik', 'BAMBANG SUMANTRI': 'Bambang Sumantri', 'STANLIB FUNDS': 'STANLIB Asset Mgt', 'CATHAY FTSE EMERGING MARKETS ETF': 'Cathay Securities Investment Trust', 'JOHNNY SUTANTO': 'Johnny Sutanto', 'FARINA WIJAYA': 'Farina Wijaya', 'RANDY SAPUTRA': 'Randy Saputra', 'HARU KOESMAHARGYO': 'Haru Koesmahargyo', 'CELSIUS INTERNATIONAL FUNDS PLC EIS FUND': 'Barclays Wealth', 'CELSIUS INVESTMENT FUNDS SICAV ESG EMERGING MARKETS FUND': 'Barclays Wealth', 'PAULUS': 'Paulus', 'REKSA DANA CAMPURAN WANTEG DIVERSITY FUND': 'Wanteg Asset Mgt', 'JEFFRI': 'Jeffri', 'SHINHAN BNPP BONJOUR SOUTH AFRICA EQUITY INVESTMENT TRUST': 'Shinhan Asset Mgt', 'PHEIM EMERGING COMPANIES BALANCED FUND': 'Pheim Asset Mgt', 'CAHYADI TEJAKUSUMA': 'Cahyadi Tejakusuma', 'REDDY SURADY': 'Reddy Surady', 'HALIM KUSUMA': 'Halim Kusuma', 'RONALD HARTONO TAN': 'Ronald Hartono Tan', 'SUGI WALUYO': 'Sugi Waluyo', 'JPMORGAN TAIWAN TOTAL EMERGING MARKETS INCOME SOLUTION FUND': 'JPMorgan Asset Mgt', 'VANGUARD INTERNATIONAL SHARES HIGH YIELD FUND': 'Vanguard Investments Australia', 'HANDY': 'Handy', 'CORPUS BOND PLUS': 'LiF Investment Mgt', 'BNP PARIBAS EASY MSCI ACWI SRI PAB UCITS ETF': 'BNP Paribas Asset Mgt', 'AGUSTINUS': 'Agustinus', 'SUBANDI': 'Subandi', 'MAN NUMERIC EXPLORER TRUST': 'Man Numeric', 'HENRY HARTONO': 'Henry Hartono', 'TRIONO JULIARSO DAWIS': 'Triono Juliarso Dawis', 'KB KITMC ASEAN EQT': 'Korea Investment Trust Mgt', 'EDWARD RAINO': 'Edward Raino', 'LISUSANTO': 'Lisusanto', 'SYLVIA': 'Sylvia', 'TEDDY': 'Teddy', 'PHILIP SURYA DJUSMAN': 'Philip Surya Djusman', 'ADE KURNIAWAN': 'Ade Kurniawan', 'THE INCUBATION FUND LTD': 'Wellington Mgt Company', 'IRMAN': 'Irman', 'ANGELIA': 'Angelia', 'SAMSUNG SECURITIES CO': 'Samsung Securities', 'BUDI MULYANA': 'Budi Mulyana', 'YUDI SUSANTO': 'Yudi Susanto', 'GOH WEE YIT': 'Goh Wee Yit', 'RICKY BASTIAN': 'Ricky Bastian', 'GUNADI WITEDJA': 'Gunadi Witedja', 'HENDARSIN LUKITO': 'Hendarsin Lukito', 'KEVIN JONATHAN': 'Kevin Jonathan', 'JHONNY VIRGO': 'Jhonny Virgo', 'HUAXU LIU': 'Huaxu Liu', 'HEO JOUNGROK': 'Heo Joungrok', 'KOJI OKUDERA': 'Koji Okudera', 'CLSA CAPITAL LTD': 'CLSA Hong Kong', 'IR ARIEF BUDIMAN': 'Ir Arief Budiman', 'SUKITO JIANNY': 'Sukito Jianny', 'JANE STREET CAPITAL': 'Jane Street Capital', 'HARDY SUNJARNI': 'Hardy Sunjarni', 'REKSA DANA PRATAMA DINAMIS SAHAM': 'Pratama Capital Assets Mgt', 'REKSA DANA PRATAMA INVESTA MANDIRI SAHAM': 'Pratama Capital Assets Mgt', 'REKSA DANA PRATAMA DANA ATRAKTIF SAHAM': 'Pratama Capital Assets Mgt', 'PRATAMA SAHAM': 'Pratama Capital Assets Mgt', 'REKSA DANA PRATAMA DANA GEMILANG SAHAM': 'Pratama Capital Assets Mgt', 'REKSA DANA PRATAMA CEMERLANG SAHAM': 'Pratama Capital Assets Mgt', 'REKSA DANA PRATAMA DANA ALPHA SAHAM': 'Pratama Capital Assets Mgt', 'SOEGITO POHARI': 'Soegito Pohari', 'HENRY DWI ADJIE FIRTANTO': 'Henry Dwi Adjie Firtanto', 'ANEN CANDRA TJEN': 'Anen Candra Tjen', 'ZUL AKHYAR MARTABAYA': 'Zul Akhyar Martabaya', 'DIAN SUGIARTO': 'Dian Sugiarto', 'A KUNTA WIDYATMAKA': 'A Kunta Widyatmaka', 'HANAFIA HOSEN': 'Hanafia Hosen', 'WONO BUDI TJAHYONO': 'Wono Budi Tjahyono', 'CHRISTIAN RICCARDO': 'Christian Riccardo', 'ARTICO SICAV ARTICO SEM': 'Christian Riccardo', 'PRESCIENT GLOBAL FUNDS ICAV': 'Christian Riccardo', 'BNP PARIBAS PARIS': 'Christian Riccardo', 'FAIRTREE GLOBAL EMERGING MARKETS': 'Christian Riccardo', 'IMPAX ENVIRONMENTAL MARKETS IRELAND FUND': 'Christian Riccardo', 'UNITED ASEAN DISCOVERY FUND': 'Christian Riccardo', 'BOSTON COMMON SUSTAINABLE EMERGING MARKETS FUND': 'Christian Riccardo', 'BOSTON COMMON ESG IMPACT INTERNATIONAL FUND': 'Christian Riccardo', 'BOSTON COMMON CATHOLIC EMERGING MARKETS FUND': 'Christian Riccardo', 'UPAMC GRAND ASEAN HIGH DIVIDEND FUND': 'Christian Riccardo', 'SANDS CAPITAL MANAGEMENT LLC': 'Christian Riccardo', 'SANDS CAPITAL EMERGING MARKETS GROWTH FUND': 'Christian Riccardo', 'TOUCHSTONE SANDS CAPITAL EMERGING MARKETS GROWTH FUND': 'Christian Riccardo', 'SANDS CAPITAL TEAM FUND': 'Christian Riccardo', 'SANDS CAPITAL EMERGING MARKETS GROWTH MASTER FUND': 'Christian Riccardo', 'SANDS CAPITAL EMERGING MARKETS EX CHINA GROWTH ETF': 'Christian Riccardo', 'SANDS CAPITAL EMERGING MARKETS EX CHINA FUND': 'Christian Riccardo', 'INTERNATIONAL EQUITY SUSTAINABLE LEADERS FUND': 'Christian Riccardo', 'FIDELITY EMERGING MARKETS QUALITY INCOME UCITS ETF': 'Christian Riccardo', 'MAMS TRUST': 'Christian Riccardo', 'LAZARD EMERING MARKETS EQUITY ADVANTAGE PORTFOLIO': 'Christian Riccardo', 'WILMINGTON EMERGING EQUITY ADVANTAGE FUND': 'Christian Riccardo', 'LAZARD WILMINGTON ACW EX US DIVERSIFIED ADVANTAGE FUND': 'Christian Riccardo', 'LACM ACWI EX US EQUITY FUND LP': 'Christian Riccardo', 'LACM ACWI EX US EQUITY CIT': 'Christian Riccardo', 'OMERS ADMINISTRATION CORPORATION': 'Christian Riccardo', 'BROWN ADVISORY GLOBAL LEADERS FUND': 'Christian Riccardo', 'BROWN ADVISORY GLOBAL LEADERS SUSTAINABLE FUND': 'Christian Riccardo', 'CATHAY LIFE INSURANCE COMPANY LTD': 'Christian Riccardo', 'DWS CONCEPT DJE GLOBALE AKTIEN': 'Christian Riccardo', 'FUH HWA RISING ASEAN FUND': 'Christian Riccardo', 'ODDO BHF ASSET MANAGEMENT GMBH': 'Christian Riccardo', 'FIDELITY SELECT GLOBAL PLUS ALL CAP EQUITY INSTITUTIONAL TRUST': 'Christian Riccardo', 'FIDELITY EMERGING MARKETS OPPORTUNITIES INSTITUTIONAL TRUST': 'Christian Riccardo', 'FIAM EMERGING MARKETS OPPORTUNITIES COMMINGLED POOL': 'Christian Riccardo', 'IG MACKENZIE PACIFIC INTERNATIONAL FUND': 'Christian Riccardo', 'GOVERNMENT OF THE PROVINCE OF ALBERTA': 'Christian Riccardo', 'CHAUTAUQUA INTERNATIONAL GROWTH EQUITY FUND': 'Christian Riccardo', 'PACE INTERNATIONAL EQUITY INVESTMENTS': 'Christian Riccardo', 'CHAUTAUQUA GLOBAL GROWTH EQUITY FUND': 'Christian Riccardo', 'NTC CIM INVESTMENT FUND ICAV': 'Christian Riccardo', 'ASSET MANAGEMENT EXCHANGE MASTER ICAV': 'Christian Riccardo', 'KAPITALFORENINGEN LAEGERNES INVEST KLT AKTIER EMERGING MARKETS INDEKS': 'Christian Riccardo', 'SUNARYO HONGGO': 'Christian Riccardo', 'WIJAYA HALIM': 'Christian Riccardo', 'FOURTON SILKKITIE ASIA EQUITY INVESTMENT FUND': 'Christian Riccardo', 'LINDRAWATI WIDJOJO': 'Christian Riccardo', 'HARBOR ETF TRUST HARBOR EMERGING MARKETS SELECT ETF': 'Christian Riccardo', 'VANDERBILT UNIVERSITY': 'Christian Riccardo', 'BUALUANG ASEAN EQUITY FUND': 'Christian Riccardo', 'BUALUANG ASEAN EQUITY RMF': 'Christian Riccardo', 'LISA SUNDARI': 'Christian Riccardo', 'CALAMOS ANTETOKOUNMPO GLOBAL SUSTAINABLE EQUITIES ETF': 'Christian Riccardo', 'TOWERS WATSON COMMON CONTRACTUAL FUND': 'Christian Riccardo', 'FINEX FUNDS ICAV': 'Christian Riccardo', 'BMO FOSSIL FUEL FREE FUND': 'Christian Riccardo', 'JANA EMERGING MARKETS SHARE TRUST': 'Christian Riccardo', 'FSSA GLOBAL EMERGING MARKET FOCUS FUND': 'Christian Riccardo', 'VCIM GLOBAL IMPACT FUND': 'Christian Riccardo', 'VIDENT INTERNATIONAL EQUITY FUND': 'Christian Riccardo', 'WILSHIRE SOLUTIONS FUNDS': 'Christian Riccardo', 'CAPE ANN GLOBAL DEVELOPING MARKETS FUND': 'Christian Riccardo', 'CAPE ANN PERIVALLON IRELAND FUND': 'Christian Riccardo', 'LUCAS SH CN': 'Christian Riccardo', 'TRINETRA EMERGING MARKETS GROWTH FUND': 'Christian Riccardo', 'HENDRAWAN WIRGHO': 'Christian Riccardo', 'DANA PENSIUN PROCTER AND GAMBLE HOME PRODUCTS INDONESIA': 'Christian Riccardo', 'MOBIUS LIFE LTD': 'Christian Riccardo', 'KRUNGTHAI AXA LIFE INSURANCE PUBLIC COMPANY LIMITED': 'Christian Riccardo', 'FANDY CENDRAWIRA': 'Christian Riccardo', 'REB EMERGING MARKETS FINANCIALS FUND': 'Christian Riccardo', 'NUVEEN MULTI ASSET INCOME FUND': 'Christian Riccardo', 'RUDY SANTOSO': 'Christian Riccardo', 'BROWN ADVISORY SUSTAINABLE INTERNATIONAL LEADERS FUND': 'Christian Riccardo', 'TJIN HWA ALIAS SUGIANTO': 'Christian Riccardo', 'MUHAMMAD SUHARTO': 'Christian Riccardo', 'THEO HANDOYO': 'Christian Riccardo', 'AGUSTINUS SIOR F': 'Christian Riccardo', 'DRS MOHAMMAD ANSOR ALAMSYAH': 'Christian Riccardo', 'LIE KWAN DIN': 'Christian Riccardo', 'BUDI YANTO P HUTAGAOL': 'Christian Riccardo', 'AGUS DWI HANDAYA SE': 'Christian Riccardo', 'EUROMOBILIARE PICTET ACTION': 'Christian Riccardo', 'EUROMOBILIARE PICTET ACTION 4 TRANSITION ATTO II': 'Christian Riccardo', 'EUROMOBILIARE PICTET GLOBAL TRENDS ESG': 'Christian Riccardo', 'JOSHA LAURENS': 'Christian Riccardo', 'WELLY LOVENNUS SURIPTO': 'Christian Riccardo', 'ELVITA': 'Christian Riccardo', 'ADI CAHYA': 'Christian Riccardo', 'LIE HIEN WENG': 'Christian Riccardo', 'KRISTOVHOL': 'Christian Riccardo', 'ANNAWATY': 'Christian Riccardo', 'GILBERT': 'Christian Riccardo', 'DIDI SATRIA SOEBAWA': 'Christian Riccardo', 'ANDREW WELLINGTON S CHAN': 'Christian Riccardo', 'PRIYASTOMO': 'Christian Riccardo', 'ISKANDAR MUDA KAMIL': 'Christian Riccardo', 'MUH GANI IRWANSYAH': 'Christian Riccardo', 'SOPHIA ALIZSA': 'Christian Riccardo', 'SUSILOWIRJONO': 'Christian Riccardo', 'IRFAN HADI WIJAYA': 'Christian Riccardo', 'G P RONALD SIHOMBING': 'Christian Riccardo', 'UNIFIED SERIES TRUST ONEASCENT EMERGING MARKETS ETF': 'Christian Riccardo', 'ANTONI': 'Christian Riccardo', 'GOLDMAN SACHS GLOBAL SOCIAL IMPACT EQUITY FUND': 'Christian Riccardo', 'CAHYO HARI PURWANTO': 'Christian Riccardo', 'EKO PRIATNO': 'Christian Riccardo', 'APRICUS CAPITAL HOLDINGS': 'Christian Riccardo', 'SAMUGRAHA DITY A KUSUMA': 'Christian Riccardo', 'BERMONT HOLDINGS 2 LIMITED': 'Christian Riccardo', 'BERMONT HOLDINGS 3 LIMITED': 'Christian Riccardo', 'BERMONT HOLDINGS 1 LIMITED': 'Christian Riccardo', 'CLEAR SKIES IWORLD EQUITY FUND': 'Christian Riccardo', 'THE BARING INTERNATIONAL UMBRELLA FUND BARING HONG KONG CHINA FUND': 'Christian Riccardo', 'FOURNIER RETIREMENT FUND': 'Christian Riccardo', 'YPW ASIAN EQUITY MOTHER FUND': 'Christian Riccardo', 'DYNAMIC EMERGING MARKETS EQUITY FUND': 'Christian Riccardo', 'DYNAMIC ACTIVE EMERGING MARKETS ETF': 'Christian Riccardo', 'BLACKROCK EMERGING MARKETS EX CHINA FUND OF BLACKROCK FUNDS': 'Christian Riccardo', 'FRANKLIN BRANDYWINE GLOBAL BALANCED FUND': 'Christian Riccardo', 'THE WELLINGTON TRUST COMPANY NATIONAL ASSOCIATION MULTIPLE COMMON INVESTMENT FUNDS GLOBAL RESEARCH EQUITY CATHOLIC VALUES': 'Christian Riccardo', 'WELLINGTON MANAGEMENT FUNDS LUXEMBOURG III SICAV': 'Christian Riccardo', 'BNP PARIBAS OBAM N V': 'Christian Riccardo', 'LCIV EMERGING MARKET EQUITY FUND': 'Christian Riccardo', 'HERMES INVESTMENT FUNDS PUBLIC': 'Christian Riccardo', 'KAPITALFORENINGEN EMERGING MARKETS AKTIER': 'Christian Riccardo', 'WEALTH INVEST AKL SEB EMERGING MARKET EQUITIES HERMES': 'Christian Riccardo', 'MARTIN CURRIE EMERGING MARKETS FUND': 'Christian Riccardo', 'STARRY HOMENTA RAMPENGAN': 'Christian Riccardo', 'RECAPITAL EQUITY': 'Christian Riccardo', 'WIDYA LAWRENZI': 'Christian Riccardo', 'FIFI AGUSTINI': 'Christian Riccardo', 'JIMMY VIRGO': 'Christian Riccardo', 'EFLI YUNEDI': 'Christian Riccardo', 'RACHMAT MAKKASAU': 'Christian Riccardo', 'IR BUDI HARTO MM': 'Christian Riccardo', 'TJIOE TJIN': 'Christian Riccardo', 'HARTARTO CIPUTRA': 'Christian Riccardo', 'RAHMAT TAHA': 'Christian Riccardo', 'KARADI HANAM': 'Christian Riccardo', 'RACHMAT GUNTUR SOEGIARTO': 'Christian Riccardo', 'DAVID SUNG': 'Christian Riccardo', 'LIE GUNAWAN SUTEJA': 'Christian Riccardo', 'DP KWI PT JUARA CAPITAL': 'Christian Riccardo', 'REKSA DANA JUARA PROVIDENTIA BALANCED FUND': 'Christian Riccardo', 'REKSA DANA JUARA EMPOWERMENT BALANCED FUND': 'Christian Riccardo', 'LENA DJAJA': 'Christian Riccardo', 'BOREALIS KAPITAL INVESTAMA': 'Christian Riccardo', 'SRI NILOWATI': 'Christian Riccardo', 'YUSAK ELSABAS': 'Christian Riccardo', 'TEDJA WIDJAJA': 'Christian Riccardo', 'PT ZEZE AJUWEL TAMBANG': 'Christian Riccardo', 'PT MANDALA MULTI CIPTA': 'Christian Riccardo', 'ARISTA RAHARJA': 'Christian Riccardo', 'RINA SUGIANTO': 'Christian Riccardo', 'ALBERT TJHIA': 'Christian Riccardo', 'LIOE KARNEN LUCKY H': 'Christian Riccardo', 'LUTHER KING': 'Christian Riccardo', 'NANAK TRISAKTI A': 'Christian Riccardo', 'BEN JASEN WANGSAJAYA': 'Christian Riccardo', 'HADI YUWONO IR': 'Christian Riccardo', 'CICILIA ANGELIA PRIBUDI': 'Christian Riccardo', 'TAN LIE SE': 'Christian Riccardo', 'VICTOR HIDAJAT': 'Christian Riccardo', 'PUTRA SABUDI': 'Christian Riccardo', 'PT DANA BANGSA SEJAHTERA': 'Christian Riccardo', 'SABAR GANDA L SITORUS': 'Christian Riccardo', 'SARANA KALYANA NUSANTARA': 'Christian Riccardo', 'RICKY TJANDRA': 'Christian Riccardo', 'KEVIN CHRISTIE MAMAHIT': 'Christian Riccardo', 'IG HADIANTO SUHARTONO': 'Christian Riccardo', 'REKSA DANA PURWANTO CAMPURAN DINAMIS': 'Christian Riccardo', 'REKSA DANA PURWANTO SAHAM BERKEMBANG': 'Christian Riccardo', 'LIDWINA ARIWIN S': 'Christian Riccardo', 'METY JUMIATI AND SPD': 'Christian Riccardo', 'UBETA ADIKUSNADI': 'Christian Riccardo', 'HARTATI': 'Christian Riccardo', 'HUBERT HARI SOEJONO': 'Christian Riccardo', 'IR ROBERT NAPITUPULU': 'Christian Riccardo', 'MILKEN JONATHAN': 'Christian Riccardo', 'KARTIKA JOHANES': 'Christian Riccardo', 'MARIANTO': 'Christian Riccardo', 'LAURA': 'Christian Riccardo', 'PT INDO MOTOR LESTARI': 'Christian Riccardo', 'CHRISTOPHER ALDO SETIAWAN': 'Christian Riccardo', 'ARIE PRIANGGA PUTRA': 'Christian Riccardo', 'IR MINTARTO HALIM': 'Christian Riccardo', 'PT ARTISAN SURYA KREASI': 'Christian Riccardo', 'ROBETH J': 'Christian Riccardo', 'DIANA NOLASARI': 'Christian Riccardo', 'OOK HANTO': 'Christian Riccardo', 'IMAN YANTO': 'Christian Riccardo', 'ADI DARMAWAN PRANATA': 'Christian Riccardo', 'PT AYODA CAPITAL GROUP': 'Christian Riccardo', 'BUN BUN TJEN': 'Christian Riccardo', 'MONICA HANDAYANI': 'Christian Riccardo', 'ANTHONY HUGO BARKI': 'Christian Riccardo', 'EVAN YUWANA': 'Christian Riccardo', 'EDY HARJITO': 'Christian Riccardo', 'VINCENT DADLANI': 'Christian Riccardo', 'GUSMAVIN WILLOPO': 'Christian Riccardo', 'MACQUARIE INTERNATIONAL MIMAL MULTI FACTOR FUND': 'Christian Riccardo', 'MACQUARIE SYSTEMATIC EMERGING MARKETS EQUITY FUND': 'Christian Riccardo', 'TWU SUPERANNUATION FUND': 'Christian Riccardo', 'DASARI DRA': 'Christian Riccardo', 'AGUNG SARYOKO': 'Christian Riccardo', 'TRI WAHYUNI DRA': 'Christian Riccardo', 'IB PUTU ANOM REDHI SH MM': 'Christian Riccardo', 'DARYANTO': 'Christian Riccardo', 'APRIL VIVADIJANTO': 'Christian Riccardo', 'JAMAR ROESLI': 'Christian Riccardo', 'MARINUS GEA': 'Christian Riccardo', 'PT ASURANSI JASA RAHARJA PUTERA': 'Christian Riccardo', 'TOMMY WITEDJA': 'Christian Riccardo', 'SUKIANTO': 'Christian Riccardo', 'ANG SOEI HOA': 'Christian Riccardo', 'MARTHA KURNIAWAN': 'Christian Riccardo', 'SALIM KUSWANTO': 'Christian Riccardo', 'PT SAPTA MANUNGGAL KARYA': 'Christian Riccardo', 'PATRICCO BARON': 'Christian Riccardo', 'ASAN': 'Christian Riccardo', 'YENNY MERYSA': 'Christian Riccardo', 'HIU JI TJHAT': 'Christian Riccardo', 'LEE MING KIT': 'Christian Riccardo', 'DJOKO SANTOSO': 'Christian Riccardo', 'PEETER LAUW': 'Christian Riccardo', 'TEGUH WIJAYA': 'Christian Riccardo', 'SBI SECURITIES': 'Christian Riccardo', 'JULIUS HALIM': 'Christian Riccardo', 'LIONG INDRA GUNAWAN': 'Christian Riccardo', 'OSCAR HUTAGAOL DRS MBA': 'Christian Riccardo', 'WILLIAM CHRISTOPHER': 'Christian Riccardo', 'WITA TUKAN': 'Christian Riccardo', 'DANI': 'Christian Riccardo', 'DJONI RION': 'Christian Riccardo', 'TATANG SURYANA': 'Christian Riccardo', 'M YUNUS': 'Christian Riccardo', 'ROBERT LIEMARTO': 'Christian Riccardo', 'EZIFRIANOS': 'Christian Riccardo', 'JARISLOWSKY FRASER EMERGING MARKETS EQUITY FUND': 'Christian Riccardo', 'FISHER INVESTMENTS INSTITUTIONAL FUNDS PLC': 'Christian Riccardo', 'FISHER INVESTMENTS EMERGING MARKETS EQUITY COLLECTIVE FUND': 'Christian Riccardo', 'FISHER INVESTMENTS ALL FOREIGN EQUITY COLLECTIVE FUND': 'Christian Riccardo', 'PLATINUM ARROW TRUST': 'Christian Riccardo', 'ALLSPRING GROUP HOLDINGS LLC': 'Christian Riccardo', 'AXA WORLD FUNDS': 'Christian Riccardo', 'INDUS SELECT MASTER FUND': 'Christian Riccardo', 'BRANDYWINE INVESTMENT TRUST': 'Christian Riccardo', 'BRANDYWINE GLOBAL OPPORTUNISTIC EQUITY FUND': 'Christian Riccardo', 'CANADA LIFE GLOBAL EQUITY AND INCOME FUND': 'Christian Riccardo', 'BRANDYWINE GLOBAL VALUE EQUITY MOTHER FUND': 'Christian Riccardo', 'BRANDYWINE GLOBAL OPPORTUNISTIC EQUITY MOTHER FUND': 'Christian Riccardo', 'IG MACKENZIE PAN ASIAN EQUITY FUND': 'Christian Riccardo', 'THE CIVIL SERVICE SUPERANNUATION FUND': 'Christian Riccardo', 'IG MACKENZIE PACIFIC INTERNATIONAL FUND II': 'Christian Riccardo', 'SUDINVEST A290': 'Christian Riccardo', 'TRANSAMERICA INTERNATIONAL SUSTAINABLE EQUITY': 'Christian Riccardo', 'EWING MARION KAUFFMAN FOUNDATION': 'Christian Riccardo', 'MACQUARIE FINANCIAL HOLDINGS LIMITED': 'Christian Riccardo', 'INVESCO MACRO ALLOCATION STRATEGY FUND': 'Christian Riccardo', 'INVESCO ADVANTAGE INTERNATIONAL FUND': 'Christian Riccardo', 'HSBC CONTINENTAL EUROPE': 'Christian Riccardo', 'OVERLOOK PARTNERS FUND': 'Christian Riccardo', 'GELA M BHD LION ADVANCED STRATEGIC FUND': 'Christian Riccardo', 'JPMORGAN CHASE PB WILLIAM VON MUEFFLING': 'Christian Riccardo', 'REST RETAIL EMPLOYEES SUPERANNUATION TRUST': 'Christian Riccardo', 'HESTA HEALTH EMPLOYEES SUPERANNUATION TRUST OF AUSTRALIA': 'Christian Riccardo', 'TRG EMERGING HORIZON SELECT EQUITY MASTER FUND': 'Christian Riccardo', 'MUTB400038708': 'Christian Riccardo', 'MTBJ400030467': 'Christian Riccardo', 'HSBC BANK PLC A C IB MAIN ACCOUNT': 'Christian Riccardo', 'NH CA POWER ASEAN PLUS SIT 1': 'Christian Riccardo', 'RHB ASSET MANAGEMENT SDN BHD': 'Christian Riccardo', 'AFFIN HWANG SELECT DIVIDEND FUND': 'Christian Riccardo', 'AFFIN HWANG ASEAN FLEXI FUND': 'Christian Riccardo', 'AFFIN HWANG SELECT ASIA PACIFIC EX JAPAN DIVIDEND FUND': 'Christian Riccardo', 'AHAM SELECT ASIA EX JAPAN OPPORTUNITY FUND': 'Christian Riccardo', 'THE TRUSTEES OF THE UNIVERSITY OF PENNSYLVANIA': 'Christian Riccardo', 'CANTILLON GLOBAL EQUITY LP': 'Christian Riccardo', 'CANTILLON FUNDS PLC CANTILLON GLOBAL EQUITY FUND': 'Christian Riccardo', 'RUTGERS THE STATE UNIVERSITY OF NEW JERSEY': 'Christian Riccardo', 'VISION FUND': 'Christian Riccardo', 'CFSIL COLONIAL FIRST STATE GLOBAL SHARES FUND 8': 'Christian Riccardo', 'AMCUMULATIVE GROWTH': 'Christian Riccardo', 'AMPRS MODERATE FUND': 'Christian Riccardo', 'AMPRS GROWTH FUND': 'Christian Riccardo', 'FIAM SELECT EMERGING MARKETS EQUITY TRUST': 'Christian Riccardo', 'FIAM SELECT EMERGING MARKETS EQUITY': 'Christian Riccardo', 'FIDELITY SAI SUSTAINABLE EMERGING MARKETS EQUITY FUND': 'Christian Riccardo', 'FIAM SELECT GLOBAL PLUS ALL CAP COMMINGLED POOL': 'Christian Riccardo', 'DEBEKA AKTIEN ASIEN ESG': 'Christian Riccardo', 'CLEARBRIDGE GLOBAL VALUE IMPROVERS FUND': 'Christian Riccardo', 'SCOTIA EMERGING MARKETS EQUITY FUND': 'Christian Riccardo', 'CLEARBRIDGE CDIP FEEDER FUND LP': 'Christian Riccardo', 'FTGF CLEARBRIDGE GLOBAL SUSTAINABILITY IMPROVERS FUND': 'Christian Riccardo', 'STICHTING JURIDISCH EIGENDOM ABN AMRO INVESTMENT SOLUTIONS': 'Christian Riccardo', 'EUROMOBILIARE INTERNATIONAL FUND QFUND90 RELOAD': 'Christian Riccardo', 'THE CHANCELLOR MASTERS AND SCHOLARS OF THE UNIVERSITY OF CAMBRIDGE': 'Christian Riccardo', 'YANTO PRANATA': 'Christian Riccardo', 'HIMAWAN YOEWONO': 'Christian Riccardo', 'TJOE WAN TJIN': 'Christian Riccardo', 'AYUB MUHAMAD ASA DARMA': 'Christian Riccardo', 'BANQUE CANTONALE VAUDOISE': 'Christian Riccardo', 'CANLIFE ASIA PACIFIC FUND': 'Christian Riccardo', 'IRONBARK ROYAL LONDON CONCENTRATED GLOBAL SHARE FUND': 'Christian Riccardo', 'ST JAMES S PLACE STRATEGIC MANAGED UNIT TRUST': 'Christian Riccardo', 'PICTET GLOBAL SELECTION FUND GLOBAL HIGH YIELD EMERGING EQUITIES FUND': 'Christian Riccardo', 'ASEAN EQUITY MOTHER FUND': 'Christian Riccardo', 'PERSIMMON PLC CM CIC ASSET MANAGEMENT': 'Christian Riccardo', 'NINETY ONE FUND SERIES CAUTIOUS MANAGED FUND': 'Christian Riccardo', 'NINETY ONE GLOBAL STRATEGY FUND': 'Christian Riccardo', 'NINETY ONE FUNDS SERIES III GLOBAL EQUITY FUND': 'Christian Riccardo', 'NINETY ONE FUNDS SERIES I DIVERSIFIED INCOME FUND': 'Christian Riccardo', 'VANGUARD GROUP': 'Vanguard Group', 'BLACKROCK INVESTMENT MGT INDEX': 'BlackRock Investment Mgt - Index', 'INVESCO': 'Invesco', 'EMPLOYEES PROVIDENT FUND BOARD': 'Employees Provident Fund Board', 'PZENA INVESTMENT MGT': 'Pzena Investment Mgt', 'ARGA INVESTMENT MGT': 'ARGA Investment Mgt', 'EASTSPRING INVESTMENTS': 'Eastspring Investments', 'STATE STREET INVESTMENT MGT': 'State Street Investment Mgt', 'ROBECO ASSET MGT': 'Robeco Asset Mgt', 'PGGM INVESTMENTS': 'PGGM Investments', 'M AND G INVESTMENTS': 'M&G Investments', 'LEGAL AND GENERAL INVESTMENT MGT': 'Legal & General Investment Mgt', 'TASPEN': 'Taspen', 'MONDRIAN INVESTMENT PARTNERS': 'Mondrian Investment Partners', 'HSBC GLOBAL ASSET MGT': 'HSBC Global Asset Mgt', 'T ROWE PRICE INTERNATIONAL': 'T Rowe Price International', 'GEODE CAPITAL MGT': 'Geode Capital Mgt', 'PYRFORD INTERNATIONAL': 'Pyrford International', 'AQR CAPITAL MGT': 'AQR Capital Mgt', 'STOREBRAND INVESTMENTS': 'Storebrand Investments', 'ACADIAN ASSET MGT': 'Acadian Asset Mgt', 'NORTHERN TRUST GLOBAL INVESTMENTS': 'Northern Trust Global Investments', 'APG ASSET MGT ASIA': 'APG Asset Mgt Asia', 'MANULIFE INVESTMENT MGT INDONESIA': 'Manulife Investment Mgt Indonesia', 'JPMORGAN ASSET MGT': 'JPMorgan Asset Mgt', 'NORGES BANK INVESTMENT MGT': 'Norges Bank Investment Mgt', 'BNP PARIBAS ASSET MGT': 'BNP Paribas Asset Mgt', 'MANDIRI INVESTASI': 'Mandiri Investasi', 'BATAVIA PROSPERINDO ASSET MGT': 'Batavia Prosperindo Asset Mgt', 'FIDELITY INVESTMENTS': 'Fidelity Investments', 'T ROWE PRICE': 'T Rowe Price', 'CHARLES SCHWAB INVESTMENT MGT': 'Charles Schwab Investment Mgt', 'DIMENSIONAL FUND ADVISORS': 'Dimensional Fund Advisors', 'INDO PREMIER INVESTMENT MGT': 'Indo Premier Investment Mgt', 'BRI INVESTMENT MGT': 'BRI Investment Mgt', 'AIA INDONESIA': 'AIA Indonesia', 'BAHANA TCW INVESTMENT MGT': 'Bahana TCW Investment Mgt', 'INVESCO ASSET MGT ASIA': 'Invesco Asset Mgt Asia', 'BUREAU OF LABOR FUNDS TAIWAN': 'Bureau of Labor Funds Taiwan', 'ARROWSTREET CAPITAL': 'Arrowstreet Capital', 'PANIN ASSET MGT': 'Panin Asset Mgt', 'ABU DHABI INVESTMENT AUTHORITY ADIA': 'Abu Dhabi Investment Authority (ADIA)', 'BRITISH COLUMBIA INVESTMENT MGT CORPORATION': 'British Columbia Investment Mgt Corporation', 'NEWTON INVESTMENT MGT': 'Newton Investment Mgt', 'MANULIFE INVESTMENT MGT': 'Manulife Investment Mgt', 'SUCORINVEST ASSET MGT': 'SucorInvest Asset Mgt', 'ALLIANZ LIFE INSURANCE': 'Allianz Life Insurance', 'COLORADO PERA': 'Colorado PERA', 'MELLON': 'Mellon', 'GLOBAL X MGT': 'Global X Mgt', 'SJUNDE AP FONDEN': 'Sjunde AP-fonden', 'ROBECO': 'Robeco', 'SCHRODER INVESTMENT MGT': 'Schroder Investment Mgt', 'JPMORGAN ASSET MGT JF ASSET MGT': 'JPMorgan Asset Mgt (JF Asset Mgt)', 'MACKENZIE INVESTMENTS': 'Mackenzie Investments', 'FIL INVESTMENT MGT': 'FIL Investment Mgt', 'NORTH CAROLINA DEPARTMENT OF STATE TREASURER': 'North Carolina Department of State Treasurer', 'AMERICAN CENTURY INVESTMENT MGT': 'American Century Investment Mgt', 'UBS GLOBAL ASSET MGT': 'UBS Global Asset Mgt', 'ALLSPRING GLOBAL INVESTMENTS': 'Allspring Global Investments', 'RESEARCH AFFILIATES': 'Research Affiliates', 'PT BANK RAKYAT INDONESIA PERSERO TBK': 'PT Bank Rakyat Indonesia (Persero) Tbk.', 'GOLDMAN SACHS ASSET MGT': 'Goldman Sachs Asset Mgt', 'ASHMORE ASSET MGT': 'Ashmore Asset Mgt', 'BNI ASSET MGT': 'BNI Asset Mgt', 'SYAILENDRA CAPITAL': 'Syailendra Capital', 'ALASKA PERMANENT FUND CORP': 'Alaska Permanent Fund Corp.', 'DWS INVESTMENTS': 'DWS Investments', 'ABERDEEN SWIP': 'aberdeen (SWIP)', 'TIAA INVESTMENT MGT': 'TIAA Investment Mgt', 'GIC': 'GIC', 'HANDELSBANKEN FONDER INDEX': 'Handelsbanken Fonder - Index', 'ILLINOIS MUNICIPAL RETIREMENT FUND IMRF': 'Illinois Municipal Retirement Fund (IMRF)', 'MITSUBISHI UFJ ASSET MGT': 'Mitsubishi UFJ Asset Mgt', 'UNIVERSAL INVESTMENT': 'Universal Investment', 'GENERATION INVESTMENT MGT': 'Generation Investment Mgt', 'PRUSIK INVESTMENT MGT': 'Prusik Investment Mgt', 'TRIMEGAH ASSET MGT': 'Trimegah Asset Mgt', 'CALPERS': 'CalPERS', 'PUBLIC MUTUAL': 'Public Mutual', 'NEW MEXICO STATE INVESTMENT COUNCIL': 'New Mexico State Investment Council', 'COLUMBIA THREADNEEDLE INVESTMENTS': 'Columbia Threadneedle Investments', 'STEWART INVESTORS': 'Stewart Investors', 'UBS SECURITIES': 'UBS Securities', 'SUMITOMO MITSUI DS ASSET MGT': 'Sumitomo Mitsui DS Asset Mgt', 'EDGBASTON INVESTMENT PARTNERS': 'Edgbaston Investment Partners', 'ENSIGN PEAK ADVISORS': 'Ensign Peak Advisors', 'PUBLIC EMPLOYEE RETIREMENT SYSTEM OF IDAHO': 'Public Employee Retirement System of Idaho', 'THE PUBLIC INSTITUTION FOR SOCIAL SECURITY': 'The Public Institution for Social Security', 'KBC ASSET MGT': 'KBC Asset Mgt', 'KUWAIT INVESTMENT OFFICE': 'Kuwait Investment Office', 'UBP ASSET MGT': 'UBP Asset Mgt', 'CIBC ASSET MGT': 'CIBC Asset Mgt', 'MORGAN STANLEY ASIA': 'Morgan Stanley Asia', 'WESPATH INVESTMENT MGT': 'Wespath Investment Mgt', 'PSP INVESTMENTS': 'PSP Investments', 'OHIO PUBLIC EMPLOYEES RETIREMENT SYSTEM OPERS': 'Ohio Public Employees Retirement System (OPERS)', 'HSBC SECURITIES': 'HSBC Securities', 'FIDEURAM ASSET MGT': 'Fideuram Asset Mgt', 'MACQUARIE BANK': 'Macquarie Bank', 'PIMCO': 'PIMCO', 'MATTHEWS INTERNATIONAL CAPITAL MGT': 'Matthews International Capital Mgt', 'HENAN PUTIHRAI ASSET MGT': 'Henan Putihrai Asset Mgt', 'KEYRIDGE ASSET MGT': 'Keyridge Asset Mgt', 'PUBLIC EMPLOYEES PENSION PLAN PEPP': 'Public Employees Pension Plan (PEPP)', 'LGPS CENTRAL': 'LGPS Central', 'RUTH ASSET MGT': 'Ruth Asset Mgt', 'BARCLAYS': 'Barclays', 'GUINNESS ASSET MGT': 'Guinness Asset Mgt', 'VICTORY CAPITAL MGT': 'Victory Capital Mgt', 'SUN LIFE FINANCIAL': 'Sun Life Financial', 'STATE OF NEW JERSEY DIVISION OF INVESTMENT': 'State of New Jersey - Division of Investment', 'BLACKROCK JAPAN INDEX': 'BlackRock Japan - Index', 'ABERDEEN': 'aberdeen', 'AVIVA INVESTORS': 'Aviva Investors', 'NOMURA ASSET MGT': 'Nomura Asset Mgt', 'INVESCO CAPITAL MGT': 'Invesco Capital Mgt', 'GMO': 'GMO', 'LAZARD ASSET MGT': 'Lazard Asset Mgt', 'ABU DHABI RETIREMENT PENSIONS AND BENEFITS FUND': 'Abu Dhabi Retirement Pensions and Benefits Fund', 'EQ ASSET MGT': 'eQ Asset Mgt', 'LION GLOBAL INVESTORS': 'Lion Global Investors', 'KOREA INVESTMENT CORPORATION': 'Korea Investment Corporation', 'PICTET ASSET MGT': 'Pictet Asset Mgt', 'AXA ASSET MGT INDONESIA': 'AXA Asset Mgt Indonesia', 'ARTISAN PARTNERS': 'Artisan Partners', 'FIM ASSET MGT': 'FIM Asset Mgt', 'GOLDMAN SACHS': 'Goldman Sachs', 'CONNOR CLARK AND LUNN INVESTMENT MGT': 'Connor Clark & Lunn Investment Mgt', 'ALLIANZ GLOBAL INVESTORS': 'Allianz Global Investors', 'SUMITOMO MITSUI TRUST ASSET MGT': 'Sumitomo Mitsui Trust Asset Mgt', 'WAYSTONE MGT': 'Waystone Mgt', 'HSBC INKA': 'HSBC INKA', 'AVRIST ASSET MGT': 'Avrist Asset Mgt', 'FIL INVESTMENT INTERNATIONAL': 'FIL Investment International', 'AMUNDI ASSET MGT': 'Amundi Asset Mgt', 'RUSSELL INVESTMENTS': 'Russell Investments', 'YAYASAN KESEHATAN PEGAWAI TELKOM': 'Yayasan Kesehatan Pegawai Telkom', 'CATERPILLAR MASTER RETIREMENT TRUST': 'Caterpillar Master Retirement Trust', 'RBC GLOBAL ASSET MGT': 'RBC Global Asset Mgt', 'KENTUCKY TEACHERS RETIREMENT SYSTEM': "Kentucky Teachers' Retirement System", 'STICHTING PENSIOENFONDS HOOGOVENS': 'Stichting Pensioenfonds Hoogovens', 'UOB ASSET MGT': 'UOB Asset Mgt', 'CONSTRUCTION AND BUILDING INDUSTRY SUPER CBUS': 'Construction and Building Industry Super - Cbus', 'MN SERVICES': 'MN Services', 'GENERAL ORGANIZATION FOR SOCIAL INSURANCE': 'General Organization for Social Insurance', 'WARUNA NUSASENTANA': 'Waruna Nusasentana', 'SEB ASSET MGT': 'SEB Asset Mgt', 'JO HAMBRO CAPITAL MGT': 'JO Hambro Capital Mgt', 'FINECO ASSET MGT': 'Fineco Asset Mgt', 'SWEDBANK ROBUR': 'Swedbank Robur', 'CARDANO': 'Cardano', 'THOMPSON SIEGEL AND WALMSLEY': 'Thompson, Siegel & Walmsley', 'SEQUISLIFE': 'Sequislife', 'DAPENBI': 'Dapenbi', 'COOK COUNTY PENSION FUND': 'Cook County Pension Fund', 'FWD ASSET MGT': 'FWD Asset Mgt', 'NYKREDIT ASSET MGT': 'Nykredit Asset Mgt', 'PARAMETRIC PORTFOLIO ASSOCIATES': 'Parametric Portfolio Associates', 'PUBLIC SCHOOL TEACHERS RETIREMENT FUND OF CHICAGO': 'Public School Teachers Retirement Fund of Chicago', 'ALLIANCEBERNSTEIN GROWTH': 'AllianceBernstein - Growth', 'PENSIONDANMARK': 'PensionDanmark', 'HARDING LOEVNER MGT': 'Harding Loevner Mgt', 'VANECK GLOBAL': 'VanEck Global', 'FIRST SENTIER INVESTORS RQI': 'First Sentier Investors RQI', 'KLP KAPITALFORVALTNING': 'KLP Kapitalforvaltning', 'SG SECURITIES': 'SG Securities', 'RESONA ASSET MGT': 'Resona Asset Mgt', 'ZKB ASSET MGT': 'ZKB Asset Mgt', 'BMO ASSET MGT': 'BMO Asset Mgt', 'AXA INVESTMENT MGRS ASIA': 'AXA Investment Mgrs Asia', 'OHIO STATE TEACHERS RETIREMENT SYSTEM': 'Ohio State Teachers Retirement System', 'GREAT EASTERN LIFE ASSURANCE': 'Great Eastern Life Assurance', 'COMGEST': 'Comgest', 'PINNACLE INVESTMENT': 'Pinnacle Investment', 'HANG SENG INVESTMENT MGT': 'Hang Seng Investment Mgt', 'TCW TRUST COMPANY OF THE WEST': 'TCW - Trust Company of the West', 'DODGE AND COX': 'Dodge & Cox', 'PICTET AND CIE': 'Pictet & Cie', 'EARNEST PARTNERS': 'Earnest Partners', 'QIC ASSET MGT': 'QIC Asset Mgt', 'TUGU PRATAMA INDONESIA': 'Tugu Pratama Indonesia', 'EURIZON CAPITAL': 'Eurizon Capital', 'BI ASSET MGT': 'BI Asset Mgt', 'ASSET MGT ONE SINGAPORE': 'Asset Mgt One Singapore', 'GENERALI INDONESIA': 'Generali Indonesia', 'INSIGHT INVESTMENTS MGT': 'Insight Investments Mgt', 'SEQUIS ASSET MGT': 'Sequis Asset Mgt', 'ALLIANZ GLOBAL INVESTORS ASSET MGT INDONESIA': 'Allianz Global Investors Asset Mgt Indonesia', 'ARIA INVESTMENTS': 'Aria Investments', 'BLACKROCK INVESTMENT MGT AUSTRALIA': 'BlackRock Investment Mgt (Australia)', 'AEGON ASSET MGT': 'AEGON Asset Mgt', 'NUVEEN ASSET MGT': 'Nuveen Asset Mgt', 'AIA': 'AIA', 'BOCI PRUDENTIAL ASSET MGT': 'BOCI Prudential Asset Mgt', 'OPAL CAPITAL': 'Opal Capital', 'WILLIAM BLAIR INVESTMENT MGT': 'William Blair Investment Mgt', 'DEKA INVESTMENT MGT': 'Deka Investment Mgt', 'DNB ASSET MGT': 'DNB Asset Mgt', 'PRINCIPAL ASSET MGT': 'Principal Asset Mgt', 'AXA INVESTMENT MGRS SCHWEIZ': 'AXA Investment Mgrs Schweiz', 'SAUDI CENTRAL BANK': 'Saudi Central Bank', 'GUINNESS ATKINSON ASSET MGT': 'Guinness Atkinson Asset Mgt', 'PT EKADHARMA INTERNATIONAL': 'Pt Ekadharma International', 'HOSKING PARTNERS': 'Hosking Partners', 'NISSAY ASSET MGT': 'Nissay Asset Mgt', 'RAIFFEISEN CAPITAL MGT': 'Raiffeisen Capital Mgt', 'SAMSUNG INVESTMENT MGT': 'Samsung Investment Mgt', 'METROPOLIS CAPITAL': 'Metropolis Capital', 'DANSKE CAPITAL MGT': 'Danske Capital Mgt', 'TOKIO MARINE ASSET MGT': 'Tokio Marine Asset Mgt', 'LUMIERE CAPITAL': 'Lumiere Capital', 'SAMUEL ASSET MGT': 'Samuel Asset Mgt', 'ASSET MGT ONE': 'Asset Mgt One', 'MORGAN STANLEY INVESTMENT MGT': 'Morgan Stanley Investment Mgt', 'POLARIS CAPITAL MGT': 'Polaris Capital Mgt', 'MAYBANK ASSET MGT': 'Maybank Asset Mgt', 'PREMIA PARTNERS': 'Premia Partners', 'GROW INVESTMENTS': 'Grow Investments', 'STAR ASSET MGT': 'Star Asset Mgt', 'AMOVA ASSET MGT': 'Amova Asset Mgt', 'QATAR INVESTMENT AUTHORITY': 'Qatar Investment Authority', 'YUANTA SECURITIES INVESTMENT TRUST': 'Yuanta Securities Investment Trust', 'DESJARDINS GLOBAL ASSET MGT': 'Desjardins Global Asset Mgt', 'ASURANSI WAHANA TATA': 'Asuransi Wahana Tata', 'JASA MARGA': 'Jasa Marga', 'SPARINVEST': 'Sparinvest', 'DAIWA ASSET MGT': 'Daiwa Asset Mgt', 'DELTA WIBAWA BERSAMA': 'Delta Wibawa Bersama', 'RHUMBLINE ADVISERS': 'RhumbLine Advisers', 'OLD MUTUAL INVESTMENT GROUP': 'Old Mutual Investment Group', 'KRESNA SECURITIES': 'Kresna Securities', 'AGGREGATE ASSET MGT': 'Aggregate Asset Mgt', 'VINVA INVESTMENT MGT': 'Vinva Investment Mgt', 'MAJ INVEST': 'Maj Invest', 'DANA PENSIUN ANGKASA PURA': 'Dana Pensiun Angkasa Pura', 'THESIS ASSET MGT': 'Thesis Asset Mgt', 'SEI INVESTMENTS': 'SEI Investments', 'LANNEBO ASSET MGT': 'Lannebo Asset Mgt', 'BAYERNINVEST': 'BayernInvest', 'FRANKLIN TEMPLETON INVESTMENTS': 'Franklin Templeton Investments', 'MACQUARIE ASSET MGT': 'Macquarie Asset Mgt', 'IFM INVESTORS': 'IFM Investors', 'MIRAE ASSET SECURITIES': 'Mirae Asset Securities', 'FUTURE FUND': 'Future Fund', 'PNM INVESTMENT MGT': 'PNM Investment Mgt', 'NORDEA INVESTMENT MGT': 'Nordea Investment Mgt', 'PREMIER MITON INVESTORS': 'Premier Miton Investors', 'CAPITAL INVESTMENT TRUST CORPORATION': 'Capital Investment Trust Corporation', 'SS AND C ALPS ADVISORS': 'SS&C ALPS Advisors', 'RAKUTEN INVESTMENT MGT': 'Rakuten Investment Mgt', 'MAJORIS ASSET MGT': 'Majoris Asset Mgt', 'OPSEU PENSION TRUST': 'OPSEU Pension Trust', 'AMUNDI HONG KONG': 'Amundi Hong Kong', 'MERLIN FIDELIS ASSET MGT': 'Merlin Fidelis Asset Mgt', 'RATHBONE INVESTMENT MGT': 'Rathbone Investment Mgt', 'PANAGORA ASSET MGT': 'PanAgora Asset Mgt', 'RESONA BANK': 'Resona Bank', 'VOYA INVESTMENT MGT': 'Voya Investment Mgt', 'AMF PENSIONS': 'AMF Pensions', 'JENSEN INVESTMENT MGT': 'Jensen Investment Mgt', 'AMUNDI SINGAPORE': 'Amundi Singapore', 'ARCA FONDI SGR': 'Arca Fondi SGR', 'TD ASSET MGT': 'TD Asset Mgt', 'HOLBERG FONDENE': 'Holberg Fondene', 'INVESCO OPPENHEIMERFUNDS': 'Invesco (OppenheimerFunds)', 'INSPIRE ADVISORS': 'Inspire Advisors', 'LOCALTAPIOLA ASSET MGT': 'LocalTapiola Asset Mgt', 'PRINCIPAL GLOBAL INVESTORS': 'Principal Global Investors', 'CSOP ASSET MGT': 'CSOP Asset Mgt', 'AURORA ASSET MGT': 'Aurora Asset Mgt', 'METZLER ASSET MGT': 'Metzler Asset Mgt', 'INTECH INVESTMENT MGT': 'INTECH Investment Mgt', 'MIGROS BANK': 'Migros Bank', 'FINREON': 'Finreon', 'GUARDIAN TRUST': 'Guardian Trust', 'ARROW INVESTMENT ADVISORS': 'Arrow Investment Advisors', 'COOPER INVESTORS': 'Cooper Investors', 'FEDERATED HERMES': 'Federated Hermes', 'LOS ANGELES COUNTY EMPLOYEES RETIREMENT ASSOCIATION LACERA': 'Los Angeles County Employees Retirement Association (LACERA)', 'CHINA INVESTMENT CORPORATION': 'China Investment Corporation', 'YAY KESEJAHTERAAN PEGAWAI PERUM PERHUTANI JATI SEJAHTERA': 'Yay Kesejahteraan Pegawai Perum Perhutani Jati Sejahtera', 'SUMITOMO MITSUI TRUST BANK': 'Sumitomo Mitsui Trust Bank', 'FULLERTON FUND MGT': 'Fullerton Fund Mgt', 'LMCG INVESTMENTS': 'LMCG Investments', 'ASSETPLUS INVESTMENT MGT': 'Assetplus Investment Mgt', 'RUSSELL INVESTMENTS IMPLEMENTATION SERVICES': 'Russell Investments Implementation Services', 'BANOR CAPITAL': 'Banor Capital', 'NATIONAL PENSION SERVICE FUND MGT': 'National Pension Service Fund Mgt', 'PROSPERA ASSET MGT': 'Prospera Asset Mgt', 'FIRST SENTIER INVESTORS': 'First Sentier Investors', 'QUONIAM ASSET MGT': 'Quoniam Asset Mgt', 'PERPETUAL INVESTMENTS': 'Perpetual Investments', 'DAI ICHI LIFE INSURANCE CO': 'Dai-Ichi Life Insurance Co.', 'CI GLOBAL ASSET MGT': 'CI Global Asset Mgt', 'FONDACO LUX': 'Fondaco Lux', 'DBX ADVISORS': 'DBX Advisors', 'LUMBUNG INDUSTRI PERSADA': 'Lumbung Industri Persada', 'CHUBB LIFE INSURANCE': 'Chubb Life Insurance', 'SETIABUDI INVESTMENT MGT': 'Setiabudi Investment Mgt', 'STRIVE ASSET MGT': 'Strive Asset Mgt', 'RELIANCE INVESTMENT MGT': 'Reliance Investment Mgt', 'KRANE FUNDS ADVISORS': 'Krane Funds Advisors', 'ANIMA': 'Anima', 'MITSUBISHI UFJ TRUST AND BANKING CORPORATION': 'Mitsubishi UFJ Trust & Banking Corporation', 'BANK OF NEW ZEALAND': 'Bank of New Zealand', 'VONTOBEL ASSET MGT': 'Vontobel Asset Mgt', 'FIRST TRUST ADVISORS': 'First Trust Advisors', 'PENDAL GROUP': 'Pendal Group', 'MOUNT MURRAY INVESTMENT': 'Mount Murray Investment', 'TUGU MANDIRI': 'Tugu Mandiri', 'GUINNESS GLOBAL INVESTORS': 'Guinness Global Investors', 'PHEIM ASSET MGT': 'Pheim Asset Mgt', 'DANAPATHI ASSET MANAGEMENT': 'Danapathi Asset Management', 'GAM INVESTMENT MGT': 'GAM Investment Mgt', 'CAPITAL DYNAMICS': 'Capital Dynamics', 'AZIMUT INVESTMENTS': 'Azimut Investments', 'AGF INTERNATIONAL ADVISORS': 'AGF International Advisors', 'ASHMORE INVESTMENT MGT': 'Ashmore Investment Mgt', 'SOLSTEIN CAPITAL': 'Solstein Capital', 'OSO INVESTMENT MGT': 'Oso Investment Mgt', 'VALVERDE INVESTMENT PARTNERS': 'Valverde Investment Partners', 'TUTTLE CAPITAL MGT': 'Tuttle Capital Mgt', 'JUPITER ASSET MGT': 'Jupiter Asset Mgt', 'FIRST SECURITIES INVESTMENT TRUST': 'First Securities Investment Trust', 'MARATHON ASSET MGT': 'Marathon Asset Mgt', 'ASIA GROWTH CAPITAL MGT': 'Asia Growth Capital Mgt', 'SHELL ASSET MGT': 'Shell Asset Mgt', 'INDOASIA ASSET MGT': 'Indoasia Asset Mgt', 'CAUSEWAY CAPITAL MGT': 'Causeway Capital Mgt', 'BOOSTER INVESTMENT MGT': 'Booster Investment Mgt', 'SINARMAS ASSET MGT': 'Sinarmas Asset Mgt', 'STATE STREET GLOBAL MARKETS TRANSITION': 'State Street Global Markets (Transition)', 'C WORLDWIDE ASSET MGT': 'C WorldWide Asset Mgt', 'CIPTADANA ASSET MGT': 'Ciptadana Asset Mgt', 'VALBURY CAPITAL MGT': 'Valbury Capital Mgt', 'OSMOSIS INVESTMENT MGT': 'Osmosis Investment Mgt', 'OFI INVEST ASSET MGT': 'OFI Invest Asset Mgt', 'FIVV': 'FIVV', 'GAMMA INVESTING': 'Gamma Investing', 'KISI ASSET MGT': 'KISI Asset Mgt', 'APERIO GROUP': 'Aperio Group', 'COMMERZBANK': 'Commerzbank', 'MNC ASSET MGT': 'MNC Asset Mgt', 'TREASURE FUND INVESTAMA': 'Treasure Fund Investama', 'NET ASSETS MGT': 'Net Assets Mgt', 'AXA IM SELECT': 'AXA IM Select', 'CAPITAL ASSET MGT': 'Capital Asset Mgt', 'DANAKITA INVESTAMA': 'Danakita Investama', 'MIRAE ASSET GLOBAL INVESTMENTS': 'Mirae Asset Global Investments', 'GLOBAL X': 'Global X', 'NARADA ASSET MGT': 'Narada Asset Mgt', 'AVANZA BANK': 'Avanza Bank', 'PNC WEALTH MGT': 'PNC Wealth Mgt', 'A J CENTRAL ASIA RAYA': 'A.J. Central Asia Raya', 'KBI GLOBAL INVESTORS': 'KBI Global Investors', 'CLARIVEST ASSET MGT': 'ClariVest Asset Mgt', 'MASOU INVESTMENT MGT': 'Masou Investment Mgt', 'INA SEKURITAS INDONESIA': 'Ina Sekuritas Indonesia', 'SWISS ASIA CAPITAL': 'Swiss Asia Capital', 'STANLIB ASSET MGT': 'STANLIB Asset Mgt', 'CATHAY SECURITIES INVESTMENT TRUST': 'Cathay Securities Investment Trust', 'BARCLAYS WEALTH': 'Barclays Wealth', 'WANTEG ASSET MGT': 'Wanteg Asset Mgt', 'SHINHAN ASSET MGT': 'Shinhan Asset Mgt', 'VANGUARD INVESTMENTS AUSTRALIA': 'Vanguard Investments Australia', 'LIF INVESTMENT MGT': 'LiF Investment Mgt', 'MAN NUMERIC': 'Man Numeric', 'WESTSIDE INVESTMENT MGT': 'Westside Investment Mgt', 'KOREA INVESTMENT TRUST MGT': 'Korea Investment Trust Mgt', 'WELLINGTON MGT COMPANY': 'Wellington Mgt Company', 'SAMSUNG SECURITIES': 'Samsung Securities', 'CLSA HONG KONG': 'CLSA Hong Kong', 'PRATAMA CAPITAL ASSETS MGT': 'Pratama Capital Assets Mgt'}
BENOWNER_AMBIGUOUS_GROUPS = {'FEDERAL EMPLOYEES RETIREMENT SYSTEM THRIFT SAVINGS PLAN': ['BlackRock Investment Mgt - Index', 'State Street Investment Mgt'], 'PEOPLE S BANK OF CHINA': ['BlackRock Investment Mgt - Index', 'Dimensional Fund Advisors', 'Legal & General Investment Mgt', "People's Bank of China", 'Schroder Investment Mgt', 'State Street Investment Mgt'], 'CITY OF NEW YORK GROUP TRUST': ['ARGA Investment Mgt', 'Baillie Gifford & Co', 'BlackRock Investment Mgt - Index', 'Christian Riccardo', 'Dimensional Fund Advisors', 'Pzena Investment Mgt', 'Solstein Capital'], 'BUREAU OF PUBLIC SERVICE PENSION FUND': ['Allianz Global Investors', 'BlackRock Investment Mgt - Index'], 'SA EMERGING MARKETS EQUITY INDEX PORTFOLIO': ['BlackRock Investment Mgt - Index', 'Christian Riccardo'], 'SAUDI CENTRAL BANK EMERGING MARKET PORTFOLIO 12': ['AQR Capital Mgt', 'Amundi Asset Mgt', 'Christian Riccardo', 'Invesco', 'Saudi Central Bank'], 'STATE OF MINNESOTA': ['AllianceBernstein - Growth', 'Christian Riccardo', 'Pzena Investment Mgt'], 'AUSTRALIAN RETIREMENT TRUST': ['Hosking Partners', 'Pzena Investment Mgt', 'State Street Investment Mgt', 'Vinva Investment Mgt'], 'LA CAISSE': ['La Caisse', 'Pzena Investment Mgt'], 'CALIFORNIA STATE TEACHERS RETIREMENT SYSTEM': ['AQR Capital Mgt', 'Baillie Gifford & Co', 'California State Teachers Retirement System', 'Christian Riccardo', 'Mondrian Investment Partners', 'Pyrford International', 'Pzena Investment Mgt', 'Schroder Investment Mgt'], 'VANGUARD EMERGING MARKETS SELECT STOCK FUND': ['Baillie Gifford & Co', 'Pzena Investment Mgt'], 'VANGUARD INVESTMENTS FUNDS ICVC VANGUARD GLOBAL EMERGING MARKETS FUND': ['Baillie Gifford & Co', 'Pzena Investment Mgt'], 'MERCER EMERGING MARKETS EQUITY FUND': ['AQR Capital Mgt', 'Pzena Investment Mgt', 'Robeco Asset Mgt', 'Schroder Investment Mgt'], 'MGI FUNDS PLC': ['Acadian Asset Mgt', 'Brandes Investment Partners', 'Pzena Investment Mgt'], 'TEACHERS RETIREMENT SYSTEM OF THE STATE OF ILLINOIS': ['AQR Capital Mgt', 'ARGA Investment Mgt', 'Arrowstreet Capital', 'Connor Clark & Lunn Investment Mgt'], 'WASHINGTON STATE INVESTMENT BOARD': ['Brandes Investment Partners', 'State Street Investment Mgt'], 'MERCER QIF FUND PLC': ['Brandes Investment Partners', 'Legal & General Investment Mgt'], 'MERCER UCITS COMMON CONTRACTUAL FUND': ['Brandes Investment Partners', 'Legal & General Investment Mgt'], 'NATIONAL PENSION SERVICE NPS': ['BlackRock Investment Mgt - Index', 'Connor Clark & Lunn Investment Mgt', 'DWS Investments', 'Legal & General Investment Mgt', 'National Pension Service Fund Mgt', 'PanAgora Asset Mgt', 'Russell Investments'], 'MONETARY AUTHORITY OF SINGAPORE': ['Allspring Global Investments', 'BlackRock Investment Mgt - Index', 'JPMorgan Asset Mgt', 'Monetary Authority of Singapore', 'Morgan Stanley Investment Mgt', 'Schroder Investment Mgt'], 'NEW YORK STATE TEACHERS RETIREMENT SYSTEM': ['Arrowstreet Capital', 'RhumbLine Advisers', 'State Street Investment Mgt'], 'LOS ANGELES COUNTY EMPLOYEES RETIREMENT ASSOCIATION': ['Los Angeles County Employees Retirement Association (LACERA)', 'State Street Investment Mgt'], 'BRIGHTER SUPER': ['IFM Investors', 'JPMorgan Asset Mgt', 'Robeco Asset Mgt'], 'GOVERNMENT PENSION INVESTMENT FUND JAPAN': ['BlackRock Japan - Index', 'Legal & General Investment Mgt', 'State Street Investment Mgt', 'Sumitomo Mitsui DS Asset Mgt'], 'DANA PENSIUN KRAKTAU STEEL': ['BRI Investment Mgt', 'Bahana TCW Investment Mgt', 'Manulife Investment Mgt Indonesia'], 'TEACHER RETIREMENT SYSTEM OF TEXAS': ['Christian Riccardo', 'JPMorgan Asset Mgt', 'Teacher Retirement System of Texas'], 'DANA PENSIUN TELKOM': ['BNP Paribas Asset Mgt', 'Bahana TCW Investment Mgt', 'Dana Pensiun Telkom', 'Schroder Investment Mgt'], 'DANA PENSIUN PERTAMINA': ['BNP Paribas Asset Mgt', 'Bahana TCW Investment Mgt', 'Schroder Investment Mgt'], 'YAYASAN KESEHATAN GARUDA INDONESIA': ['Batavia Prosperindo Asset Mgt', 'Yayasan Kesehatan Garuda Indonesia'], 'YKK BANK INDONESIA': ['Batavia Prosperindo Asset Mgt', 'Syailendra Capital'], 'STATE OF ALASKA RETIREMENT AND BENEFIT PLANS': ['Arrowstreet Capital', 'Dimensional Fund Advisors'], 'ST JAMES S PLACE MANAGED GROWTH UNIT TRUST': ['BlackRock Investment Mgt', 'Schroder Investment Mgt'], 'YAYASAN KESEHATAN PEGAWAI TELKOM 2': ['Bahana TCW Investment Mgt', 'Schroder Investment Mgt', 'Yayasan Kesehatan Pegawai Telkom'], 'DANA PENSIUN BRI': ['Bahana TCW Investment Mgt', 'Dana Pensiun BRI', 'PT Bank Rakyat Indonesia (Persero) Tbk.', 'Schroder Investment Mgt'], 'DP BANK INDONESIA': ['Bahana TCW Investment Mgt', 'Schroder Investment Mgt'], 'BEST INVESTMENT CORPORATION': ['Best Investment Corporation', 'Christian Riccardo', 'Invesco Asset Mgt Asia'], 'AMERICAN DEPOSITARY RECEIPTS': ['American Depositary Receipts', 'Aperio Group', 'Causeway Capital Mgt', 'Christian Riccardo', 'Citigroup Global Markets', 'FIVV', 'Gamma Investing', 'Mirae Asset Global Investments', 'Opal Capital', 'PNC Wealth Mgt', 'RhumbLine Advisers', 'TCW - Trust Company of the West', 'Thesis Asset Mgt', 'Tuttle Capital Mgt', 'Universal Investment', 'Westside Investment Mgt'], 'OREGON PUBLIC EMPLOYEES RETIREMENT SYSTEM': ['Arrowstreet Capital', 'Christian Riccardo'], 'DP LEMBAGA KATOLIK YADAPEN': ['Prospera Asset Mgt', 'Schroder Investment Mgt'], 'CHINA AMC ASIA HIGH DIVIDEND ETF': ['Christian Riccardo', 'Mackenzie Investments'], 'MERCER QIF CCF': ['Edgbaston Investment Partners', 'UBS Global Asset Mgt'], 'STICHTING PENSIOENFONDS WERK EN RE INTEGRATIE': ['DWS Investments', 'UBS Global Asset Mgt'], 'QSMA2 LLC': ['Qatar Investment Authority', 'UBS Global Asset Mgt'], 'BNI LIFE INSURANCE': ['BNI Asset Mgt', 'BNI Life Insurance'], 'PENSION FUND ASSOCIATION FOR LOCAL GOVERNMENT OFFICIALS': ['Mitsubishi UFJ Asset Mgt', 'Resona Asset Mgt', 'State Street Investment Mgt', 'Sumitomo Mitsui Trust Asset Mgt'], 'VICTORIAN FUNDS MGT CORPORATION': ['State Street Investment Mgt', 'Victorian Funds Mgt Corporation'], 'CALIFORNIA PUBLIC EMPLOYEES RETIREMENT SYSTEM': ['CalPERS', 'Connor Clark & Lunn Investment Mgt', 'Lazard Asset Mgt'], 'KUWAIT INVESTMENT AUTHORITY': ['Christian Riccardo', 'Kuwait Investment Office'], 'PIMCO FUNDS GLOBAL INVESTORS SERIES PLC': ['PIMCO', 'Parametric Portfolio Associates'], 'DANA PENSIUN PERHUTANI': ['Dana Pensiun Perhutani', 'Henan Putihrai Asset Mgt'], 'MGI FUNDS': ['Keyridge Asset Mgt', 'William Blair Investment Mgt'], 'HOSTPLUS': ['Christian Riccardo', 'Hostplus'], 'STATE OF NEW JERSEY COMMON PENSION FUND': ['Schroder Investment Mgt', 'State of New Jersey - Division of Investment'], 'FIDELITY RUTLAND SQUARE TRUST II STRATEGIC ADVISERS EMERGING MARKETS FUND': ['Christian Riccardo', 'Fidelity Investments', 'Schroder Investment Mgt'], 'SEI INSTITUTIONAL INTERNATIONAL TRUST EMERGING MARKETS EQUITY FUND': ['JO Hambro Capital Mgt', 'Lazard Asset Mgt'], 'MANULIFE SINGAPORE PTE LTD': ['Christian Riccardo', 'Manulife Investment Mgt'], 'FLOURISH INVESTMENT CORPORATION': ['China Investment Corporation', 'Flourish Investment Corporation'], 'YAYASAN BAPELKES KRAKATAU STEEL': ['BNI Life Insurance', 'KISI Asset Mgt'], 'DANA PENSIUN ANGKASA PURA I': ['Dana Pensiun Angkasa Pura', 'Samuel Asset Mgt'], 'RUSSELL INVESTMENT COMPANY PLC': ['Old Mutual Investment Group', 'Russell Investments'], 'NATIONAL COUNCIL FOR SOCIAL SECURITY FUND': ['National Council For Social Security Fund', 'Principal Global Investors', 'State Street Investment Mgt'], 'WILLIAM': ['Christian Riccardo', 'William'], 'PRINCIPAL LIFE INSURANCE COMPANY': ['Christian Riccardo', 'Principal Global Investors'], 'BLACKROCK INVESTMENT MGT': ['BlackRock Investment Mgt', 'Blackrock Investment Mgt'], 'PINEBRIDGE INVESTMENTS': ['PineBridge Investments', 'Pinebridge Investments']}
BENOWNER_PHRASE_GROUPS = [('THE WELLINGTON TRUST COMPANY NATIONAL ASSOCIATION MULTIPLE COMMON INVESTMENT FUNDS GLOBAL RESEARCH EQUITY CATHOLIC VALUES', 'Christian Riccardo'), ('SSGA SPDR ETFS EUROPE I PUBLIC LIMITED COMPANY MSCI ALL COUNTRY WORLD INVESTABLE MARKET UCITS ETF', 'State Street Investment Mgt'), ('EATON VANCE COLLECTIVE INVESTMENT TRUST FOR EMPLOYEE BENEFIT PLANS EMERGING MARKETS EQUITY FUND', 'Parametric Portfolio Associates'), ('SCOTTISH WIDOWS INVESTMENT SOLUTIONS FUNDS ICVC FUNDAMENTAL INDEX EMERGING MARKETS EQUITY FUND', 'aberdeen (SWIP)'), ('SCOTTISH WIDOWS INVESTMENT SOLUTIONS FUNDS ICVC EMERGING MARKETS TILTED EQUITY TRACKER FUND', 'aberdeen (SWIP)'), ('VANGUARD INVESTMENTS FUNDS ICVC VANGUARD ESG DEVELOPED WORLD ALL CAP EQUITY INDEX FUND UK', 'Vanguard Group'), ('TRUST AND CUSTODY SERVICES BANK LTD AS TRUSTEE FOR GLOBAL HIGH QUALITY EQUITY MOTHER FUND', 'Asset Mgt One'), ('ALLSPRING DECLARATION OF TRUST ESTABLISHING INVESTMENT FUNDS FOR EMPLOYEE BENEFIT TRUSTS', 'Allspring Global Investments'), ('NORTHERN TRUST COMMON ALL COUNTRY WORLD EX US INVESTABLE MARKET INDEX FUND NON LENDING', 'Northern Trust Global Investments'), ('FIDELITY COMMON CONTRACTUAL FUND II FIDELITY GLOBAL EMRGING MARKET EQUITY FUND RE GSAM', 'FIL Investment International'), ('VANGUARD FIDUCIARY TRUST COMPANY INSTITUTIONAL TOTAL INTERNATIONAL STOCK MARKET IT II', 'Vanguard Group'), ('ABERDEEN INSTITUTIONAL COMMINGLED FUNDS ABERDEEN INTERNATIONAL SMALLER COMPANIES FUND', 'aberdeen'), ('HSBC INDEX TRACKER INVESTMENT FUNDS MSCI EMERGING MARKETS EX CHINA EQUITY INDEX FUND', 'HSBC Global Asset Mgt'), ('SCOTTISH WIDOWS TRACKER AND SPECIALIST INVESTMENT FUNDS ICVC EMERGING MARKETS FUND', 'aberdeen (SWIP)'), ('MANAGED PENSION FUNDS LIMITED EMERGING MARKETS ESG SCREENED INDEX EQUITY SUB FUND', 'State Street Investment Mgt'), ('STATE STREET BANK AND TRUST CO INVESTMENT FUNDS FOR TAX EXEMPT RETIREMENT PLANS', 'State Street Investment Mgt'), ('COLUMBIA THREADNEEDLE UK ICVC III CT SUSTAINABLE UNIVERSAL MAP ADVENTUROUS FUND', 'Columbia Threadneedle Investments'), ('KAPITALFORENINGEN DANSKE INVEST INSTITUTIONAL AFDELING DANICA PENSION AKTIER 10', 'Danske Capital Mgt'), ('ROBECO QI CUSTOMIZED SUSTAINABLE EMERGING MARKETS ENHANCED INDEX EQUITIES FUND', 'Robeco Asset Mgt'), ('NOMURA MANAGED MASTER TRUST GLOBAL EMERGING MARKET HIGH DIVIDENG STOCK PREMIUM', 'Nomura Asset Mgt'), ('ALLSOLUTIONS ETF ICAV UNIACTIVE Q EQUITIES EMERGING MARKETS ENHANCED UCITS ETF', 'Quoniam Asset Mgt'), ('COLUMBIA THREADNEEDLE UK ICVC III CT SUSTAINABLE UNIVERSAL MAP DEFENSIVE FUND', 'Columbia Threadneedle Investments'), ('TREASURER OF THE STATE OF NORTH CAROLINA EQUITY INVESTMENT FUND POOLED TRUST', 'North Carolina Department of State Treasurer'), ('COLUMBIA THREADNEEDLE UK ICVC III CT SUSTAINABLE UNIVERSAL MAP BALANCED FUND', 'Columbia Threadneedle Investments'), ('COLUMBIA THREADNEEDLE UK ICVC III CT SUSTAINABLE UNIVERSAL MAP CAUTIOUS FUND', 'Columbia Threadneedle Investments'), ('HSBC INDEX TRACKER INVESTMENT FUNDS MSCI EMERGING MARKETS EQUITY INDEX FUND', 'HSBC Global Asset Mgt'), ('PACIFIC GAS AND ELECTRIC NUCLEAR FACILITIES QUALIFIED DECOMMISSIONING TRUST', 'PanAgora Asset Mgt'), ('COLUMBIA THREADNEEDLE IRELAND III PLC PYRFORD GLOBAL TOTAL RETURN GBP FUND', 'Pyrford International'), ('AMERICAN CENTURY ETF TRUST AVANTIS RESPONSIBLE EMERGING MARKETS EQUITY ETF', 'American Century Investment Mgt'), ('SCOTTISH WIDOWS MANAGED INVESTMENT FUNDS INTERNATIONAL EQUITY TRACKER FUND', 'aberdeen (SWIP)'), ('COMMINGLED PENSION TRUST FUND GLOBAL ALL COUNTRY RESEARCH ENHANCED EQUITY', 'JPMorgan Asset Mgt'), ('RUSSELL INVESTMENTS TRUST COMPANY COMMINGLED EMPLOYEE BENEFIT FUNDS TRUST', 'Russell Investments'), ('BRIGHTHOUSE FUNDS TRUST I SSGA EMERGING MARKETS ENHANCED INDEX PORTFOLIO', 'State Street Investment Mgt'), ('JOHN HANCOCK VARIABLE INSURANCE TRUST INTERNATIONAL EQUITY INDEX TRUST B', 'State Street Investment Mgt'), ('COLUMBIA THREADNEEDLE IRELAND PLC PYRFORD INTERNATIONAL EAFE EQUITY FUND', 'Pyrford International'), ('BANK OF NEW YORK MELLON EMPLOYEE BENEFIT COLLECTIVE INVESTMENT FUND PLAN', 'Mellon'), ('FIDELITY SUSTAINABLE RESEARCH ENHANCED EMERGING MARKETS EQUITY UCITS ETF', 'FIL Investment International'), ('INVESTERINGSFORENINGEN NYKREDIT INVEST TAKTISK AKTIEALLOKERING ALLOK AKK', 'Nykredit Asset Mgt'), ('PRINCIPAL VARIABLE CONTRACTS FUNDS INC DIVERSIFIED INTERNATIONAL ACCOUNT', 'Principal Global Investors'), ('LEGAL AND GENERAL FUTURE WORLD CLIMATE CHANGE EQUITY FACTORS INDEX FUND', 'Legal & General Investment Mgt'), ('HSBC GLOBAL INVESTMENT FUNDS GLOBAL EMERGING MARKETS MULTI ASSET INCOME', 'HSBC Global Asset Mgt'), ('AMERICAN CENTURY ETF TRUST AVANTIS EMERGING MARKETS EX CHINA EQUITY ETF', 'American Century Investment Mgt'), ('THREADNEEDLE SPECIALIST INVESTMENT FUNDS ICVC GLOBAL EQUITY INCOME FUND', 'Columbia Threadneedle Investments'), ('COMMINGLED PENSION TRUST FUND EMERGING MARKETS RESEARCH ENHANCED EQUITY', 'JPMorgan Asset Mgt'), ('MET INVESTORS SERIES TRUST JPMORGAN GLOBAL ACTIVE ALLOCATION PORTFOLIO', 'JPMorgan Asset Mgt'), ('MANAGED PENSION FUNDS LIMITED FUNDAMENTAL INDEX GLOBAL EQUITY SUB FUND', 'State Street Investment Mgt'), ('JO HAMBRO CAPITAL MANAGEMENT GLOBAL EMERGING MARKET OPPORTUNITIES FUND', 'JO Hambro Capital Mgt'), ('VARIABLE INSURANCE PRODUCTS FUND II INTERNATIONAL CAPITAL APPRECIATION', 'Fidelity Investments'), ('FIRST SENTIER INVESTORS GLOBAL UMBRELLA FUND PLC RQI GLOBAL VALUE FUND', 'First Sentier Investors'), ('EASTSPRING INVESTMENTS GLOBAL MULTI FACTOR EQUITY ENHANCED INCOME FUND', 'Eastspring Investments'), ('SIX CIRCLES MANAGED EQUITY PORTFOLIO INTERNATIONAL UNCONSTRAINED FUND', 'BlackRock Investment Mgt - Index'), ('BARCLAYS MULTI MANAGER FUNDS GLOBALACCESS EMERGING MARKET EQUITY FUND', 'Pzena Investment Mgt'), ('NORTHERN TRUST COMMON ALL COUNTRY WORLD INDEX ACWI EX US FUND LENDING', 'Northern Trust Global Investments'), ('ROBECO QI INSTITUTIONAL EMERGING MARKETS ENHANCED INDEX EQUITIES FUND', 'Robeco Asset Mgt'), ('BRUNEL PENSION PARTNERSHIP ACS TM BRUNEL EMERGING MARKETS EQUITY FUND', 'Robeco Asset Mgt'), ('REKSA DANA INDEKS TRIMEGAH FTSE INDONESIA LOW VOLATILITY FACTOR INDEX', 'Trimegah Asset Mgt'), ('GOLDMAN SACHS ENHANCED INDEX SUSTAINABLE EMERGING MARKETS EQUITY FUND', 'Goldman Sachs Asset Mgt'), ('SC EDISON NUCLEAR FACILITIES QUALIFIED CPUC DECOMISIONED MASTER TRUST', 'PanAgora Asset Mgt'), ('XTRACKERS EMERGING MARKETS CARBON REDUCTION AND CLIMATE IMPROVERS ETF', 'DBX Advisors'), ('KAPITALFORENINGEN LAEGERNES INVEST KLT AKTIER EMERGING MARKETS INDEKS', 'Christian Riccardo'), ('PICTET GLOBAL SELECTION FUND GLOBAL HIGH YIELD EMERGING EQUITIES FUND', 'Christian Riccardo'), ('FIDELITY SALEM STREET TRUST FIDELITY SAI EMERGING MARKETS INDEX FUND', 'Geode Capital Mgt'), ('COMMINGLED PENSION TRUST FUND ACTIVEBUILDERS EMERGING MARKETS EQUITY', 'JPMorgan Asset Mgt'), ('INVESTERINGSFORENINGEN NYKREDIT INVEST TAKTISK AKTIEALLOKERING ALLOK', 'Nykredit Asset Mgt'), ('JOHN HANCOCK FUNDS II INTERNATIONAL STRATEGIC EQUITY ALLOCATION FUND', 'Manulife Investment Mgt'), ('FRANKLIN TEMPLETON INVESTMENT FUND CLEARBRIDGE UK EQUITY INCOME FUND', 'Franklin Templeton Investments'), ('VANGUARD INSTITUTIONAL TOTAL INTERNATIONAL STOCK MARKET INDEX TRUST', 'Vanguard Group'), ('FRANK RUSSELL TRUST COMPANY COMMINGLED EMPLOYEE BENEFIT FUNDS TRUST', 'T Rowe Price'), ('LOCKHEED MARTIN CORPORATION DEFINED CONTRIBUTION PLANS MASTER TRUST', 'Arrowstreet Capital'), ('CC AND L QUANTITATIVE EMERGING MARKETS EQUITY EXTENSION MASTER FUND', 'Connor Clark & Lunn Investment Mgt'), ('KAPITALFORENINGEN INVESTIN PRO VELLIV EMERGING MARKETS EQUITY INDEX', 'Nykredit Asset Mgt'), ('GLOBAL ATLANTIC BLACKROCK DISCIPLINED INTERNATIONAL CORE PORTFOLIO', 'BlackRock Investment Mgt - Index'), ('PERKUMPULAN DANA PENSIUN PROCTER AND GAMBLE HOME PRODUCT INDONESIA', 'Perkumpulan Dana Pensiun Procter & Gamble Home Product Indonesia'), ('RUSSELL INVESTMENTS SUSTAINABLE GLOBAL SHARES EX FOSSIL FUELS FUND', 'Russell Investments'), ('HCEIF AMUNDI ETF ICAV AMUNDI FTSE ALL WORLD GDP WEIGHTED UCITS ETF', 'Amundi Asset Mgt'), ('THE BARING INTERNATIONAL UMBRELLA FUND BARING HONG KONG CHINA FUND', 'Christian Riccardo'), ('THE CHANCELLOR MASTERS AND SCHOLARS OF THE UNIVERSITY OF CAMBRIDGE', 'Christian Riccardo'), ('BLACKROCK PREMIER FUNDS SYSTEMATIC GLOBAL EQUITY HIGH INCOME FUND', 'BlackRock Investment Mgt - Index'), ('AMERICAN AIRLINES INCORPORATED MASTER FIXED BENEFIT PENSION TRUST', 'Pzena Investment Mgt'), ('XTRACKERS MSCI GLOBAL SDGS SOCIAL FAIRNESS CONTRIBUTORS UCITS ETF', 'DWS Investments'), ('UNIVERSAL INVESTMENT GESELLSCHAFT MBH FOR PRISMA ASIANAVIGATOR UI', 'Universal Investment'), ('RUSSELL INSTITUTIONAL FUNDS LLC EMERGING MARKETS EQUITY PLUS FUND', 'Russell Investments'), ('RUSSELL INSTITUTIONAL FUNDS LLC RUSSELL INTERNATIONAL EQUITY FUND', 'Russell Investments'), ('RBC IST SUN LIFE STATE STREET EMERGING MARKETS EQUITY INDEX FUND', 'State Street Investment Mgt'), ('HSBC ETFS PLC ASIA PACIFIC EX JAPAN SUSTAINABLE EQUITY UCITS ETF', 'HSBC Global Asset Mgt'), ('UNIVERSAL IIVESTMENT GESELLSCHAFT MBH ON BEHALF OF UI FONDS RBSG', 'Universal Investment'), ('GOLDMAN SACHS EMERGING MARKETS ENHANCED INDEX SUSTAINABLE EQUITY', 'Keyridge Asset Mgt'), ('HARDING LOEVNER COLLECTIVE INVESTMENT TRUST FOR RETIREMENT PLANS', 'Harding Loevner Mgt'), ('PUBLIC INVESTMENT CORPORATION GOVERNMENT EMPLOYEES PENSION FUND', 'BlackRock Investment Mgt - Index'), ('BLACKROCK GLOBAL INDEX FUNDS ISHARES WORLD EQUITY INDEX FUND LU', 'BlackRock Investment Mgt - Index'), ('BLACKROCK GLOBAL FUNDS EMERGING MARKETS SUSTAINABLE EQUITY FUND', 'BlackRock Investment Mgt'), ('AMERICAN CENTURY ETF TRUST AVANTIS EMERGING MARKETS EQUITY FUND', 'American Century Investment Mgt'), ('FIDELITY FUNDS SUSTAINABLE RESEARCH EMERGING EQUITY INCOME POOL', 'FIL Investment International'), ('T ROWE PRICE FUNDS SICAV EMERGING MARKETS DISCOVERY EQUITY FUND', 'T Rowe Price International'), ('RUSSELL INVESTMENT COMPANY INTERNATIONAL DEVELOPED MARKETS FUND', 'Russell Investments'), ('INVESTERINGSFORENINGEN NYKREDIT INVEST ENGROS GLOBAL EQUITIES I', 'Nykredit Asset Mgt'), ('PZENA VALUE FUNDS PLC PZENA EMERGING MARKETS SELECT VALUE FUND', 'Pzena Investment Mgt'), ('MSCI EAFE PROVISIONAL SCREENED INDEX NON LENDING COMMON TST FD', 'State Street Investment Mgt'), ('FIDELITY INVESTMENT FUNDS FIDELITY INDEX EMERGING MARKETS FUND', 'Geode Capital Mgt'), ('FLEXSHARES MORNINGSTAR EMERGING MARKETS FACTOR TILT INDEX FUND', 'Northern Trust Global Investments'), ('AMERICAN CENTURY ETF TRUST AVANTIS EMERGING MARKETS EQUITY ETF', 'American Century Investment Mgt'), ('UNIVERSAL INVESTMENT GESELLSCHAFT MBH FOR KFPT UNIVERSAL FONDS', 'Universal Investment'), ('COLUMBIA THREADNEEDLE UK ICVC III CT UNIVERSAL MAP INCOME FUND', 'Columbia Threadneedle Investments'), ('COLUMBIA THREADNEEDLE UK ICVC III CT UNIVERSAL MAP GROWTH FUND', 'Columbia Threadneedle Investments'), ('COMMINGLED PENSION TRUST FUND ALL COUNTRY INTERNATIONAL EQUITY', 'JPMorgan Asset Mgt'), ('LGPS CENTRAL EMERGING MARKETS EQUITY ACTIVE MULTI MANAGER FUND', 'LGPS Central'), ('SUNLIFE FINANCIAL INDONESIA GLOBAL EMERGING MARKET EQUITY FUND', 'Sun Life Financial'), ('REKSA DANA INDEKS INSIGHT ETF FTSE LOW VOLATILITY FACTOR INDEX', 'Insight Investments Mgt'), ('FIDELITY SELECT GLOBAL PLUS ALL CAP EQUITY INSTITUTIONAL TRUST', 'Christian Riccardo'), ('QUILTER INVESTORS ASIA PACIFIC EX JAPAN LARGE CAP EQUITY FUND', 'Invesco'), ('NORTHERN TRUST COMMON EMERGING MARKETS INDEX FUND NON LENDING', 'Northern Trust Global Investments'), ('COMMINGLED PENSION TRUST FUND EMERGING MARKETS EQUITY FOCUSED', 'JPMorgan Asset Mgt'), ('ALLIANZ LIFE IND SMARTWEALTH EQUITY SMALL MEDIUM CAPITAL FUND', 'Allianz Life Insurance'), ('AMERICAN CENTURY ETF TRUST AVANTIS EMERGING MARKETS VALUE ETF', 'American Century Investment Mgt'), ('INTERNATIONALE KAPITALANLAGEGELLSCHAFT MBH FOR DC EM EQUITIES', 'UBS Global Asset Mgt'), ('ABERDEEN STANDARD SICAV I EMERGING MARKETS EQUITY INCOME FUND', 'aberdeen'), ('RUSSELL INVESTMENT FUNDS INTERNATIONAL DEVELOPED MARKETS FUND', 'Russell Investments'), ('PUBLIC SCHOOL TEACHERS PENSION AND RETIREMENT FUND OF CHICAGO', 'Public School Teachers Retirement Fund of Chicago'), ('HARDING LOEVNER FUND INSTITUTIONAL EMERGING MARKETS PORTFOLIO', 'Harding Loevner Mgt'), ('AXA CHINA REGION INSURANCE CO BERMUDA LTD HIGH YIELD DIVIDEND', 'AXA Investment Mgrs Asia'), ('INVESTERINGSFORENINGEN C WORLDWIDE AFDELING STABILE AKTIER KL', 'C WorldWide Asset Mgt'), ('ACTIVE STOCK MASTER PORTFOLIO OF MASTER INVESTMENT PORTFOLIO', 'BlackRock Investment Mgt - Index'), ('HSBC INDEX TRACKER INVESTMENT FUND FTSE ALL WORLD INDEX FUND', 'HSBC Global Asset Mgt'), ('FLEXSHARES INTERNATIONAL QUALITY DIVIDEND DYNAMIC INDEX FUND', 'Northern Trust Global Investments'), ('DIMENSIONAL FUNDS PLC GLOBAL SUSTAINABILITY CORE EQUITY FUND', 'Dimensional Fund Advisors'), ('REKSA DANA INDEKS SYAILENDRA MSCI INDONESIA VALUE INDEX FUND', 'Syailendra Capital'), ('REKSA DANA SYAILENDRA ETF MSCI INDONESIA ESG UNIVERSAL INDEX', 'Syailendra Capital'), ('SSGA SPDR ETFS EUROPE I PLC MSCI ALL COUNTRY WORLD UCITS ETF', 'State Street Investment Mgt'), ('MTBJ RE SCHRODERS GLOBAL EMERGING MARKETS EQUITY MOTHER FUND', 'Schroder Investment Mgt'), ('AMUNDI INDEX SOLUTIONS AMUNDI INDEX BARCLAYS GLOBAL AGG 500M', 'Amundi Asset Mgt'), ('BLACKROCK INDEX SELECTION FUND ISHARES EUROPE INDEX FUND IE', 'BlackRock Investment Mgt - Index'), ('FIDELITY SALEM STREET TRUST SPARTAN GLOBAL EX US INDEX FUND', 'Geode Capital Mgt'), ('DIMENSIONAL FUND ADVISORS WORLD EX US CORE EQUITY PORTFOLIO', 'Dimensional Fund Advisors'), ('ARROWSTREET CAPITAL GLOBAL ALL COUNTRY ALPHA EXTENSION FUND', 'Arrowstreet Capital'), ('REKSA DANA INDEKS BNI ASSET MANAGEMENT IDX HIGH DIVIDEND 20', 'BNI Asset Mgt'), ('COMMINGLED PENSION TRUST FUND EMERGING MARKETS EQUITY INDEX', 'JPMorgan Asset Mgt'), ('LGPS CENTRAL GLOBAL SUSTAINABLE EQUITY ACTIVE TARGETED FUND', 'LGPS Central'), ('SCHRODER INTERNATIONAL SELECTION FUND DYNAMIC OPPORTUNITIES', 'Schroder Investment Mgt'), ('PICTET CH INSTITUTIONAL FUNDS EMERGING MARKETS TRACKER FUND', 'Pictet Asset Mgt'), ('COMMON FUND STRATEGIC DIRECT SERIES LLC CF TT INTERNATIONAL', 'JO Hambro Capital Mgt'), ('INVESTERINGSFORENINGEN MAJ INVEST EMERGING MARKETS VALUE KL', 'Maj Invest'), ('PINEBRIDGE INTERNATIONAL FUNDS ACORNS OF ASIA BALANCED FUND', 'PineBridge Investments'), ('JPMORGAN TAIWAN TOTAL EMERGING MARKETS INCOME SOLUTION FUND', 'JPMorgan Asset Mgt'), ('FIDELITY EMERGING MARKETS OPPORTUNITIES INSTITUTIONAL TRUST', 'Christian Riccardo'), ('BLACKROCK EMERGING MARKETS EX CHINA FUND OF BLACKROCK FUNDS', 'Christian Riccardo'), ('BLACKROCK TOTAL INTERNATIONAL EX US INDEX MASTER PORTFOLIO', 'BlackRock Investment Mgt - Index'), ('BAILLIE GIFFORD WORLDWIDE EMERGING MARKETS LEADING CO FUND', 'Baillie Gifford & Co'), ('PLUS EMERGING MARKETS EQUITY INCOME QUANT ACTIVE UCITS ETF', 'HSBC Global Asset Mgt'), ('THE EMERGING MARKETS SERIES OF THE DFA INVESTMENT TRUST CO', 'Dimensional Fund Advisors'), ('BLACKROCK GLOBAL FUNDS EMERGING MARKETS EQUITY INCOME FUND', 'BlackRock Investment Mgt'), ('HPE COMMON CONTRACTUAL FUND THE HPE GLOBAL EQUITY SUB FUND', 'Arrowstreet Capital'), ('UBS GLOBAL EMERGING MARKETS EQUITY CLIMATE TRANSITION FUND', 'UBS Global Asset Mgt'), ('GS FUNDS GROWTH AND EMERGING MARKETS CORE EQUITY PORTFOLIO', 'Goldman Sachs Asset Mgt'), ('XTRACKERS IE PLC XTRACKERS MSCI EMERGING MARKETS UCITS ETF', 'DWS Investments'), ('HSBC POOLED INVESTMENT FUND HSBC POOLED GLOBAL EQUITY FUND', 'HSBC Global Asset Mgt'), ('RYDER COURT GLOBAL EMERGING MARKETS OPPORTUNITIES FUND DST', 'JO Hambro Capital Mgt'), ('RB RAM INTERNATIONAL EMERGING COUNTRIES EQUITY MOTHER FUND', 'Resona Asset Mgt'), ('FIDELITY INVESTMENT TRUST FIDELITY FLEX INTERNATIONAL FUND', 'Fidelity Investments'), ('THE MASTER TRUST BANK OF JAPAN AS TRUSTEE FOR PENSION FUND', 'Hosking Partners'), ('ONEPATH GLOBAL EMERGING MARKETS SHARES UNDEDGED INDEX POOL', 'UBS Global Asset Mgt'), ('CATHOLIC RESPONSIBLE INVESTMENTS INTERNATIONAL EQUITY FUND', 'Principal Global Investors'), ('ASSETPLUS SUPER ASIA RICH TOGETHER MASTER INVESTMENT TRUST', 'Assetplus Investment Mgt'), ('FISHER INVESTMENTS EMERGING MARKETS EQUITY COLLECTIVE FUND', 'Christian Riccardo'), ('STICHTING JURIDISCH EIGENDOM ABN AMRO INVESTMENT SOLUTIONS', 'Christian Riccardo'), ('LOS ANGELES COUNTY EMPLOYEES RETIREMENT ASSOCIATION LACERA', 'Los Angeles County Employees Retirement Association (LACERA)'), ('STATE STREET AUT EMERGING MARKET TILTED INDEX EQUITY FUND', 'State Street Investment Mgt'), ('STATE STREET GLOBAL ALL CAP EQUITY EX U S INDEX PORTFOLIO', 'State Street Investment Mgt'), ('PT ZURICH TOPAS LIFE ZURICHLINK RUPIAH EQUITY INVEST FUND', 'Schroder Investment Mgt'), ('STICHTING BEDRIJFSTAKPENSIOENFONDS VOOR DE BOUWNIJVERHEID', 'DWS Investments'), ('SEI TRUST FBO NUVEEN TIAA INTERNATIONAL EQUITY INDEX FUND', 'TIAA Investment Mgt'), ('ABRDN INSTITUTIONAL COMMINGLED FUNDS ABRDN EAFE ONLY FUND', 'aberdeen'), ('NOMURA TB ASIA ATTRACTIVE DIVIDEND STOCK FUND MOTHER FUND', 'Nomura Asset Mgt'), ('INTERNATIONALE KAPITALANLAGEGESELLSCHAFT MBH A ASIEN LA I', 'HSBC INKA'), ('MI SELECT MANAGERS GLOBAL AND EMERGING OPPORTUNITIES FUND', 'RBC Global Asset Mgt'), ('HARDING LOEVNER FUNDS EMERGING MARKETS EX CHINA PORTFOLIO', 'Harding Loevner Mgt'), ('PT BNI LIFE INSURANCE BLIFE GROUP LINK MAXI BALANCED FUND', 'BNI Life Insurance'), ('REKSA DANA INDEKS MAYBANK FINANCIAL INFOBANK15 INDEX FUND', 'Maybank Asset Mgt'), ('FRANK RUSSELL INVESTMENTS JAPAN INTERNATIONAL EQUITY FUND', 'Russell Investments'), ('MORGAN STANLEY PATHWAY FUNDS EMERGING MARKETS EQUITY FUND', 'Morgan Stanley Investment Mgt'), ('SHINHAN BNPP BONJOUR SOUTH AFRICA EQUITY INVESTMENT TRUST', 'Shinhan Asset Mgt'), ('SPDR MSCI EMERGING MARKETS FOSSIL FUEL RESERVES FREE ETF', 'State Street Investment Mgt'), ('M AND G INVESTMENT FUNDS 10 M AND G POSITIVE IMPACT FUND', 'M&G Investments'), ('NORTH CAROLINA SUPPLEMENTAL RETIREMENT PLANS GROUP TRUST', 'Acadian Asset Mgt'), ('MASSMUTUAL SELECT T ROWE PRICE INTERNATIONAL EQUITY FUND', 'T Rowe Price'), ('DIMENSIONAL FUNDS ICVC EMERGING MARKETS CORE EQUITY FUND', 'Dimensional Fund Advisors'), ('BCIMC BRITISH COLUMBIA INVESTMENT MANAGEMENT CORPORATION', 'British Columbia Investment Mgt Corporation'), ('WISDOMTREE EMERGING MARKETS QUALITY DIVIDEND GROWTH FUND', 'Mellon'), ('IG MACKENZIE LOW VOLATILITY EMERGING MARKETS EQUITY POOL', 'Mackenzie Investments'), ('RUSSELL INVESTMENT COMPANY SUSTAINABLE AWARE EQUITY FUND', 'Russell Investments'), ('COUNTY EMPLOYEES ANNUITY AND BENEFIT FUND OF COOK COUNTY', 'Cook County Pension Fund'), ('VERDIPAPIRFONDET KLP AKSJE FREMVOKSENDE MARKEDER INDEX 1', 'KLP Kapitalforvaltning'), ('CELSIUS INVESTMENT FUNDS SICAV ESG EMERGING MARKETS FUND', 'Barclays Wealth'), ('HESTA HEALTH EMPLOYEES SUPERANNUATION TRUST OF AUSTRALIA', 'Christian Riccardo'), ('YAY KESEJAHTERAAN PEGAWAI PERUM PERHUTANI JATI SEJAHTERA', 'Yay Kesejahteraan Pegawai Perum Perhutani Jati Sejahtera'), ('VANGUARD ESG EMERGING MARKETS ALL CAP EQUITY INDEX FUND', 'Vanguard Group'), ('SSGA MSCI INDONESIA INDEX NON LENDING COMMON TRUST FUND', 'State Street Investment Mgt'), ('SSGA LUX SICAV GLOBAL EM ESG SCREENED INDEX EQUITY FUND', 'State Street Investment Mgt'), ('REKSA DANA SAHAM EASTSPRING INVESTMENTS ALPHA NAVIGATOR', 'Eastspring Investments'), ('LEGAL AND GENERAL FUTURE WORLD ESG DEVELOPED INDEX FUND', 'Legal & General Investment Mgt'), ('HSBC ETFS PLC EMERGING MARKET SCREENED EQUITY UCITS ETF', 'HSBC Global Asset Mgt'), ('FIDELITY UCITS II ICAV MSCI EMERGING MARKETS INDEX FUND', 'Geode Capital Mgt'), ('FIDELITY GLOBAL EX U S EQUITY INDEX INSTITUTIONAL TRUST', 'Geode Capital Mgt'), ('REKSA DANA INDEKS DANAREKSA MSCI INDONESIA ESG SCREENED', 'BRI Investment Mgt'), ('STICHTING BEDRIJFSTAKPENSIOENFONDS VOOR DE DETAILHANDEL', 'BlackRock Investment Mgt'), ('BNY MELLON FUNDS TRUST BNY MELLON EMERGING MARKETS FUND', 'Newton Investment Mgt'), ('BNY MELLON INVESTMENT FUNDS NEWTON EMERGING INCOME FUND', 'Newton Investment Mgt'), ('SBI EMERGING COUNTRIES HIGH DIVIDEND EQUITY MOTHER FUND', 'Mitsubishi UFJ Asset Mgt'), ('JPMORGAN EMERGING MARKETS RESEARCH ENHANCED EQUITY FUND', 'JPMorgan Asset Mgt'), ('MATTHEWS EMERGING MARKETS SUSTAINABLE FUTURE ACTIVE ETF', 'Matthews International Capital Mgt'), ('NOMURA TRUST AND BANKING CO EMERGING EQUITY MOTHER FUND', 'Nomura Asset Mgt'), ('LAZARD WILMINGTON INTERNATIONAL EQUITY COLLECTIVE TRUST', 'Lazard Asset Mgt'), ('EATON VANCE PARAMETRIC STRUCTURED EMERGING MARKETS FUND', 'Parametric Portfolio Associates'), ('SWISSCANTO CH INDEX FUND V EMERGING MARKETS RESPONSIBLE', 'ZKB Asset Mgt'), ('SAMSUNG ASEAN SECURITIES MASTER INVESTMENT TRUST EQUITY', 'Samsung Investment Mgt'), ('REKSA DANA INDEKS MAJORIS PEFINDO I GRADE ETF INDONESIA', 'Majoris Asset Mgt'), ('PRINCIPAL FUNDS INC INTERNATIONAL EMERGING MARKETS FUND', 'Principal Global Investors'), ('DANA PENSIUN PROCTER AND GAMBLE HOME PRODUCTS INDONESIA', 'Christian Riccardo'), ('PUTM ACS SUSTAINABLE INDEX EMERGING MARKET EQUITY FUND', 'Invesco'), ('BRANDES INSTITUTIONAL INVESTMENT EMERGING MARKETS FUND', 'Brandes Investment Partners'), ('THE BOARD OF REGENTS OF THE UNIVERSITY OF TEXAS SYSTEM', 'Acadian Asset Mgt'), ('REKSA DANA INDEKS PREMIER ETF MSCI INDONESIA LARGE CAP', 'Indo Premier Investment Mgt'), ('ARROWSTREET INTERNATIONAL EQUITY ACWI EX US TRUST FUND', 'Arrowstreet Capital'), ('REKSA DANA SUCORINVEST PREMIUM SOVEREIGN BALANCED FUND', 'SucorInvest Asset Mgt'), ('UBS LUX FUND SOLUTIONS MSCI EMERGING MARKETS UCITS ETF', 'UBS Global Asset Mgt'), ('UBS LIFE ALL WORLD EQUITY FACTOR MIX CARBON AWARE FUND', 'UBS Global Asset Mgt'), ('REKSA DANA ASHMORE DANA OBLIGASI PROVIDENTIA NUSANTARA', 'Ashmore Asset Mgt'), ('TIAA CREF FUNDS TIAA CREF EMERGING MARKETS EQUITY FUND', 'TIAA Investment Mgt'), ('LAZARD INVESTMENT FUNDS LAZARD DEVELOPING MARKETS FUND', 'Lazard Asset Mgt'), ('LION OCBC SECURITIES APAC FINANCIALS DIVIDEND PLUS ETF', 'Lion Global Investors'), ('FAM SERIES UCITS ICAV EMERGING MARKETS EQUITY FAM FUND', 'Fineco Asset Mgt'), ('JAPAN MUTUAL AID ASSOCIATION OF PUBLIC SCHOOL TEACHERS', 'Resona Asset Mgt'), ('SWISSCANTO CH EQUITY FUND SUSTAINABLE EMERGING MARKETS', 'ZKB Asset Mgt'), ('JOHN HANCOCK TRUST COMPANY COLLECTIVE INVESTMENT TRUST', 'Manulife Investment Mgt'), ('PRINCIPAL GLOBAL INVESTORS COLLECTIVE INVESTMENT TRUST', 'Principal Global Investors'), ('LAZARD WILMINGTON ACW EX US DIVERSIFIED ADVANTAGE FUND', 'Christian Riccardo'), ('AFFIN HWANG SELECT ASIA PACIFIC EX JAPAN DIVIDEND FUND', 'Christian Riccardo'), ('VANGUARD INTERNATIONAL HIGH DIVIDEND YIELD INDEX FUND', 'Vanguard Group'), ('STATE OF CONNECTICUT RETIREMENT PLANS AND TRUST FUNDS', 'ARGA Investment Mgt'), ('M AND G LUX INVESTMENT FUNDS 1 INCOME ALLOCATION FUND', 'M&G Investments'), ('FAM SERIES UCITS ICAV DIVERSITY AND ICLUSION FAM FUND', 'M&G Investments'), ('MONDRIAN FOCUSED ALL COUNTRIES WORLD EX US EQUITY CIT', 'Mondrian Investment Partners'), ('INTERNATIONAL BANK FOR RECONSTRUCTION AND DEVELOPMENT', 'Northern Trust Global Investments'), ('STICHTING DEPOSITARY APG EMERGING MARKETS EQUITY POOL', 'APG Asset Mgt Asia'), ('INVESTORS GROUP IPROFILE EMERGING MARKETS EQUITY POOL', 'JPMorgan Asset Mgt'), ('SCHWAB FUNDAMENTAL EMERGING MARKETS EQUITY INDEX FUND', 'Charles Schwab Investment Mgt'), ('ROBECO CAPITAL GROWTH FUNDS EMERGING MARKETS EQUITIES', 'Robeco Asset Mgt'), ('DIMENSIONAL EMERGING MARKETS EX CHINA CORE EQUITY ETF', 'Dimensional Fund Advisors'), ('ARROWSTREET DELAWARE CUSTOM ALPHA EXTENSION FUND Q LP', 'Arrowstreet Capital'), ('BNY MELLON GLOBAL EMERGING MARKETS OPPORTUNITIES FUND', 'Newton Investment Mgt'), ('SEVENTH SWEDISH NATIONAL PENSION FUND AP7 EQUITY FUND', 'Sjunde AP-fonden'), ('BPER INTERNATIONAL SICAV GLOBAL BALANCED RISK CONTROL', 'UBS Global Asset Mgt'), ('XTRACKERS IE PLC MSCI AC WORLD ESG SCREENED UCITS ETF', 'DWS Investments'), ('HARTFORD SCHRODERS INTERNATIONAL MULTI CAP VALUE FUND', 'Schroder Investment Mgt'), ('SMTB PICTET GLOBAL THEMATIC OPPORTUNITIES MOTHER FUND', 'Pictet Asset Mgt'), ('RUSSELL INVESTMENTS MENSURAM BONAM GLOBAL SHARES FUND', 'Russell Investments'), ('AXA VORSORGE FONDS II EQUITIES GLOBAL MARKETS PASSIVE', 'AXA Investment Mgrs Schweiz'), ('FRANKLIN EMERGING MARKET CORE DIVIDEND TILT INDEX ETF', 'Franklin Templeton Investments'), ('GLOBAL ALPHA STRATEGY FUND GLOBAL EQUITY SATELLITE II', 'Fondaco Lux'), ('GLOBAL ALPHA STRATEGY FUND GLOBAL EQUITY SATELLITE IV', 'Fondaco Lux'), ('MARATHON UCITS FUNDS GLOBAL OPPORTUNITIES EQUITY FUND', 'Marathon Asset Mgt'), ('TOUCHSTONE SANDS CAPITAL EMERGING MARKETS GROWTH FUND', 'Christian Riccardo'), ('CALAMOS ANTETOKOUNMPO GLOBAL SUSTAINABLE EQUITIES ETF', 'Christian Riccardo'), ('BROWN ADVISORY SUSTAINABLE INTERNATIONAL LEADERS FUND', 'Christian Riccardo'), ('WEALTH INVEST AKL SEB EMERGING MARKET EQUITIES HERMES', 'Christian Riccardo'), ('FISHER INVESTMENTS ALL FOREIGN EQUITY COLLECTIVE FUND', 'Christian Riccardo'), ('FIDELITY SAI SUSTAINABLE EMERGING MARKETS EQUITY FUND', 'Christian Riccardo'), ('FTGF CLEARBRIDGE GLOBAL SUSTAINABILITY IMPROVERS FUND', 'Christian Riccardo'), ('VANGUARD FUNDS PLC VANGUARD FTSE ALL WORLD EX US ETF', 'Vanguard Group'), ('ARGA FUNDS TRUST ARGA EMERGING MARKETS EX CHINA FUND', 'ARGA Investment Mgt'), ('LEGAL AND GENERAL GLOBAL EMERGING MARKETS INDEX FUND', 'Legal & General Investment Mgt'), ('COLLEGES OF APPLIED ARTS AND TECHNOLOGY PENSION PLAN', 'Acadian Asset Mgt'), ('FLEXSHARES INTERNATIONAL QUALITY DIVIDEND INDEX FUND', 'Northern Trust Global Investments'), ('REKSA DANA CAMPURAN SUCORINVEST DANA PRIORITAS SEHAT', 'SucorInvest Asset Mgt'), ('SMARTWEALTH RUPIAH EQUITY SUSTAINABLE INDONESIA FUND', 'Allianz Life Insurance'), ('POLICE AND FIREMEN S RETIREMENT SYSTEM OF NEW JERSEY', 'Mellon'), ('JPMORGAN ACTIVE BUILDERS EMERGING MARKETS EQUITY ETF', 'JPMorgan Asset Mgt'), ('GOLDMAN SACHS ACTIVEBETA EMERGING MARKETS EQUITY ETF', 'Goldman Sachs Asset Mgt'), ('GOLDMAN SACHS MARKETBETA EMERGING MARKETS EQUITY ETF', 'Goldman Sachs Asset Mgt'), ('MUAM EMERGING EQUITY INDEX MOTHER FUND MTBJ101111080', 'Mitsubishi UFJ Asset Mgt'), ('ABRDN EVOLVE ASIA PACIFIC EX JAPAN EQUITY INDEX FUND', 'aberdeen'), ('CBUS CONSTRUCTION AND BUILDING UNIONS SUPERANNUATION', 'Construction and Building Industry Super - Cbus'), ('VARIABLE PORTFOLIO PARTNERS INTERNATIONAL VALUE FUND', 'Thompson, Siegel & Walmsley'), ('BRIDGE BUILDER TAX MANAGED INTERNATIONAL EQUITY FUND', 'Parametric Portfolio Associates'), ('THE TRUSTEE OF GLOBAL SUSTAINABLE EQUITY MOTHER FUND', 'Nomura Asset Mgt'), ('SWISSCANTO IE ESGEN SDG INDEX EQUITY WORLD UCITS ETF', 'ZKB Asset Mgt'), ('DEKA MSCI EMERGING MARKETS ESG FOCUS SELECT UCTS ETF', 'Deka Investment Mgt'), ('RUSSELL INVESTMENTS LOW CARBON AUSTRALIAN SHARE FUND', 'Russell Investments'), ('RUSSELL INVESTMENTS CONTINENTAL EUROPEAN EQUITY FUND', 'UBS Global Asset Mgt'), ('KESEJAHTERAAN PEGAWAI PERUM PERHUTANI JATI SEJAHTERA', 'Yay Kesejahteraan Pegawai Perum Perhutani Jati Sejahtera'), ('ASSETPLUS SUPER ASIA PRIVATE EQUITY INVESTMENT TRUST', 'Assetplus Investment Mgt'), ('KRANESHARES MSCI EMERGING MARKETS EX CHINA INDEX ETF', 'Krane Funds Advisors'), ('MOUNT MURRAY INVESTMENT EMERGING MARKETS EQUITY FUND', 'Mount Murray Investment'), ('IRONBARK ROYAL LONDON CONCENTRATED GLOBAL SHARE FUND', 'Christian Riccardo'), ('ISHARES MSCI INDONESIA INVESTABLE MARKET INDEX FUND', 'BlackRock Investment Mgt - Index'), ('ST JAMES S PLACE EMERGING MARKETS EQUITY UNIT TRUST', 'ARGA Investment Mgt'), ('ARGA INTERNATIONAL SECTOR NEUTRAL VALUE EQUITY FUND', 'ARGA Investment Mgt'), ('BLACKROCK EMERGING MARKETS ESG INSIGHTS EQUITY FUND', 'BlackRock Investment Mgt - Index'), ('SSGA MSCI ACWI EX USA INDEX NON LENDING DAILY TRUST', 'State Street Investment Mgt'), ('WEST MIDLANDS METROPOLITAN AUTHORITIES PENSION FUND', 'Mondrian Investment Partners'), ('T ROWE PRICE EMERGING MARKETS DISCOVERY STOCK TRUST', 'T Rowe Price International'), ('STICHTING DEPOSITARY APG EMERGING MULTI CLIENT POOL', 'APG Asset Mgt Asia'), ('ASURANSI JIWA MANULIFE INDONESIA IDR NON PAR EQUITY', 'Manulife Investment Mgt Indonesia'), ('ASURANSI JIWA MANULIFE INDONESIA IDR SURPLUS EQUITY', 'Manulife Investment Mgt Indonesia'), ('REKSA DANA INDEKS MANDIRI INDEKS FTSE INDONESIA ESG', 'Mandiri Investasi'), ('ST JAMES S PLACE GLOBAL EMERGING MARKETS UNIT TRUST', 'Robeco Asset Mgt'), ('1895 FONDS FGR WERELD AANDELEN ENHANCED INDEX FONDS', 'BlackRock Investment Mgt'), ('PUBLIC EMPLOYEES RETIREMENT ASSOCIATION OF COLORADO', 'Colorado PERA'), ('GOLDMAN SACHS EMERGING MARKETS EQUITY INSIGHTS FUND', 'Goldman Sachs Asset Mgt'), ('ABERDEEN SELECT PORTFOLIO ABERDEEN INCOME PLUS FUND', 'aberdeen'), ('LION CHINA MERCHANTS EMERGING ASIA SELECT INDEX ETF', 'Lion Global Investors'), ('NOMURA FUNDS IRELAND GLOBAL SUSTAINABLE EQUITY FUND', 'Nomura Asset Mgt'), ('REKSA DANA INDEKS FWD ASSET IDX30 INDEX EQUITY FUND', 'FWD Asset Mgt'), ('KAPITALFORENINGEN INVESTIN PRO GLOBALE AKTIER INDEX', 'Nykredit Asset Mgt'), ('REKSA DANA SAHAM UOBAM SUSTAINABLE EQUITY INDONESIA', 'UOB Asset Mgt'), ('INVESTORS WHOLESALE EMERGING MARKETS EQUITIES TRUST', 'BlackRock Investment Mgt (Australia)'), ('FIDELITY SELECT PORTFOLIOS SEMICONDUCTORS PORTFOLIO', 'Fidelity Investments'), ('BOCI PRUDENTIAL MSCI AC ASIA PACIFIC EX JAPAN INDEX', 'BOCI Prudential Asset Mgt'), ('DESJARDINS SUSTAINABLE EMERGING MARKETS EQUITY FUND', 'Desjardins Global Asset Mgt'), ('DAIWA EMERGING EQUITY FUNDAMENTAL INDEX MOTHER FUND', 'Daiwa Asset Mgt'), ('RHUMBLINE CV ALL GLOBAL EX US EQUITY INDEX FUND LLC', 'RhumbLine Advisers'), ('TD EMERALD EMERGING MARKETS INDEX POOLED FUND TRUST', 'TD Asset Mgt'), ('MARGETTS AFH DA GLOBAL EMERGING MARKETS EQUITY FUND', 'Jupiter Asset Mgt'), ('FIAM EMERGING MARKETS OPPORTUNITIES COMMINGLED POOL', 'Christian Riccardo'), ('HARBOR ETF TRUST HARBOR EMERGING MARKETS SELECT ETF', 'Christian Riccardo'), ('KRUNGTHAI AXA LIFE INSURANCE PUBLIC COMPANY LIMITED', 'Christian Riccardo'), ('UNIFIED SERIES TRUST ONEASCENT EMERGING MARKETS ETF', 'Christian Riccardo'), ('EASTSPRING INVESTMENTS SINGAPORE ASEAN EQUITY FUND', 'Eastspring Investments'), ('EASTSPRING INVESTMENTS ASIA PACIFIC EQUITY MY FUND', 'Eastspring Investments'), ('STATE STREET AUT EMERGING MARKET INDEX EQUITY FUND', 'State Street Investment Mgt'), ('HSBC LIFE INTERNATIONAL WEALTH GOAL INSURANCE PLAN', 'HSBC Global Asset Mgt'), ('NORTHERN TRUST GLOBAL INVESTMENTS COLLECTIVE FUNDS', 'Northern Trust Global Investments'), ('SUNAMERICA SERIES TRUST EMERGING MARKETS PORTFOLIO', 'JPMorgan Asset Mgt'), ('REKSA DANA INDEKS BATAVIA INDEX PEFINDO PRIME BANK', 'Batavia Prosperindo Asset Mgt'), ('T ROWE PRICE EMERGING MARKETS DISCOVERY STOCK FUND', 'T Rowe Price'), ('REKSA DANA INDEKS PREMIER ETF IDX HIGH DIVIDEND 20', 'Indo Premier Investment Mgt'), ('REKSA DANA SUCORINVEST AGGRESSIVE BALANCED AI FUND', 'SucorInvest Asset Mgt'), ('CANADA LIFE EMERGING MARKETS LARGE CAP EQUITY FUND', 'Mackenzie Investments'), ('WELLS FARGO ADVANTAGE EMERGING MARKETS EQUITY FUND', 'Allspring Global Investments'), ('GOLDMAN SACHS ACTIVE BETA INTERNATIONAL EQUITY ETF', 'Goldman Sachs Asset Mgt'), ('UBS LUX INST SICAV EMERGING MARKETS EQUITY PASSIVE', 'UBS Global Asset Mgt'), ('CMA STATE STREET EMERGING EQUITY INDEX MOTHER FUND', 'State Street Investment Mgt'), ('RUTH ASSET MANAGEMENT SICAV EMERGING MARKET EQUITY', 'Ruth Asset Mgt'), ('AVIVA INVESTORS GLOBAL EMERGING MARKETS INDEX FUND', 'Aviva Investors'), ('BORDER TO COAST EMERGING MARKETS EQUITY ALPHA FUND', 'Goldman Sachs Asset Mgt'), ('CC AND L QUANTITATIVE EMERGING MARKETS EQUITY FUND', 'Connor Clark & Lunn Investment Mgt'), ('FAM SERIES UCITS ICAV FINECO AM GLOBAL EQUITY FUND', 'Fineco Asset Mgt'), ('BNP PARIBAS EASY MSCI EMERGING ESG FILTERED MIN TE', 'BNP Paribas Asset Mgt'), ('SWISSCANTO CH INDEX FUND V EQUITY EMERGING MARKETS', 'ZKB Asset Mgt'), ('AXA CHINA REGION INSURANCE COMPANY BERMUDA LIMITED', 'AXA Investment Mgrs Asia'), ('TEACHERS RETIREMENT SYSTEM OF THE CITY OF NEW YORK', 'Teachers Retirement System of the City of New York'), ('AJ GENERALI INDONESIA PT GENERALY EQUITY BETA PLUS', 'Generali Indonesia'), ('PEAR TREE POLARIS INTERNATIONAL OPPORTUNITIES FUND', 'Polaris Capital Mgt'), ('RHUMBLINE ALL COUNTRY INTERNATIONAL INDEX FUND LLC', 'RhumbLine Advisers'), ('RATHBONE SICAV GLOBAL EMERGING MARKETS EQUITY FUND', 'Rathbone Investment Mgt'), ('PRINCIPAL CHINA INDIA INDONESIA OPPORTUNITIES FUND', 'Principal Asset Mgt'), ('PERPETUAL SMART BETA EMERGING MARKET EQUITIES FUND', 'Perpetual Investments'), ('THE MARATHON LONDON GLOBAL SELECT INVESTMENT TRUST', 'Marathon Asset Mgt'), ('RUSSELL INVESTMENTS SUSTAINABLE GLOBAL SHARES FUND', 'Russell Investments'), ('SANDS CAPITAL EMERGING MARKETS EX CHINA GROWTH ETF', 'Christian Riccardo'), ('FIDELITY EMERGING MARKETS QUALITY INCOME UCITS ETF', 'Christian Riccardo'), ('BRANDYWINE GLOBAL OPPORTUNISTIC EQUITY MOTHER FUND', 'Christian Riccardo'), ('ISHARES ESG AWARE MSCI EMERGING MARKETS INDEX ETF', 'BlackRock Investment Mgt - Index'), ('BLACKROCK EMERGING MARKETS LONG SHORT EQUITY FUND', 'BlackRock Investment Mgt - Index'), ('ISHARES MSCI ALL COUNTRY ASIA EX JAPAN INDEX FUND', 'BlackRock Investment Mgt - Index'), ('PZENA INTERNATIONAL VALUE ALL COUNTRY EX U S FUND', 'Pzena Investment Mgt'), ('MERCER GLOBAL EQUITY ALL COUNTRY WORLD INDEX FUND', 'Brandes Investment Partners'), ('ISHARES IV PLC MSCI EM IMI ESG SCREENED UCITS ETF', 'BlackRock Investment Mgt - Index'), ('STICHTING PENSIOENFONDS RAIL AND OPENBAAR VERVOER', 'State Street Investment Mgt'), ('BAILLIE GIFFORD OVERSEAS GROWTH FUND PACIFIC FUND', 'Baillie Gifford & Co'), ('ROBECO CAPITAL GROWTH FUNDS ASIA PACIFIC EQUITIES', 'Robeco Asset Mgt'), ('REKSA DANA EASTSPRING INVESTMENTS VALUE DISCOVERY', 'Eastspring Investments'), ('LEGAL AND GENERAL ASIA PACIFIC EQUITY INCOME FUND', 'Legal & General Investment Mgt'), ('MONDRIAN ALL COUNTRIES WORLD EX US EQUITY FUND LP', 'Mondrian Investment Partners'), ('MONDRIAN EMERGING MARKETS EX CHINA EQUITY FUND LP', 'Mondrian Investment Partners'), ('PACIFIC SELECT FUND PD EMERGING MARKETS PORTFOLIO', 'Geode Capital Mgt'), ('NORTHERN TRUST EMERGING MARKETS EQUITY INDEX FUND', 'Northern Trust Global Investments'), ('NTCC COLLECTIVE FUNDS FOR EMPLOYEE BENEFIT TRUSTS', 'Northern Trust Global Investments'), ('JPMORGAN GLOBAL EMERGING MARKETS INCOME TRUST PLC', 'JPMorgan Asset Mgt'), ('T ROWE PRICE FUNDS OEIC GLOBAL IMPACT EQUITY FUND', 'T Rowe Price'), ('DIMENSIONAL FUNDS PLC WORLD ALLOCATION 60 40 FUND', 'Dimensional Fund Advisors'), ('REKSA DANA DANAREKSA BALANCED REGULAR INCOME FUND', 'BRI Investment Mgt'), ('MOOREA FUND MULTI MANAGER EMERGING MARKETS EQUITY', 'BlackRock Investment Mgt'), ('ARROWSTREET ACWI EX US ALPHA EXTENSION TRUST FUND', 'Arrowstreet Capital'), ('PFI MEGA LIFE INSURANCE PFI MEGA LIFE EQUITY FUND', 'Schroder Investment Mgt'), ('REKSA DANA ASHMORE PROVIDENTIA BALANCED NUSANTARA', 'Ashmore Asset Mgt'), ('UNIVERSAL FONDS UNIVERSAL INVESTMENT GESELLSCHAFT', 'Universal Investment'), ('NORTHERN TRUST EUROPE EQUITY INDEX UCITS FGR FUND', 'Northern Trust Global Investments'), ('JPMORGAN BETABUILDERS EMERGING MARKETS EQUITY ETF', 'JPMorgan Asset Mgt'), ('JPMORGAN GLOBAL EMERGING MARKETS HYBRID FUND QDII', 'JPMorgan Asset Mgt'), ('MATTHEWS EMERGING MARKETS SUSTAINABLE FUTURE FUND', 'Matthews International Capital Mgt'), ('ABRDN EMERGING MARKETS EQUITY ENHANCED INDEX FUND', 'aberdeen'), ('AVIVA INVESTORS GLOBAL EMERGING MARKETS CORE FUND', 'Aviva Investors'), ('PENDAL GLOBAL EMERGING MARKETS OPPORTUNITIES FUND', 'JO Hambro Capital Mgt'), ('AB SICAV I EMERGING MARKETS MULTI ASSET PORTFOLIO', 'AllianceBernstein - Growth'), ('AJ GENERALI INDONESIA PT GENERALI FORTUNA EQ FUND', 'Generali Indonesia'), ('FIDELITY SUSTAINABLE EMERGING MARKETS EQUITY FUND', 'Fidelity Investments'), ('FIDELITY EMERGING MARKETS EQUITY INVESTMENT TRUST', 'Fidelity Investments'), ('ABERDEEN STANDARD SICAV 1 DIVERSIFIED INCOME FUND', 'aberdeen'), ('REKSA DANA STAR VALUE HIGH CONVICTION EQUITY FUND', 'Star Asset Mgt'), ('INVESTERINGSFORENINGEN SPARINVEST INDEX MSCI ACWI', 'Sparinvest'), ('COLUMBIA RESEARCH ENHANCED EMERGING ECONOMIES ETF', 'Columbia Threadneedle Investments'), ('RAKUTEN EMERGING MARKETS EQUITY INDEX MOTHER FUND', 'Rakuten Investment Mgt'), ('XTRACKERS MSCI EMERGING MARKETS HEDGED EQUITY ETF', 'DBX Advisors'), ('SANDS CAPITAL EMERGING MARKETS GROWTH MASTER FUND', 'Christian Riccardo'), ('LAZARD EMERING MARKETS EQUITY ADVANTAGE PORTFOLIO', 'Christian Riccardo'), ('MACQUARIE SYSTEMATIC EMERGING MARKETS EQUITY FUND', 'Christian Riccardo'), ('NINETY ONE FUNDS SERIES I DIVERSIFIED INCOME FUND', 'Christian Riccardo'), ('PUBLIC SCHOOL TEACHERS RETIREMENT FUND OF CHICAGO', 'Public School Teachers Retirement Fund of Chicago'), ('ISHARES CORE MSCI EMERGING MARKETS IMI INDEX ETF', 'BlackRock Investment Mgt - Index'), ('ISHARES MSCI EMERGING MARKETS QUALITY FACTOR ETF', 'BlackRock Investment Mgt - Index'), ('WELLS FARGO MANAGED EMERGING MARKETS EQUITY FUND', 'State Street Investment Mgt'), ('HSBC POOLED VC ASIA PACIFIC EX JAPAN EQUITY FUND', 'HSBC Global Asset Mgt'), ('VERDIPAPIRFONDET STOREBRAND INDEKS ALLE MARKEDER', 'Storebrand Investments'), ('DANA PENSIUN LEMBAGA KEUANGAN MANULIFE INDONESIA', 'Manulife Investment Mgt Indonesia'), ('REKSA DANA INDEKS PREMIER ETF FTSE INDONESIA ESG', 'Indo Premier Investment Mgt'), ('REKSA DANA BALANCED FUND SUCORINVEST ANAK PINTAR', 'SucorInvest Asset Mgt'), ('GLOBAL X MSCI SUPERDIVIDEND EMERGING MARKETS ETF', 'Global X Mgt'), ('RUSSELL INVESTMENT COMPANY EMERGING MARKETS FUND', 'UBS Global Asset Mgt'), ('ABERDEEN ASIA PACIFIC EQUITY ENHANCED INDEX FUND', 'aberdeen (SWIP)'), ('UNIVERSAL INVESTMENT GMBH FOR APT UNIVERSAL FUND', 'Universal Investment'), ('BMO SUSTAINABLE OPPORTUNITIES GLOBAL EQUITY FUND', 'Columbia Threadneedle Investments'), ('PUBLIC EMPLOYEES RETIREMENT SYSTEM OF OHIO OPERS', 'Ohio Public Employees Retirement System (OPERS)'), ('HARTFORD SCHRODERS DIVERSIFIED OPPRTUNITIES FUND', 'Schroder Investment Mgt'), ('PICTET GLOBAL THEMATIC OPPORTUNITIES MASTER FUND', 'Pictet Asset Mgt'), ('SMTB PICTET POSITIVE CHANGE STRATEGY MOTHER FUND', 'Pictet Asset Mgt'), ('INTERNATIONALE KAPITALANLAGEGESELLSCHAFT MBH EMA', 'HSBC INKA'), ('RUSSELL INVESTMENTS EMERGING MARKETS EQUITY POOL', 'Russell Investments'), ('KAPITALFORENINGEN INVESTIN PRO GLOBAL EQUITIES I', 'Nykredit Asset Mgt'), ('NISSAY EMERGING MARKETS EQUITY INDEX MOTHER FUND', 'Nissay Asset Mgt'), ('DESJARDINS RI EMERGING MARKETS LOW CO2 INDEX ETF', 'Desjardins Global Asset Mgt'), ('ARCA FONDI SOCIETA DI GESTIONE DEL RISPARMIO SPA', 'Arca Fondi SGR'), ('FIRST SENTIER INVESTORS GLOBAL UMBRELLA FUND PLC', 'First Sentier Investors'), ('XTRACKERS MSCI ALL WORLD EX US HEDGED EQUITY ETF', 'DBX Advisors'), ('SCOTIA EMERGING MARKETS EQUITY INDEX TRACKER ETF', 'State Street Investment Mgt'), ('EUROMOBILIARE PICTET ACTION 4 TRANSITION ATTO II', 'Christian Riccardo'), ('WELLINGTON MANAGEMENT FUNDS LUXEMBOURG III SICAV', 'Christian Riccardo'), ('CANTILLON FUNDS PLC CANTILLON GLOBAL EQUITY FUND', 'Christian Riccardo'), ('ISHARES CORE MSCI TOTAL INTERNATIONAL STOCK ETF', 'BlackRock Investment Mgt - Index'), ('EASTSPRING INVESTMENTS ASIA PACIFIC EQUITY FUND', 'Eastspring Investments'), ('ISHARES IV PLC MSCI EMERGING EX CHINA UCITS ETF', 'BlackRock Investment Mgt - Index'), ('STATE STREET EMERGING MARKETS EQUITY INDEX FUND', 'State Street Investment Mgt'), ('M AND G INVESTMENT FUNDS 11 EPISODE INCOME FUND', 'M&G Investments'), ('MONDRIAN GLOBAL ALL COUNTRIES WORLD EQUITY FUND', 'Mondrian Investment Partners'), ('VERDIPAPIRFONDET STOREBRAND INDEKS NYE MARKEDER', 'Storebrand Investments'), ('VERDIPAPIRFONDET STOREBRAND EQUAL OPPORTUNITIES', 'Storebrand Investments'), ('ASURANSI JIWA MANULIFE INDONESIA IDR PAR EQUITY', 'Manulife Investment Mgt Indonesia'), ('IG JPMORGAN EMERGING MARKETS EQUITY FUND GROWTH', 'JPMorgan Asset Mgt'), ('REKSA DANA INDEKS BATAVIA INDEX PEFINDO I GRADE', 'Batavia Prosperindo Asset Mgt'), ('MANULIFE MULTIFACTOR EMERGING MARKETS INDEX ETF', 'Dimensional Fund Advisors'), ('EMERGING MARKETS EX CHINA CORE EQUITY PORTFOLIO', 'Dimensional Fund Advisors'), ('PANIN DAI ICHI LIFE PANIN SPECIAL BALANCED FUND', 'Panin Asset Mgt'), ('SMARTWEALTH RUPIAH EQUITY ROTATION CLASS B FUND', 'Allianz Life Insurance'), ('ALLIANZ LIFE SMARTWEALTH EQUITY INDOGLOBAL FUND', 'Allianz Life Insurance'), ('HC CAPITAL TRUST THE EMERGING MARKETS PORTFOLIO', 'Mellon'), ('WISDOMTREE EMERGING MARKETS EFFICIENT CORE FUND', 'Mellon'), ('JPMORGAN FUNDS CHINA A SHARE OPPORTUNITIES FUND', 'JPMorgan Asset Mgt (JF Asset Mgt)'), ('MACKENZIE EMERGING MARKETS EX CHINA EQUITY FUND', 'Mackenzie Investments'), ('WELLS FARGO EMERGING MARKETS LARGE MID CAP FUND', 'Allspring Global Investments'), ('UBS CH INDEX FUND EQUITIES EMERGING MARKETS NSL', 'UBS Global Asset Mgt'), ('REKSA DANA INDEKS BNI AM IDX PEFINDO PRIME BANK', 'BNI Asset Mgt'), ('HGIF ASIA PACIFIC EX JAPAN EQUITY HIGH DIVIDEND', 'HSBC Global Asset Mgt'), ('PIMCO BERMUDA DYNAMIC MULTI ASSET STRATEGY FUND', 'PIMCO'), ('SSGA SPDR ETFS EUROPE II PUBLIC LIMITED COMPANY', 'State Street Investment Mgt'), ('INVESCO MARKETS III PLC FTSE RAFI EUROPE MID SM', 'Invesco Capital Mgt'), ('SOUTHERN COMPANY SYSTEM MASTER RETIREMENT TRUST', 'Southern Company System Master Retirement Trust'), ('KAPITALFORENINGEN EMD INVEST GLOBALE AKTIER III', 'Nykredit Asset Mgt'), ('HARDING LOEVNER FUND EMERGING MARKETS PORTFOLIO', 'Harding Loevner Mgt'), ('HANG SENG INSURANCE COMPANY ASIA EQUITY PASSIVE', 'Hang Seng Investment Mgt'), ('AJ GENERALI INDONESIA PT GENERALY EQUITY ULTIMA', 'Generali Indonesia'), ('AJ GENERALI INDONESIA PT GENERALI EQUITY GROWTH', 'Generali Indonesia'), ('REKSA DANA INSIGHT TUNAS BANGSA BALANCED FUND 2', 'Insight Investments Mgt'), ('DANA PENSIUN LEMBAGA KEUANGAN ALLIANZ INDONESIA', 'Allianz Global Investors Asset Mgt Indonesia'), ('FIRE AND POLICE PENSION ASSOCIATION OF COLORADO', 'Russell Investments'), ('BAYERNINVEST KAPITALVERWALTUNGSGESELLSCHAFT MBH', 'BayernInvest'), ('PRINCIPAL MPF ASIA PACIFIC EX JAPAN EQUITY FUND', 'Principal Asset Mgt'), ('BNZ WHOLESALE INTERNATIONAL EQUITIES INDEX FUND', 'Bank of New Zealand'), ('NGS SUPER NON GOVERNMENT SCHOOLS SUPERANNUATION', 'ClariVest Asset Mgt'), ('BOSTON COMMON SUSTAINABLE EMERGING MARKETS FUND', 'Christian Riccardo'), ('MACQUARIE INTERNATIONAL MIMAL MULTI FACTOR FUND', 'Christian Riccardo'), ('JARISLOWSKY FRASER EMERGING MARKETS EQUITY FUND', 'Christian Riccardo'), ('CFSIL COLONIAL FIRST STATE GLOBAL SHARES FUND 8', 'Christian Riccardo'), ('FIAM SELECT GLOBAL PLUS ALL CAP COMMINGLED POOL', 'Christian Riccardo'), ('EUROMOBILIARE INTERNATIONAL FUND QFUND90 RELOAD', 'Christian Riccardo'), ('ABU DHABI RETIREMENT PENSIONS AND BENEFITS FUND', 'Abu Dhabi Retirement Pensions and Benefits Fund'), ('VANGUARD FUNDS PLC VANGUARD FTSE ALL WORLD ETF', 'Vanguard Group'), ('BLACKROCK CDN MSCI EMERGING MARKETS INDEX FUND', 'BlackRock Investment Mgt - Index'), ('REGIME DE RETRAITE DE L UNIVERSITE DE MONTREAL', 'ARGA Investment Mgt'), ('CAISSE DE RETRAITE DES EMPLOYES D HYDRO QUEBEC', 'State Street Investment Mgt'), ('BAILLIE GIFFORD EMERGING MARKETS EX CHINA FUND', 'Baillie Gifford & Co'), ('BAILLIE GIFFORD WORLDWIDE POSITIVE CHANGE FUND', 'Baillie Gifford & Co'), ('T ROWE PRICE INTERNATIONAL FUNDS NEW ASIA FUND', 'T Rowe Price International'), ('FIDELITY FLEX INTERNATIONAL FOCUSED INDEX FUND', 'Geode Capital Mgt'), ('JPMORGAN EMERGING MARKETS INVESTMENT TRUST PLC', 'JPMorgan Asset Mgt'), ('JPMORGAN INVESTMENT FUNDS GLOBAL BALANCED FUND', 'JPMorgan Asset Mgt'), ('REKSA DANA INDEKS BNP PARIBAS IDX30 FILANTROPI', 'BNP Paribas Asset Mgt'), ('REKSA DANA MANDIRI INVESTA EQUITY ASEAN 5 PLUS', 'Mandiri Investasi'), ('T ROWE PRICE INTERNATIONAL EQUITY RESEARCH ETF', 'T Rowe Price'), ('SCHWAB FUNDAMENTAL EMERGING MARKETS EQUITY ETF', 'Charles Schwab Investment Mgt'), ('DIMENSIONAL EMERGING MARKETS CORE EQUITY 2 ETF', 'Dimensional Fund Advisors'), ('RD PREMIER ETF INDONESIA STATE OWNED COMPANIES', 'Indo Premier Investment Mgt'), ('ARROWSTREET ACWI ALPHA EXTENSION FUND V CAYMAN', 'Arrowstreet Capital'), ('WISDOMTREE EMERGING MARKETS HIGH DIVIDEND FUND', 'Mellon'), ('PACIFIC SELECT FUND EMERGING MARKETS PORTFOLIO', 'Goldman Sachs Asset Mgt'), ('REKSA DANA SYAILENDRA PENDAPATAN TETAP PREMIUM', 'Syailendra Capital'), ('REKSA DANA TRIMEGAH BALANCED ABSOLUTE STRATEGY', 'Trimegah Asset Mgt'), ('STATE STREET GLOBAL ADVISORS EXEMPT UNIT TRUST', 'State Street Investment Mgt'), ('SCHRODER INSTITUTIONAL DEVELOPING MARKETS FUND', 'Schroder Investment Mgt'), ('ABU DHABI RETIREMENT PENSION AND BENEFITS FUND', 'Abu Dhabi Retirement Pensions and Benefits Fund'), ('OBLATE INTERNATIONAL PASTORAL INVESTMENT TRUST', 'Artisan Partners'), ('CC AND L QUANTITATIVE GLOBAL EQUITY UCITS FUND', 'Connor Clark & Lunn Investment Mgt'), ('JNL MULTI MANAGER EMERGING MARKETS EQUITY FUND', 'T Rowe Price International'), ('RUSSELL INVESTMENTS FOCUSED GLOBAL EQUITY POOL', 'Russell Investments'), ('THE BUNTING FAMILY VI SOCIALLY RESPONSIBLE LLC', 'Parametric Portfolio Associates'), ('EARNEST EMERGING MARKETS INVESTMENT TRUST FUND', 'Earnest Partners'), ('AJ GENERALI INDONESIA PT GENERALI EQUITY INDEX', 'Generali Indonesia'), ('NOMURA SYSTEMATIC EMERGING MARKETS EQUITY FUND', 'Nomura Asset Mgt'), ('REKSA DANA PRINCIPAL INDO DOMESTIC EQUITY FUND', 'Principal Asset Mgt'), ('PREMIA DOW JONES EMERGING ASEAN TITANS 100 ETF', 'Premia Partners'), ('DESJARDINS EMERGING MARKETS OPPORTUNITIES FUND', 'Desjardins Global Asset Mgt'), ('FLOURISH INVESTMENT CORPORATION CIC SMART BETA', 'China Investment Corporation'), ('AMOVA MSCI AC ASIA EX JAPAN EX CHINA INDEX ETF', 'Amova Asset Mgt'), ('BROWN ADVISORY GLOBAL LEADERS SUSTAINABLE FUND', 'Christian Riccardo'), ('GOLDMAN SACHS GLOBAL SOCIAL IMPACT EQUITY FUND', 'Christian Riccardo'), ('TRG EMERGING HORIZON SELECT EQUITY MASTER FUND', 'Christian Riccardo'), ('THE TRUSTEES OF THE UNIVERSITY OF PENNSYLVANIA', 'Christian Riccardo'), ('NINETY ONE FUNDS SERIES III GLOBAL EQUITY FUND', 'Christian Riccardo'), ('VANGUARD TOTAL INTERNATIONAL STOCK INDEX FUND', 'Vanguard Group'), ('ISHARES MSCI ASIA EX JAPAN CLIMATE ACTION ETF', 'BlackRock Investment Mgt - Index'), ('ISHARES EMERGING MARKET IMI EQUITY INDEX FUND', 'BlackRock Investment Mgt - Index'), ('BLACKROCK EMERGING MARKETS ALPHA TILTS FUND B', 'BlackRock Investment Mgt - Index'), ('ISHARES MSCI EMERGING MARKETS ASIA INDEX FUND', 'BlackRock Investment Mgt - Index'), ('ISHARES IV PLC MSCI EM ESG ENHANCED UCITS ETF', 'BlackRock Investment Mgt - Index'), ('STATE STREET GLOBAL ADVISORS LUXEMBOURG SICAV', 'State Street Investment Mgt'), ('EMERGING MARKETS INDEX PLUS COMMON TRUST FUND', 'State Street Investment Mgt'), ('INTERNATIONAL EXPATRIATE BENEFIT MASTER TRUST', 'State Street Investment Mgt'), ('BAILLIE GIFFORD POSITIVE CHANGE EQUITIES FUND', 'Baillie Gifford & Co'), ('BAILLIE GIFFORD IMPACT INVESTMENT MOTHER FUND', 'Baillie Gifford & Co'), ('REKSA DANA INDEKS EASTSPRING ESGQ45 IDXKEHATI', 'Eastspring Investments'), ('QUEEN S UNIVERSITY OF BELFAST RETIREMENT PLAN', 'Legal & General Investment Mgt'), ('REKSA DANA MANULIFE INSTITUTIONAL EQUITY FUND', 'Manulife Investment Mgt Indonesia'), ('T ROWE PRICE INTERNATIONAL VALUE EQUITY TRUST', 'T Rowe Price'), ('ROBECO GLOBAL EMERGING MARKETS EQUITY FUND II', 'Robeco Asset Mgt'), ('JOHN HANCOCK MULTIFACTOR EMERGING MARKETS ETF', 'Dimensional Fund Advisors'), ('REKSA DANA INDEKS PREMIER ETF PEFINDO I GRADE', 'Indo Premier Investment Mgt'), ('SPW MANAGED ICVC SPW GLOBAL EX UK EQUITY FUND', 'BlackRock Investment Mgt'), ('ARROWSTREET CANADA GLOBAL ALL COUNTRY FUND II', 'Arrowstreet Capital'), ('REKSA DANA PANIN DANA INFRASTRUKTUR BERTUMBUH', 'Panin Asset Mgt'), ('LF ACCESS EMERGING MARKETS EQUITY FUND ROBECO', 'Robeco'), ('MACKENZIE EMERGING MARKETS LARGE CAPITAL FUND', 'Mackenzie Investments'), ('FIDELITY FUNDS SICAV ASIAN SPECIAL SITUATIONS', 'FIL Investment Mgt'), ('UBS IRL ETF PLC MSCI ACWI UNIVERSAL UCITS ETF', 'UBS Global Asset Mgt'), ('REKSA DANA INDEKS BNI AM INDEKS IDX GROWTH 30', 'BNI Asset Mgt'), ('REKSA DANA SYAILENDRA ALPHA FOCUS EQUITY FUND', 'Syailendra Capital'), ('DB ADVISORS EMERGING MARKETS EQUITIES PASSIVE', 'DWS Investments'), ('JPMORGAN ACTIVE DEVELOPING MARKETS EQUITY ETF', 'JPMorgan Asset Mgt'), ('SCHRODER SUSTAINABLE MULTI FACTOR EQUITY FUND', 'Schroder Investment Mgt'), ('ASI ASIA PACIFIC EX JAPAN EQUITY TRACKER FUND', 'aberdeen'), ('LAZARD EMERGING MARKETS EQUITY ADVANTAGE FUND', 'Lazard Asset Mgt'), ('SUMISHIN SHINKOUKOKUKABUSHI INDEX MOTHER FUND', 'Sumitomo Mitsui Trust Asset Mgt'), ('STB EMERGING MARKETS EQUITY INDEX MOTHER FUND', 'Sumitomo Mitsui Trust Asset Mgt'), ('YAYASAN KESEJAHTERAAN KARYAWAN BANK INDONESIA', 'Yayasan Kesejahteraan Karyawan Bank Indonesia'), ('NATIONAL RAILROAD RETIREMENT INVESTMENT TRUST', 'National Railroad Retirement Investment Trust'), ('EATON VANCE TAX MANAGED EMERGING MARKETS FUND', 'Parametric Portfolio Associates'), ('BMO PRIVATE EMERGING MARKETS EQUITY PORTFOLIO', 'Comgest'), ('FIDELITY EMERGING MARKETS EQUITY CENTRAL FUND', 'Fidelity Investments'), ('REKSA DANA PRINCIPAL TOTAL RETURN EQUITY FUND', 'Principal Asset Mgt'), ('SCHRODER INTERNATIONAL MULTI CAP EQUITY TRUST', 'Schroder Investment Mgt'), ('RUSSELL TAX MANAGED INTERNATIONAL EQUITY FUND', 'Russell Investments'), ('DANA PENSIUN MANFAAT PASTI UNILEVER INDONESIA', 'Dana Pensiun Manfaat Pasti Unilever Indonesia'), ('JAPAN POLICE PERSONNEL MUTUAL AID ASSOCIATION', 'Resona Bank'), ('VANGUARD INTERNATIONAL SHARES HIGH YIELD FUND', 'Vanguard Investments Australia'), ('INTERNATIONAL EQUITY SUSTAINABLE LEADERS FUND', 'Christian Riccardo'), ('FOURTON SILKKITIE ASIA EQUITY INVESTMENT FUND', 'Christian Riccardo'), ('TRANSAMERICA INTERNATIONAL SUSTAINABLE EQUITY', 'Christian Riccardo'), ('ST JAMES S PLACE STRATEGIC MANAGED UNIT TRUST', 'Christian Riccardo'), ('OHIO PUBLIC EMPLOYEES RETIREMENT SYSTEM OPERS', 'Ohio Public Employees Retirement System (OPERS)'), ('CONSTRUCTION AND BUILDING INDUSTRY SUPER CBUS', 'Construction and Building Industry Super - Cbus'), ('ISHARES CORE MSCI AC ASIA EX JAPAN INDEX ETF', 'BlackRock Investment Mgt - Index'), ('ISHARES EMERGING MARKETS DIVIDEND INDEX FUND', 'BlackRock Investment Mgt - Index'), ('DJS KETENAGAKERJAAN PROGRAM JAMINAN KEMATIAN', 'BPJS Ketenagakerjaan'), ('MICROSOFT CORPORATION SAVINGS PLUS 401K PLAN', 'Pzena Investment Mgt'), ('PZENA INTERNATIONAL EX US FOCUSED VALUE FUND', 'Pzena Investment Mgt'), ('ARGA GLOBAL SECTOR NEUTRAL VALUE EQUITY FUND', 'ARGA Investment Mgt'), ('BAILLIE GIFFORD EMERGING MARKETS GROWTH FUND', 'Baillie Gifford & Co'), ('STICHTING PENSIOENFONDS ZORG EN WELZIJN PFZW', 'PGGM Investments'), ('M AND G FUNDS 1 ASIA PACIFIC EX JAPAN EQUITY', 'M&G Investments'), ('HSBC MULTI FACTOR WORLDWIDE EQUITY UCITS ETF', 'HSBC Global Asset Mgt'), ('VERDIPAPIRFONDET STOREBRAND GLOBAL SOLUTIONS', 'Storebrand Investments'), ('KOP KESEHATAN PEG AND PENSIUNAN BANK MANDIRI', 'Mandiri Investasi'), ('MANDIRI ACTIVE BALANCED MONEY SYARIAH RUPIAH', 'Mandiri Investasi'), ('REKSA DANA BATAVIA PROVIDENTIA BALANCED FUND', 'Batavia Prosperindo Asset Mgt'), ('T ROWE PRICE INTERNATIONAL VALUE EQUITY FUND', 'T Rowe Price'), ('TRP INTERNATIONAL STRUCTURED RESEARCH EQUITY', 'T Rowe Price'), ('T ROWE PRICE GLOBAL EX U S EQUITY MACRO POOL', 'T Rowe Price'), ('DFA EMERGING MARKETS CORE EQUITY 2 PORTFOLIO', 'Dimensional Fund Advisors'), ('REKSA DANA INDEKS BAHANA ETF PEFINDO I GRADE', 'Bahana TCW Investment Mgt'), ('ARROWSTREET CANADA GLOBAL ALL COUNTRY FUND I', 'Arrowstreet Capital'), ('REKSA DANA CAMPURAN PANIN DANA BERIMBANG DUA', 'Panin Asset Mgt'), ('WISDOMTREE EMERGING MARKETS MULTIFACTOR FUND', 'Mellon'), ('NBI DIVERSIFIED EMERGING MARKETS EQUITY FUND', 'Goldman Sachs Asset Mgt'), ('REKSA DANA ASHMORE DANA USD EQUITY NUSANTARA', 'Ashmore Asset Mgt'), ('REKSA DANA SYAILENDRA DANA EKUITAS SEJAHTERA', 'Syailendra Capital'), ('REKSA DANA SYAILENDRA DANA INVESTASI DINAMIS', 'Syailendra Capital'), ('DANA PENSIUN KONFERENSI WALIGEREJA INDONESIA', 'Syailendra Capital'), ('SCHRODER CAPITAL MANAGEMENT COLLECTIVE TRUST', 'Schroder Investment Mgt'), ('FIDELITY INVESTMENT FUNDS FIDELITY ASIA FUND', 'FIL Investment Mgt'), ('INTERNATIONALE KAPITALANLAGEGESELLSCHAFT MBH', 'HSBC INKA'), ('RUSSELL INVESTMENTS GLOBAL SHARES INDEX FUND', 'Russell Investments'), ('EATON VANCE PARAMETRIC EMERGING MARKETS FUND', 'Parametric Portfolio Associates'), ('HARDING LOEVNER EMERGING MARKETS EQUITY FUND', 'Harding Loevner Mgt'), ('COLONIAL FIRST STATE EMERGING MARKETS FUND 6', 'First Sentier Investors RQI'), ('REKSA DANA INDEX PINNACLE FTSE INDONESIA ETF', 'Pinnacle Investment'), ('EURIZON EMERGING LEADERS ESG 50 GENNAIO 2027', 'Eurizon Capital'), ('FIDELITY MANAGEMENT AND RESEARCH COMPANY LLC', 'Fidelity Investments'), ('FIDELITY SELECT EMERGING MARKETS EQUITY FUND', 'Fidelity Investments'), ('RUSSELL EMERGING DIVIDEND GROWTH MOTHER FUND', 'Russell Investments'), ('DESJARDINS EMERGING MARKETS EQUITY INDEX ETF', 'Desjardins Global Asset Mgt'), ('WILMINGTON TRUST COLLECTIVE INVESTMENT TRUST', 'ClariVest Asset Mgt'), ('BNP PARIBAS EASY MSCI ACWI SRI PAB UCITS ETF', 'BNP Paribas Asset Mgt'), ('BOSTON COMMON CATHOLIC EMERGING MARKETS FUND', 'Christian Riccardo'), ('SANDS CAPITAL EMERGING MARKETS EX CHINA FUND', 'Christian Riccardo'), ('NINETY ONE FUND SERIES CAUTIOUS MANAGED FUND', 'Christian Riccardo'), ('NORTH CAROLINA DEPARTMENT OF STATE TREASURER', 'North Carolina Department of State Treasurer'), ('ALLIANZ GLOBAL INVESTORS ASSET MGT INDONESIA', 'Allianz Global Investors Asset Mgt Indonesia'), ('MITSUBISHI UFJ TRUST AND BANKING CORPORATION', 'Mitsubishi UFJ Trust & Banking Corporation'), ('ISHARES MSCI TOTAL INTERNATIONAL INDEX FUND', 'BlackRock Investment Mgt - Index'), ('BROWN ADVISORY EMERGING MARKETS SELECT FUND', 'Pzena Investment Mgt'), ('PACS GEM SMART BETA EQUITY PORTFOLIO ASGSBF', 'Eastspring Investments'), ('SPDR PORTFOLIO MSCI GLOBAL STOCK MARKET ETF', 'State Street Investment Mgt'), ('LVIP SSGA EMERGING MARKET EQUITY INDEX FUND', 'State Street Investment Mgt'), ('BAILLIE GIFFORD COLLECTIVE INVESTMENT TRUST', 'Baillie Gifford & Co'), ('OKLAHOMA PUBLIC EMPLOYEES RETIREMENT SYSTEM', 'Mondrian Investment Partners'), ('MONDRIAN EMERGING MARKETS VALUE EQUITY FUND', 'Mondrian Investment Partners'), ('MANULIFE SCHRODER DANA EKUITAS PREMIER FUND', 'Manulife Investment Mgt Indonesia'), ('MANULIFE DANA EKUITAS INDO DEVELOPED MARKET', 'Manulife Investment Mgt Indonesia'), ('REKSA DANA BNP PARIBAS INDONESIA ESG EQUITY', 'BNP Paribas Asset Mgt'), ('BUREAU OF LABOR FUNDS LABOR RETIREMENT FUND', 'Bureau of Labor Funds Taiwan'), ('REKSA DANA SUCORINVEST DANA SEHAT BERIMBANG', 'SucorInvest Asset Mgt'), ('ALLIANZ SMARTLINK RUPIAH BALANCED PLUS FUND', 'Allianz Life Insurance'), ('MACKENZIE EMERGING MARKETS EQUITY INDEX ETF', 'Mackenzie Investments'), ('GOTTHARD INSTITUTIONAL FUND GLOBAL EQUITIES', 'UBS Global Asset Mgt'), ('REKSA DANA ASHMORE SAHAM SEJAHTERA NUSATARA', 'Ashmore Asset Mgt'), ('REKSA DANA ASHMORE DANA PROGRESIF NUSANTARA', 'Ashmore Asset Mgt'), ('STICHTING PENSIOENFONDS HORECA AND CATERING', 'Northern Trust Global Investments'), ('TRIMEGAH FTSE4 GOOD INDONESIA INDEX KELAS M', 'Trimegah Asset Mgt'), ('PUBLIC EMPLOYEES RETIREMENT SYSTEM OF IDAHO', 'Public Employee Retirement System of Idaho'), ('SCHRODER EMERGING MARKETS EQUITY ALPHA FUND', 'Schroder Investment Mgt'), ('AIR PRODUCTS AND CHEMICALS INC MASTER TRUST', 'FIL Investment Mgt'), ('RBC VFS INTERNATIONAL EQUITY FUND INCUBATOR', 'RBC Global Asset Mgt'), ('SWEDBANK ROBUR ACCESS EDGE EMERGING MARKETS', 'Swedbank Robur'), ('RQI EMERGING MARKETS DIVERSIFIED ALPHA FUND', 'First Sentier Investors RQI'), ('EURIZON EMERGING LEADERS ESG 50 LUGLIO 2026', 'Eurizon Capital'), ('REKSA DANA INDEKS RHB SRI KEHATI INDEX FUND', 'Allianz Global Investors Asset Mgt Indonesia'), ('SMART WEALTH RUPIAH MULTI ASSET INCOME FUND', 'Allianz Global Investors Asset Mgt Indonesia'), ('REKSA DANA CIMB PRINCIPAL BALANCED FOCUS II', 'Principal Asset Mgt'), ('SMARTETFS ASIA PACIFIC DIVIDEND BUILDER ETF', 'Guinness Atkinson Asset Mgt'), ('COLUMBIA INTERNATIONAL DIVIDEND INCOME FUND', 'Columbia Threadneedle Investments'), ('COLUMBIA EMERGING MARKETS CORE EX CHINA ETF', 'Columbia Threadneedle Investments'), ('FINREON WORLD EQUITY ECONOMIC EXPSR PENSION', 'Finreon'), ('REKSA DANA SETIABUDI DANA KOMBINASI DINAMIS', 'Setiabudi Investment Mgt'), ('PENDAL SUSTAINABLE INTERNATIONAL SHARE FUND', 'Pendal Group'), ('BOSTON COMMON ESG IMPACT INTERNATIONAL FUND', 'Christian Riccardo'), ('CHAUTAUQUA INTERNATIONAL GROWTH EQUITY FUND', 'Christian Riccardo'), ('BRANDYWINE GLOBAL OPPORTUNISTIC EQUITY FUND', 'Christian Riccardo'), ('BRITISH COLUMBIA INVESTMENT MGT CORPORATION', 'British Columbia Investment Mgt Corporation'), ('RUSSELL INVESTMENTS IMPLEMENTATION SERVICES', 'Russell Investments Implementation Services'), ('VANGUARD EMERGING MARKETS STOCK INDEX FUND', 'Vanguard Group'), ('VANGUARD EMERGING MARKETS SHARE INDEX FUND', 'Vanguard Group'), ('ISHARES MSCI EMERGING MARKETS EX CHINA ETF', 'BlackRock Investment Mgt - Index'), ('ISHARES MSCI ALL COUNTRY ASIA EX JAPAN ETF', 'BlackRock Investment Mgt - Index'), ('EMERGING MARKETS INDEX NON LENDABLE FUND B', 'BlackRock Investment Mgt - Index'), ('ISHARES GLOBAL MONTHLY DIVIDEND INDEX FUND', 'BlackRock Investment Mgt - Index'), ('ISHARES LOW CARBON OPTIMIZED MSCI ACWI ETF', 'BlackRock Investment Mgt - Index'), ('PACE INTERNATIONAL EMERGING MARKETS EQUITY', 'ARGA Investment Mgt'), ('SPDR MSCI EMERGING MARKETS QUALITY MIX ETF', 'State Street Investment Mgt'), ('SPDR S AND P EMERGING MARKETS EX CHINA ETF', 'State Street Investment Mgt'), ('BAILLIE GIFFORD EMERGING MARKETS LARGE CAP', 'Baillie Gifford & Co'), ('DESJARDINS SOCIETERRA POSITIVE CHANGE FUND', 'Baillie Gifford & Co'), ('BMO PRIVATE INTERNATIONAL EQUITY PORTFOLIO', 'Pyrford International'), ('CLINTON NUCLEAR POWER PLANT QUALIFIED FUND', 'Northern Trust Global Investments'), ('REKSA DANA MANULIFE GREATER INDONESIA FUND', 'Manulife Investment Mgt Indonesia'), ('REKSA DANA INDEKS BNP PARIBAS IDX GROWTH30', 'BNP Paribas Asset Mgt'), ('T ROWE PRICE INTEGRATED GLOBAL EQUITY FUND', 'T Rowe Price'), ('ROBECO INSTITUTIONAL EMERGING MARKETS FUND', 'Robeco Asset Mgt'), ('DFA EMERGING MARKETS SOCIAL CORE PORTFOLIO', 'Dimensional Fund Advisors'), ('GLOBAL X ETFS ICAV SUPERDIVIDEND UCITS ETF', 'Global X Mgt'), ('REKSA DANA ASHMORE SAHAM UNGGULAN NUSATARA', 'Ashmore Asset Mgt'), ('REKSA DANA SYAILENDRA EQUITY MOMENTUM FUND', 'Syailendra Capital'), ('THREADNEEDLE LUX GLOBAL MULTI ASSET INCOME', 'Columbia Threadneedle Investments'), ('ABRDN EMERGING MARKETS EQUITY TRACKER FUND', 'aberdeen'), ('LAZARD DEVELOPING MARKETS EQUITY PORTFOLIO', 'Lazard Asset Mgt'), ('REGNAN GLOBAL EQUITY IMPACT SOLUTIONS FUND', 'JO Hambro Capital Mgt'), ('REKSA DANA PINNACLE CORE HIGH DIVIDEND ETF', 'Pinnacle Investment'), ('SIGNATURE SELECT APAC ALLOCATION PLUS FUND', 'BlackRock Investment Mgt (Australia)'), ('ABRDN EMERGING MARKETS DIVIDEND ACTIVE ETF', 'aberdeen'), ('FRANKLIN CLEARBRIDGE EMERGING MARKETS FUND', 'Franklin Templeton Investments'), ('MACQUARIE TRUE INDEX EMERGING MARKETS FUND', 'Macquarie Asset Mgt'), ('FRANKLIN EMERGING MARKETS EQUITY INDEX ETF', 'Franklin Templeton Investments'), ('FIRST TRUST GLOBAL EQUITY INCOME UCITS ETF', 'First Trust Advisors'), ('SANDS CAPITAL EMERGING MARKETS GROWTH FUND', 'Christian Riccardo'), ('REKSA DANA JUARA PROVIDENTIA BALANCED FUND', 'Christian Riccardo'), ('REKSA DANA JUARA EMPOWERMENT BALANCED FUND', 'Christian Riccardo'), ('FISHER INVESTMENTS INSTITUTIONAL FUNDS PLC', 'Christian Riccardo'), ('BRANDYWINE GLOBAL VALUE EQUITY MOTHER FUND', 'Christian Riccardo'), ('IG MACKENZIE PACIFIC INTERNATIONAL FUND II', 'Christian Riccardo'), ('REST RETAIL EMPLOYEES SUPERANNUATION TRUST', 'Christian Riccardo'), ('AHAM SELECT ASIA EX JAPAN OPPORTUNITY FUND', 'Christian Riccardo'), ('RUTGERS THE STATE UNIVERSITY OF NEW JERSEY', 'Christian Riccardo'), ('PUBLIC EMPLOYEE RETIREMENT SYSTEM OF IDAHO', 'Public Employee Retirement System of Idaho'), ('THE PUBLIC INSTITUTION FOR SOCIAL SECURITY', 'The Public Institution for Social Security'), ('STATE OF NEW JERSEY DIVISION OF INVESTMENT', 'State of New Jersey - Division of Investment'), ('INVESCO EMERGING MARKETS EX CHINA FUND UK', 'Invesco'), ('STATE OF NEW JERSEY COMMON PENSION FUND D', 'Pzena Investment Mgt'), ('PZENA EMERGING MARKETS FOCUSED VALUE FUND', 'Pzena Investment Mgt'), ('MONDRIAN EMERGING MARKETS EQUITY ESG FUND', 'Mondrian Investment Partners'), ('AQR GLOBAL RISK PREMIUM MASTER ACCOUNT II', 'AQR Capital Mgt'), ('REKSA DANA MANULIFE DANA TUMBUH BERIMBANG', 'Manulife Investment Mgt Indonesia'), ('REKSA DANA BNP PARIBAS INFRASTRUKTUR PLUS', 'BNP Paribas Asset Mgt'), ('MANDIRI PROGRESSIVE BALANCED MONEY RUPIAH', 'Mandiri Investasi'), ('DFA INTERNATIONAL VECTOR EQUITY PORTFOLIO', 'Dimensional Fund Advisors'), ('REKSA DANA INDEKS PREMIER ETF INDEX IDX30', 'Indo Premier Investment Mgt'), ('REKSA DANA DANAREKSA ETF INDONESIA TOP 40', 'BRI Investment Mgt'), ('PT AIA FINANCIAL HY HY NON PAR IDR EQUITY', 'AIA Indonesia'), ('YAYASAN KESEHATAN PENSIUNAN ANEKA TAMBANG', 'Bahana TCW Investment Mgt'), ('ARROWSTREET ACWI ALPHA EXTENSION FUND III', 'Arrowstreet Capital'), ('ARROWSTREET GLOBAL EQUITY ACWI TRUST FUND', 'Arrowstreet Capital'), ('ARROWSTREET DELAWARE ALPHA EXTENSION FUND', 'Arrowstreet Capital'), ('REKSA DANA INDEKS PANIN ETF IDX30 DINAMIS', 'Panin Asset Mgt'), ('REKSA DANA CAMPURAN PANIN PRIORITAS SEHAT', 'Panin Asset Mgt'), ('DPLK ALLIANZ IND HM SAMPOERNA EQUITY FUND', 'Allianz Life Insurance'), ('ALLIANZ LIFE IND SAVINGS PLAN EQUITY FUND', 'Allianz Life Insurance'), ('REKSA DANA SCHRODER DYNAMIC BALANCED FUND', 'Schroder Investment Mgt'), ('REKSA DANA ASHMORE DANA EKUITAS NUSANTARA', 'Ashmore Asset Mgt'), ('NUVEEN EMERGING MARKETS EQUITY INDEX FUND', 'TIAA Investment Mgt'), ('TIAA CREF FUNDS INTERNATIONAL EQUITY FUND', 'TIAA Investment Mgt'), ('FOREIGN AND COLONIAL INVESTMENT TRUST PLC', 'Columbia Threadneedle Investments'), ('CIBC EMERGING MARKETS EQUITY PRIVATE POOL', 'CIBC Asset Mgt'), ('BMO GUARDIAN ASIAN GROWTH AND INCOME FUND', 'Matthews International Capital Mgt'), ('STATE STREET AUT GLOBAL INDEX EQUITY FUND', 'State Street Investment Mgt'), ('AZL SCHRODER EMERGING MARKETS EQUITY FUND', 'Schroder Investment Mgt'), ('ABRDN EMERGING MARKETS EQUITY INCOME FUND', 'aberdeen'), ('NOMURA TB NAMCO GLOBAL HIGH PROFIT MOTHER', 'Nomura Asset Mgt'), ('LIONGLOBAL ASIA HIGH DIVIDEND EQUITY FUND', 'Lion Global Investors'), ('ARTISAN SUSTAINABLE EMERGING MARKETS FUND', 'Artisan Partners'), ('RBC EMERGING MARKETS EX CHINA EQUITY FUND', 'RBC Global Asset Mgt'), ('GENERAL ORGANISATION FOR SOCIAL INSURANCE', 'General Organization for Social Insurance'), ('JOHCM EMERGING MARKETS OPPORTUNITIES FUND', 'JO Hambro Capital Mgt'), ('SOCIETE GENERALE PARIS STK BASKET TRADING', 'SG Securities'), ('DBI FONDS DSPT 8 ALLIANZ GLOBAL INVESTORS', 'AXA Investment Mgrs Asia'), ('REKSA DANA PINNACLE STRATEGIC EQUITY FUND', 'Pinnacle Investment'), ('DODGE AND COX EMERGING MARKETS STOCK FUND', 'Dodge & Cox'), ('REKSA DANA INSIGHT BHINNEKA BALANCED FUND', 'Insight Investments Mgt'), ('REKSA DANA INSIGHT GENERATE BALANCED FUND', 'Insight Investments Mgt'), ('BLACKROCK ADVANTAGE EMERGING MARKETS FUND', 'BlackRock Investment Mgt'), ('YUANTA P SHARES MSCI INDONESIA INDEX FUND', 'Yuanta Securities Investment Trust'), ('CHUBB LIFE INSURANCE IND RUPIAH EQUITY GF', 'Chubb Life Insurance'), ('REKSA DANA CAMPURAN WANTEG DIVERSITY FUND', 'Wanteg Asset Mgt'), ('WILMINGTON EMERGING EQUITY ADVANTAGE FUND', 'Christian Riccardo'), ('KAPITALFORENINGEN EMERGING MARKETS AKTIER', 'Christian Riccardo'), ('CANADA LIFE GLOBAL EQUITY AND INCOME FUND', 'Christian Riccardo'), ('FIAM SELECT EMERGING MARKETS EQUITY TRUST', 'Christian Riccardo'), ('GENERAL ORGANIZATION FOR SOCIAL INSURANCE', 'General Organization for Social Insurance'), ('VANGUARD FTSE ALL WORLD EX US INDEX FUND', 'Vanguard Group'), ('VANGUARD FTSE EMERGING MARKETS UCITS ETF', 'Vanguard Group'), ('VANGUARD ASIA EX JAPAN SHARES INDEX FUND', 'Vanguard Group'), ('VANGUARD FTSE GLOBAL SMALL CAP UCITS ETF', 'Vanguard Group'), ('ISHARES MSCI EMERGING MARKETS INDEX FUND', 'BlackRock Investment Mgt - Index'), ('ISHARES ESG ADVANCED MSCI EAFE INDEX ETF', 'BlackRock Investment Mgt - Index'), ('ST JAMES S PLACE GLOBAL VALUE UNIT TRUST', 'Pzena Investment Mgt'), ('BRIDGE BUILDER INTERNATIONAL EQUITY FUND', 'Pzena Investment Mgt'), ('UN RELIEF AND WORKS AGENCY FOR PALESTINE', 'State Street Investment Mgt'), ('HSBC GLOBAL EMERGING MARKETS EQUITY FUND', 'HSBC Global Asset Mgt'), ('COLONIAL FIRST STATE INVESTMENT FUND 187', 'Acadian Asset Mgt'), ('REKSA DANA INDEKS BNP PARIBAS SRI KEHATI', 'BNP Paribas Asset Mgt'), ('REKSA DANA MANDIRI INVESTA CERDAS BANGSA', 'Mandiri Investasi'), ('REKSA DANA INDEKS MANDIRI ETF SRI KEHATI', 'Mandiri Investasi'), ('REKSA DANA MANDIRI INVESTA DYNAMO FACTOR', 'Mandiri Investasi'), ('SCHWAB INTERNATIONAL DIVIDEND EQUITY ETF', 'Charles Schwab Investment Mgt'), ('ALL COUNTRY EX US EQUITY MARKET SUBTRUST', 'Dimensional Fund Advisors'), ('REKSA DANA DANAREKSA MAWAR EKUITAS UTAMA', 'BRI Investment Mgt'), ('PT AIA FINANCIAL USD ONSHORE EQUITY FUND', 'AIA Indonesia'), ('BUREAU OF LABOR FUNDS LABOR PENSION FUND', 'Bureau of Labor Funds Taiwan'), ('ARROWSTREET CAPITAL NEWBURY FUND LIMITED', 'Arrowstreet Capital'), ('REKSA DANA CAMPURAN PANIN DANA BERIMBANG', 'Panin Asset Mgt'), ('REKSA DANA SUCORINVEST USD BALANCED FUND', 'SucorInvest Asset Mgt'), ('REKSA DANA INDEKS BNI AM PEFINDO I GRADE', 'BNI Asset Mgt'), ('REKSA DANA SYAILENDRA EQUITY GARUDA FUND', 'Syailendra Capital'), ('PUBLIC MUTUAL ASEAN DIVIDEND SEQUEL FUND', 'Public Mutual'), ('THE PUBLIC INSTITUTE FOR SOCIAL SECURITY', 'The Public Institution for Social Security'), ('AVANTIS CIBC EMERGING MARKETS EQUITY ETF', 'CIBC Asset Mgt'), ('REKSA DANA SAHAM HPAM SMART BETA EKUITAS', 'Henan Putihrai Asset Mgt'), ('WS WALES PP EMERGING MARKETS EQUITY FUND', 'Waystone Mgt'), ('METZLER INVESTMENT GMBH FOR MI FONDS F29', 'HSBC INKA'), ('INTERNATIONALE KAPITALANLAGEGESELLSCHAFT', 'HSBC INKA'), ('T ROWE PRICE EMERGING MARKETS STOCK FUND', 'T Rowe Price International'), ('HALLMARK CARDS INCORPORATED MASTER TRUST', 'Russell Investments'), ('RUSSELL INVESTMENTS OVERSEAS EQUITY POOL', 'Russell Investments'), ('COLONIAL FIRST STATE INVESTMENT FUND 110', 'JO Hambro Capital Mgt'), ('CFSIL COMMONWEALTH GLOBAL SHARES FUND 22', 'First Sentier Investors RQI'), ('COLONIAL FIRST STATE INVESTMENT FUND 122', 'First Sentier Investors RQI'), ('CFSIL COMMONWEALTH GLOBAL SHARES FUND 23', 'First Sentier Investors RQI'), ('STATE TEACHERS RETIREMENT SYSTEM OF OHIO', 'Ohio State Teachers Retirement System'), ('REKSA DANA INDEKS UOBAM INDEKS BISNIS 27', 'UOB Asset Mgt'), ('REKSA DANA INSIGHT BENEFIT BALANCED FUND', 'Insight Investments Mgt'), ('REKSA DANA INSIGHT COMMUNITY DEVELOPMENT', 'Insight Investments Mgt'), ('REKSA DANA INSIGHT TUNAS BANGSA I NEXT G', 'Insight Investments Mgt'), ('PT ASURANSI JIWA SEQUIS LIFE EQUITY FUND', 'Sequis Asset Mgt'), ('AEGON EQUITY EMERGING MARKETS INDEX FUND', 'AEGON Asset Mgt'), ('DEKA INVESTMENT GMBH RE LWH FONDS AKTIEN', 'Deka Investment Mgt'), ('REKSA DANA INDEKS PRINCIPAL INDEX IDX 30', 'Principal Asset Mgt'), ('NISSAY GLOBAL LEADERS EQUITY MOTHER FUND', 'Nissay Asset Mgt'), ('DANSKE INVEST INDEX GLOBAL AC RESTRICTED', 'Danske Capital Mgt'), ('TCSB EMERGING EQUITY PASSIVE MOTHER FUND', 'Asset Mgt One'), ('AMUNDI PRIME ALL COUNTRY WORLD UCITS ETF', 'Amundi Asset Mgt'), ('MTBOJ PENSION FUND ASSOCIATION 400039078', 'Sumitomo Mitsui Trust Bank'), ('RUDOLPH J AND DAPHNE A MUNZER FOUNDATION', 'Russell Investments Implementation Services'), ('SHELL OVERSEAS CONTRIBUTORY PENSION FUND', 'Shell Asset Mgt'), ('C WORLDWIDE AFDELING EMERGING MARKETS KL', 'C WorldWide Asset Mgt'), ('RUSSELL MULTI ASSET FACTOR EXPOSURE FUND', 'Russell Investments'), ('GLOBAL X MSCI EMERGING MARKETS INDEX ETF', 'Global X'), ('REKSA DANA NIKKO INDONESIA BALANCED FUND', 'Ina Sekuritas Indonesia'), ('CELSIUS INTERNATIONAL FUNDS PLC EIS FUND', 'Barclays Wealth'), ('REKSA DANA PRATAMA INVESTA MANDIRI SAHAM', 'Pratama Capital Assets Mgt'), ('IMPAX ENVIRONMENTAL MARKETS IRELAND FUND', 'Christian Riccardo'), ('FRANKLIN BRANDYWINE GLOBAL BALANCED FUND', 'Christian Riccardo'), ('VANGUARD FTSE GLOBAL ALL CAP INDEX FUND', 'Vanguard Group'), ('BIFS EMERGING MARKETS EQUITY INDEX FUND', 'BlackRock Investment Mgt - Index'), ('ISHARES MSCI ACWI LOW CARBON TARGET ETF', 'BlackRock Investment Mgt - Index'), ('REGENTS OF THE UNIVERSITY OF CALIFORNIA', 'State Street Investment Mgt'), ('HSBC GIF GLOBAL EMERGING MARKETS EQUITY', 'HSBC Global Asset Mgt'), ('FIDELITY TOTAL INTERNATIONAL INDEX FUND', 'Geode Capital Mgt'), ('GUIDESTONE EMERGING MARKETS EQUITY FUND', 'AQR Capital Mgt'), ('3M EMPLOYEE RETIREMENT INCOME PLAN ERIP', 'Acadian Asset Mgt'), ('REKSA DANA MANULIFE DANA CAMPURAN UTAMA', 'Manulife Investment Mgt Indonesia'), ('MANULIFE DANA EKUITAS SMALL MID CAPITAL', 'Manulife Investment Mgt Indonesia'), ('REKSA DANA BNP PARIBAS KENCANA BALANCED', 'BNP Paribas Asset Mgt'), ('DIMENSIONAL FUNDS PLC WORLD EQUITY FUND', 'Dimensional Fund Advisors'), ('BLACKROCK EMERGING MARKETS TRACKER FUND', 'BlackRock Investment Mgt'), ('PT ASURANSI JIWA IFG IFG LINK BERIMBANG', 'Bahana TCW Investment Mgt'), ('ARROWSTREET COLLECTIVE INVESTMENT TRUST', 'Arrowstreet Capital'), ('REKSA DANA SAHAM PANIN DANA BERDEDIKASI', 'Panin Asset Mgt'), ('BNY MELLON GLOBAL EMERGING MARKETS FUND', 'Newton Investment Mgt'), ('SMARTWEALTH RUPIAH EQUITY ROTATION FUND', 'Allianz Life Insurance'), ('CF DV EMERGING MARKETS STOCK INDEX FUND', 'Mellon'), ('REKSA DANA SCHRODER 90 PLUS EQUITY FUND', 'Schroder Investment Mgt'), ('BVK PERSONALVORSORGE DES KANTONS ZURICH', 'UBS Global Asset Mgt'), ('ASHMORE DIGITAL EQUITY SUSTAINABLE FUND', 'Ashmore Asset Mgt'), ('REKSA DANA INDEKS SYAILENDRA SRI KEHATI', 'Syailendra Capital'), ('XTRACKERS UCITS COMMON CONTRACTUAL FUND', 'DWS Investments'), ('NUVEEN INTERNATIONAL OPPORTUNITIES FUND', 'TIAA Investment Mgt'), ('CFS WHOLESALE INDEXED GLOBAL SHARE FUND', 'State Street Investment Mgt'), ('CFSIL COMMONWEALTH GLOBAL SHARE FUND 16', 'State Street Investment Mgt'), ('COLONIAL FIRST STATE INVESTMENT FUND 50', 'State Street Investment Mgt'), ('FONDITALIA EQUITY PACIFIC EX JAPAN FUND', 'Fideuram Asset Mgt'), ('GREAT WEST EMERGING MARKETS EQUITY FUND', 'Lazard Asset Mgt'), ('S BANK EMERGING MARKETS ESG EQUITY FUND', 'FIM Asset Mgt'), ('GOLDMAN SACHS EXECUTION AND CLEARING LP', 'Goldman Sachs'), ('ALLIANZ GLOBAL INVESTORS GMBH VKA FONDS', 'Allianz Global Investors'), ('CATERPILLAR INC MASTER RETIREMENT TRUST', 'Caterpillar Master Retirement Trust'), ('REKSA DANA PINNACLE ENHANCED LIQUID ETF', 'Pinnacle Investment'), ('REKSA DANA INSIGHT GROWTH BALANCED FUND', 'Insight Investments Mgt'), ('DEKA INVESTMENT GMBH RE ELB FUNDMASTER1', 'Deka Investment Mgt'), ('MAESTORLINK EQUITY PLUS PRINCIPAL IDX30', 'Principal Asset Mgt'), ('VALIC COMPANY I EMERGING ECONOMIES FUND', 'BlackRock Investment Mgt'), ('SPARINVEST INDEX GLOBALE AKTIER ETIK KL', 'Sparinvest'), ('FRANK RUSSELL INVESTMENT COMPANY II PLC', 'Old Mutual Investment Group'), ('CSOP FTSE ASIA PACIFIC SELECT INDEX ETF', 'CSOP Asset Mgt'), ('METZLER ASIAN BALANCED INVESTMENTS FUND', 'Metzler Asset Mgt'), ('FEDERATED HERMES GLOBAL ALLOCATION FUND', 'Federated Hermes'), ('ARISTEA SICAV NEW FRONTIERS EQUITY FUND', 'Banor Capital'), ('REKSA DANA SETIABUDI DANA OBLIGASI PLUS', 'Setiabudi Investment Mgt'), ('ICAPITAL ASIA PACIFIC EX JAPAN BTB FUND', 'Capital Dynamics'), ('REKSA DANA I AM BUMN BALANCED PLUS FUND', 'Indoasia Asset Mgt'), ('REKSA DANA DANAKITA INVESTASI FLEKSIBEL', 'Danakita Investama'), ('GLOBAL X MSCI ASIA PACIFIC EX JAPAN ETF', 'Mirae Asset Global Investments'), ('IG MACKENZIE PACIFIC INTERNATIONAL FUND', 'Christian Riccardo'), ('CAPE ANN GLOBAL DEVELOPING MARKETS FUND', 'Christian Riccardo'), ('GELA M BHD LION ADVANCED STRATEGIC FUND', 'Christian Riccardo'), ('JPMORGAN CHASE PB WILLIAM VON MUEFFLING', 'Christian Riccardo'), ('CLEARBRIDGE GLOBAL VALUE IMPROVERS FUND', 'Christian Riccardo'), ('ILLINOIS MUNICIPAL RETIREMENT FUND IMRF', 'Illinois Municipal Retirement Fund (IMRF)'), ('VANGUARD EMERGING MARKETS EX CHINA ETF', 'Vanguard Group'), ('ISHARES CORE MSCI EMERGING MARKETS ETF', 'BlackRock Investment Mgt - Index'), ('ISHARES MSCI EMERGING MARKETS ASIA ETF', 'BlackRock Investment Mgt - Index'), ('M AND G CLIMATE AWARE MULTI ASSET FUND', 'M&G Investments'), ('FIDELITY ZERO INTERNATIONAL INDEX FUND', 'Geode Capital Mgt'), ('CT PYRFORD GLOBAL ABSOLUTE RETURN FUND', 'Pyrford International'), ('AQR GLOBAL RISK PREMIUM MASTER ACCOUNT', 'AQR Capital Mgt'), ('PENSION RESERVES INVESTMENT TRUST FUND', 'Acadian Asset Mgt'), ('IMCO EMERGING MARKETS PUBLIC EQUITY LP', 'Northern Trust Global Investments'), ('REKSA DANA MANULIFE DANA EKUITAS UTAMA', 'Manulife Investment Mgt Indonesia'), ('NORGES BANK INVESTMENT MANAGEMENT NBIM', 'Norges Bank Investment Mgt'), ('MANDIRI ATTRACTIVE EQUITY MONEY RUPIAH', 'Mandiri Investasi'), ('PT ASURANSI BRI LIFE DARLINK AGRESIF 1', 'Batavia Prosperindo Asset Mgt'), ('PT ASURANSI BRI LIFE DARLINK AGRESIF 2', 'Batavia Prosperindo Asset Mgt'), ('T ROWE PRICE GLOBAL IMPACT EQUITY FUND', 'T Rowe Price'), ('AZL DFA INTERNATIONAL CORE EQUITY FUND', 'Dimensional Fund Advisors'), ('REKSA DANA DANAREKSA MAWAR KONSUMER 10', 'BRI Investment Mgt'), ('REKSA DANA DANAREKSA ANGGREK FLEKSIBEL', 'BRI Investment Mgt'), ('AIA FINANCIAL IDR ULTIMATE EQUITY FUND', 'AIA Indonesia'), ('AXA CH STRATEGY FUND GLOBAL EQUITY CHF', 'BlackRock Investment Mgt'), ('REKSA DANA INDEKS BAHANA ETF BISNIS 27', 'Bahana TCW Investment Mgt'), ('REKSA DANA BAHANA EXPLORER EQUITY FUND', 'Bahana TCW Investment Mgt'), ('REKSA DANA BAHANA DANA EKUITAS ANDALAN', 'Bahana TCW Investment Mgt'), ('ARROWSTREET EMERGING MARKET TRUST FUND', 'Arrowstreet Capital'), ('ROCHE US RETIREMENT PLANS MASTER TRUST', 'Arrowstreet Capital'), ('REKSA DANA SAHAM PANIN DANA BERKEMBANG', 'Panin Asset Mgt'), ('MANULIFE GLOBAL FUND ASEAN EQUITY FUND', 'Manulife Investment Mgt'), ('SUCORINVEST SUSTAINABILITY EQUITY FUND', 'SucorInvest Asset Mgt'), ('SMARTWEALTH EQUITY INFRASTRUCTURE FUND', 'Allianz Life Insurance'), ('JNL MELLON EMERGING MARKETS INDEX FUND', 'Mellon'), ('REKSA DANA SCHRODER DANA PRESTASI PLUS', 'Schroder Investment Mgt'), ('MACKENZIE FUTUREPATH GLOBAL VALUE FUND', 'Mackenzie Investments'), ('US BANK GUIDEMARK EMERGING MARKET FUND', 'Goldman Sachs Asset Mgt'), ('DEKA INVESTMENT GMBH RE BF MASTERFONDS', 'UBS Global Asset Mgt'), ('NIKKEI ASIA 300 INVESTABLE MOTHER FUND', 'Mitsubishi UFJ Asset Mgt'), ('STATE OF NEW MEXICO INVESTMENT COUNCIL', 'New Mexico State Investment Council'), ('CIBC EMERGING MARKETS EQUITY INDEX ETF', 'CIBC Asset Mgt'), ('PUBLIC SECTOR PENSION INVESTMENT BOARD', 'PSP Investments'), ('L NSF RS KRINGAR TILLV XTMARKNAD INDEX', 'State Street Investment Mgt'), ('NOMURA ASEAN GROWTH EQUITY MOTHER FUND', 'Nomura Asset Mgt'), ('CONNOR CLARK AND LUNN INVESTMENT TRUST', 'Connor Clark & Lunn Investment Mgt'), ('VENERABLE EMERGING MARKETS EQUITY FUND', 'Russell Investments'), ('CARDANO GLOBAL SUSTAINABLE EQUITY FUND', 'Cardano'), ('AB SICAV I ALL MARKET INCOME PORTFOLIO', 'AllianceBernstein - Growth'), ('PENSION DANMARK PERSFORSKRGSAKTSELSKAB', 'PensionDanmark'), ('EMERGING MARKETS EQUITY PLUS PORTFOLIO', 'Earnest Partners'), ('ALLSPRING EMERGING MARKETS EQUITY FUND', 'Allspring Global Investments'), ('NUVEEN TIAA EMERGING MARKET INDEX FUND', 'Nuveen Asset Mgt'), ('MAESTROLINK EQUITY PLUS PRINCIPAL LQ45', 'Principal Asset Mgt'), ('ASI ASIA PACIFIC DYNAMIC DIVIDEND FUND', 'aberdeen'), ('SPARINVEST INDEX B REDYGTIGE GLOBAL KL', 'Sparinvest'), ('TEACHERS RETIREMENT SYSTEM OF OKLAHOMA', 'Teachers Retirement System Of Oklahoma'), ('MAJ INVEST EMERGING MARKETS VALUE FUND', 'Maj Invest'), ('SEI SELECT EMERGING MARKETS EQUITY ETF', 'SEI Investments'), ('ALPS EMERGING SECTOR DIVIDEND DOGS ETF', 'SS&C ALPS Advisors'), ('VERDIPAPIRFONDET HOLBERG VEKSTMARKEDER', 'Holberg Fondene'), ('MIGROS PENSIONSKASSE FONDS AKTIEN WELT', 'Migros Bank'), ('FINREON EMERGING MARKETS EQUITY ISOPRO', 'Finreon'), ('CIMB PRINCIPAL ASSET MANAGEMENT BERHAD', 'Principal Asset Mgt'), ('FRANKLIN LIBERTYQ EMERGING MARKETS ETF', 'Franklin Templeton Investments'), ('CI EMERGING MARKETS DIVIDEND INDEX ETF', 'CI Global Asset Mgt'), ('CHUBB LIFE INSURANCE IND RUPIAH EQUITY', 'Chubb Life Insurance'), ('DANA PENSIUN KARYAWAN STAF KEBON AGUNG', 'Dana Pensiun Karyawan Staf Kebon Agung'), ('REKSA DANA INDEKS CIPTA ETF INDEX LQ45', 'Ciptadana Asset Mgt'), ('REKSA DANA VALBURY INVESTASI BERIMBANG', 'Valbury Capital Mgt'), ('ONEMARKETS AMUNDI FLEXIBLE INCOME FUND', 'Amundi Asset Mgt'), ('REKSA DANA NIKKO INDONESIA EQUITY FUND', 'Ina Sekuritas Indonesia'), ('PHEIM EMERGING COMPANIES BALANCED FUND', 'Pheim Asset Mgt'), ('REKSA DANA PRATAMA DANA ATRAKTIF SAHAM', 'Pratama Capital Assets Mgt'), ('REKSA DANA PRATAMA DANA GEMILANG SAHAM', 'Pratama Capital Assets Mgt'), ('FSSA GLOBAL EMERGING MARKET FOCUS FUND', 'Christian Riccardo'), ('EUROMOBILIARE PICTET GLOBAL TRENDS ESG', 'Christian Riccardo'), ('INVESCO MACRO ALLOCATION STRATEGY FUND', 'Christian Riccardo'), ('STATE STREET GLOBAL MARKETS TRANSITION', 'State Street Global Markets (Transition)'), ('VANGUARD TOTAL WORLD STOCK INDEX FUND', 'Vanguard Group'), ('VANGUARD ESG GLOBAL ALL CAP UCITS ETF', 'Vanguard Group'), ('ISHARES EMERGING MARKETS DIVIDEND ETF', 'BlackRock Investment Mgt - Index'), ('DELA NATURA EN LEVENSVERZEKERINGEN NV', 'State Street Investment Mgt'), ('BAILLIE GIFFORD EMERGING MARKETS FUND', 'Baillie Gifford & Co'), ('BAILLIE GIFFORD CAUTIOUS MANAGED FUND', 'Baillie Gifford & Co'), ('M AND G GLOBAL POSITIVE IMPACT EQUITY', 'M&G Investments'), ('MONDRIAN EMERGING MARKETS EQUITY FUND', 'Mondrian Investment Partners'), ('MANULIFE DANA EKUITAS INDONESIA CHINA', 'Manulife Investment Mgt Indonesia'), ('MANULIFE ASIA BEST SELECT EQUITY FUND', 'Manulife Investment Mgt Indonesia'), ('JPMORGAN EMERGING MARKETS INCOME FUND', 'JPMorgan Asset Mgt'), ('JPMORGAN EMERGING MARKETS EQUITY FUND', 'JPMorgan Asset Mgt'), ('REKSA DANA INDEKS MANDIRI INDEKS LQ45', 'Mandiri Investasi'), ('REKSA DANA BATAVIA CAMPURAN NUSANTARA', 'Batavia Prosperindo Asset Mgt'), ('REKSA DANA BATAVIA CAMPURAN CEMERLANG', 'Batavia Prosperindo Asset Mgt'), ('WHOLESALE INTERNATIONAL CORE EQUITIES', 'Dimensional Fund Advisors'), ('REKSA DANA BAHANA STELLAR EQUITY FUND', 'Bahana TCW Investment Mgt'), ('PT ASURANSI JIWA IFG IFG LINK EKUITAS', 'Bahana TCW Investment Mgt'), ('REKSA DANA INDEKS BAHANA INDEKS IDX30', 'Bahana TCW Investment Mgt'), ('ASURANSI BRI LIFE KPD BTIM EQUITY IBS', 'Bahana TCW Investment Mgt'), ('SUSTAINABLE GLOBAL EQUITY INCOME FUND', 'Newton Investment Mgt'), ('MANULIFE MPF PACIFIC ASIA EQUITY FUND', 'Manulife Investment Mgt'), ('LATTICE EMERGING MARKETS STRATEGY ETF', 'Mellon'), ('PENSIOENFONDS METAAL EN TECHNIEK PFMT', 'UBS Global Asset Mgt'), ('AEIS INSTITUTIONAL FUND AKTIEN GLOBAL', 'UBS Global Asset Mgt'), ('REKSA DANA INDEKS BNI AM INDEKS IDX30', 'BNI Asset Mgt'), ('COLLEGE RETIREMENT EQUITIES FUND CREF', 'TIAA Investment Mgt'), ('NUVEEN LIFE INTERNATIONAL EQUITY FUND', 'TIAA Investment Mgt'), ('NEW YORK STATE COMMON RETIREMENT FUND', 'New York State Common Retirement Fund'), ('COLUMBIA THREADNEEDLE IRELAND III PLC', 'Columbia Threadneedle Investments'), ('MATTHEWS EMERGING MARKETS EQUITY FUND', 'Matthews International Capital Mgt'), ('SCHRODER QEP GLOBAL ACTIVE VALUE FUND', 'Schroder Investment Mgt'), ('SCHRODER EMERGING MARKETS EQUITY FUND', 'Schroder Investment Mgt'), ('SCHRODER GLOBAL EMERGING MARKETS FUND', 'Schroder Investment Mgt'), ('SCHRODER EMERGING MARKETS FUND CANADA', 'Schroder Investment Mgt'), ('SCHRODER GLOBAL EQUITY COMPONENT FUND', 'Schroder Investment Mgt'), ('RED LIFE REINSURANCE SERVICES LIMITED', 'Artisan Partners'), ('AVRIST LINK EQUITY PREMIER 2 IDR FUND', 'Avrist Asset Mgt'), ('AVRIST LINK EQUITY PREMIER 3 IDR FUND', 'Avrist Asset Mgt'), ('SUTTER HEALTH MASTER RETIREMENT TRUST', 'Sutter Health Master Retirement Trust'), ('WINNIPEG CIVIC EMPLOYEES PENSION PLAN', 'Winnipeg Civic Employees Pension Plan'), ('REKSA DANA PINNACLE INDONESIA ESG ETF', 'Pinnacle Investment'), ('CC AND L DIVERSIFIED INCOME PORTFOLIO', 'Connor Clark & Lunn Investment Mgt'), ('EARNEST PARTNERS EMERGING MARKET FUND', 'Earnest Partners'), ('REKSA DANA INDEKS SEQUIS EQUITY IDX30', 'Sequis Asset Mgt'), ('FIDELITY SERIES EMERGING MARKETS FUND', 'Fidelity Investments'), ('REKSA DANA SAM INDONESIAN EQUITY FUND', 'Samuel Asset Mgt'), ('PREMIER MITON DIVERSIFIED INCOME FUND', 'Premier Miton Investors'), ('SELIGSON AND CO EMERGING MARKETS FUND', 'LocalTapiola Asset Mgt'), ('AGF SYSTEMATIC GLOBAL ESG FACTORS ETF', 'AGF International Advisors'), ('GOVERNMENT OF THE PROVINCE OF ALBERTA', 'Christian Riccardo'), ('PACE INTERNATIONAL EQUITY INVESTMENTS', 'Christian Riccardo'), ('ASSET MANAGEMENT EXCHANGE MASTER ICAV', 'Christian Riccardo'), ('TOWERS WATSON COMMON CONTRACTUAL FUND', 'Christian Riccardo'), ('TRINETRA EMERGING MARKETS GROWTH FUND', 'Christian Riccardo'), ('THE CIVIL SERVICE SUPERANNUATION FUND', 'Christian Riccardo'), ('PERSIMMON PLC CM CIC ASSET MANAGEMENT', 'Christian Riccardo'), ('OHIO STATE TEACHERS RETIREMENT SYSTEM', 'Ohio State Teachers Retirement System'), ('VANGUARD ESG INTERNATIONAL STOCK ETF', 'Vanguard Group'), ('PUBLIC EMPLOYEES LONG TERM CARE FUND', 'BlackRock Investment Mgt - Index'), ('INVESCO GLOBAL EMERGING MARKETS FUND', 'Invesco'), ('1832 AM EMERGING MARKETS EQUITY POOL', 'Pzena Investment Mgt'), ('ARGA EMERGING MARKETS SELECT FUND LP', 'ARGA Investment Mgt'), ('PRULINK SINGAPORE ASIAN MANAGED FUND', 'Eastspring Investments'), ('CBSG GEM BETA SMART BETA EQ PFASGBSB', 'Eastspring Investments'), ('BRANDES EMERGING MARKETS EQUITY FUND', 'Brandes Investment Partners'), ('STATE STREET TOTAL RETURN V I S FUND', 'State Street Investment Mgt'), ('PACIFIC HORIZON INVESTMENT TRUST PLC', 'Baillie Gifford & Co'), ('BAILLIE GIFFORD EMERGING MARKETS ETF', 'Baillie Gifford & Co'), ('VANGUARD ACTIVE POSITIVE IMPACT FUND', 'Baillie Gifford & Co'), ('M AND G GLOBAL EMERGING MARKETS FUND', 'M&G Investments'), ('PT PRUDENTIAL LIFE ASSURANCE REP III', 'Eastspring Investments'), ('L AND G ASSURANCE PENSION MANAGEMENT', 'Legal & General Investment Mgt'), ('FIDELITY EMERGING MARKETS INDEX FUND', 'Geode Capital Mgt'), ('BMO PYRFORD INTERNATIONAL STOCK FUND', 'Pyrford International'), ('FIREFIGHTERS PENSION INVESTMENT FUND', 'Northern Trust Global Investments'), ('REKSA DANA MANULIFE DANA CAMPURAN II', 'Manulife Investment Mgt Indonesia'), ('REKSA DANA BATAVIA USA BALANCED ASIA', 'Batavia Prosperindo Asset Mgt'), ('REKSA DANA BATAVIA DISRUPTIVE EQUITY', 'Batavia Prosperindo Asset Mgt'), ('T ROWE PRICE ASIA OPPORTUNITIES FUND', 'T Rowe Price'), ('T ROWE PRICE RETIREMENT HYBRID TRUST', 'T Rowe Price'), ('PT AIA FINANCIAL UL EQUITY PROGRAMME', 'AIA Indonesia'), ('DEUTSCHE BANK TRUST COMPANY AMERICAS', 'American Depositary Receipts'), ('BNY MELLON GLOBAL EQUITY INCOME FUND', 'Newton Investment Mgt'), ('REKSA DANA INDEKS SUCORINVEST IDX 30', 'SucorInvest Asset Mgt'), ('WISDOMTREE GLOBAL HIGH DIVIDEND FUND', 'Mellon'), ('PT ASURANSI BRI LIFE DARLINK DINAMIS', 'Schroder Investment Mgt'), ('GALLUS AKTIEN EMERGING MARKETS AKTIV', 'UBS Global Asset Mgt'), ('SYAILENDRA BALANCED OPPORTUNITY FUND', 'Syailendra Capital'), ('HANDELSBANKEN EMERGING MARKETS INDEX', 'Handelsbanken Fonder - Index'), ('INVESCO EQV ASIA PACIFIC EQUITY FUND', 'Invesco'), ('UBAM POSITIVE IMPACT EMERGING EQUITY', 'UBP Asset Mgt'), ('MATTHEWS ASIA EX JAPAN DIVIDEND FUND', 'Matthews International Capital Mgt'), ('SCHRODER DIVERSIFIED GROWTH PORFOLIO', 'Schroder Investment Mgt'), ('NOMURA TB NAMCO GLOBAL EQUITY MOTHER', 'Nomura Asset Mgt'), ('SEI GMF EMERGING MARKETS EQUITY FUND', 'Lazard Asset Mgt'), ('THE GREAT EASTERN LIFE SLIF FUND PAR', 'Lion Global Investors'), ('SBI PICTET ASIA HI TECH VENTURE FUND', 'Pictet Asset Mgt'), ('S BANK GROWING ECONOMIES EQUITY FUND', 'FIM Asset Mgt'), ('LOMBARD ODIER DARIER HENTSCH AND CIE', 'Lombard Odier Darier Hentsch & Cie'), ('REKSA DANA AVRIST EQUITY GROWTH FUND', 'Avrist Asset Mgt'), ('COMMONWEALTH EMERGING MARKETS FUND 9', 'JO Hambro Capital Mgt'), ('NOMURA GLOBAL HIGH INCOME STOCK FUND', 'Nomura Asset Mgt'), ('GREATLINK SUSTAINABILITY EQUITY FUND', 'Great Eastern Life Assurance'), ('REKSA DANA INDEKS INSIGHT SRI KEHATI', 'Insight Investments Mgt'), ('REKSA DANA RHB INDO ASIA EQUITY FUND', 'Allianz Global Investors Asset Mgt Indonesia'), ('HOSKING PARTNERS GLOBAL EQUITY TRUST', 'Hosking Partners'), ('REKSA DANA INDEKS SAM ETF SRI KEHATI', 'Samuel Asset Mgt'), ('SUN LIFE ASSURANCE COMPANY OF CANADA', 'Schroder Investment Mgt'), ('BNPPF S FUND EQUITY EMERGING MARKETS', 'BNP Paribas Asset Mgt'), ('PRINCIPAL TRUST COMPANY ASIA LIMITED', 'Principal Asset Mgt'), ('HOSTPLUS POOLED SUPERANNUATION TRUST', 'IFM Investors'), ('MERLIN FIDELIS EMERGING MARKETS FUND', 'Merlin Fidelis Asset Mgt'), ('STRIVE EMERGING MARKETS EX CHINA ETF', 'Strive Asset Mgt'), ('UPAMC GRAND ASEAN HIGH DIVIDEND FUND', 'Christian Riccardo'), ('CHAUTAUQUA GLOBAL GROWTH EQUITY FUND', 'Christian Riccardo'), ('REB EMERGING MARKETS FINANCIALS FUND', 'Christian Riccardo'), ('DYNAMIC EMERGING MARKETS EQUITY FUND', 'Christian Riccardo'), ('REKSA DANA PURWANTO CAMPURAN DINAMIS', 'Christian Riccardo'), ('REKSA DANA PURWANTO SAHAM BERKEMBANG', 'Christian Riccardo'), ('MACQUARIE FINANCIAL HOLDINGS LIMITED', 'Christian Riccardo'), ('INVESCO ADVANTAGE INTERNATIONAL FUND', 'Christian Riccardo'), ('PT BANK RAKYAT INDONESIA PERSERO TBK', 'PT Bank Rakyat Indonesia (Persero) Tbk.'), ('CONNOR CLARK AND LUNN INVESTMENT MGT', 'Connor Clark & Lunn Investment Mgt'), ('CAPITAL INVESTMENT TRUST CORPORATION', 'Capital Investment Trust Corporation'), ('MERCER EMERGING MARKETS SHARES FUND', 'BlackRock Investment Mgt - Index'), ('BLACKROCK CDN ACWI ALPHA TILTS FUND', 'BlackRock Investment Mgt - Index'), ('ISHARES BROAD EMERGING MARKETS FUND', 'BlackRock Investment Mgt - Index'), ('CONSOLIDATED EDISON RETIREMENT PLAN', 'BlackRock Investment Mgt - Index'), ('INVESCO ASIAN EQUITY INCOME FUND UK', 'Invesco'), ('BRANDES EMERGING MARKETS VALUE FUND', 'Brandes Investment Partners'), ('BAILLIE GIFFORD WORLDWIDE FUNDS PLC', 'Baillie Gifford & Co'), ('L AND G COLLECTIVE INVESTMENT TRUST', 'Legal & General Investment Mgt'), ('LEGAL AND GENERAL ASSURANCE SOCIETY', 'Legal & General Investment Mgt'), ('SPARTAN EMERGING MARKETS INDEX POOL', 'Geode Capital Mgt'), ('PT ASURANSI JIWA MANULIFE INDONESIA', 'Manulife Investment Mgt Indonesia'), ('MANULIFE DANA PRIMA DINAMIS MODERAT', 'Manulife Investment Mgt Indonesia'), ('DPLK MANULIFE CITIBANK BALANCE DKPK', 'Manulife Investment Mgt Indonesia'), ('MANULIFE DANA SMALL MID CAPITAL USD', 'Manulife Investment Mgt Indonesia'), ('EMERGING MARKETS EQUITY CANADA FUND', 'JPMorgan Asset Mgt'), ('REKSA DANA MANDIRI INVESTA ATRAKTIF', 'Mandiri Investasi'), ('REKSA DANA INDEKS BATAVIA IDX30 ETF', 'Batavia Prosperindo Asset Mgt'), ('REKSA DANA BATAVIA SAHAM ESG IMPACT', 'Batavia Prosperindo Asset Mgt'), ('FIDELITY ADVISOR EMERGING ASIA FUND', 'Fidelity Investments'), ('T ROWE PRICE GLOBAL ALLOCATION FUND', 'T Rowe Price'), ('REKSA DANA PREMIER CAMPURAN DINAMIS', 'Indo Premier Investment Mgt'), ('REKSA DANA BRI ETF INDONESIA TOP 40', 'BRI Investment Mgt'), ('BNY MELLON ASIAN OPPORTUNITIES FUND', 'Newton Investment Mgt'), ('REKSA DANA SCHRODER DANA TERPADU II', 'Schroder Investment Mgt'), ('MACKENZIE GQE EMERGING MARKETS FUND', 'Mackenzie Investments'), ('REKSA DANA INDEKS BNI AM SRI KEHATI', 'BNI Asset Mgt'), ('UNIVERSAL INVESTMENT BAYVK A2 FONDS', 'Universal Investment'), ('REKSA DANA TRIMEGAH SAHAM NUSANTARA', 'Trimegah Asset Mgt'), ('EDGBASTON ASIAN EQUITY JERSEY TRUST', 'Edgbaston Investment Partners'), ('GUINNESS ASSET MANAGEMENT FUNDS PLC', 'Guinness Asset Mgt'), ('USAA GLOBAL MANAGED VOLATILITY FUND', 'Victory Capital Mgt'), ('GREAT EASTERN TRUST PRIVATE LIMITED', 'Lion Global Investors'), ('PICTET EMERGING MARKETS MULTI ASSET', 'Pictet Asset Mgt'), ('CC AND L Q GROUP GLOBAL EQUITY FUND', 'Connor Clark & Lunn Investment Mgt'), ('FIDELITY COMMON CONTRACTUAL FUND II', 'FIL Investment International'), ('MULTI STYLE MULTI MANAGER FUNDS PLC', 'Russell Investments'), ('STICHTING JURIDISCH EIGENAAR ACTIAM', 'Cardano'), ('PT AJ SEQUIS LIFE S A KPD BPAM SQEF', 'Sequislife'), ('PT AJ SEQUIS LIFE S A KPD SQAM SQEF', 'Sequislife'), ('PT AJ SEQUIS LIFE S A KPD BPAM SQMF', 'Sequislife'), ('PT AJ SEQUIS LIFE S A KPD SQAM SQMF', 'Sequislife'), ('EMERGING MARKETS COMPLETION FUND LP', 'Parametric Portfolio Associates'), ('BMO MSCI EMERGING MARKETS INDEX ETF', 'BMO Asset Mgt'), ('AXA WF FRAMLINGTON EMERGING MARKETS', 'AXA Investment Mgrs Asia'), ('HARBOR EMERGING MARKETS EQUITY FUND', 'Earnest Partners'), ('ASI ASIA PACIFIC INCOME EQUITY FUND', 'aberdeen'), ('REKSA DANA SAM DANA SAHAM NUSANTARA', 'Samuel Asset Mgt'), ('DANA PENSIUN LEMBAGA KEUANGAN SAHAM', 'Dana Pensiun BRI'), ('PT ASURANSI MITRA PELINDUNG MUSTIKA', 'PT Asuransi Mitra Pelindung Mustika'), ('PT VICTORINDO INVESTAMA SURYA ARTHA', 'Pt Victorindo Investama Surya Artha'), ('AMP WHOLESALE EMERGING MARKETS FUND', 'Guardian Trust'), ('AZ FUND 1 AZ EQUITY ASEAN COUNTRIES', 'Azimut Investments'), ('REKSA DANA DANAKITA SAHAM PRIORITAS', 'Danakita Investama'), ('REKSA DANA PRATAMA DANA ALPHA SAHAM', 'Pratama Capital Assets Mgt'), ('DYNAMIC ACTIVE EMERGING MARKETS ETF', 'Christian Riccardo'), ('MARTIN CURRIE EMERGING MARKETS FUND', 'Christian Riccardo'), ('FIAM SELECT EMERGING MARKETS EQUITY', 'Christian Riccardo'), ('SCOTIA EMERGING MARKETS EQUITY FUND', 'Christian Riccardo'), ('ABU DHABI INVESTMENT AUTHORITY ADIA', 'Abu Dhabi Investment Authority (ADIA)'), ('NEW MEXICO STATE INVESTMENT COUNCIL', 'New Mexico State Investment Council'), ('CATERPILLAR MASTER RETIREMENT TRUST', 'Caterpillar Master Retirement Trust'), ('KENTUCKY TEACHERS RETIREMENT SYSTEM', "Kentucky Teachers' Retirement System"), ('MSCI EQUITY INDEX FUND B INDONESIA', 'BlackRock Investment Mgt - Index'), ('MSCI ACWI EX U S IMI INDEX FUND B2', 'BlackRock Investment Mgt - Index'), ('BRANDES INSTITUTIONAL EQUITY TRUST', 'Brandes Investment Partners'), ('ISHARES III PUBLIC LIMITED COMPANY', 'BlackRock Investment Mgt - Index'), ('SPDR S AND P EMERGING ASIA PACIFIC', 'State Street Investment Mgt'), ('SPDR S AND P EMERGING MARKETS FUND', 'State Street Investment Mgt'), ('M AND G FUNDS 1 M AND G ASIAN FUND', 'M&G Investments'), ('T ROWE PRICE ASIAN EX JAPAN EQUITY', 'T Rowe Price International'), ('WOODBRIDGE INVESTMENTS CORPORATION', 'Pyrford International'), ('COMMONWEALTH GLOBAL SHARES FUND 30', 'Acadian Asset Mgt'), ('REKSA DANA MANULIFE SAHAM SMC PLUS', 'Manulife Investment Mgt Indonesia'), ('RD MANDIRI INVESTA EQUITY MOVEMENT', 'Mandiri Investasi'), ('T ROWE PRICE IG GLOBAL IMPACT POOL', 'T Rowe Price'), ('SCHWAB EMERGING MARKETS EQUITY ETF', 'Charles Schwab Investment Mgt'), ('RD PREMIER ETF INDONESIA FINANCIAL', 'Indo Premier Investment Mgt'), ('REKSA DANA INDEKS PANIN SRI KEHATI', 'Panin Asset Mgt'), ('REKSA DANA SUCORINVEST EQUITY FUND', 'SucorInvest Asset Mgt'), ('DPLK ALLIANZ IND PPUKP EQUITY FUND', 'Allianz Life Insurance'), ('DPLK ALLIANZ IND PPUKP NESTLE FUND', 'Allianz Life Insurance'), ('MIGROS AST FONDS II AKTIEN AUSLAND', 'UBS Global Asset Mgt'), ('REKSA DANA PENDAPATAN TETAP BNI AM', 'BNI Asset Mgt'), ('SYAILENDRA EQUITY OPPORTUNITY FUND', 'Syailendra Capital'), ('MERRILL LYNCH INTERNATIONAL LONDON', 'BofA Securities'), ('HANDELSBANKEN TILLV XTMARKNADSFOND', 'Handelsbanken Fonder - Index'), ('ILLINOIS MUNICIPAL RETIREMENT FUND', 'Illinois Municipal Retirement Fund (IMRF)'), ('REKSA DANA TRIMEGAH EQUITY FOCUS 2', 'Trimegah Asset Mgt'), ('PUBLIC SOUTH EAST ASIA SELECT FUND', 'Public Mutual'), ('VICTORY MARKET NEUTRAL INCOME FUND', 'Victory Capital Mgt'), ('GMO EMERGING MARKETS EX CHINA FUND', 'GMO'), ('NYKREDIT INVEST ENGROS VAEKSTLANDE', 'Lazard Asset Mgt'), ('NEW SHIPS INC PENSION MASTER TRUST', 'Artisan Partners'), ('WS ACCESS GLOBAL ACTIVE VALUE FUND', 'Waystone Mgt'), ('AVRIST LINK EQUITY GROWTH IDR FUND', 'Avrist Asset Mgt'), ('YAYASAN KESEHATAN PEGAWAI TELKOM 3', 'Yayasan Kesehatan Pegawai Telkom'), ('SEB EMERGING MARKETS EXPOSURE FUND', 'SEB Asset Mgt'), ('NYKREDIT INVEST TAKTISK ALLOKERING', 'Nykredit Asset Mgt'), ('VANECK VECTORS INDONESIA INDEX ETF', 'VanEck Global'), ('GREATLINK INTELLIGENCE EQUITY FUND', 'Great Eastern Life Assurance'), ('REKSA DANA SEQUIS EQUITY INDONESIA', 'Sequis Asset Mgt'), ('NOMURA INDONESIA STOCK MOTHER FUND', 'Nomura Asset Mgt'), ('DANA PENSIUN SEMEN GRESIK KPAI SAM', 'Samuel Asset Mgt'), ('ZIF AKTIEN EMERGING MARKETS PASSIV', 'DWS Investments'), ('VOYA VACS INDEX SERIES I PORTFOLIO', 'Voya Investment Mgt'), ('COOPER INVESTORS ASIAN EQUITY FUND', 'Cooper Investors'), ('EMERGING MARKETS EQUITY ACTIVE ETF', 'Russell Investments Implementation Services'), ('IDA AYU KETUT SURYA ADNYANI SE MAP', 'Ida Ayu Ketut Surya Adnyani Se Map'), ('REKSA DANA SETIABUDI DANA CAMPURAN', 'Setiabudi Investment Mgt'), ('EMERGING MARKETS EQUITY SELECT ETF', 'First Trust Advisors'), ('REKSA DANA DANAPATHI EQUITY GROWTH', 'Danapathi Asset Management'), ('HEREFORD FUNDS VALVERDE ASEAN FUND', 'Valverde Investment Partners'), ('REKSA DANA PRATAMA CEMERLANG SAHAM', 'Pratama Capital Assets Mgt'), ('BROWN ADVISORY GLOBAL LEADERS FUND', 'Christian Riccardo'), ('IG MACKENZIE PAN ASIAN EQUITY FUND', 'Christian Riccardo'), ('MATTHEWS INTERNATIONAL CAPITAL MGT', 'Matthews International Capital Mgt'), ('PUBLIC EMPLOYEES PENSION PLAN PEPP', 'Public Employees Pension Plan (PEPP)'), ('BLACKROCK INVESTMENT MGT AUSTRALIA', 'BlackRock Investment Mgt (Australia)'), ('YUANTA SECURITIES INVESTMENT TRUST', 'Yuanta Securities Investment Trust'), ('CATHAY SECURITIES INVESTMENT TRUST', 'Cathay Securities Investment Trust'), ('VANGUARD INTERNATIONAL VALUE FUND', 'Vanguard Group'), ('VANGUARD FTSE ALL WORLD UCITS ETF', 'Vanguard Group'), ('ISHARES MSCI EMERGING MARKETS ETF', 'BlackRock Investment Mgt - Index'), ('PZENA EMERGING MARKETS VALUE FUND', 'Pzena Investment Mgt'), ('ARGA EMERGING MARKETS EQUITY FUND', 'ARGA Investment Mgt'), ('BRANDES GLOBAL OPPORTUNITIES FUND', 'Brandes Investment Partners'), ('ISHARES II PUBLIC LIMITED COMPANY', 'BlackRock Investment Mgt - Index'), ('ISHARES III PLC MSCI EM UCITS ETF', 'BlackRock Investment Mgt - Index'), ('ISHARES V PLC MSCI ACWI UCITS ETF', 'BlackRock Investment Mgt - Index'), ('ISHARES IV PUBLIC LIMITED COMPANY', 'BlackRock Investment Mgt - Index'), ('BRITISH AIRWAYS PENSIONS TRUSTEES', 'BlackRock Investment Mgt - Index'), ('SPDR S AND P EMERGING MARKETS ETF', 'State Street Investment Mgt'), ('LF ACCESS GLOBAL EQUITY CORE FUND', 'Baillie Gifford & Co'), ('BAILLIE GIFFORD MANAGED FUND OEIC', 'Baillie Gifford & Co'), ('MONDRIAN GLOBAL EQUITY VALUE FUND', 'Mondrian Investment Partners'), ('COLUMBIA THREADNEEDLE IRELAND PLC', 'Pyrford International'), ('NOVA SCOTIA TEACHERS PENSION FUND', 'Pyrford International'), ('AQR UCITS FUNDS EMERGING EQUITIES', 'AQR Capital Mgt'), ('NATIONAL EMPLOYMENT SAVINGS TRUST', 'Northern Trust Global Investments'), ('TEXAS MUNICIPAL RETIREMENT SYSTEM', 'Northern Trust Global Investments'), ('IG JPMORGAN EMERGING MARKETS FUND', 'JPMorgan Asset Mgt'), ('REKSA DANA INDEKS PREMIER ETF IDX', 'Indo Premier Investment Mgt'), ('RD PREMIER ETF INDONESIA CONSUMER', 'Indo Premier Investment Mgt'), ('YAYASAN KESEJAHTERAAN PEKERJA BRI', 'BRI Investment Mgt'), ('REKSA DANA DANAREKSA KOMODITAS 10', 'BRI Investment Mgt'), ('DPLK AIA DANAKARYA SEHATSEJAHTERA', 'AIA Indonesia'), ('ASURANSI BRI LIFE KPD BTIM EQUITY', 'Bahana TCW Investment Mgt'), ('REKSADANA BAHANA TRAILBLAZER FUND', 'Bahana TCW Investment Mgt'), ('REKSA DANA SCHRODER DANA PRESTASI', 'Schroder Investment Mgt'), ('GENUS EMERGING MARKET EQUITY FUND', 'Allspring Global Investments'), ('ALASKA PERMANENT FUND CORPORATION', 'Alaska Permanent Fund Corp.'), ('STICHTING AHOLD DELHAIZE PENSIOEN', 'Northern Trust Global Investments'), ('CIBC EMERGING MARKETS EQUITY IETF', 'CIBC Asset Mgt'), ('RENAISSANCE EMERGING MARKETS FUND', 'CIBC Asset Mgt'), ('INVESCO RAFI EMERGING MARKETS ETF', 'Invesco Capital Mgt'), ('MDPIM EMERGING MARKET EQUITY POOL', 'GMO'), ('PT AXA MANDIRI FINANCIAL SERVICES', 'AXA Asset Mgt Indonesia'), ('MAESTROLINK EQUITY PLUS SCHRODERS', 'AXA Asset Mgt Indonesia'), ('ALLIANZ GLOBAL INVEST EUROPE GMBH', 'Allianz Global Investors'), ('WAYSTONE UCITS PLATFORM LUX SICAV', 'Waystone Mgt'), ('PING AN FTSE ASEAN DIVIDEND INDEX', 'UOB Asset Mgt'), ('SEB SICAV 1 EMERGING MARKETS FUND', 'SEB Asset Mgt'), ('DODGE AND COX WORLDWIDE FUNDS PLC', 'Dodge & Cox'), ('INVESTERINGSFORENINGEN BANKINVEST', 'BI Asset Mgt'), ('DEKA NACHHALTIGKEIT IMPACT AKTIEN', 'Deka Investment Mgt'), ('REKSA DANA INDEKS GROW SRI KEHATI', 'Grow Investments'), ('REKSA DANA INDEKS STAR INFOBANK15', 'Star Asset Mgt'), ('DAI ICHI MUTUAL LIFE INSURANCE CO', 'Dai-Ichi Life Insurance Co.'), ('GUINNESS ASIAN EQUITY INCOME FUND', 'Guinness Global Investors'), ('REKSA DANA DANAPATHI BALANCE FUND', 'Danapathi Asset Management'), ('REKSA DANA OSO FLORES EQUITY FUND', 'Oso Investment Mgt'), ('AMUNDI ESG GLOBAL LOW CARBON FUND', 'Amundi Asset Mgt'), ('REKSA DANA ARCHITAS SAHAM DINAMIS', 'AXA IM Select'), ('REKSA DANA CAPITAL OPTIMAL EQUITY', 'Capital Asset Mgt'), ('CATHAY LIFE INSURANCE COMPANY LTD', 'Christian Riccardo'), ('JANA EMERGING MARKETS SHARE TRUST', 'Christian Riccardo'), ('HSBC BANK PLC A C IB MAIN ACCOUNT', 'Christian Riccardo'), ('NORTHERN TRUST GLOBAL INVESTMENTS', 'Northern Trust Global Investments'), ('MANULIFE INVESTMENT MGT INDONESIA', 'Manulife Investment Mgt Indonesia'), ('COLUMBIA THREADNEEDLE INVESTMENTS', 'Columbia Threadneedle Investments'), ('STICHTING PENSIOENFONDS HOOGOVENS', 'Stichting Pensioenfonds Hoogovens'), ('NATIONAL PENSION SERVICE FUND MGT', 'National Pension Service Fund Mgt'), ('FIRST SECURITIES INVESTMENT TRUST', 'First Securities Investment Trust'), ('CENTRAL PROVIDENT FUND BOARD CPF', 'BlackRock Investment Mgt - Index'), ('ARGA EMERGING MARKETS VALUE FUND', 'ARGA Investment Mgt'), ('PENSIONSKASSE DES BUNDES PUBLICA', 'BlackRock Investment Mgt - Index'), ('ISHARES ESG ADVANCED MSCI EM ETF', 'BlackRock Investment Mgt - Index'), ('KNIGHTS OF COLUMBUS MASTER TRUST', 'State Street Investment Mgt'), ('POOL REINSURANCE COMPANY LIMITED', 'Legal & General Investment Mgt'), ('NORTHERN TRUST QUANTITATIVE FUND', 'Northern Trust Global Investments'), ('REKSADANA MANULIFE SAHAM ANDALAN', 'Manulife Investment Mgt Indonesia'), ('JPMORGAN MULTI ASSET INCOME FUND', 'JPMorgan Asset Mgt'), ('ESKOM PENSION AND PROVIDENT FUND', 'Robeco Asset Mgt'), ('PT AIA FINANCIAL UL EQUITY SOLID', 'AIA Indonesia'), ('REKSA DANA BAHANA PRIMAVERA FUND', 'Bahana TCW Investment Mgt'), ('REKSA DANA BAHANA PRIMAVERA PLUS', 'Bahana TCW Investment Mgt'), ('REKSA DANA SUCORINVEST MAXI FUND', 'SucorInvest Asset Mgt'), ('ASURANSI CIPUTRA INDONESIA SUCOR', 'SucorInvest Asset Mgt'), ('SMARTWEALTH EQUITY INDOASIA FUND', 'Allianz Life Insurance'), ('GLOBAL X FTSE SOUTHEAST ASIA ETF', 'Global X Mgt'), ('AVA EQUITY DOLLAR NUSANTARA FUND', 'Ashmore Asset Mgt'), ('INVESCO ASIA PACIFIC GROWTH FUND', 'Invesco'), ('IMPERIAL EMERGING ECONOMIES POOL', 'CIBC Asset Mgt'), ('CIBC EMERGING MARKETS INDEX FUND', 'CIBC Asset Mgt'), ('SCHRODER DIVERSIFIED GROWTH FUND', 'Schroder Investment Mgt'), ('AQUILA LIFE UK EQUITY INDEX FUND', 'Aviva Investors'), ('LION GLOBAL SOUTH EAST ASIA FUND', 'Lion Global Investors'), ('SCOTIA DIVERSIFIED BALANCED FUND', 'Connor Clark & Lunn Investment Mgt'), ('CC AND L Q GLOBAL EQUITY FUND LP', 'Connor Clark & Lunn Investment Mgt'), ('CGMI PRIME FINANCE CLEARANCE ACC', 'Citigroup Global Markets'), ('AVRIST ADA SAHAM BLUE SAFIR FUND', 'Avrist Asset Mgt'), ('RUSSELL INVESTMENT COMPANY V PLC', 'Russell Investments'), ('STICHTING PENSIONFONDS HOOGOVENS', 'Stichting Pensioenfonds Hoogovens'), ('REKSA DANA SEQUIS BALANCE ULTIMA', 'Sequis Asset Mgt'), ('AEGON WORLD EQUITY INDEX FUND II', 'AEGON Asset Mgt'), ('DNB GLOBAL EMERGING MARKET INDEX', 'DNB Asset Mgt'), ('HOSKING PARTNERS EQUITY FUND LLC', 'Hosking Partners'), ('VALIDA AKTIEN EMERGING MARKETS 1', 'Raiffeisen Capital Mgt'), ('AFDELING DANICA PENSION AKTIER 2', 'Danske Capital Mgt'), ('SCHRODER ASIAN ASSET INCOME FUND', 'Schroder Investment Mgt'), ('TAN SIAUW DJIANG ALS ANDI TJIPTO', 'Tan Siauw Djiang Als Andi Tjipto'), ('RD MAJORIS USD BALANCE INDONESIA', 'Majoris Asset Mgt'), ('YUWONO SUMODIJOYO AND BENG YOE T', 'Yuwono Sumodijoyo & Beng Yoe T'), ('ARROW DOW JONES GLOBAL YIELD ETF', 'Arrow Investment Advisors'), ('DFC EMERGING MARKETS EQUITY FUND', 'LMCG Investments'), ('FSITC ASIAN EMERGING MARKET FUND', 'First Securities Investment Trust'), ('BNY MELLON EMERGING MARKETS FUND', 'Mellon'), ('SEI EMERGING MARKETS EQUITY FUND', 'Mellon'), ('REKSA DANA MNC SMART EQUITY FUND', 'MNC Asset Mgt'), ('CATHAY FTSE EMERGING MARKETS ETF', 'Cathay Securities Investment Trust'), ('REKSA DANA PRATAMA DINAMIS SAHAM', 'Pratama Capital Assets Mgt'), ('FAIRTREE GLOBAL EMERGING MARKETS', 'Christian Riccardo'), ('OMERS ADMINISTRATION CORPORATION', 'Christian Riccardo'), ('VIDENT INTERNATIONAL EQUITY FUND', 'Christian Riccardo'), ('CAPE ANN PERIVALLON IRELAND FUND', 'Christian Riccardo'), ('LCIV EMERGING MARKET EQUITY FUND', 'Christian Riccardo'), ('EWING MARION KAUFFMAN FOUNDATION', 'Christian Riccardo'), ('AFFIN HWANG SELECT DIVIDEND FUND', 'Christian Riccardo'), ('LEGAL AND GENERAL INVESTMENT MGT', 'Legal & General Investment Mgt'), ('YAYASAN KESEHATAN PEGAWAI TELKOM', 'Yayasan Kesehatan Pegawai Telkom'), ('GLOBAL EX US ALPHA TILTS FUND B', 'BlackRock Investment Mgt - Index'), ('INVESCO DEVELOPING MARKETS FUND', 'Invesco'), ('TA ASIA HIGH CONVICTION MANDATE', 'Invesco'), ('MDPIM INTERNATIONAL EQUITY POOL', 'Pzena Investment Mgt'), ('PZENA GLOBAL FOCUSED VALUE FUND', 'Pzena Investment Mgt'), ('PZENA INVESTMENT MANAGEMENT LLC', 'Pzena Investment Mgt'), ('AMERICAN HEART ASSOCIATION AHAI', 'State Street Investment Mgt'), ('EASTSPRING IDX ESG LEADERS PLUS', 'Eastspring Investments'), ('STAGECOACH GROUP PENSION SCHEME', 'Pyrford International'), ('AQR COLLECTIVE INVESTMENT TRUST', 'AQR Capital Mgt'), ('REKSA DANA MANULIFE SAHAM UTAMA', 'Manulife Investment Mgt Indonesia'), ('MANULIFE DANA BERIMBANG SYARIAH', 'Manulife Investment Mgt Indonesia'), ('PT HEKSA SOLUTION INSURANCE MMI', 'Mandiri Investasi'), ('MANDIRI EXCELLENT EQUITY RUPIAH', 'Mandiri Investasi'), ('STICHTING PHILIPS PENSIOENFONDS', 'BlackRock Investment Mgt'), ('DUNHAM INTERNATIONAL STOCK FUND', 'Arrowstreet Capital'), ('REKSA DANA SAHAM PANIN BETA ONE', 'Panin Asset Mgt'), ('REKSA DANA PANIN DANA PRIORITAS', 'Panin Asset Mgt'), ('JPM GEM DIVERSIFIED MOTHER FUND', 'JPMorgan Asset Mgt (JF Asset Mgt)'), ('PIMCO RAE EMERGING MARKETS FUND', 'Research Affiliates'), ('DANA PENSIUN FREEPORT INDONESIA', 'Ashmore Asset Mgt'), ('MAESTROLINK EQUITY PLUS ASHMORE', 'Ashmore Asset Mgt'), ('HANDELSBANKEN GLOBAL INDEX CORE', 'Handelsbanken Fonder - Index'), ('PRUSIK ASIAN EQUITY INCOME FUND', 'Prusik Investment Mgt'), ('CT LUX ASIAN EQUITY INCOME FUND', 'Columbia Threadneedle Investments'), ('REKSA DANA HENAN ULTIMA EKUITAS', 'Henan Putihrai Asset Mgt'), ('REKSA DANA HPAM TACTICAL EQUITY', 'Henan Putihrai Asset Mgt'), ('BARCLAYS CAPITAL SECURITIES LTD', 'Barclays'), ('BCT PIF SMART ASIAN EQUITY FUND', 'Schroder Investment Mgt'), ('ABERDEEN EM SMA COMPLETION FUND', 'aberdeen'), ('BAYERNINVEST FONDS AKTIEN ASIEN', 'FIL Investment Mgt'), ('AI STRATEGIC GLOBAL EQUITY FUND', 'Aviva Investors'), ('SEB FUND 3 GLOBAL EXPOSURE FUND', 'SEB Asset Mgt'), ('PARAMETRIC SEM CORE EQUITY FUND', 'Parametric Portfolio Associates'), ('QIC INTERNATIONAL EQUITIES FUND', 'QIC Asset Mgt'), ('REKSA DANA SEQUIS EQUITY MAXIMA', 'Sequis Asset Mgt'), ('REKSA DANA MAYBANK DANA EKUITAS', 'Maybank Asset Mgt'), ('REKSA DANA GROW SAHAM INDONESIA', 'Grow Investments'), ('SAMUEL TIRTA AGUNG BUDIHARDJO H', 'Samuel Tirta Agung Budihardjo H'), ('PT DWIMURIA INVESTAMA ANDALAN 2', 'PT Dwimuria Investama Andalan 2'), ('PELAYARAN NASIONAL SINAR PAGODA', 'Pelayaran Nasional Sinar Pagoda'), ('AMF AKTIEFOND TILLV XTMARKNADER', 'AMF Pensions'), ('INVESCO DEVELOPING MARKET CLASS', 'Invesco (OppenheimerFunds)'), ('PINEBRIDGE SOUTH EAST ASIA FUND', 'PineBridge Investments'), ('FRANKLIN FTSE ASIA EX JAPAN ETF', 'Franklin Templeton Investments'), ('AUGUSTINUS BEDJO TJOKRODIHARDJO', 'Augustinus Bedjo Tjokrodihardjo'), ('DPLK TUGU MANDIRI DANA SITAMPAN', 'Tugu Mandiri'), ('PHEIM ASEAN ALL CAP EQUITY FUND', 'Pheim Asset Mgt'), ('MORGAN STANLEY INVESTMENT FUNDS', 'Morgan Stanley Investment Mgt'), ('PT ASURANSI JASA RAHARJA PUTERA', 'Christian Riccardo'), ('CLEARBRIDGE CDIP FEEDER FUND LP', 'Christian Riccardo'), ('NINETY ONE GLOBAL STRATEGY FUND', 'Christian Riccardo'), ('JPMORGAN ASSET MGT JF ASSET MGT', 'JPMorgan Asset Mgt (JF Asset Mgt)'), ('AMERICAN CENTURY INVESTMENT MGT', 'American Century Investment Mgt'), ('SUMITOMO MITSUI TRUST ASSET MGT', 'Sumitomo Mitsui Trust Asset Mgt'), ('PARAMETRIC PORTFOLIO ASSOCIATES', 'Parametric Portfolio Associates'), ('INDONESIA INVESTMENT AUTHORITY', 'Indonesia Investment Authority'), ('PZENA INTERNATIONAL VALUE FUND', 'Pzena Investment Mgt'), ('ALASKA ELECTRICAL PENSION FUND', 'Brandes Investment Partners'), ('AEGON WORLD EQUITY BETA POOL P', 'BlackRock Investment Mgt - Index'), ('M AND G LUX INVESTMENT FUNDS 1', 'M&G Investments'), ('REKSA DANA MANULIFE DANA SAHAM', 'Manulife Investment Mgt Indonesia'), ('MANULIFE DANA EKUITAS UNGGULAN', 'Manulife Investment Mgt Indonesia'), ('REKSA DANA BNP PARIBAS EKUITAS', 'BNP Paribas Asset Mgt'), ('KPD SAHAM DANA MAKSIMA BATAVIA', 'Batavia Prosperindo Asset Mgt'), ('STATE OF CONNECTICUT TREASURER', 'T Rowe Price'), ('PT ASURANSI BRI LIFE KPD BRIMI', 'BRI Investment Mgt'), ('DPLK AIA FINANCIAL EQUITY PPUK', 'AIA Indonesia'), ('BLACKROCK SOLUTIONS FUNDS ICAV', 'BlackRock Investment Mgt'), ('DIVERSIFIED GLOBAL SHARE TRUST', 'Arrowstreet Capital'), ('ARROWSTREET GLOBAL EQUITY FUND', 'Arrowstreet Capital'), ('REKSA DANA INDEKS PANIN IDX 30', 'Panin Asset Mgt'), ('REKSA DANA PANIN DANA UNGGULAN', 'Panin Asset Mgt'), ('ABU DHABI INVESTMENT AUTHORITY', 'Abu Dhabi Investment Authority (ADIA)'), ('MANULIFE INTERNATIONAL LIMITED', 'Manulife Investment Mgt'), ('SMARTLINK RUPIAH BALANCED FUND', 'Allianz Life Insurance'), ('DANA PENSIUN CENTRAL BANK ASIA', 'Schroder Investment Mgt'), ('JPMORGAN ASEAN FUND A C 604124', 'JPMorgan Asset Mgt'), ('INSTITUTIONAL RETIREMENT TRUST', 'Invesco'), ('THREADNEEDLE ASIA PACIFIC FUND', 'Columbia Threadneedle Investments'), ('PLATO INSTITUTIONAL INDEX FUND', 'KBC Asset Mgt'), ('CIBC INTERNATIONAL EQUITY FUND', 'CIBC Asset Mgt'), ('ABERDEEN INDONESIA EQUITY FUND', 'aberdeen'), ('LAZARD GLOBAL ACTIVE FUNDS PLC', 'Lazard Asset Mgt'), ('GE GREATLINK ASEAN GROWTH FUND', 'Lion Global Investors'), ('TA SOUTH EAST ASIA EQUITY FUND', 'Lion Global Investors'), ('MAESTROLINK EQUITY PLUS BAHANA', 'AXA Asset Mgt Indonesia'), ('WS WALES PP GLOBAL GROWTH FUND', 'Waystone Mgt'), ('REKSA DANA INDEKS AVRIST IDX30', 'Avrist Asset Mgt'), ('PIMCO DIVIDEND AND INCOME FUND', 'Parametric Portfolio Associates'), ('I GEDE SAGUS YUDISTIRA LAKSANA', 'I Gede Sagus Yudistira Laksana'), ('DPLK GENERALI EQUITY BETA PLUS', 'Generali Indonesia'), ('AEGON WORLD EQUITY BEAT POOL C', 'AEGON Asset Mgt'), ('ASURANSI KERUGIAN JASA RAHARJA', 'Asuransi Kerugian Jasa Raharja'), ('PT EKADHARMA INTERNATIONAL TBK', 'Pt Ekadharma International'), ('PUTM ACS EMERGING MARKETS FUND', 'aberdeen'), ('RUSSELL INVESTMENTS UCITS ICAV', 'Russell Investments'), ('YUANTA NEW ASEAN BALANCED FUND', 'Yuanta Securities Investment Trust'), ('FUTURE FUND BOARD OF GUARDIANS', 'Future Fund'), ('ANDI MELDA DARDIANSYAH SOUISSA', 'Andi Melda Dardiansyah Souissa'), ('PT SUMBER PERINTIS RAGAMPANGAN', 'PT Sumber Perintis Ragampangan'), ('MASOU ASTA PACIFIC EQUITY FUND', 'Masou Investment Mgt'), ('LACM ACWI EX US EQUITY FUND LP', 'Christian Riccardo'), ('DWS CONCEPT DJE GLOBALE AKTIEN', 'Christian Riccardo'), ('ODDO BHF ASSET MANAGEMENT GMBH', 'Christian Riccardo'), ('NUVEEN MULTI ASSET INCOME FUND', 'Christian Riccardo'), ('CLEAR SKIES IWORLD EQUITY FUND', 'Christian Riccardo'), ('HERMES INVESTMENT FUNDS PUBLIC', 'Christian Riccardo'), ('BLACKROCK INVESTMENT MGT INDEX', 'BlackRock Investment Mgt - Index'), ('EMPLOYEES PROVIDENT FUND BOARD', 'Employees Provident Fund Board'), ('FRANKLIN TEMPLETON INVESTMENTS', 'Franklin Templeton Investments'), ('MIRAE ASSET GLOBAL INVESTMENTS', 'Mirae Asset Global Investments'), ('VANGUARD INVESTMENTS AUSTRALIA', 'Vanguard Investments Australia'), ('ISHARES ESG AWARE MSCI EM ETF', 'BlackRock Investment Mgt - Index'), ('PZENA GLOBAL VALUE FUND 2 DST', 'Pzena Investment Mgt'), ('ARGA INTERNATIONAL VALUE FUND', 'ARGA Investment Mgt'), ('ESPRING INV ASIAN EQUITY FUND', 'Eastspring Investments'), ('HSBC MSCI INDONESIA UCITS ETF', 'HSBC Global Asset Mgt'), ('NS PENSION PUBLIC EQUITY FUND', 'Pyrford International'), ('AQR EMERGING EQUITIES FUND II', 'AQR Capital Mgt'), ('INDUSTRIENS PENSIONFORSIKRING', 'JPMorgan Asset Mgt'), ('RD BATAVIA DANA SAHAM OPTIMAL', 'Batavia Prosperindo Asset Mgt'), ('RD PREMIER EKUITAS MAKRO PLUS', 'Indo Premier Investment Mgt'), ('REKSA DANA DANA EKUITAS PRIMA', 'Bahana TCW Investment Mgt'), ('BNP PARIBAS FINANCIAL MARKETS', 'BNP Paribas Financial Markets'), ('REKSA DANA PANIN DANA TELADAN', 'Panin Asset Mgt'), ('REKSA DANA PANIN DANA BERSAMA', 'Panin Asset Mgt'), ('REKSA DANA INDEKS PANIN IDX30', 'Panin Asset Mgt'), ('REKSA DANA PANIN DANA MAKSIMA', 'Panin Asset Mgt'), ('BNYM MELLON ASIAN INCOME FUND', 'Newton Investment Mgt'), ('PRIME EQUITY RUPIAH SCHRODERS', 'Schroder Investment Mgt'), ('HSBC GIF ASIAN FOCUSED INCOME', 'HSBC Global Asset Mgt'), ('PUBLIC INDONESIA BALANCE FUND', 'Public Mutual'), ('COLUMBIA THREADNEEDLE LUX III', 'Columbia Threadneedle Investments'), ('PUBLIC EMPLOYEES PENSION PLAN', 'Public Employees Pension Plan (PEPP)'), ('BCSL CLIENT PB CAYMAN CLIENTS', 'Barclays'), ('SUNLINK IDR FIXED INCOME PLUS', 'Sun Life Financial'), ('PICTET EMERGING MARKETS INDEX', 'Pictet Asset Mgt'), ('CAPACITY FONDS INSTITUTIONNEL', 'Pictet Asset Mgt'), ('ARTISAN PARTNERS GLOBAL FUNDS', 'Artisan Partners'), ('CC AND L Q GLOBAL EQUITY FUND', 'Connor Clark & Lunn Investment Mgt'), ('ALLIANZ GLOBAL INVESTORS FUND', 'Allianz Global Investors'), ('REKSA DANA INDEKS AVRIST LQ45', 'Avrist Asset Mgt'), ('PT TANRISE PROPERTY INDONESIA', 'Pt Tanrise Property Indonesia'), ('HUDSON EAST RIVER SYSTEMS LLC', 'Parametric Portfolio Associates'), ('KPD GREATLINK SAHAM NUSANTARA', 'Great Eastern Life Assurance'), ('GREATLINK SUPREME EQUITY FUND', 'Great Eastern Life Assurance'), ('VICTORY EMERGING MARKETS FUND', 'Victory Capital Mgt'), ('PELAYARAN MULTI JAYA SAMUDERA', 'Pelayaran Multi Jaya Samudera'), ('INSIGHT COMMUNITY DEVELOPMENT', 'Insight Investments Mgt'), ('REKSA DANA INSIGHT FELLOWSHIP', 'Insight Investments Mgt'), ('ALLIANZ ALPHA SECTOR ROTATION', 'Allianz Global Investors Asset Mgt Indonesia'), ('AIA ASIA EX JAPAN EQUITY FUND', 'AIA'), ('PRIME EQUITY RUPIAH PRINCIPAL', 'Principal Asset Mgt'), ('NAM WORLDWIDE INVESTMENT FUND', 'Nissay Asset Mgt'), ('ASSET MANAGEMENT EXCHANGE CCF', 'Metropolis Capital'), ('REKSA DANA PNM SAHAM UNGGULAN', 'PNM Investment Mgt'), ('MAJORIS SAHAM ALOKASI DINAMIK', 'Majoris Asset Mgt'), ('OPSEU PENSION PLAN TRUST FUND', 'OPSEU Pension Trust'), ('PT RAJAWALI MUTIARA SEJAHTERA', 'Pt Rajawali Mutiara Sejahtera'), ('LEONARDY PUTRA NEGARA SIREGAR', 'Leonardy Putra Negara Siregar'), ('ANDREW PRASETIO SUWANDI PUGUH', 'Andrew Prasetio Suwandi Puguh'), ('SHELL FOUNDATION SHELL CENTRE', 'Shell Asset Mgt'), ('REKSA DANA INDEKS SIMAS IDX30', 'Sinarmas Asset Mgt'), ('REKSA DANA KISI BALANCED FUND', 'KISI Asset Mgt'), ('BATAVIA PROSPERINDO ASSET MGT', 'Batavia Prosperindo Asset Mgt'), ('CHARLES SCHWAB INVESTMENT MGT', 'Charles Schwab Investment Mgt'), ('EDGBASTON INVESTMENT PARTNERS', 'Edgbaston Investment Partners'), ('TCW TRUST COMPANY OF THE WEST', 'TCW - Trust Company of the West'), ('MORGAN STANLEY INVESTMENT MGT', 'Morgan Stanley Investment Mgt'), ('EMPLOYEES PROVIDENT FUND EPF', 'Employees Provident Fund Board'), ('PRUDENTIAL ASSURANCE COMPANY', 'Eastspring Investments'), ('BLACKROCK GLOBAL INDEX FUNDS', 'BlackRock Investment Mgt - Index'), ('MERCER EMERGING MARKETS FUND', 'Robeco Asset Mgt'), ('PT PRUDENTIAL LIFE ASSURANCE', 'Eastspring Investments'), ('DANA PENSIUN KARYAWAN TASPEN', 'Taspen'), ('BMO INTERNATIONAL VALUE FUND', 'Pyrford International'), ('AQR GRP EL 20 MASTER ACCOUNT', 'AQR Capital Mgt'), ('KPD BPAM RUPIAH MANAGED FUND', 'Batavia Prosperindo Asset Mgt'), ('REKSA DANA BATAVIA LQ45 PLUS', 'Batavia Prosperindo Asset Mgt'), ('REKSA DANA PREMIER ETF IDX30', 'Indo Premier Investment Mgt'), ('REKSA DANA PREMIER ETF LQ 45', 'Indo Premier Investment Mgt'), ('YKP BRI QQ BANTUAN LAINNYA 1', 'BRI Investment Mgt'), ('REKSA DANA PANIN DANA ULTIMA', 'Panin Asset Mgt'), ('SMARTLINK RUPIAH EQUITY FUND', 'Allianz Life Insurance'), ('DPLK ALLIANZ IND EQUITY FUND', 'Allianz Life Insurance'), ('FIDELITY FUND INDONESIA FUND', 'FIL Investment Mgt'), ('EMERGING MARKETS EQUITY FUND', 'Allspring Global Investments'), ('DANA PENSIUN BRI SWAKELOLA 3', 'PT Bank Rakyat Indonesia (Persero) Tbk.'), ('DANA PENSIUN BRI SWAKELOLA 2', 'PT Bank Rakyat Indonesia (Persero) Tbk.'), ('KPD SAHAM DANA MAKSIMA BNIAM', 'BNI Asset Mgt'), ('DWS INVEST EUROPEAN EQUITIES', 'DWS Investments'), ('REKSA DANA TRIM KAPITAL PLUS', 'Trimegah Asset Mgt'), ('TRIMEGAH SEKURITAS INDONESIA', 'Trimegah Asset Mgt'), ('PUBLIC INDONESIA SELECT FUND', 'Public Mutual'), ('CIBC ASIA PACIFIC INDEX FUND', 'CIBC Asset Mgt'), ('JPMORGAN INCOME BUILDER FUND', 'JPMorgan Asset Mgt'), ('DPLK AXA MANDIRI EQUITY FUND', 'AXA Asset Mgt Indonesia'), ('MAESTROLINK MAXIADVANTAGE RP', 'AXA Asset Mgt Indonesia'), ('CATERPILLAR INVESTMENT TRUST', 'Caterpillar Master Retirement Trust'), ('UNITED INDONESIA EQUITY FUND', 'UOB Asset Mgt'), ('PT AJ SEQUIS LIFEE AAMI SQMF', 'Sequislife'), ('PT STUDILMU TEKNOLOGI GLOBAL', 'PT Studilmu Teknologi Global'), ('KPD AAMI RUPIAH MANAGED FUND', 'Sequis Asset Mgt'), ('KPD SQAM RUPIAH MANAGED FUND', 'Sequis Asset Mgt'), ('ALLSPRING LUX WORLDWIDE FUND', 'Allspring Global Investments'), ('RAIFFEISEN NACHHALTIG AKTIEN', 'Raiffeisen Capital Mgt'), ('PT BERLIAN TAMBANG INDONESIA', 'PT Berlian Tambang Indonesia'), ('REKSA DANA MAM BALANCED FUND', 'Maybank Asset Mgt'), ('PT PELAYARAN MARINDO PACIFIC', 'PT Pelayaran Marindo Pacific'), ('REKSA DANA PNM SAHAM AGRESIF', 'PNM Investment Mgt'), ('PINEBRIDGE WORLD EQUITY OPEN', 'PineBridge Investments'), ('MARGARETHA PRATIWI SEKARWATI', 'Margaretha Pratiwi Sekarwati'), ('SWANDANA WITANANTA SEJAHTERA', 'Swandana Witananta Sejahtera'), ('REKSA DANA AURORA SMC EQUITY', 'Aurora Asset Mgt'), ('WM POOL EQUITIES TRUST NO 75', 'INTECH Investment Mgt'), ('PRINCIPAL ASEAN DYNAMIC FUND', 'Principal Asset Mgt'), ('CHUBB RUPIAH EQUITY FUND III', 'Chubb Life Insurance'), ('ANGKASA STRATEGIK INVESTINDO', 'Angkasa Strategik Investindo'), ('GIOVANNO OROH VICKY HERMAWAN', 'Giovanno Oroh Vicky Hermawan'), ('BINTANG PRIMA PUTRA HUTASOIT', 'Bintang Prima Putra Hutasoit'), ('AGCM FUND ASIA DIVIDEND FUND', 'Asia Growth Capital Mgt'), ('DANA PENSIUN LEMBAGA KEUNGAN', 'Pinebridge Investments'), ('VALBURY PRIME DYNAMIC EQUITY', 'Valbury Capital Mgt'), ('SANDS CAPITAL MANAGEMENT LLC', 'Christian Riccardo'), ('NTC CIM INVESTMENT FUND ICAV', 'Christian Riccardo'), ('YPW ASIAN EQUITY MOTHER FUND', 'Christian Riccardo'), ('ALLSPRING GROUP HOLDINGS LLC', 'Christian Riccardo'), ('NH CA POWER ASEAN PLUS SIT 1', 'Christian Riccardo'), ('RHB ASSET MANAGEMENT SDN BHD', 'Christian Riccardo'), ('AFFIN HWANG ASEAN FLEXI FUND', 'Christian Riccardo'), ('MONDRIAN INVESTMENT PARTNERS', 'Mondrian Investment Partners'), ('BUREAU OF LABOR FUNDS TAIWAN', 'Bureau of Labor Funds Taiwan'), ('ALLSPRING GLOBAL INVESTMENTS', 'Allspring Global Investments'), ('SUMITOMO MITSUI DS ASSET MGT', 'Sumitomo Mitsui DS Asset Mgt'), ('KOREA INVESTMENT CORPORATION', 'Korea Investment Corporation'), ('FIL INVESTMENT INTERNATIONAL', 'FIL Investment International'), ('THOMPSON SIEGEL AND WALMSLEY', 'Thompson, Siegel & Walmsley'), ('GREAT EASTERN LIFE ASSURANCE', 'Great Eastern Life Assurance'), ('WILLIAM BLAIR INVESTMENT MGT', 'William Blair Investment Mgt'), ('CHINA INVESTMENT CORPORATION', 'China Investment Corporation'), ('VALVERDE INVESTMENT PARTNERS', 'Valverde Investment Partners'), ('ISHARES MSCI ACWI EX US ETF', 'BlackRock Investment Mgt - Index'), ('VANGUARD GLOBAL EQUITY FUND', 'Pzena Investment Mgt'), ('PZENA GLOBAL VALUE FUND DST', 'Pzena Investment Mgt'), ('BRANDES INVESTMENT PARTNERS', 'Brandes Investment Partners'), ('INTERNATIONAL MONETARY FUND', 'State Street Investment Mgt'), ('BAILLIE GIFFORD GROUP TRUST', 'Baillie Gifford & Co'), ('LEGAL AND GENERAL UCITS ETF', 'Legal & General Investment Mgt'), ('PYRFORD INTERNATIONAL TRUST', 'Pyrford International'), ('STICHTING PENSIOENFONDS ABP', 'APG Asset Mgt Asia'), ('MANULIFE DANA PRIMA DINAMIS', 'Manulife Investment Mgt Indonesia'), ('MANDIRI EQUITY MONEY RUPIAH', 'Mandiri Investasi'), ('KPD BPAM RUPIAH EQUITY FUND', 'Batavia Prosperindo Asset Mgt'), ('FIDELITY EMERGING ASIA FUND', 'Fidelity Investments'), ('FIDELITY PACIFIC BASIN FUND', 'Fidelity Investments'), ('REKSA DANA PANIN DANA PRIMA', 'Panin Asset Mgt'), ('PFI MEGA LIFE BALANCED FUND', 'Schroder Investment Mgt'), ('UBS GLOBAL ASSET MANAGEMENT', 'UBS Global Asset Mgt'), ('PRIME EQUITY RUPIAH ASHMORE', 'Ashmore Asset Mgt'), ('DANA PENSIUN KRAKATAU STEEL', 'BNI Asset Mgt'), ('HSBC LIFE VALUE GROWTH FUND', 'HSBC Global Asset Mgt'), ('REKSA DANA TRIM KOMBINASI 2', 'Trimegah Asset Mgt'), ('MATTHEWS ASIA DIVIDEND FUND', 'Matthews International Capital Mgt'), ('LAZARD ASSET MANAGEMENT LLC', 'Lazard Asset Mgt'), ('AVRIST LINK EQUITY IDR FUND', 'Avrist Asset Mgt'), ('SWEDBANK ROBUR ACCESS ASIEN', 'Swedbank Robur'), ('DANA PENSUIN BANK INDONESIA', 'Dapenbi'), ('FWD EXCELLENT BALANCED FUND', 'FWD Asset Mgt'), ('CATHOLIC VALUES EQUITY FUND', 'Parametric Portfolio Associates'), ('KPD AAMI RUPIAH EQUITY FUND', 'Sequis Asset Mgt'), ('KPD SQAM RUPIAH EQUITY FUND', 'Sequis Asset Mgt'), ('WEST YORKSHIRE PENSION FUND', 'West Yorkshire Pension Fund'), ('ALBERTUS ANDRI ANGGASAPUTRO', 'Albertus Andri Anggasaputro'), ('OHMAN ETISK EMERGING MARKET', 'Lannebo Asset Mgt'), ('ERICK KRISTIANTO ADITYATAMA', 'Erick Kristianto Adityatama'), ('FRANKLIN LIBERTYSHARES ICAV', 'Franklin Templeton Investments'), ('AMUNDI MODERATE GROWTH FUND', 'Amundi Hong Kong'), ('MD INTERNATIONAL VALUE FUND', 'Jensen Investment Mgt'), ('PHILIP AGUSTINUS P TRISNADI', 'Philip Agustinus P Trisnadi'), ('IVONNE OLIVIA INDRADIHARDJO', 'Ivonne Olivia Indradihardjo'), ('DHANI MUHAMMAD KURNIAWAN ST', 'Dhani Muhammad Kurniawan St'), ('YOHANES BINTORO BUDIRAHARJO', 'Yohanes Bintoro Budiraharjo'), ('TJUN FUNG AND IWAN PURNAWAN', 'Tjun Fung & Iwan Purnawan'), ('PT LUMBUNG INDUSTRI PERSADA', 'Lumbung Industri Persada'), ('PT CHANDRA ANUGERAH SENTOSA', 'Pt Chandra Anugerah Sentosa'), ('DANIEL MONIAGA FREDDY PUTRA', 'Daniel Moniaga Freddy Putra'), ('TJANDRA STRATEGIK INVESTAMA', 'Tjandra Strategik Investama'), ('BAMBANG TEDJO NARSOYO ST MT', 'Bambang Tedjo Narsoyo St Mt'), ('EDWARD BARCELLO SIMORANGKIR', 'Edward Barcello Simorangkir'), ('REKSA DANA I AM EQUITY FUND', 'Indoasia Asset Mgt'), ('AMUNDI CDA HIGH INCOME FUND', 'Amundi Asset Mgt'), ('REKSA DANA KISI EQUITY FUND', 'KISI Asset Mgt'), ('PRESCIENT GLOBAL FUNDS ICAV', 'Christian Riccardo'), ('UNITED ASEAN DISCOVERY FUND', 'Christian Riccardo'), ('DRS MOHAMMAD ANSOR ALAMSYAH', 'Christian Riccardo'), ('EUROMOBILIARE PICTET ACTION', 'Christian Riccardo'), ('BRANDYWINE INVESTMENT TRUST', 'Christian Riccardo'), ('STATE STREET INVESTMENT MGT', 'State Street Investment Mgt'), ('INDO PREMIER INVESTMENT MGT', 'Indo Premier Investment Mgt'), ('FIRST SENTIER INVESTORS RQI', 'First Sentier Investors RQI'), ('AXA INVESTMENT MGRS SCHWEIZ', 'AXA Investment Mgrs Schweiz'), ('GUINNESS ATKINSON ASSET MGT', 'Guinness Atkinson Asset Mgt'), ('DESJARDINS GLOBAL ASSET MGT', 'Desjardins Global Asset Mgt'), ('OLD MUTUAL INVESTMENT GROUP', 'Old Mutual Investment Group'), ('INVESCO GLOBAL INCOME FUND', 'Invesco'), ('BRANDES GLOBAL EQUITY FUND', 'Brandes Investment Partners'), ('AQR EMERGING EQUITIES FUND', 'AQR Capital Mgt'), ('JPMORGAN ETFS IRELAND ICAV', 'JPMorgan Asset Mgt'), ('RDI BATAVIA SRI KEHATI ETF', 'Batavia Prosperindo Asset Mgt'), ('PRIME EQUITY RUPIAH BAHANA', 'Bahana TCW Investment Mgt'), ('ARROWSTREET US GROUP TRUST', 'Arrowstreet Capital'), ('UNIVERSAL S AKTIE NFONDS I', 'Manulife Investment Mgt'), ('SMARTWEALTH LIQUIFLEX LQ45', 'Allianz Life Insurance'), ('DPLK ALLIANZ IND HERO FUND', 'Allianz Life Insurance'), ('GLOBAL X SUPERDIVIDEND ETF', 'Global X Mgt'), ('DANA PENSIUN BRI SWAKELOLA', 'PT Bank Rakyat Indonesia (Persero) Tbk.'), ('GOLDMAN SACHS ETF TRUST II', 'Goldman Sachs Asset Mgt'), ('HSBC LIFE AND ANNUITY FUND', 'HSBC Global Asset Mgt'), ('JPMSE LUX RE JPMSE RE UI S', 'Universal Investment'), ('FOVERUKA PENSION UNIVERSAL', 'Universal Investment'), ('EBK AKTIEN UNIVERSAL FONDS', 'Universal Investment'), ('ASABRI PERSERO DAPEN POLRI', 'Asabri'), ('PB CHINA ASEAN EQUITY FUND', 'Public Mutual'), ('CIBC EMERGING MARKETS FUND', 'CIBC Asset Mgt'), ('ASIAN EQUITIES DERIVATIVES', 'HSBC Securities'), ('MACQUARIE BANK LIMITED DBU', 'Macquarie Bank'), ('RUTH CORE EMERGING MARKETS', 'Ruth Asset Mgt'), ('SCHRODER ASIAN INCOME FUND', 'Schroder Investment Mgt'), ('SCHRODER GLOBAL VALUE FUND', 'Schroder Investment Mgt'), ('ABERDEEN ASIAN INCOME FUND', 'aberdeen'), ('PT AVRIST ASSET MANAGEMENT', 'Avrist Asset Mgt'), ('KENTUCKY RETIREMENT SYSTEM', "Kentucky Teachers' Retirement System"), ('VIRGINIA RETIREMENT SYSTEM', 'Virginia Retirement System'), ('FWD USD BALANCED PLUS FUND', 'FWD Asset Mgt'), ('TRINITY HEALTH CORPORATION', 'Trinity Health Corporation'), ('PICTET AND CIE UNDISCLOSED', 'Pictet & Cie'), ('PRUDENTIAL UNIT TRUST FUND', 'BOCI Prudential Asset Mgt'), ('INDONESIA MANDIRI NETWORKS', 'Indonesia Mandiri Networks'), ('RAIFFEISEN EURASIEN AKTIEN', 'Raiffeisen Capital Mgt'), ('DARMAYANTI ANYA ADISUSANTO', 'Darmayanti Anya Adisusanto'), ('NTB IESI MSCI ENHMF 936561', 'Amova Asset Mgt'), ('REKSA DANA KRESNA INDEX 45', 'Kresna Securities'), ('MINEWORKERS PENSION SCHEME', 'Mineworkers Pension Scheme'), ('AGUSTINUS SIOR FERDINANDUS', 'Agustinus Sior Ferdinandus'), ('RADITYA DIKA ANGKASA PUTRA', 'Raditya Dika Angkasa Putra'), ('KSD MIRAE ASSET SECURITIES', 'Mirae Asset Securities'), ('YURICHO ALEXANDER YOGIANTO', 'Yuricho Alexander Yogianto'), ('PT ARTHAWENASAKTI GEMILANG', 'Pt Arthawenasakti Gemilang'), ('PT INDOVESTA UTAMA MANDIRI', 'PT Indovesta Utama Mandiri'), ('TEGUH BUDI PAMUNGKAS SE MM', 'Teguh Budi Pamungkas SE MM'), ('RICHARD NELSON MANURUNG IR', 'Richard Nelson Manurung Ir'), ('ELDA CHRISTIAN TIARAWATY C', 'Elda Christian Tiarawaty C'), ('PT BINTANG DELAPAN HARMONI', 'PT Bintang Delapan Harmoni'), ('DRA MARIA GORETTI TINAWATI', 'Dra Maria Goretti Tinawati'), ('HENGKY GOENAWAN HENOEK S E', 'Hengky Goenawan Henoek S E'), ('PRABOWO WICAKSONO YUWONO P', 'Prabowo Wicaksono Yuwono P'), ('PT GISTEX CHEWON SYNTHETIC', 'Pt Gistex Chewon Synthetic'), ('MARISA ANTA SARI AND S SOS', 'Marisa Anta Sari & S SOS'), ('PT JAVA AGRI SUKSES MAKMUR', 'Pt Java Agri Sukses Makmur'), ('MICHAEL CHRISTIAN SOEKAMTO', 'Michael Christian Soekamto'), ('ERIC HARTANTO NYOTOSETIADI', 'Eric Hartanto Nyotosetiadi'), ('CHANDRA RAMAZETA PAMUNGKAS', 'Chandra Ramazeta Pamungkas'), ('PT BERJUANG SUKSES BERSAMA', 'Pt Berjuang Sukses Bersama'), ('REKSADANA MNC DANA EKUITAS', 'MNC Asset Mgt'), ('RD CAPITAL BALANCED GROWTH', 'Capital Asset Mgt'), ('PT ADIKARYA SUKSES BERSAMA', 'PT Adikarya Sukses Bersama'), ('MAN NUMERIC EXPLORER TRUST', 'Man Numeric'), ('LACM ACWI EX US EQUITY CIT', 'Christian Riccardo'), ('BUALUANG ASEAN EQUITY FUND', 'Christian Riccardo'), ('BERMONT HOLDINGS 2 LIMITED', 'Christian Riccardo'), ('BERMONT HOLDINGS 3 LIMITED', 'Christian Riccardo'), ('BERMONT HOLDINGS 1 LIMITED', 'Christian Riccardo'), ('BOREALIS KAPITAL INVESTAMA', 'Christian Riccardo'), ('CANTILLON GLOBAL EQUITY LP', 'Christian Riccardo'), ('T ROWE PRICE INTERNATIONAL', 'T Rowe Price International'), ('NORGES BANK INVESTMENT MGT', 'Norges Bank Investment Mgt'), ('ALASKA PERMANENT FUND CORP', 'Alaska Permanent Fund Corp.'), ('HANDELSBANKEN FONDER INDEX', 'Handelsbanken Fonder - Index'), ('PT EKADHARMA INTERNATIONAL', 'Pt Ekadharma International'), ('QATAR INVESTMENT AUTHORITY', 'Qatar Investment Authority'), ('PRINCIPAL GLOBAL INVESTORS', 'Principal Global Investors'), ('SUMITOMO MITSUI TRUST BANK', 'Sumitomo Mitsui Trust Bank'), ('DAI ICHI LIFE INSURANCE CO', 'Dai-Ichi Life Insurance Co.'), ('DANAPATHI ASSET MANAGEMENT', 'Danapathi Asset Management'), ('AGF INTERNATIONAL ADVISORS', 'AGF International Advisors'), ('KOREA INVESTMENT TRUST MGT', 'Korea Investment Trust Mgt'), ('PRATAMA CAPITAL ASSETS MGT', 'Pratama Capital Assets Mgt'), ('MERCER GLOBAL EQUITY FUND', 'Brandes Investment Partners'), ('BRANDES GLOBAL VALUE FUND', 'Brandes Investment Partners'), ('PRUDENTIAL LIFE ASSURANCE', 'Eastspring Investments'), ('BMO AM GLOBAL EQUITY FUND', 'Pyrford International'), ('AQR GRP EL MASTER ACCOUNT', 'AQR Capital Mgt'), ('SPP EMERGING MARKETS PLUS', 'Storebrand Investments'), ('MANULIFE DANA EKUITAS USD', 'Manulife Investment Mgt Indonesia'), ('TUGU REASURANSI INDONESIA', 'Manulife Investment Mgt Indonesia'), ('RD BNP PARIBAS MAXI SAHAM', 'BNP Paribas Asset Mgt'), ('REKSA DANA FORTIS SOLARIS', 'BNP Paribas Asset Mgt'), ('DANA PENSIUN BANK MANDIRI', 'Mandiri Investasi'), ('RD PREMIER ETF SRI KEHATI', 'Indo Premier Investment Mgt'), ('DPLK AIA EQUITY FUND ER13', 'AIA Indonesia'), ('DANA PENSIUN BRI BAHANA 2', 'Bahana TCW Investment Mgt'), ('AVA SMALL CAP EQUITY FUND', 'Schroder Investment Mgt'), ('FIDELITY FUNDS ASEAN FUND', 'FIL Investment Mgt'), ('UBS LUX FUND SOLUTIONS II', 'UBS Global Asset Mgt'), ('BBH MTBJ RE MTBJ400025521', 'State Street Investment Mgt'), ('IAM NATIONAL PENSION FUND', 'Northern Trust Global Investments'), ('CT LUX ASIA EQUITIES FUND', 'Columbia Threadneedle Investments'), ('THREADNEEDLE PENSIONS LTD', 'Columbia Threadneedle Investments'), ('MORGAN STANLEY AND CO INC', 'Morgan Stanley Asia'), ('EXPANSION CRAFTED BY RUTH', 'Ruth Asset Mgt'), ('PT TANCORP GLOBAL SENTOSA', 'PT Tancorp Global Sentosa'), ('SEB EMERGING MARKETSFONDS', 'SEB Asset Mgt'), ('FLORIDA RETIREMENT SYSTEM', 'Florida Retirement System'), ('PT TANRISE MAKMUR SENTOSA', 'Pt Tanrise Makmur Sentosa'), ('ASURANSI JIWA SEQUIS LIFE', 'Sequislife'), ('FWD EXCELLENT EQUITY FUND', 'FWD Asset Mgt'), ('RB INDONESIAN EQUITY FUND', 'Resona Asset Mgt'), ('EARNEST INSTITUTIONAL LLC', 'Earnest Partners'), ('PT TUGU PRATAMA INDONESIA', 'Tugu Pratama Indonesia'), ('REKSA DANA INSIGHT WEALTH', 'Insight Investments Mgt'), ('WIN BAGHRAF BERSAUDARA PT', 'Win Baghraf Bersaudara Pt'), ('JOAO ANGELO DE SOUSA MOTA', 'Joao Angelo De Sousa Mota'), ('TM EQUITY INDONESIAN FUND', 'Tokio Marine Asset Mgt'), ('TM EQUITY AGGRESSIVE FUND', 'Tokio Marine Asset Mgt'), ('MALTA PENSION INVESTMENTS', 'BNP Paribas Asset Mgt'), ('CEMERLANG PRIMA INVESTAMA', 'Cemerlang Prima Investama'), ('AHMAD SOLICHIN LUTFIYANTO', 'Ahmad Solichin Lutfiyanto'), ('TATANG BUDIONO TJAHAJANTO', 'Tatang Budiono Tjahajanto'), ('PT OPTIK TUNGGAL SEMPURNA', 'Pt Optik Tunggal Sempurna'), ('HANDAYANI TJANDRASA DR IR', 'Handayani Tjandrasa Dr Ir'), ('PT SUARA VISUAL INDONESIA', 'PT Suara Visual Indonesia'), ('INDUK KOPERASI KEPOLISIAN', 'Induk Koperasi Kepolisian'), ('GUNUNG PAHALA TUMPAL S IR', 'Gunung Pahala Tumpal S, Ir'), ('FERRY MARANGIN SIMATUPANG', 'Ferry Marangin Simatupang'), ('ARIEF CHRISTOPHEL SARAGIH', 'Arief Christophel Saragih'), ('RATNA LISTIANI SUGITO DRG', 'Ratna Listiani Sugito Drg'), ('FERDYNAN JULYADY SETIAWAN', 'Ferdynan Julyady Setiawan'), ('PT BERKAT NIRWANA SENTOSA', 'Pt Berkat Nirwana Sentosa'), ('IDA BAGUS PUTU ANOM REDHI', 'Ida Bagus Putu Anom Redhi'), ('CHUBB RUPIAH MANAGED FUND', 'Chubb Life Insurance'), ('STEVEN ANTONIUS KURNIAWAN', 'Steven Antonius Kurniawan'), ('RUBEN ARTIA LUMBAN TORUAN', 'Ruben Artia Lumban Toruan'), ('ALEXANDER SOETOPO PUTRA P', 'Alexander Soetopo Putra P'), ('LIEM SALIM HADRIAN WIJAYA', 'Liem Salim Hadrian Wijaya'), ('PT MERPATI MARINE SERVICE', 'Pt Merpati Marine Service'), ('NICODEMUS KASAN KURNIAWAN', 'Nicodemus Kasan Kurniawan'), ('WENDY FEBRIYANTI LAOLI ST', 'Wendy Febriyanti Laoli ST'), ('IMPRESITAMA VISI KARYA PT', 'Impresitama Visi Karya Pt'), ('BHAGWAN RAMCHAND BHARWANI', 'Bhagwan Ramchand Bharwani'), ('INDINVEST STRATEGY CENTRE', 'Indinvest Strategy Centre'), ('REKSA DANA CIPTA DINAMIKA', 'Ciptadana Asset Mgt'), ('CV SHENGPO SUKSES BERSAMA', 'CV Shengpo Sukses Bersama'), ('TREASURE FUND SUPER MAXXI', 'Treasure Fund Investama'), ('REKSA DANA NET DANA FLEXI', 'Net Assets Mgt'), ('FUH HWA RISING ASEAN FUND', 'Christian Riccardo'), ('BUALUANG ASEAN EQUITY RMF', 'Christian Riccardo'), ('BMO FOSSIL FUEL FREE FUND', 'Christian Riccardo'), ('CHRISTOPHER ALDO SETIAWAN', 'Christian Riccardo'), ('BANQUE CANTONALE VAUDOISE', 'Christian Riccardo'), ('CANLIFE ASIA PACIFIC FUND', 'Christian Riccardo'), ('DIMENSIONAL FUND ADVISORS', 'Dimensional Fund Advisors'), ('BAHANA TCW INVESTMENT MGT', 'Bahana TCW Investment Mgt'), ('GENERATION INVESTMENT MGT', 'Generation Investment Mgt'), ('BOCI PRUDENTIAL ASSET MGT', 'BOCI Prudential Asset Mgt'), ('DANA PENSIUN ANGKASA PURA', 'Dana Pensiun Angkasa Pura'), ('ARROW INVESTMENT ADVISORS', 'Arrow Investment Advisors'), ('GUINNESS GLOBAL INVESTORS', 'Guinness Global Investors'), ('BRANDES INVESTMENT FUNDS', 'Brandes Investment Partners'), ('SPDR MSCI ACWI EX US ETF', 'State Street Investment Mgt'), ('MBB PUBLIC MARKETS I LLC', 'State Street Investment Mgt'), ('INTERNATIONAL STOCK FUND', 'Pyrford International'), ('SPP EMERGING MARKETS SRI', 'Storebrand Investments'), ('GSP DANA SAHAM BERIMBANG', 'Manulife Investment Mgt Indonesia'), ('REKSA DANA FORTIS PESONA', 'BNP Paribas Asset Mgt'), ('MANDIRI DINAMIC MONEY RP', 'Mandiri Investasi'), ('BATAVIA SMART LIQUID ETF', 'Batavia Prosperindo Asset Mgt'), ('T ROWE PRICE FUNDS SICAV', 'T Rowe Price'), ('ALASKA COMMON TRUST FUND', 'T Rowe Price'), ('RD PREMIER ETF SMINFRA18', 'Indo Premier Investment Mgt'), ('ASABRI PERSERO DAPEN TNI', 'Asabri'), ('PUBLIC ASEAN GROWTH FUND', 'Public Mutual'), ('BMO GLOBAL BALANCED FUND', 'Columbia Threadneedle Investments'), ('ENSIGN PEAK ADVISORS INC', 'Ensign Peak Advisors'), ('AI BALANCED PENSION FUND', 'Schroder Investment Mgt'), ('SCHRODER GAIA HELIX FUND', 'Schroder Investment Mgt'), ('CITIGROUP GLOBAL MARKETS', 'Citigroup Global Markets'), ('INVESCO BCPL BADGER FUND', 'Invesco Capital Mgt'), ('MAESTROLINK BALANCED IDR', 'AXA Asset Mgt Indonesia'), ('REKSA DANA AVRIST EQUITY', 'Avrist Asset Mgt'), ('PT INVESTASI KARYA UTAMA', 'PT Investasi Karya Utama'), ('FLEXIBLE INCOME FAM FUND', 'Fineco Asset Mgt'), ('PARAMETRIC TMEMC FUND LP', 'Parametric Portfolio Associates'), ('AXA AFFIN LIFE INSURANCE', 'AXA Investment Mgrs Asia'), ('KPD GREATLINK DANA SAHAM', 'Great Eastern Life Assurance'), ('QIC LISTED EQUITIES FUND', 'QIC Asset Mgt'), ('TSCB DIAM VIP FOCUS FUND', 'Asset Mgt One Singapore'), ('AJ GENERALI INDONESIA PT', 'Generali Indonesia'), ('REKSA DANA INSIGHT GREEN', 'Insight Investments Mgt'), ('FERDIAN TIMUR SATYAGRAHA', 'Ferdian Timur Satyagraha'), ('STEVEN SETIAWAN SUTRISNA', 'Steven Setiawan Sutrisna'), ('STANLEY HANDIONO ANGKASA', 'Stanley Handiono Angkasa'), ('LANNEBO EMERGING MARKETS', 'Lannebo Asset Mgt'), ('DANIEL ALEXANDER SANTOSA', 'Daniel Alexander Santosa'), ('NINIK MURNI AYU HARTANTO', 'Ninik Murni Ayu Hartanto'), ('DEEPAK PRITAMDAS GURBANI', 'Deepak Pritamdas Gurbani'), ('YUDHI IRAWAN AMIN WIJAYA', 'Yudhi Irawan Amin Wijaya'), ('INCOME INSURANCE LIMITED', 'Fullerton Fund Mgt'), ('YAFINDO GLOBAL INVESTAMA', 'Yafindo Global Investama'), ('VINI TABADIRI INVESTA PT', 'Vini Tabadiri Investa Pt'), ('PT INDONESIA OCEAN TRUCK', 'PT Indonesia Ocean Truck'), ('IMELDA FRANCISCA BUDIMAN', 'Prospera Asset Mgt'), ('BARNABAS SIMON KURNIAWAN', 'Barnabas Simon Kurniawan'), ('ANAS WIDJAJA BARAK RIMBA', 'Anas Widjaja Barak Rimba'), ('AWALUDDIN OCTO RACHNALIM', 'Awaluddin Octo Rachnalim'), ('ENRICO FAJAR FERDIANSYAH', 'Enrico Fajar Ferdiansyah'), ('TJAHJANA KURNIAWAN TIRTA', 'Tjahjana Kurniawan Tirta'), ('IR HARJANTO TEGUH RAHAJU', 'Ir Harjanto Teguh Rahaju'), ('PT SYNERGY INTI PROPERTI', 'Pt Synergy Inti Properti'), ('HARRIS KRISTANTO WIDJAJA', 'Harris Kristanto Widjaja'), ('PHEIM ASIA EX JAPAN FUND', 'Pheim Asset Mgt'), ('IR EDI WAHYU CAHYONO M P', 'Ir Edi Wahyu Cahyono M P'), ('BOOSTER WHOLESALE SCHEME', 'Booster Investment Mgt'), ('THE BING TIK AND BUDIMAN', 'The Bing Tik & Budiman'), ('DANA PENSIUN KIMIA FARMA', 'Narada Asset Mgt'), ('AGUS IWAN SUTANTO KUSUMA', 'Agus Iwan Sutanto Kusuma'), ('THE MANZANITA FOUNDATION', 'The Manzanita Foundation'), ('CV ARTHA FIBER INDONESIA', 'CV Artha Fiber Indonesia'), ('CAISHEN SPC LTCP FUND SP', 'Swiss Asia Capital'), ('HENRY DWI ADJIE FIRTANTO', 'Henry Dwi Adjie Firtanto'), ('WILSHIRE SOLUTIONS FUNDS', 'Christian Riccardo'), ('ANDREW WELLINGTON S CHAN', 'Christian Riccardo'), ('APRICUS CAPITAL HOLDINGS', 'Christian Riccardo'), ('FOURNIER RETIREMENT FUND', 'Christian Riccardo'), ('STARRY HOMENTA RAMPENGAN', 'Christian Riccardo'), ('RACHMAT GUNTUR SOEGIARTO', 'Christian Riccardo'), ('PT DANA BANGSA SEJAHTERA', 'Christian Riccardo'), ('SARANA KALYANA NUSANTARA', 'Christian Riccardo'), ('IB PUTU ANOM REDHI SH MM', 'Christian Riccardo'), ('PT SAPTA MANUNGGAL KARYA', 'Christian Riccardo'), ('INDUS SELECT MASTER FUND', 'Christian Riccardo'), ('ASEAN EQUITY MOTHER FUND', 'Christian Riccardo'), ('MITSUBISHI UFJ ASSET MGT', 'Mitsubishi UFJ Asset Mgt'), ('KUWAIT INVESTMENT OFFICE', 'Kuwait Investment Office'), ('HENAN PUTIHRAI ASSET MGT', 'Henan Putihrai Asset Mgt'), ('ALLIANZ GLOBAL INVESTORS', 'Allianz Global Investors'), ('COOK COUNTY PENSION FUND', 'Cook County Pension Fund'), ('ALLIANCEBERNSTEIN GROWTH', 'AllianceBernstein - Growth'), ('AXA INVESTMENT MGRS ASIA', 'AXA Investment Mgrs Asia'), ('HANG SENG INVESTMENT MGT', 'Hang Seng Investment Mgt'), ('MERLIN FIDELIS ASSET MGT', 'Merlin Fidelis Asset Mgt'), ('INVESCO OPPENHEIMERFUNDS', 'Invesco (OppenheimerFunds)'), ('ASSETPLUS INVESTMENT MGT', 'Assetplus Investment Mgt'), ('LUMBUNG INDUSTRI PERSADA', 'Lumbung Industri Persada'), ('SETIABUDI INVESTMENT MGT', 'Setiabudi Investment Mgt'), ('GOVERNMENT OF INDONESIA', 'Government of Indonesia'), ('PZENA GLOBAL VALUE FUND', 'Pzena Investment Mgt'), ('BURROUGHS WELLCOME FUND', 'Pzena Investment Mgt'), ('ARGA INTERNATIONAL FUND', 'ARGA Investment Mgt'), ('ARGA ASIA EX JAPAN FUND', 'ARGA Investment Mgt'), ('BLACKROCK BALANCED FUND', 'BlackRock Investment Mgt - Index'), ('PT ASURANSI JIWA TASPEN', 'Taspen'), ('DPLK MANULIFE INDONESIA', 'Manulife Investment Mgt Indonesia'), ('REKSA DANA INDEKS IDX30', 'BNP Paribas Asset Mgt'), ('DANA PENSIUN PERKEBUNAN', 'Mandiri Investasi'), ('BATAVIA SAHAM CEMERLANG', 'Batavia Prosperindo Asset Mgt'), ('BATAVIA SAHAM SEJAHTERA', 'Batavia Prosperindo Asset Mgt'), ('RD BATAVIA DANA DINAMIS', 'Batavia Prosperindo Asset Mgt'), ('DPLK AIA FINANCIAL DEXA', 'AIA Indonesia'), ('KPD DAPEN WIKA PPMP S A', 'Bahana TCW Investment Mgt'), ('ARERO DER WELTFONDS ESG', 'DWS Investments'), ('DEAM FONDS KG PENSIONEN', 'DWS Investments'), ('GOVERNMENT OF SINGAPORE', 'GIC'), ('HANDELSBANKEN ASIENFOND', 'Handelsbanken Fonder - Index'), ('MERSEYSIDE PENSION FUND', 'Northern Trust Global Investments'), ('GENERATION IM ASIA FUND', 'Generation Investment Mgt'), ('UI FONDS BAV RBI AKTIEN', 'State Street Investment Mgt'), ('BALANCE CRAFTED BY RUTH', 'Ruth Asset Mgt'), ('ROBERT CHRISTIAN TANOKO', 'Robert Christian Tanoko'), ('LAZARD ASSET MANAGEMENT', 'Lazard Asset Mgt'), ('PKG FONDS AKTIEN GLOBAL', 'Pictet Asset Mgt'), ('DB ADVISORY MULTIBRANDS', 'Pictet Asset Mgt'), ('AXA MAESTROLINK DYNAMIC', 'AXA Asset Mgt Indonesia'), ('AXA FINANCIAL INDONESIA', 'AXA Asset Mgt Indonesia'), ('ALLIANZGI FONDS APNIESA', 'Allianz Global Investors'), ('PT SURYAPARAMITRA ABADI', 'PT Suryaparamitra Abadi'), ('PETRUS KURNIAWAN SUSILO', 'Petrus Kurniawan Susilo'), ('HOSKING GLOBAL FUND PLC', 'Hosking Partners'), ('UTAH RETIREMENT SYSTEMS', 'Utah Retirement Systems'), ('LUMIERE ASIA VALUE FUND', 'Lumiere Capital'), ('YKP3 JATI SEJAHTERA SAM', 'Samuel Asset Mgt'), ('EDHIE BASKORO YUDHOYONO', 'Edhie Baskoro Yudhoyono'), ('DEWATA MEKARYA AGENG PT', 'Dewata Mekarya Ageng PT'), ('PT ASURANSI WAHANA TATA', 'Asuransi Wahana Tata'), ('PT DELTA WIBAWA BERSAMA', 'Delta Wibawa Bersama'), ('HARESH CHANDRA NANDWANI', 'Haresh Chandra Nandwani'), ('SCHRODER ASIA MORE PLUS', 'Schroder Investment Mgt'), ('FRANKLIN TEMPLETON ICAV', 'Franklin Templeton Investments'), ('PT ERINDO MULIA PRATAMA', 'Pt Erindo Mulia Pratama'), ('PT MITRA MAJU INVESTAMA', 'Pt Mitra Maju Investama'), ('SANTIKA TANTRA SETIAWAN', 'Santika Tantra Setiawan'), ('HYACINTHA SUSANTI JAHJA', 'Hyacintha Susanti Jahja'), ('INSPIRE GLOBAL HOPE ETF', 'Inspire Advisors'), ('UNIVERSITAS GADJAH MADA', 'Universitas Gadjah Mada'), ('IWAN SUGIARTO WIJATMIKO', 'Iwan Sugiarto Wijatmiko'), ('DEWINTA ANGELIA AND GAN', 'Dewinta Angelia & Gan'), ('IR NELSON IDRIS SILABAN', 'Ir Nelson Idris Silaban'), ('PT HIT GLOBAL INDONESIA', 'PT Hit Global Indonesia'), ('PT KEDAI MIMPI TERWUJUD', 'Pt Kedai Mimpi Terwujud'), ('SRI REDJEKI BUDIMARTONO', 'Sri Redjeki Budimartono'), ('DILIP LACHMANDAS DODANI', 'Dilip Lachmandas Dodani'), ('MELANIA AMANDA MANDAGIE', 'Melania Amanda Mandagie'), ('DHARMAYANTO TIRTAWISATA', 'Dharmayanto Tirtawisata'), ('VICTOR ANDRIANTO WIBAWA', 'Victor Andrianto Wibawa'), ('TJIOK ARISTO SUNGKONO S', 'Tjiok Aristo Sungkono S'), ('BRAM RAHARDIAN MAHENDRA', 'Bram Rahardian Mahendra'), ('I MADE KOMPIANG GAUTAMA', 'I Made Kompiang Gautama'), ('BUDI SUTRISNO RACHMAT K', 'Budi Sutrisno Rachmat K'), ('THE INCUBATION FUND LTD', 'Wellington Mgt Company'), ('ARTICO SICAV ARTICO SEM', 'Christian Riccardo'), ('SANDS CAPITAL TEAM FUND', 'Christian Riccardo'), ('VCIM GLOBAL IMPACT FUND', 'Christian Riccardo'), ('TJIN HWA ALIAS SUGIANTO', 'Christian Riccardo'), ('SAMUGRAHA DITY A KUSUMA', 'Christian Riccardo'), ('DP KWI PT JUARA CAPITAL', 'Christian Riccardo'), ('CICILIA ANGELIA PRIBUDI', 'Christian Riccardo'), ('PT ARTISAN SURYA KREASI', 'Christian Riccardo'), ('TWU SUPERANNUATION FUND', 'Christian Riccardo'), ('HSBC CONTINENTAL EUROPE', 'Christian Riccardo'), ('DEBEKA AKTIEN ASIEN ESG', 'Christian Riccardo'), ('MANULIFE INVESTMENT MGT', 'Manulife Investment Mgt'), ('SCHRODER INVESTMENT MGT', 'Schroder Investment Mgt'), ('GOLDMAN SACHS ASSET MGT', 'Goldman Sachs Asset Mgt'), ('AXA ASSET MGT INDONESIA', 'AXA Asset Mgt Indonesia'), ('ASSET MGT ONE SINGAPORE', 'Asset Mgt One Singapore'), ('INSIGHT INVESTMENTS MGT', 'Insight Investments Mgt'), ('PREMIER MITON INVESTORS', 'Premier Miton Investors'), ('RATHBONE INVESTMENT MGT', 'Rathbone Investment Mgt'), ('FIRST SENTIER INVESTORS', 'First Sentier Investors'), ('RELIANCE INVESTMENT MGT', 'Reliance Investment Mgt'), ('MOUNT MURRAY INVESTMENT', 'Mount Murray Investment'), ('ASIA GROWTH CAPITAL MGT', 'Asia Growth Capital Mgt'), ('TREASURE FUND INVESTAMA', 'Treasure Fund Investama'), ('INA SEKURITAS INDONESIA', 'Ina Sekuritas Indonesia'), ('WESTSIDE INVESTMENT MGT', 'Westside Investment Mgt'), ('INVESCO ASIA TRUST PLC', 'Invesco'), ('BLACKROCK GLOBAL FUNDS', 'BlackRock Investment Mgt - Index'), ('BLACKROCK FUNDS I ICAV', 'BlackRock Investment Mgt - Index'), ('SPDR MSCI ACWI IMI ETF', 'State Street Investment Mgt'), ('BAILLIE GIFFORD AND CO', 'Baillie Gifford & Co'), ('LEGAL AND GENERAL ICAV', 'Legal & General Investment Mgt'), ('HYDRO ONE PENSION PLAN', 'Pyrford International'), ('FIDELITY FAR EAST FUND', 'Fidelity Investments'), ('SCHRODER DANA ISTIMEWA', 'Schroder Investment Mgt'), ('UBS LUX FUND SOLUTIONS', 'UBS Global Asset Mgt'), ('BNI LIFE INSURANCE SHF', 'BNI Asset Mgt'), ('PB ASEAN DIVIDEND FUND', 'Public Mutual'), ('WISDOMTREE ISSUER ICAV', 'Keyridge Asset Mgt'), ('INVESCO MARKETS II PLC', 'Invesco Capital Mgt'), ('MAYO CLINIC FOUNDATION', 'Lazard Asset Mgt'), ('PICTET POSITIVE CHANGE', 'Pictet Asset Mgt'), ('AXA BUMN TITANS EQUITY', 'AXA Asset Mgt Indonesia'), ('GOLDMAN SACHS FUNDS II', 'JO Hambro Capital Mgt'), ('PT AJ SEQUIS AAMI SQEF', 'Sequislife'), ('BNP PARIBAS FUNDS ASIA', 'BNP Paribas Asset Mgt'), ('FWD SPRINT EQUITY FUND', 'FWD Asset Mgt'), ('JONATHAN HANDOKO PUTRA', 'Jonathan Handoko Putra'), ('ARIA INVESTMENTS TRUST', 'Aria Investments'), ('DEKA RETURN LIGHT BETA', 'Deka Investment Mgt'), ('DANA PENSIUN PEGADAIAN', 'Dana Pensiun Pegadaian'), ('BOENNARIO NYOTOSETIADI', 'Boennario Nyotosetiadi'), ('RUDY SETIA WIBISONO SH', 'Rudy Setia Wibisono Sh'), ('LUSI INGAWATI HADITORO', 'Lusi Ingawati Haditoro'), ('ROSNA MARLYANI HARTOJO', 'Rosna Marlyani Hartojo'), ('PT BERKAT MITRA PATRIA', 'PT Berkat Mitra Patria'), ('WELLIJ ICHWAN NOORJONO', 'Wellij Ichwan Noorjono'), ('POLYANSKI CAKRA KESUMA', 'Polyanski Cakra Kesuma'), ('I GEDE ALIT ARDIMAYASA', 'I Gede Alit Ardimayasa'), ('AIG ASIA BALANCED FUND', 'PineBridge Investments'), ('LISA WIDIASARI ATMADJA', 'Lisa Widiasari Atmadja'), ('FILIPUS HUNADI SETIADI', 'Filipus Hunadi Setiadi'), ('L SURYAWIDJAJA DJUHADI', 'L Suryawidjaja Djuhadi'), ('RAHMAT SUMENGKAR ANHAR', 'Rahmat Sumengkar Anhar'), ('TJOANDI BUNGSU SANTOSO', 'Tjoandi Bungsu Santoso'), ('HENNY SARI DEWI IRAWAN', 'Henny Sari Dewi Irawan'), ('JOHN CORNELIUS YONATAN', 'John Cornelius Yonatan'), ('BAMBANG SETYAWAN DJOJO', 'Bambang Setyawan Djojo'), ('PT KURNIA LUWUK SEJATI', 'Pt Kurnia Luwuk Sejati'), ('BURHAN GUNAWAN KOSIDIN', 'Burhan Gunawan Kosidin'), ('TRI CAHYA WISNHU GROHO', 'Tri Cahya Wisnhu Groho'), ('EDDY SOERYANTO SOEGOTO', 'Eddy Soeryanto Soegoto'), ('RABIN INDRAJAD HATTARI', 'Rabin Indrajad Hattari'), ('ERNI KURNIAWATI AZISAH', 'Erni Kurniawati Azisah'), ('PT VERITAS INTI AMERTA', 'PT Veritas Inti Amerta'), ('ADRI ACHMAD DARADJATUN', 'Adri Achmad Daradjatun'), ('BUDI HARIONO LIMAN LIM', 'Budi Hariono Liman Lim'), ('AGUS RIWAYANTO DIPUTRA', 'Agus Riwayanto Diputra'), ('FEDERAL HOLD INDONESIA', 'Federal Hold Indonesia'), ('REINO RAMAPUTRA BARACK', 'Reino Ramaputra Barack'), ('ALBERT HARANDA LINCOLN', 'Albert Haranda Lincoln'), ('LISA SELIYANTI MULYONO', 'Lisa Seliyanti Mulyono'), ('SAMUEL HARRY MULANA SH', 'Samuel Harry Mulana Sh'), ('SIAT ALAN DARMASAPUTRA', 'Siat Alan Darmasaputra'), ('R HERDI SUHENDRI SAKTI', 'R Herdi Suhendri Sakti'), ('RUDY HARTONO BUDIYANTO', 'Rudy Hartono Budiyanto'), ('BENNY HANJAYA MARDJUKI', 'Benny Hanjaya Mardjuki'), ('HANAFI LIBRARI PRATAMA', 'Hanafi Librari Pratama'), ('ALBERT KRISTIANTO GANI', 'Albert Kristianto Gani'), ('JOANES JUSTIRA GUNAWAN', 'Joanes Justira Gunawan'), ('WELLY LOVENNUS SURIPTO', 'Christian Riccardo'), ('PT ZEZE AJUWEL TAMBANG', 'Christian Riccardo'), ('PT MANDALA MULTI CIPTA', 'Christian Riccardo'), ('KEVIN CHRISTIE MAMAHIT', 'Christian Riccardo'), ('PT AYODA CAPITAL GROUP', 'Christian Riccardo'), ('OSCAR HUTAGAOL DRS MBA', 'Christian Riccardo'), ('OVERLOOK PARTNERS FUND', 'Christian Riccardo'), ('AYUB MUHAMAD ASA DARMA', 'Christian Riccardo'), ('EASTSPRING INVESTMENTS', 'Eastspring Investments'), ('STOREBRAND INVESTMENTS', 'Storebrand Investments'), ('INVESCO ASSET MGT ASIA', 'Invesco Asset Mgt Asia'), ('ALLIANZ LIFE INSURANCE', 'Allianz Life Insurance'), ('WESPATH INVESTMENT MGT', 'Wespath Investment Mgt'), ('KLP KAPITALFORVALTNING', 'KLP Kapitalforvaltning'), ('TUGU PRATAMA INDONESIA', 'Tugu Pratama Indonesia'), ('RAIFFEISEN CAPITAL MGT', 'Raiffeisen Capital Mgt'), ('SAMSUNG INVESTMENT MGT', 'Samsung Investment Mgt'), ('TOKIO MARINE ASSET MGT', 'Tokio Marine Asset Mgt'), ('MIRAE ASSET SECURITIES', 'Mirae Asset Securities'), ('SS AND C ALPS ADVISORS', 'SS&C ALPS Advisors'), ('RAKUTEN INVESTMENT MGT', 'Rakuten Investment Mgt'), ('LOCALTAPIOLA ASSET MGT', 'LocalTapiola Asset Mgt'), ('ASHMORE INVESTMENT MGT', 'Ashmore Investment Mgt'), ('BOOSTER INVESTMENT MGT', 'Booster Investment Mgt'), ('OSMOSIS INVESTMENT MGT', 'Osmosis Investment Mgt'), ('WELLINGTON MGT COMPANY', 'Wellington Mgt Company'), ('ISHARES MSCI ACWI ETF', 'BlackRock Investment Mgt - Index'), ('BANK RAKYAT INDONESIA', 'Bank Rakyat Indonesia'), ('BLACKROCK UCITS FUNDS', 'BlackRock Investment Mgt - Index'), ('PRUFUND CAUTIOUS FUND', 'M&G Investments'), ('DP FREEPORT INDONESIA', 'BNP Paribas Asset Mgt'), ('MANDIRI INVESTA AKTIF', 'Mandiri Investasi'), ('RD BATAVIA DANA SAHAM', 'Batavia Prosperindo Asset Mgt'), ('DIMENSIONAL ETF TRUST', 'Dimensional Fund Advisors'), ('DIMENSIONAL FUNDS PLC', 'Dimensional Fund Advisors'), ('AIA HY PAR IDR EQUITY', 'AIA Indonesia'), ('PANIN NEO EQUITY FUND', 'Panin Asset Mgt'), ('AMERICAN CENTURY ICAV', 'American Century Investment Mgt'), ('HSBC GIF ASEAN EQUITY', 'HSBC Global Asset Mgt'), ('HSBC LIFE PAR WL FUND', 'HSBC Global Asset Mgt'), ('TRAM CONSUMPTION PLUS', 'Trimegah Asset Mgt'), ('MANULIFE PAR 3 EQUITY', 'Manulife Investment Mgt'), ('MANULIFE PAR 2 EQUITY', 'Manulife Investment Mgt'), ('PT WARUNA NUSASENTANA', 'Waruna Nusasentana'), ('ANDREAS BOBBY CIPUTRA', 'Andreas Bobby Ciputra'), ('KPD GREATLINK BALANCE', 'Great Eastern Life Assurance'), ('HARSHA EDWANA JOESOEF', 'Harsha Edwana Joesoef'), ('KURNIA LUKMAN GOUTAMA', 'Kurnia Lukman Goutama'), ('GENERALI EQUITY ALPHA', 'Generali Indonesia'), ('SEQUIS Q DYNAMIC FUND', 'Sequis Asset Mgt'), ('RAM TECHCHAND MANWANI', 'Ram Techchand Manwani'), ('PT SINAR INTI KORPORA', 'PT Sinar Inti Korpora'), ('WIDYANTI LISTYANI LIE', 'Widyanti Listyani Lie'), ('NICHOLAS JOSHUA WEEKS', 'Nicholas Joshua Weeks'), ('NN PARAPLUFONDS 1 N V', 'Goldman Sachs Asset Mgt'), ('TJIANG LIKSON CHANDRA', 'Tjiang Likson Chandra'), ('WILLY PURNAMA CHANDRA', 'Willy Purnama Chandra'), ('SULAIMAN ARIF ARIANTO', 'Sulaiman Arif Arianto'), ('ABDUL RAHMAN NASUTION', 'Abdul Rahman Nasution'), ('WIDYA KASUMA LAWRENZI', 'Widya Kasuma Lawrenzi'), ('ARGA MAHANANA NUGRAHA', 'Arga Mahanana Nugraha'), ('GEMA MERDEKA GOEYARDI', 'Gema Merdeka Goeyardi'), ('EKA SETYA ADRIANTO SE', 'Eka Setya Adrianto Se'), ('LEVINA LAWRENT KUSUMA', 'Levina Lawrent Kusuma'), ('DJOKOSANTOSO MOELJONO', 'Djokosantoso Moeljono'), ('AMUNDI INDONESIA FUND', 'Amundi Singapore'), ('KENNY ANTONIO RULLY S', 'Kenny Antonio Rully S'), ('TAN GUNAWAN TANWIJAYA', 'Tan Gunawan Tanwijaya'), ('RITA LACHMAN BHARWANI', 'Rita Lachman Bharwani'), ('IWAN SANTOSA KUSWANTO', 'Iwan Santosa Kuswanto'), ('PINEBRIDGE ASIAN FUND', 'PineBridge Investments'), ('RONNY HERMAWAN ANDOYO', 'Ronny Hermawan Andoyo'), ('AUGUSTINA SIMANJUNTAK', 'Augustina Simanjuntak'), ('REGINA SARAH TAMBUNAN', 'Regina Sarah Tambunan'), ('JAUW DHANNY KURNIAWAN', 'Jauw Dhanny Kurniawan'), ('MUHAMMAD CANDRA UTAMA', 'Muhammad Candra Utama'), ('LEONARD AGUS SETIAWAN', 'Leonard Agus Setiawan'), ('DJOHAN TEGUH SUGIANTO', 'Djohan Teguh Sugianto'), ('PT METROCOM INDONESIA', 'PT Metrocom Indonesia'), ('TORAS S PANGGABEAN SH', 'Toras S Panggabean Sh'), ('YOHANES FREDDY WIJAYA', 'Yohanes Freddy Wijaya'), ('SUDIRMAN SIMANGUNSONG', 'Sudirman Simangunsong'), ('ANDREE SEMIKAR WIJAYA', 'Andree Semikar Wijaya'), ('ANTONIUS JUSUF JOHARI', 'Antonius Jusuf Johari'), ('STEVEN WIJAYA GUNAWAN', 'Steven Wijaya Gunawan'), ('EDWARD THEJASURYA LIM', 'Edward Thejasurya Lim'), ('PT AGUNG BERSATU JAYA', 'Pt Agung Bersatu Jaya'), ('ROY SETIAWAN PRASETYA', 'Roy Setiawan Prasetya'), ('KRIS JACOBUS SILFANUS', 'Kris Jacobus Silfanus'), ('HATTA WENTIO BUDIANTO', 'Hatta Wentio Budianto'), ('CHRISTINA TANUWIDJAJA', 'Christina Tanuwidjaja'), ('MUHAMMAD NADHIF TAQWA', 'Muhammad Nadhif Taqwa'), ('IVAN ALAMSYAH SIREGAR', 'Ivan Alamsyah Siregar'), ('ANTONIUS AGUS SUSANTO', 'Antonius Agus Susanto'), ('IRAWATI DARMAWAN BCHK', 'Irawati Darmawan Bchk'), ('I KADEK SARENDRA JAYA', 'I Kadek Sarendra Jaya'), ('PAULA NANIWATI DASUKI', 'Paula Naniwati Dasuki'), ('JILLY VANESSI LOPRANG', 'Jilly Vanessi Loprang'), ('INDRA KRISMAYADI S IK', 'Indra Krismayadi S Ik'), ('MUHAMMAD PURI ANDAMAS', 'Muhammad Puri Andamas'), ('ONG SANG SANG BUDIMAN', 'Ong Sang Sang Budiman'), ('PITER GOENAWAN HENOEK', 'Piter Goenawan Henoek'), ('HUSDI STEVEN TIORONTO', 'Husdi Steven Tioronto'), ('CANDRA AGUSTIAN DATAM', 'Candra Agustian Datam'), ('PANDRI PRABONO MOELYO', 'Pandri Prabono Moelyo'), ('ASURANSI CENTRAL ASIA', 'A.J. Central Asia Raya'), ('NICOLAUS T B HARJANTO', 'Nicolaus T B Harjanto'), ('TRIONO JULIARSO DAWIS', 'Triono Juliarso Dawis'), ('SAMSUNG SECURITIES CO', 'Samsung Securities'), ('VANDERBILT UNIVERSITY', 'Christian Riccardo'), ('BUDI YANTO P HUTAGAOL', 'Christian Riccardo'), ('SABAR GANDA L SITORUS', 'Christian Riccardo'), ('IG HADIANTO SUHARTONO', 'Christian Riccardo'), ('PT INDO MOTOR LESTARI', 'Christian Riccardo'), ('HSBC GLOBAL ASSET MGT', 'HSBC Global Asset Mgt'), ('PYRFORD INTERNATIONAL', 'Pyrford International'), ('BNP PARIBAS ASSET MGT', 'BNP Paribas Asset Mgt'), ('NEWTON INVESTMENT MGT', 'Newton Investment Mgt'), ('SUCORINVEST ASSET MGT', 'SucorInvest Asset Mgt'), ('MACKENZIE INVESTMENTS', 'Mackenzie Investments'), ('PRUSIK INVESTMENT MGT', 'Prusik Investment Mgt'), ('BLACKROCK JAPAN INDEX', 'BlackRock Japan - Index'), ('LION GLOBAL INVESTORS', 'Lion Global Investors'), ('JO HAMBRO CAPITAL MGT', 'JO Hambro Capital Mgt'), ('NORDEA INVESTMENT MGT', 'Nordea Investment Mgt'), ('JENSEN INVESTMENT MGT', 'Jensen Investment Mgt'), ('INTECH INVESTMENT MGT', 'INTECH Investment Mgt'), ('PERPETUAL INVESTMENTS', 'Perpetual Investments'), ('C WORLDWIDE ASSET MGT', 'C WorldWide Asset Mgt'), ('A J CENTRAL ASIA RAYA', 'A.J. Central Asia Raya'), ('BPJS KETENAGAKERJAAN', 'BPJS Ketenagakerjaan'), ('INVESCO PACIFIC FUND', 'Invesco'), ('AQR LEELANAU FUND LP', 'AQR Capital Mgt'), ('SPP GLOBAL SOLUTIONS', 'Storebrand Investments'), ('SPP AKTIEFOND GLOBAL', 'Storebrand Investments'), ('MANULIFE GLOBAL FUND', 'Manulife Investment Mgt Indonesia'), ('INDORAMA EQUITY FUND', 'Manulife Investment Mgt Indonesia'), ('AVA GROWTH PLUS FUND', 'Schroder Investment Mgt'), ('UBS AG LONDON BRANCH', 'UBS Securities'), ('IRISH LIFE ASSURANCE', 'Keyridge Asset Mgt'), ('JASA RAHARJA PERSERO', 'Jasa Raharja (Persero)'), ('STATE OF CONNECTICUT', 'Schroder Investment Mgt'), ('EQ EMERGING DIVIDEND', 'eQ Asset Mgt'), ('GE LIFE SLIF NON PAR', 'Lion Global Investors'), ('ALLIANZGI FONDS PTV2', 'Allianz Global Investors'), ('RICO RIZAL BUDIDARMO', 'Rico Rizal Budidarmo'), ('PICTET ET CIE GENEVA', 'Pictet & Cie'), ('YOSEPH NEIL SANMIKHA', 'Yoseph Neil Sanmikha'), ('DEKA INVESTMENT GMBH', 'Deka Investment Mgt'), ('ANDRE RIZKY RAMADHAN', 'Andre Rizky Ramadhan'), ('ANDRI WIJONO SUTIONO', 'Andri Wijono Sutiono'), ('BENNY PONTIAN MUSLIM', 'Benny Pontian Muslim'), ('JEANNE NATALIE PUTRI', 'Jeanne Natalie Putri'), ('AGGREGATE VALUE FUND', 'Aggregate Asset Mgt'), ('SUGIANTO ADI SUWIRYO', 'Sugianto Adi Suwiryo'), ('ADRIAN PUTRA BRASALI', 'Adrian Putra Brasali'), ('HEE RENALD MAMANGKEY', 'Hee Renald Mamangkey'), ('PRUDENTIAL HONG KONG', 'Prudential Hong Kong'), ('MARGARETHA PRATIWI S', 'Margaretha Pratiwi S'), ('PERSEK DAPENMA PAMSI', 'Persek Dapenma Pamsi'), ('VIVIANA DYAH AYU R K', 'Viviana Dyah Ayu R K'), ('BAMBANG SOERJADI ALI', 'Bambang Soerjadi Ali'), ('YOSSY ANDREW GIRSANG', 'Yossy Andrew Girsang'), ('BAMBANG SUSILO HALIM', 'Bambang Susilo Halim'), ('HENDRIK BIMA HERAWAN', 'Hendrik Bima Herawan'), ('DWIJANTO GONDOKUSUMO', 'Dwijanto Gondokusumo'), ('LAUDHI ADRIAN SUSILO', 'Laudhi Adrian Susilo'), ('PT INOCREA INDONESIA', 'PT Inocrea Indonesia'), ('TEDDY KURNIA TJANDRA', 'Teddy Kurnia Tjandra'), ('ANTON HARTONO ANSARI', 'Anton Hartono Ansari'), ('RISMA ERAWATI SINAGA', 'Risma Erawati Sinaga'), ('ANTHONIUS HEROE W SH', 'Anthonius Heroe W Sh'), ('OEI SETIAWAN WINARTO', 'Oei Setiawan Winarto'), ('MI FONDS K18 KAE SPK', 'Metzler Asset Mgt'), ('CEPI JAMALUDIN MALIK', 'Cepi Jamaludin Malik'), ('DJOHAN EMIR SETIJOSO', 'Djohan Emir Setijoso'), ('DIOCESE OF ARLINGTON', 'Federated Hermes'), ('HANSEL SOSRO SAPUTRO', 'Hansel Sosro Saputro'), ('DAVIN ALDO ANDRIANTO', 'Davin Aldo Andrianto'), ('DWI HENDRY NATHANIEL', 'Dwi Hendry Nathaniel'), ('PT ASURANSI BRI LIFE', 'PT Asuransi BRI Life'), ('RATNA SARI SUHARTONO', 'Ratna Sari Suhartono'), ('MUKIT HENDRAYATNO ST', 'Mukit Hendrayatno ST'), ('DRS H ARMED WJAYA MH', 'Drs H Armed Wjaya Mh'), ('SUWALIM HADI PRATAMA', 'Suwalim Hadi Pratama'), ('SUKENGSI HARI SLAMET', 'Sukengsi Hari Slamet'), ('ARIEF DARU WIBAWANTO', 'Arief Daru Wibawanto'), ('SARANAKELOLA INVESTA', 'Saranakelola Investa'), ('ALEX PURNADI CHANDRA', 'Alex Purnadi Chandra'), ('RIEDNO GRAAL TALIAWO', 'Riedno Graal Taliawo'), ('LILI MULYADI SUTANTO', 'Lili Mulyadi Sutanto'), ('ANIMA PRIMA FUNDS PL', 'Anima'), ('JONI INDRA SE AKT MM', 'Joni Indra Se Akt Mm'), ('ELLY IVONNE SITOMPUL', 'Elly Ivonne Sitompul'), ('CORNELLIUS HENRY KHO', 'Cornellius Henry Kho'), ('TANTIO BAGUS CAHYONO', 'Tantio Bagus Cahyono'), ('PAULUS RAGIL SAPUTRO', 'Paulus Ragil Saputro'), ('CV ULTRA ENGINEERING', 'CV Ultra Engineering'), ('ROBERT PETER RAJIMIN', 'Robert Peter Rajimin'), ('OMAR HIDAJAT WANGSIT', 'Omar Hidajat Wangsit'), ('WILSON CAHYADI LIAUW', 'Wilson Cahyadi Liauw'), ('PT MULYA HUSADA JAYA', 'PT Mulya Husada Jaya'), ('KUNARDI SASTRAWIJAYA', 'Kunardi Sastrawijaya'), ('SUSANTO GONDO WIJAYA', 'Susanto Gondo Wijaya'), ('ANNA MARIA TJIADARMA', 'Anna Maria Tjiadarma'), ('I NYOMAN SUGIRI YASA', 'I Nyoman Sugiri Yasa'), ('ERMIRA KURNIA SUSILO', 'Ermira Kurnia Susilo'), ('RUDY GUNAWAN WIDJAJA', 'Rudy Gunawan Widjaja'), ('AMI NOERDHINI MARLIM', 'Ami Noerdhini Marlim'), ('FJERNOSTEN INDEKS KL', 'Danske Capital Mgt'), ('LISTIARINI DEWAJANTI', 'Listiarini Dewajanti'), ('CAROLUS ALDO WINDURA', 'Carolus Aldo Windura'), ('MNC SAHAM DANA AKTIF', 'MNC Asset Mgt'), ('PHILIP SURYA DJUSMAN', 'Philip Surya Djusman'), ('ZUL AKHYAR MARTABAYA', 'Zul Akhyar Martabaya'), ('G P RONALD SIHOMBING', 'Christian Riccardo'), ('BNP PARIBAS OBAM N V', 'Christian Riccardo'), ('BEN JASEN WANGSAJAYA', 'Christian Riccardo'), ('METY JUMIATI AND SPD', 'Christian Riccardo'), ('IR ROBERT NAPITUPULU', 'Christian Riccardo'), ('ADI DARMAWAN PRANATA', 'Christian Riccardo'), ('PLATINUM ARROW TRUST', 'Christian Riccardo'), ('PZENA INVESTMENT MGT', 'Pzena Investment Mgt'), ('FIDELITY INVESTMENTS', 'Fidelity Investments'), ('UBS GLOBAL ASSET MGT', 'UBS Global Asset Mgt'), ('UNIVERSAL INVESTMENT', 'Universal Investment'), ('ENSIGN PEAK ADVISORS', 'Ensign Peak Advisors'), ('RBC GLOBAL ASSET MGT', 'RBC Global Asset Mgt'), ('ASURANSI WAHANA TATA', 'Asuransi Wahana Tata'), ('DELTA WIBAWA BERSAMA', 'Delta Wibawa Bersama'), ('VINVA INVESTMENT MGT', 'Vinva Investment Mgt'), ('CHUBB LIFE INSURANCE', 'Chubb Life Insurance'), ('KRANE FUNDS ADVISORS', 'Krane Funds Advisors'), ('FIRST TRUST ADVISORS', 'First Trust Advisors'), ('CAUSEWAY CAPITAL MGT', 'Causeway Capital Mgt'), ('OFI INVEST ASSET MGT', 'OFI Invest Asset Mgt'), ('KBI GLOBAL INVESTORS', 'KBI Global Investors'), ('MASOU INVESTMENT MGT', 'Masou Investment Mgt'), ('YKP BRI THT TRADING', 'BRI Investment Mgt'), ('JPMORGAN SECURITIES', 'JPMorgan Securities'), ('FUTURELEGACY 6 FUND', 'Newton Investment Mgt'), ('FUTURELEGACY 7 FUND', 'Newton Investment Mgt'), ('FUTURELEGACY 5 FUND', 'Newton Investment Mgt'), ('FUTURELEGACY 4 FUND', 'Newton Investment Mgt'), ('FUTURELEGACY 3 FUND', 'Newton Investment Mgt'), ('GREENFIELDS CAPITAL', 'Mellon'), ('SWISS NATIONAL BANK', 'Swiss National Bank'), ('ARERO DER WELTFONDS', 'DWS Investments'), ('BIL UNIVERSAL FONDS', 'Universal Investment'), ('WESPATH FUNDS TRUST', 'Wespath Investment Mgt'), ('MATTHEWS ASIA FUNDS', 'Matthews International Capital Mgt'), ('SPW TACTICAL FUND 1', 'Schroder Investment Mgt'), ('SPW TACTICAL FUND 2', 'Schroder Investment Mgt'), ('PICTET REGENERATION', 'Pictet Asset Mgt'), ('GOLDMAN SACHS FUNDS', 'Goldman Sachs Asset Mgt'), ('LOMBARD ODIER FUNDS', 'Lombard Odier Darier Hentsch & Cie'), ('ALLIANZ CGI FONDS 1', 'Allianz Global Investors'), ('ALLIANZ PV WS FONDS', 'Allianz Global Investors'), ('ALLIANZ VGI 1 FONDS', 'Allianz Global Investors'), ('PT AVRIST ASSURANCE', 'Avrist Asset Mgt'), ('AMUNDI AUSTRIA GMBH', 'Amundi Asset Mgt'), ('KARYA DUMOGI RAHAYU', 'Karya Dumogi Rahayu'), ('MANULIFE PAR EQUITY', 'Manulife Investment Mgt'), ('INDRAJONO SANKAWANG', 'Indrajono Sankawang'), ('SUWIGNYO BUDIMAN IR', 'Suwignyo Budiman Ir'), ('SANDI SUWITO SUSILO', 'Sandi Suwito Susilo'), ('HENDRAWAN TRANGGANA', 'Hendrawan Tranggana'), ('JONATHAN ANDERSEN P', 'Jonathan Andersen P'), ('SOEGIHARTO SOEGONDO', 'Soegiharto Soegondo'), ('IR I PUTU ABDI ANOM', 'IR I Putu Abdi Anom'), ('DARWIN SHP MANURUNG', 'Darwin Shp Manurung'), ('ANDY WIJAYA CHANDRA', 'Andy Wijaya Chandra'), ('DONNY CAHYADI FOENG', 'Donny Cahyadi Foeng'), ('SIMON MARK DELICATO', 'Simon Mark Delicato'), ('A M NOVA CHRISTIANA', 'A M Nova Christiana'), ('PT INDOTARA PERSADA', 'PT Indotara Persada'), ('RATNAWATI TANIARDJA', 'Ratnawati Taniardja'), ('ALEXANDER FULBERTUS', 'Alexander Fulbertus'), ('VINSEN ALFREDO ARYA', 'Vinsen Alfredo Arya'), ('CHRISTIAN ANGKY ANG', 'Christian Angky Ang'), ('WAHID BROTO KUNCORO', 'Wahid Broto Kuncoro'), ('HARI SIAGA AMIJARSO', 'Hari Siaga Amijarso'), ('HENNY WIDYAWATI OEI', 'Henny Widyawati Oei'), ('ROMMY HARTONO THEOS', 'Rommy Hartono Theos'), ('MARINI RYANI IRAWAN', 'Marini Ryani Irawan'), ('TJINDARSAH MINTARNO', 'Tjindarsah Mintarno'), ('PRAWIRAYUDHA HALIEM', 'Prawirayudha Haliem'), ('M JAROT EKO WINARNO', 'M Jarot Eko Winarno'), ('YENNY HAMIDAH KOEAN', 'Yenny Hamidah Koean'), ('RANDY MULYADI JATIM', 'Randy Mulyadi Jatim'), ('OEN MARIA CHRISTINA', 'Oen Maria Christina'), ('ARIEF TJATUR WIDODO', 'Arief Tjatur Widodo'), ('MUTB300000069 STOCK', 'Mitsubishi UFJ Trust & Banking Corporation'), ('MUTB300000120 STOCK', 'Mitsubishi UFJ Trust & Banking Corporation'), ('DAVID FERNANDO AUDY', 'David Fernando Audy'), ('EY SARWONO SOEDARTO', 'Ey Sarwono Soedarto'), ('LIE PETER LEEANSYAH', 'Lie Peter Leeansyah'), ('MOH IRFAN SEPTIAWAN', 'Moh Irfan Septiawan'), ('THOENG CONRAD ITUNG', 'Thoeng Conrad Itung'), ('ELIA DJONLIE NAULIE', 'Elia Djonlie Naulie'), ('REYN ALBERT BONARDO', 'Reyn Albert Bonardo'), ('I DEWA PUTU PRAYOGA', 'I Dewa Putu Prayoga'), ('K T ISMAIL DJALIMAN', 'K T Ismail Djaliman'), ('FATKUL MUBIN AND ST', 'Fatkul Mubin & ST'), ('JECKY EDISON LENENG', 'Jecky Edison Leneng'), ('WILLIARTO KUOSASTRA', 'Williarto Kuosastra'), ('TEGUH WAHJU NUGRAHA', 'Teguh Wahju Nugraha'), ('PUTU DENIK SUASTIKA', 'Putu Denik Suastika'), ('RISWAN DJAYASAPUTRA', 'Riswan Djayasaputra'), ('NABILA ILMI HAKIMAH', 'Nabila Ilmi Hakimah'), ('J G NARA NILANDAROE', 'J. G. Nara Nilandaroe'), ('JANE STREET CAPITAL', 'Jane Street Capital'), ('AGUS DWI HANDAYA SE', 'Christian Riccardo'), ('DIDI SATRIA SOEBAWA', 'Christian Riccardo'), ('ISKANDAR MUDA KAMIL', 'Christian Riccardo'), ('CAHYO HARI PURWANTO', 'Christian Riccardo'), ('LIOE KARNEN LUCKY H', 'Christian Riccardo'), ('HUBERT HARI SOEJONO', 'Christian Riccardo'), ('ARIE PRIANGGA PUTRA', 'Christian Riccardo'), ('LIONG INDRA GUNAWAN', 'Christian Riccardo'), ('WILLIAM CHRISTOPHER', 'Christian Riccardo'), ('AMCUMULATIVE GROWTH', 'Christian Riccardo'), ('AMPRS MODERATE FUND', 'Christian Riccardo'), ('ARGA INVESTMENT MGT', 'ARGA Investment Mgt'), ('M AND G INVESTMENTS', 'M&G Investments'), ('ARROWSTREET CAPITAL', 'Arrowstreet Capital'), ('RESEARCH AFFILIATES', 'Research Affiliates'), ('TIAA INVESTMENT MGT', 'TIAA Investment Mgt'), ('MORGAN STANLEY ASIA', 'Morgan Stanley Asia'), ('VICTORY CAPITAL MGT', 'Victory Capital Mgt'), ('INVESCO CAPITAL MGT', 'Invesco Capital Mgt'), ('RUSSELL INVESTMENTS', 'Russell Investments'), ('HARDING LOEVNER MGT', 'Harding Loevner Mgt'), ('PINNACLE INVESTMENT', 'Pinnacle Investment'), ('DEKA INVESTMENT MGT', 'Deka Investment Mgt'), ('PRINCIPAL ASSET MGT', 'Principal Asset Mgt'), ('POLARIS CAPITAL MGT', 'Polaris Capital Mgt'), ('AGGREGATE ASSET MGT', 'Aggregate Asset Mgt'), ('MACQUARIE ASSET MGT', 'Macquarie Asset Mgt'), ('OPSEU PENSION TRUST', 'OPSEU Pension Trust'), ('VOYA INVESTMENT MGT', 'Voya Investment Mgt'), ('CI GLOBAL ASSET MGT', 'CI Global Asset Mgt'), ('BANK OF NEW ZEALAND', 'Bank of New Zealand'), ('CIPTADANA ASSET MGT', 'Ciptadana Asset Mgt'), ('VALBURY CAPITAL MGT', 'Valbury Capital Mgt'), ('CLARIVEST ASSET MGT', 'ClariVest Asset Mgt'), ('PT TASPEN ASURANSI', 'Taspen'), ('T ROWE PRICE FUNDS', 'T Rowe Price'), ('YKP BRI P4 TRADING', 'BRI Investment Mgt'), ('YKP BRI P3 TRADING', 'BRI Investment Mgt'), ('DPLK AIA FINANCIAL', 'AIA Indonesia'), ('PT INDRA INVESTAMA', 'SucorInvest Asset Mgt'), ('AGGRESSIVE EKUITAS', 'Sun Life Financial'), ('GUNAWAN TJOKROHADI', 'Gunawan Tjokrohadi'), ('GMS DISCOVERY FUND', 'Artisan Partners'), ('STICHTING PENSIOEN', 'MN Services'), ('HENDRY CHRISTIAN W', 'Hendry Christian ,W'), ('NOW PENSIONS TRUST', 'Cardano'), ('DWIDJAJA SETIABUDI', 'Dwidjaja Setiabudi'), ('TJHIN SYLVESTER TH', 'Tjhin Sylvester Th'), ('CHANDRA ADISUSANTO', 'Chandra Adisusanto'), ('LIONG BUDI SUHARJA', 'Liong Budi Suharja'), ('SATRIA AMI PUTRA A', 'Satria Ami Putra A'), ('WIDAYANTI DARMAWAN', 'Widayanti Darmawan'), ('CAPITAL ASEAN FUND', 'Capital Investment Trust Corporation'), ('PT DIMA INVESTINDO', 'PT Dima Investindo'), ('FX BOEDI SOEBIANTO', 'Fx Boedi Soebianto'), ('BAMBANG HERUAWAN H', 'Bambang Heruawan H'), ('SIS APIK WIJAYANTO', 'Sis Apik Wijayanto'), ('KURNIA JAYA SUTANA', 'Kurnia Jaya Sutana'), ('HENGKY KARTAWINATA', 'Hengky Kartawinata'), ('YOSEPH MARIO YAPPY', 'Yoseph Mario Yappy'), ('EKO HANDONO TJAHJO', 'Eko Handono Tjahjo'), ('AXEL ILHAM GUNARDI', 'Axel Ilham Gunardi'), ('FANDRAWATI WIDJOJO', 'Fandrawati Widjojo'), ('ADI GUNAWAN TANZIL', 'Adi Gunawan Tanzil'), ('IGN SUGIAN SINANTO', 'Ign Sugian Sinanto'), ('PANGKAL MULTIKARYA', 'Pangkal Multikarya'), ('KT ISMAIL DJALIMAN', 'KT Ismail Djaliman'), ('FATHA BUDI ANUGRAH', 'Fatha Budi Anugrah'), ('ANTARES HANAEDI II', 'Antares Hanaedi II'), ('SENG BENNY SUSANTO', 'Seng Benny Susanto'), ('EDDY KOKO YOENAEDY', 'Eddy Koko Yoenaedy'), ('YANDRAMIN HALIM IR', 'Yandramin Halim IR'), ('ANGGA INDO SUCIPTO', 'Angga Indo Sucipto'), ('ANTHONIUS S TJAHJA', 'Anthonius S Tjahja'), ('WAHYUDI LUKMANTARA', 'Wahyudi Lukmantara'), ('GUNAWAN SETIONO IR', 'Gunawan Setiono Ir'), ('RIDWAN SOMADIBRATA', 'Ridwan Somadibrata'), ('PRIMARTONO GUNAWAN', 'Primartono Gunawan'), ('ILYAS ABDUR RAHMAN', 'Ilyas Abdur Rahman'), ('NICO GUNNAR HUTOMO', 'Nico Gunnar Hutomo'), ('IGNATIUS YANSEN NG', 'Ignatius Yansen Ng'), ('JAHJA SETIAATMADJA', 'Jahja Setiaatmadja'), ('DARWIN HARIANJA ST', 'Darwin Harianja St'), ('FERINA TEJA WIJAYA', 'Ferina Teja Wijaya'), ('LISTIJANTI HIDAYAT', 'Listijanti Hidayat'), ('CAHYADI TEJAKUSUMA', 'Cahyadi Tejakusuma'), ('RONALD HARTONO TAN', 'Ronald Hartono Tan'), ('KB KITMC ASEAN EQT', 'Korea Investment Trust Mgt'), ('A KUNTA WIDYATMAKA', 'A Kunta Widyatmaka'), ('WONO BUDI TJAHYONO', 'Wono Budi Tjahyono'), ('CHRISTIAN RICCARDO', 'Christian Riccardo'), ('LINDRAWATI WIDJOJO', 'Christian Riccardo'), ('MUH GANI IRWANSYAH', 'Christian Riccardo'), ('LIE GUNAWAN SUTEJA', 'Christian Riccardo'), ('ANTHONY HUGO BARKI', 'Christian Riccardo'), ('APG ASSET MGT ASIA', 'APG Asset Mgt Asia'), ('JPMORGAN ASSET MGT', 'JPMorgan Asset Mgt'), ('BRI INVESTMENT MGT', 'BRI Investment Mgt'), ('FIL INVESTMENT MGT', 'FIL Investment Mgt'), ('SYAILENDRA CAPITAL', 'Syailendra Capital'), ('TRIMEGAH ASSET MGT', 'Trimegah Asset Mgt'), ('FIDEURAM ASSET MGT', 'Fideuram Asset Mgt'), ('KEYRIDGE ASSET MGT', 'Keyridge Asset Mgt'), ('GUINNESS ASSET MGT', 'Guinness Asset Mgt'), ('SUN LIFE FINANCIAL', 'Sun Life Financial'), ('WARUNA NUSASENTANA', 'Waruna Nusasentana'), ('NYKREDIT ASSET MGT', 'Nykredit Asset Mgt'), ('GENERALI INDONESIA', 'Generali Indonesia'), ('SAUDI CENTRAL BANK', 'Saudi Central Bank'), ('METROPOLIS CAPITAL', 'Metropolis Capital'), ('DANSKE CAPITAL MGT', 'Danske Capital Mgt'), ('RHUMBLINE ADVISERS', 'RhumbLine Advisers'), ('PNM INVESTMENT MGT', 'PNM Investment Mgt'), ('PANAGORA ASSET MGT', 'PanAgora Asset Mgt'), ('FULLERTON FUND MGT', 'Fullerton Fund Mgt'), ('PROSPERA ASSET MGT', 'Prospera Asset Mgt'), ('VONTOBEL ASSET MGT', 'Vontobel Asset Mgt'), ('GAM INVESTMENT MGT', 'GAM Investment Mgt'), ('AZIMUT INVESTMENTS', 'Azimut Investments'), ('OSO INVESTMENT MGT', 'Oso Investment Mgt'), ('TUTTLE CAPITAL MGT', 'Tuttle Capital Mgt'), ('MARATHON ASSET MGT', 'Marathon Asset Mgt'), ('INDOASIA ASSET MGT', 'Indoasia Asset Mgt'), ('SINARMAS ASSET MGT', 'Sinarmas Asset Mgt'), ('DANAKITA INVESTAMA', 'Danakita Investama'), ('SWISS ASIA CAPITAL', 'Swiss Asia Capital'), ('LIF INVESTMENT MGT', 'LiF Investment Mgt'), ('SAMSUNG SECURITIES', 'Samsung Securities')]

def match_benowner_report_group(cleaned_name):
    key = normalize_match_text(cleaned_name)
    if not key:
        return None, ''
    exact = BENOWNER_UNIQUE_GROUP_MAP.get(key)
    if exact:
        return exact, key
    # Longest unambiguous report phrase wins. Short aliases are intentionally
    # excluded to avoid recreating the previous broad overgrouping.
    padded_key = f' {key} '
    hits = [(phrase, group) for phrase, group in BENOWNER_PHRASE_GROUPS
            if f' {phrase} ' in padded_key]
    if not hits:
        return None, ''
    best_len = len(hits[0][0])
    best_groups = {group for phrase, group in hits if len(phrase) == best_len}
    if len(best_groups) != 1:
        return None, ''
    return next(iter(best_groups)), hits[0][0]

class ReportGroupingClassifier(InvestorClassifier):
    """Use the uploaded BenOwners hierarchy as the institutional source of truth.

    Priority:
    1. Existing private/undisclosed account rules.
    2. Existing individual SID rules (IDD/IDF).
    3. Unambiguous parent/child grouping from the BenOwners workbook.
    4. Conservative direct private-entity handling from the existing code.
    5. Otherwise ML_REQUIRED rather than forcing a broad parent group.
    """
    def decide(self, sid, investor_name, country=None):
        c = self.make_context(sid, investor_name, country)
        if not c.sid or c.sid == 'NAN':
            return ClassificationDecision(UNRESOLVED, 'invalid_input', 'missing SID')
        if not c.name:
            return ClassificationDecision(UNRESOLVED, 'invalid_input', 'missing investor name')

        # Preserve the reviewed private-account logic.
        private_decision = self._rule_undisclosed_account(c)
        if private_decision is not None:
            return private_decision
        private_decision = self._rule_undisclosed_intermediary(c)
        if private_decision is not None:
            return private_decision

        # Preserve individual investors exactly as handled by the original code.
        individual_decision = self._rule_domestic_individual(c)
        if individual_decision is not None:
            return individual_decision

        group, evidence = match_benowner_report_group(c.name)
        if group:
            return ClassificationDecision(group, 'benowners_report_group', evidence)

        # Keep conservative named-entity behavior for non-fund private entities.
        direct = self._rule_direct_entity_fallback(c)
        if direct is not None:
            return direct

        return ClassificationDecision(UNRESOLVED, 'benowners_no_unambiguous_match', c.name)

DEFAULT_CLASSIFIER = ReportGroupingClassifier()

def classify_fund_manager(sid, investor_name, country=None):
    return DEFAULT_CLASSIFIER.classify(sid, investor_name, country)


"""## 6. Accuracy evaluation against labeled test data

This evaluation uses normalized exact-match accuracy. It ignores capitalization, leading/trailing spaces, repeated whitespace, and minor Unicode formatting differences, but it does **not** use fuzzy matching. This keeps the metric strict while avoiding false errors caused only by formatting.
"""

# Colab paths: upload BBRI_combined.xlsx before running this cell.
INPUT_XLSX = "/content/BBRI_combined.xlsx"
OUTPUT_XLSX = "/content/classified_test_output_manager_first.xlsx"
WRONG_XLSX = "/content/wrong_outputs.xlsx"

# For local/Jupyter execution, automatically use the current folder if needed.
from pathlib import Path
if not Path(INPUT_XLSX).exists() and Path("BBRI_combined.xlsx").exists():
    INPUT_XLSX = "BBRI_combined.xlsx"
    OUTPUT_XLSX = "classified_test_output_manager_first.xlsx"
    WRONG_XLSX = "wrong_outputs.xlsx"

print("Test file:", INPUT_XLSX)

def normalize_label(value):
    """Normalize labels for strict but formatting-tolerant comparison."""
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    value = value.strip().upper()
    value = re.sub(r"\s+", " ", value)
    return value


def evaluate_predictions(classified_df, schema):
    """Add correctness columns and return overall evaluation metrics."""
    if not schema.true_label:
        raise ValueError(
            "No true-label column found. Expected one of: "
            + ", ".join(COLUMN_ALIASES["true_label"])
        )

    result = classified_df.copy()
    result["True_Label_Normalized"] = result[schema.true_label].map(normalize_label)
    result["Prediction_Normalized"] = result["Fund_Manager_Guess"].map(normalize_label)

    labeled_mask = result["True_Label_Normalized"].ne("")
    resolved_mask = ~result["Needs_ML"]
    comparable_mask = labeled_mask & resolved_mask

    result["Is_Correct"] = False
    result.loc[labeled_mask, "Is_Correct"] = (
        result.loc[labeled_mask, "True_Label_Normalized"]
        == result.loc[labeled_mask, "Prediction_Normalized"]
    )

    labeled_rows = int(labeled_mask.sum())
    resolved_labeled_rows = int(comparable_mask.sum())
    correct_all = int(result.loc[labeled_mask, "Is_Correct"].sum())
    correct_resolved = int(result.loc[comparable_mask, "Is_Correct"].sum())
    wrong_all = labeled_rows - correct_all

    metrics = {
        "Total rows": len(result),
        "Labeled rows": labeled_rows,
        "Resolved rows": int(resolved_mask.sum()),
        "Unresolved rows": int((~resolved_mask).sum()),
        "Coverage (%)": resolved_mask.mean() * 100 if len(result) else 0.0,
        "Overall accuracy (%)": correct_all / labeled_rows * 100 if labeled_rows else float("nan"),
        "Resolved-only accuracy (%)": correct_resolved / resolved_labeled_rows * 100 if resolved_labeled_rows else float("nan"),
        "Correct predictions": correct_all,
        "Wrong predictions": wrong_all,
    }
    return result, pd.DataFrame({"Metric": metrics.keys(), "Value": metrics.values()})

# Run the classifier and measure accuracy.
classified_df, schema = run_pipeline(input_excel=INPUT_XLSX,output_xlsx=OUTPUT_XLSX,include_trace=True,)

# Apply clean_name to Nama Investor
classified_df["Fund Name Guess"] = (classified_df[schema.investor_name].fillna("").map(clean_name))

classified_df, accuracy_summary = evaluate_predictions(classified_df,schema)

# Remove internal accuracy columns from the exported Excel file
classified_output_df = classified_df.drop(
    columns=[
        "True_Label_Normalized",
        "Prediction_Normalized",
        "Is_Correct",
    ],
    errors="ignore",
)

classified_output_df.to_excel(
    OUTPUT_XLSX,
    index=False,
    engine="openpyxl"
)

wrong_mask = (
    classified_df["True_Label_Normalized"].ne("")
    & ~classified_df["Is_Correct"]
)

# Get the original columns from BBRI_combined.xlsx
original_columns = pd.read_excel(
    INPUT_XLSX,
    nrows=0,
    engine="openpyxl"
).columns.tolist()

# Only include original columns plus prediction diagnostics
wrong_output_columns = original_columns + [
    "Fund_Manager_Guess",
    "Fund Name Guess",
    "Needs_ML",
    "Matched_Rule",
    "Match_Evidence",
]

wrong_df = classified_df.loc[
    wrong_mask,
    wrong_output_columns
].copy()

wrong_df.to_excel(
    WRONG_XLSX,
    index=False,
    engine="openpyxl"
)

display(accuracy_summary)
print(f"Saved classified output: {OUTPUT_XLSX}")
print(f"Saved wrong predictions: {WRONG_XLSX}")
print(f"Wrong rows: {len(wrong_df):,}")

# Accuracy by SID type helps identify which rule families need improvement.
classified_df["SID_Type"] = classified_df[schema.sid].astype(str).str[:3]
accuracy_by_sid = (
    classified_df[classified_df["True_Label_Normalized"].ne("")]
    .groupby("SID_Type", dropna=False)
    .agg(
        Rows=("Is_Correct", "size"),
        Correct=("Is_Correct", "sum"),
        Unresolved=("Needs_ML", "sum"),
    )
)
accuracy_by_sid["Accuracy_Pct"] = accuracy_by_sid["Correct"] / accuracy_by_sid["Rows"] * 100
accuracy_by_sid["Coverage_Pct"] = (accuracy_by_sid["Rows"] - accuracy_by_sid["Unresolved"]) / accuracy_by_sid["Rows"] * 100
display(accuracy_by_sid.sort_values("Accuracy_Pct"))

# Rule-level diagnostic: low-accuracy rules are the best place to improve mappings.
accuracy_by_rule = (
    classified_df[classified_df["True_Label_Normalized"].ne("")]
    .groupby("Matched_Rule", dropna=False)
    .agg(
        Rows=("Is_Correct", "size"),
        Correct=("Is_Correct", "sum"),
        Unresolved=("Needs_ML", "sum"),
    )
)
accuracy_by_rule["Accuracy_Pct"] = accuracy_by_rule["Correct"] / accuracy_by_rule["Rows"] * 100
display(accuracy_by_rule.sort_values(["Accuracy_Pct", "Rows"], ascending=[True, False]))

display(wrong_df.head(100))

# Download both review files when running in Google Colab.
try:
    from google.colab import files
    files.download(OUTPUT_XLSX)
    files.download(WRONG_XLSX)
except ImportError:
    print("Not running in Google Colab. Files remain in the current folder.")

Keyword configuration validation passed.
Direct mapping validation passed.
Keyword configuration validation passed.
Direct mapping validation passed.
Manager-first regression tests passed.
Test file: /content/BBRI_combined.xlsx


,Metric,Value
0,Total rows,1000.000000
1,Labeled rows,1000.000000
2,Resolved rows,606.000000
3,Unresolved rows,394.000000
4,Coverage (%),60.600000
5,Overall accuracy (%),35.200000
6,Resolved-only accuracy (%),58.085809
7,Correct predictions,352.000000
8,Wrong predictions,648.000000


Saved classified output: /content/classified_test_output_manager_first.xlsx
Saved wrong predictions: /content/wrong_outputs.xlsx
Wrong rows: 648


,Rows,Correct,Unresolved,Accuracy_Pct,Coverage_Pct
SID_Type,,,,,
FDD,4,0,3,0.000000,25.000000
PFD,22,0,13,0.000000,40.909091
OTD,1,0,0,0.000000,100.000000
ISF,14,0,9,0.000000,35.714286
SCD,2,0,2,0.000000,0.000000
PFF,110,3,73,2.727273,33.636364
MFF,342,56,169,16.374269,50.584795
OTF,66,11,36,16.666667,45.454545
CPD,39,7,20,17.948718,48.717949


,Rows,Correct,Unresolved,Accuracy_Pct
Matched_Rule,,,,
benowners_no_unambiguous_match,394,0,394,0.000000
benowners_report_group,368,117,0,31.793478
direct_entity_fallback,14,11,0,78.571429
domestic_individual,198,198,0,100.000000
undisclosed_intermediary,15,15,0,100.000000
undisclosed_account,11,11,0,100.000000


,SID,9/2/2026,9/8/2026,Chg,Nama Investor,Negara,True Cient (Group),Fund_Manager_Guess,Fund Name Guess,Needs_ML,Matched_Rule,Match_Evidence
0,CPD0110AK337922,6958500.0,6958500,0,PT INDRA INVESTAMA - SUCOR ASSET,INDONESIA,PT INDRA INVESTAMA,SucorInvest Asset Mgt,PT INDRA INVESTAMA - SUCOR ASSET,False,benowners_report_group,PT INDRA INVESTAMA
2,CPD0204K5219044,12154000.0,12154000,0,PT PELAYARAN MULTI JAYA SAMUDERA,INDONESIA,PT PELAYARAN MULTI JAYA SAMUDERA,Pelayaran Multi Jaya Samudera,PT PELAYARAN MULTI JAYA SAMUDERA,False,benowners_report_group,PELAYARAN MULTI JAYA SAMUDERA
3,CPD0206F3660246,23600000.0,23600000,0,"WAHANA SENTOSA CEMERLANG, PT",INDONESIA,"WAHANA SENTOSA CEMERLANG, PT",ML_REQUIRED,"WAHANA SENTOSA CEMERLANG, PT",True,benowners_no_unambiguous_match,"WAHANA SENTOSA CEMERLANG, PT"
4,CPD0211BHS26606,11650000.0,11050000,-600000,PT. HASJRAT ABADI,INDONESIA,PT. HASJRAT ABADI,ML_REQUIRED,PT. HASJRAT ABADI,True,benowners_no_unambiguous_match,PT. HASJRAT ABADI
5,CPD0301I1007782,5000000.0,5000000,0,PT MERAPI AGUNG,INDONESIA,PT MERAPI AGUNG,ML_REQUIRED,PT MERAPI AGUNG,True,benowners_no_unambiguous_match,PT MERAPI AGUNG
...,...,...,...,...,...,...,...,...,...,...,...,...
343,ISD281110109094,35336019.0,36754419,1418400,PT BNI LIFE INSURANCE-BLIFE GROUP LINK MAXI - ...,INDONESIA,Bank Negara Indonesia,BNI Life Insurance,PT BNI LIFE INSURANCE-BLIFE GROUP LINK MAXI - ...,False,benowners_report_group,PT BNI LIFE INSURANCE BLIFE GROUP LINK MAXI BA...
344,ISD2901FMV14716,8000000.0,8000000,0,PT ASURANSI MULTI ARTHA GUNA Tbk,INDONESIA,PT ASURANSI MULTI ARTHA GUNA Tbk,ML_REQUIRED,PT ASURANSI MULTI ARTHA GUNA TBK,True,benowners_no_unambiguous_match,PT ASURANSI MULTI ARTHA GUNA TBK
345,ISD2912XS038792,11042900.0,11042900,0,PT ASURANSI JIWA CENTRAL ASIA RAYA - NEW CARLI...,INDONESIA,A.J. Central Asia Raya,ML_REQUIRED,PT ASURANSI JIWA CENTRAL ASIA RAYA - NEW CARLI...,True,benowners_no_unambiguous_match,PT ASURANSI JIWA CENTRAL ASIA RAYA - NEW CARLI...
346,ISD300875997883,10465000.0,10465000,0,ASURANSI SINAR MAS ASM EQUITY RUPIAH FUND,INDONESIA,Sinarmas Asset Mgt,ML_REQUIRED,ASURANSI SINAR MAS ASM EQUITY RUPIAH FUND,True,benowners_no_unambiguous_match,ASURANSI SINAR MAS ASM EQUITY RUPIAH FUND


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>